# SimCLR-Vision-SSL: the complete code

Code behind the paper *Where Do Scarce Labels Belong? A Compute-Matched Study of Supervised and Self-Supervised Contrastive Pretraining*
(Mahmoud Alyosify, Mirna Imbabi, Natalie Nashed, Hazem M. Abbas; Queen's University), together with the earlier
course project it grew out of.

**What this notebook is.** Every source file of the project, grouped by its role in the pipeline, each with a
one-line description. A code cell holds one file, unchanged, behind a `%%writefile <path>` line.

**How to use it.**
* *To read:* go through the sections in order; the table of contents follows the order in which the pipeline runs.
* *To rebuild the code tree:* run all cells. The first code cell creates the folder `SimCLR-Vision-SSL-code/`
  next to the notebook and moves into it; every other code cell only writes one file there. Nothing is trained,
  downloaded or evaluated. The commands that run the experiments are in Markdown blocks (Section 9), so they never
  run by accident.

The notebook was generated from the repository by a script and saved without outputs; it has not been run.

## Contents

0. [Set-up: target folder](#0.-Set-up)
1. [Environment](#1.-Environment)
2. [The `dlt` package: data, augmentations, models, loss, training, evaluation](#2.-The-dlt-package)
3. [Tests](#3.-Tests)
4. [Datasets and labeled pools](#4.-Datasets-and-labeled-pools)
5. [The GPU job queue](#5.-The-GPU-job-queue)
6. [Experiment definitions (job files)](#6.-Experiment-definitions)
7. [Analysis: re-evaluation, statistics, tables and figures](#7.-Analysis)
8. [Building the paper](#8.-Building-the-paper)
9. [Running the whole study (commands, not executed)](#9.-Running-the-whole-study)
10. [Appendix A: the original course project](#Appendix-A:-the-original-course-project)
11. [Appendix B: the visual-search demo](#Appendix-B:-the-visual-search-demo)
12. [Appendix C: Weights & Biases sync and the first asset generator](#Appendix-C:-W&B-sync-and-the-first-asset-generator)
13. [Appendix D: code cells of the earlier notebooks](#Appendix-D:-code-cells-of-the-earlier-notebooks)

### How the pieces fit together

```
datasets (CIFAR-10/100 auto-download; STL-10, Tiny-ImageNet via scripts/prepare_*.py)
   │
   ▼
jobs/*.txt ──► scripts/run_queue.py / run_phases.sh ──► python -m dlt.train | dlt.finetune | dlt.semisup | dlt.distill | scripts.ft_search
                                                           │   (one run = results/runs/<name>/result.json + preds.npz; checkpoint model.pt)
                                                           ▼
                                      scripts/reeval.py (label-limited probes, CDNV split) · scripts/diversity.py (pool statistics)
                                                           │
                                                           ▼
                                      scripts/make_assets.py (+ scripts/stats.py) ──► paper/generated/*.tex, paper/figures/*.pdf
                                                           │
                                                           ▼
                                                  paper/build.sh ──► arxiv.pdf, submission.pdf
```

| question in the paper | code | experiments |
|---|---|---|
| SimCLR vs. SupCon vs. SupCon+U at equal steps | `dlt/losses.py`, `dlt/train.py` | `jobs/01_main.txt`, `jobs/05_revision_main.txt`, `jobs/rev_p01_short.txt`, `rev_p06`, `rev_p10` |
| evaluation protocols (probes, kNN, fine-tuning) | `dlt/evaluate.py`, `dlt/finetune.py`, `scripts/ft_search.py` | `jobs/03_finetune.txt`, `rev_p02`–`rev_p05`, `rev_p07`, `rev_p11` |
| semi-supervised training and distillation | `dlt/semisup.py`, `dlt/distill.py` | `jobs/02_semisup.txt`, `rev_p04_fixmatch_from`, `rev_p05b/c` |
| representation geometry (CDNV) | `dlt/evaluate.py` (`geometry`), `scripts/reeval.py` | all encoders |
| pool coverage | `scripts/make_pools.py`, `scripts/diversity.py` | `rev_p09_coverage` |
| label fraction × budget grid | `dlt/train.py`, `dlt/finetune.py` | `rev_p12`, `rev_p13`, `rev_p14b` |
| SupCon regularisation, ResNet-50 | `dlt/train.py`, `dlt/models.py` | `rev_p14` |
| other datasets | `scripts/prepare_stl10.py`, `scripts/prepare_tinyimagenet.py` | `rev_p08*` (STL-10), `rev_p15/16` (CIFAR-100), `rev_p17`–`rev_p20` (Tiny-ImageNet) |
| statistics, every number, table and figure | `scripts/stats.py`, `scripts/make_assets.py` | — |

## 0. Set-up

In [ ]:
import os

# Every %%writefile cell below writes one file of the project into this folder.
TARGET = "SimCLR-Vision-SSL-code"
DIRS = [
    "configs",
    "deployment",
    "dlt",
    "jobs",
    "paper",
    "scripts",
    "src/Final Work",
    "src/Final Work/configs",
    "src/Final Work/supcon",
    "src/Midterm Work",
    "src/Midterm Work/configs",
    "tests",
    "wandb"
]

os.makedirs(TARGET, exist_ok=True)
os.chdir(TARGET)
for d in DIRS:
    os.makedirs(d, exist_ok=True)
print("writing the code tree into", os.getcwd())


## 1. Environment

Python 3.11 with PyTorch 2.x on one CUDA GPU (all reported numbers: one NVIDIA A40). `requirements.txt` and
`environment.yml` list the packages; the `Dockerfile` builds the same environment.

**`requirements.txt`** (14 lines) — Python packages.

In [ ]:
%%writefile requirements.txt
# Versions used for every result in paper/ (NVIDIA A40, CUDA 12.6, Python 3.12).
torch==2.11.0
torchvision==0.26.0
numpy==2.5.2
scipy==1.18.0
pandas==3.0.5
matplotlib==3.11.1
pillow==12.3.0
pyyaml>=6.0
# only for the CLIP reference in scripts/ood_resolution.py
transformers==5.15.1
# only for the original course-project code in src/ and deployment/
scikit-learn
tqdm


**`environment.yml`** (9 lines) — Conda environment of the paper (`conda env create -f environment.yml`).

In [ ]:
%%writefile environment.yml
name: spend-labels-last
channels:
  - conda-forge
dependencies:
  - python=3.12
  - pip
  - pip:
      - --extra-index-url https://download.pytorch.org/whl/cu126
      - -r requirements.txt


**`Dockerfile`** (12 lines) — Container with the same environment.

In [ ]:
%%writefile Dockerfile
# Reproduces the paper's experiments on any machine with an NVIDIA GPU.
#   docker build -t spend-labels-last .
#   docker run --gpus all -v $PWD/results:/work/results spend-labels-last bash scripts/reproduce.sh
FROM pytorch/pytorch:2.11.0-cuda12.6-cudnn9-runtime
WORKDIR /work
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY dlt dlt
COPY scripts scripts
COPY jobs jobs
COPY tests tests
ENV DLT_DATA=/work/data


## 2. The `dlt` package

Everything that trains or evaluates a network. Datasets are kept on the GPU as uint8 tensors and augmented there
in batches. Budgets are gradient steps: `--epochs E` means `E * (N // 512)` steps, where `N` is the size of the
full training set (97 steps per unit on CIFAR).

**`dlt/__init__.py`** (13 lines) — package description; sets the CUDA allocator option that reduces out-of-memory failures when two runs share the GPU.

In [ ]:
%%writefile dlt/__init__.py
"""dlt -- experiment code for the diversity-label trade-off study.

Everything in this package runs on a single GPU. Datasets are kept on the GPU
as uint8 tensors and augmented there in batches, so one training step is
limited by the network and not by data loading.
"""

import os as _os

# fewer out-of-memory failures from fragmentation when two runs share the GPU (must be set before CUDA starts)
_os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

__version__ = "1.0.0"


**`dlt/data.py`** (192 lines) — CIFAR-10/100, STL-10 and Tiny-ImageNet as uint8 GPU tensors; stratified, nested label subsets (the 1% subset is inside the 5% subset, and so on).

In [ ]:
%%writefile dlt/data.py
"""Dataset loading and label/pool subsampling.

CIFAR-10/100 are read from the original python pickles and returned as uint8
tensors of shape [N, 3, 32, 32]. Subsets are stratified (same number of images
per class) and *nested*: for a fixed seed, the 1% subset is contained in the 5%
subset, which is contained in the 10% subset, and so on.
"""

import os
import pickle
import tarfile
import urllib.request

import numpy as np
import torch

URLS = {
    "cifar10": "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz",
    "cifar100": "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz",
}
FOLDERS = {"cifar10": "cifar-10-batches-py", "cifar100": "cifar-100-python"}

# Channel statistics used for normalisation. The CIFAR-10 values are the ones
# used by the original code base of this project.
STATS = {
    "cifar10": ((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    "cifar100": ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762)),
}

CIFAR10_CLASSES = ["airplane", "automobile", "bird", "cat", "deer",
                   "dog", "frog", "horse", "ship", "truck"]


def _unpickle(path):
    with open(path, "rb") as f:
        return pickle.load(f, encoding="bytes")


def _maybe_download(name, root):
    folder = os.path.join(root, FOLDERS[name])
    if os.path.isdir(folder):
        return folder
    os.makedirs(root, exist_ok=True)
    archive = os.path.join(root, os.path.basename(URLS[name]))
    if not os.path.exists(archive):
        print(f"[data] downloading {URLS[name]}")
        urllib.request.urlretrieve(URLS[name], archive)
    with tarfile.open(archive, "r:gz") as tar:
        tar.extractall(root)
    return folder


def _load_stl10(root):
    """STL-10 (Coates et al., 2011): 5,000 labeled + 100,000 unlabeled training images and
    8,000 test images at 96 px. The training tensor holds the labeled images first; the
    unlabeled images follow with label -1. Built by scripts/prepare_stl10.py."""
    path = os.path.join(root, "stl10_uint8.npz")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found: run scripts/prepare_stl10.py first")
    d = np.load(path)
    n_lab = d["train_x"].shape[0]
    train_x = np.concatenate([d["train_x"], d["unlabeled_x"]])
    train_y = np.concatenate([d["train_y"], -np.ones(d["unlabeled_x"].shape[0], dtype=np.int64)])
    return {
        "name": "stl10",
        "train_x": torch.from_numpy(train_x),
        "train_y": torch.from_numpy(train_y).long(),
        "test_x": torch.from_numpy(d["test_x"]),
        "test_y": torch.from_numpy(d["test_y"]).long(),
        "num_classes": 10,
        "mean": tuple(float(v) for v in d["mean"]),
        "std": tuple(float(v) for v in d["std"]),
        "labeled_idx": np.arange(n_lab),
    }


def _load_tinyimagenet(root):
    """Tiny-ImageNet (Le & Yang, 2015): 200 classes, 100,000 training images of 64 px; the official
    validation split (10,000 images) is used as the test set. Built by scripts/prepare_tinyimagenet.py."""
    path = os.path.join(root, "tinyimagenet_uint8.npz")
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found: run scripts/prepare_tinyimagenet.py first")
    d = np.load(path)
    return {
        "name": "tinyimagenet",
        "train_x": torch.from_numpy(d["train_x"]),
        "train_y": torch.from_numpy(d["train_y"]).long(),
        "test_x": torch.from_numpy(d["test_x"]),
        "test_y": torch.from_numpy(d["test_y"]).long(),
        "num_classes": 200,
        "mean": tuple(float(v) for v in d["mean"]),
        "std": tuple(float(v) for v in d["std"]),
    }


def labeled_subset(data, fraction, seed):
    """Indices of the training images whose labels may be used.

    CIFAR: a nested class-balanced random subset. STL-10: the fixed labeled split
    (any positive fraction selects all of it)."""
    if "labeled_idx" in data:
        return data["labeled_idx"] if fraction > 0 else np.arange(0)
    n = data["train_y"].shape[0]
    if fraction >= 1.0:
        return np.arange(n)
    if fraction <= 0.0:
        return np.arange(0)
    return stratified_subset(data["train_y"].cpu().numpy(), fraction, seed)


def load_dataset(name, root):
    """Return a dict with uint8 image tensors [N,3,H,W] and int64 labels."""
    if name == "stl10":
        return _load_stl10(root)
    if name == "tinyimagenet":
        return _load_tinyimagenet(root)
    folder = _maybe_download(name, root)
    if name == "cifar10":
        xs, ys = [], []
        for i in range(1, 6):
            d = _unpickle(os.path.join(folder, f"data_batch_{i}"))
            xs.append(d[b"data"])
            ys.extend(d[b"labels"])
        train_x, train_y = np.concatenate(xs), np.array(ys)
        d = _unpickle(os.path.join(folder, "test_batch"))
        test_x, test_y = d[b"data"], np.array(d[b"labels"])
        num_classes = 10
    elif name == "cifar100":
        d = _unpickle(os.path.join(folder, "train"))
        train_x, train_y = d[b"data"], np.array(d[b"fine_labels"])
        d = _unpickle(os.path.join(folder, "test"))
        test_x, test_y = d[b"data"], np.array(d[b"fine_labels"])
        num_classes = 100
    else:
        raise ValueError(f"unknown dataset {name}")

    def to_tensor(x):
        return torch.from_numpy(np.ascontiguousarray(x.reshape(-1, 3, 32, 32)))

    mean, std = STATS[name]
    return {
        "name": name,
        "train_x": to_tensor(train_x),
        "train_y": torch.from_numpy(train_y).long(),
        "test_x": to_tensor(test_x),
        "test_y": torch.from_numpy(test_y).long(),
        "num_classes": num_classes,
        "mean": mean,
        "std": std,
    }


def stratified_subset(labels, fraction, seed):
    """Indices of a class-balanced random subset.

    For a fixed seed the subsets are nested across fractions, because every
    class keeps the first ``round(fraction * n_c)`` entries of one fixed random
    permutation of its images.
    """
    labels = np.asarray(labels)
    rng = np.random.RandomState(seed)
    out = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        perm = rng.permutation(idx)
        k = int(round(fraction * len(idx)))
        out.append(perm[:k])
    return np.sort(np.concatenate(out))


class IndexSampler:
    """Yields batches of indices by cycling through shuffled copies of a pool.

    If the pool is smaller than the batch, an image can appear more than once
    in a batch. The loss groups views by image index, so such repeats are
    treated as extra positives and never as negatives.
    """

    def __init__(self, pool, batch_size, generator, device):
        self.pool = torch.as_tensor(pool, dtype=torch.long, device=device)
        self.batch_size = batch_size
        self.g = generator
        self.device = device
        self.buf = torch.empty(0, dtype=torch.long, device=device)

    def next(self, n=None):
        n = self.batch_size if n is None else n
        while self.buf.numel() < n:
            perm = torch.randperm(self.pool.numel(), generator=self.g, device=self.device)
            self.buf = torch.cat([self.buf, self.pool[perm]])
        out, self.buf = self.buf[:n], self.buf[n:]
        return out


**`dlt/gpu_aug.py`** (358 lines) — batched GPU augmentations (crop, flip, colour jitter, grayscale, blur, erasing) with per-image random parameters, checked against torchvision in `tests/`; named pipelines (`simclr`, `simclr_strong`, ...).

In [ ]:
%%writefile dlt/gpu_aug.py
"""Batched data augmentation on the GPU.

Every transform draws its random parameters *per image*, exactly like the
torchvision transforms it replaces (RandomResizedCrop, RandomHorizontalFlip,
ColorJitter, RandomGrayscale, GaussianBlur, RandomErasing), but the whole batch
is processed at once. ``tests/test_gpu_aug.py`` checks each operation against
torchvision for fixed parameters.

Order of operations inside a pipeline:
    rotation -> crop/flip -> colour jitter -> grayscale -> blur -> normalise -> erasing
"""

import math
from dataclasses import dataclass, replace
from typing import Optional, Tuple

import torch
import torch.nn.functional as F


@dataclass(frozen=True)
class AugConfig:
    crop_scale: Optional[Tuple[float, float]] = None   # RandomResizedCrop area range
    crop_ratio: Tuple[float, float] = (3.0 / 4.0, 4.0 / 3.0)
    flip: bool = False                                 # RandomHorizontalFlip(p=0.5)
    jitter: Optional[Tuple[float, float, float, float]] = None  # brightness, contrast, saturation, hue
    jitter_p: float = 0.8
    gray_p: float = 0.0                                # RandomGrayscale
    blur: bool = False                                 # GaussianBlur(3, sigma in blur_sigma), always applied
    blur_sigma: Tuple[float, float] = (0.1, 2.0)
    erase_p: float = 0.0                               # RandomErasing after normalisation
    erase_scale: Tuple[float, float] = (0.02, 0.2)
    erase_ratio: Tuple[float, float] = (0.3, 3.3)
    rot90: bool = False                                # random rotation by 0/90/180/270 degrees
    pad_crop: int = 0                                  # RandomCrop(32, padding=pad_crop)
    pad_mode: str = "zeros"                            # "zeros" or "reflection"


# Colour distortion strength s = 0.5 (Chen et al. 2020, CIFAR-10 setting):
# ColorJitter(0.8 s, 0.8 s, 0.8 s, 0.2 s) applied with probability 0.8.
CJ = (0.4, 0.4, 0.4, 0.1)

# Pipelines. The "expNN" names are the experiment ids of the original
# 42-configuration augmentation sweep of this project.
PIPELINES = {
    "none": AugConfig(),
    "sup": AugConfig(pad_crop=4, flip=True),
    # --- canonical SimCLR / SupContrast CIFAR recipe (= exp8) ----------------
    "simclr": AugConfig(crop_scale=(0.2, 1.0), flip=True, jitter=CJ, gray_p=0.2),
    # stronger variant (ImageNet-style crop range, full-strength colour distortion)
    "simclr_strong": AugConfig(crop_scale=(0.08, 1.0), flip=True, jitter=(0.8, 0.8, 0.8, 0.2), gray_p=0.2),
    # --- original 8-configuration study --------------------------------------
    "exp1": AugConfig(crop_scale=(0.2, 1.0)),
    "exp2": AugConfig(crop_scale=(0.2, 1.0), flip=True),
    "exp3": AugConfig(crop_scale=(0.2, 1.0), jitter=CJ),
    "exp4": AugConfig(crop_scale=(0.2, 1.0), gray_p=0.2),
    "exp5": AugConfig(crop_scale=(0.2, 1.0), flip=True, jitter=CJ),
    "exp6": AugConfig(crop_scale=(0.2, 1.0), flip=True, gray_p=0.2),
    "exp7": AugConfig(crop_scale=(0.2, 1.0), jitter=CJ, gray_p=0.2),
    "exp8": AugConfig(crop_scale=(0.2, 1.0), flip=True, jitter=CJ, gray_p=0.2),
    # --- structural pipelines without colour jitter ("base") ------------------
    "exp9": AugConfig(crop_scale=(0.6, 1.0), blur=True),
    "exp10": AugConfig(crop_scale=(0.6, 1.0), erase_p=0.7),
    "exp13": AugConfig(crop_scale=(0.6, 1.0), flip=True, blur=True),
    "exp35": AugConfig(crop_scale=(0.8, 1.0)),
    "exp36": AugConfig(rot90=True),
    # --- the same pipelines with colour jitter added --------------------------
    "exp38": AugConfig(rot90=True, jitter=CJ),
    "exp39": AugConfig(crop_scale=(0.8, 1.0), jitter=CJ),
    "exp40": AugConfig(crop_scale=(0.6, 1.0), blur=True, jitter=CJ),
    "exp41": AugConfig(crop_scale=(0.6, 1.0), flip=True, blur=True, jitter=CJ),
    "exp42": AugConfig(crop_scale=(0.6, 1.0), erase_p=0.7, jitter=CJ),
}


def sup_pipeline(size, pad_mode="zeros"):
    """Supervised augmentation: random crop with padding of size/8 (4 px at 32 px, 12 px at 96 px) and flip."""
    return AugConfig(pad_crop=size // 8, flip=True, pad_mode=pad_mode)


def get_pipeline(name):
    """Look up a pipeline. ``name+nojitter`` / ``name+nogray`` remove one component."""
    base, *mods = name.split("+")
    cfg = PIPELINES[base]
    for m in mods:
        if m == "nojitter":
            cfg = replace(cfg, jitter=None)
        elif m == "nogray":
            cfg = replace(cfg, gray_p=0.0)
        elif m == "noflip":
            cfg = replace(cfg, flip=False)
        else:
            raise ValueError(f"unknown modifier {m}")
    return cfg


# ----------------------------------------------------------------------------
# colour helpers (float images in [0, 1], shape [B, 3, H, W])
# ----------------------------------------------------------------------------

def rgb_to_gray(x):
    r, g, b = x.unbind(dim=1)
    return (0.2989 * r + 0.587 * g + 0.114 * b).unsqueeze(1)


def _rgb2hsv(img):
    r, g, b = img.unbind(dim=-3)
    maxc = torch.max(img, dim=-3).values
    minc = torch.min(img, dim=-3).values
    eqc = maxc == minc
    cr = maxc - minc
    ones = torch.ones_like(maxc)
    s = cr / torch.where(eqc, ones, maxc)
    cr_div = torch.where(eqc, ones, cr)
    rc = (maxc - r) / cr_div
    gc = (maxc - g) / cr_div
    bc = (maxc - b) / cr_div
    hr = (maxc == r) * (bc - gc)
    hg = ((maxc == g) & (maxc != r)) * (2.0 + rc - bc)
    hb = ((maxc != g) & (maxc != r)) * (4.0 + gc - rc)
    h = torch.fmod((hr + hg + hb) / 6.0 + 1.0, 1.0)
    return torch.stack((h, s, maxc), dim=-3)


def _hsv2rgb(img):
    h, s, v = img.unbind(dim=-3)
    i = torch.floor(h * 6.0)
    f = h * 6.0 - i
    i = i.to(torch.int32) % 6
    p = torch.clamp(v * (1.0 - s), 0.0, 1.0)
    q = torch.clamp(v * (1.0 - s * f), 0.0, 1.0)
    t = torch.clamp(v * (1.0 - s * (1.0 - f)), 0.0, 1.0)
    mask = i.unsqueeze(-3) == torch.arange(6, device=i.device).view(-1, 1, 1)
    a1 = torch.stack((v, q, p, p, t, v), dim=-3)
    a2 = torch.stack((t, v, v, q, p, p), dim=-3)
    a3 = torch.stack((p, p, t, v, v, q), dim=-3)
    a4 = torch.stack((a1, a2, a3), dim=-4)
    return torch.einsum("...ijk,...xijk->...xjk", mask.to(img.dtype), a4)


def adjust_brightness(x, f):
    return (x * f.view(-1, 1, 1, 1)).clamp_(0, 1)


def adjust_contrast(x, f):
    f = f.view(-1, 1, 1, 1)
    mean = rgb_to_gray(x).mean(dim=(1, 2, 3), keepdim=True)
    return (f * x + (1 - f) * mean).clamp_(0, 1)


def adjust_saturation(x, f):
    f = f.view(-1, 1, 1, 1)
    return (f * x + (1 - f) * rgb_to_gray(x)).clamp_(0, 1)


def adjust_hue(x, f):
    hsv = _rgb2hsv(x)
    h = (hsv[:, 0] + f.view(-1, 1, 1)) % 1.0
    return _hsv2rgb(torch.stack((h, hsv[:, 1], hsv[:, 2]), dim=1))


_JITTER_OPS = (adjust_brightness, adjust_contrast, adjust_saturation, adjust_hue)


# ----------------------------------------------------------------------------
# parameter samplers
# ----------------------------------------------------------------------------

def _rand(shape, g, device):
    return torch.rand(shape, generator=g, device=device)


def _uniform(shape, lo, hi, g, device):
    return lo + (hi - lo) * _rand(shape, g, device)


def _first_success(ok):
    """For a [T, B] success table return (index of first success per column, any success)."""
    any_ok = ok.any(dim=0)
    first = ok.float().argmax(dim=0)
    return first, any_ok


def sample_rrc(B, scale, ratio, size, g, device, attempts=10):
    """Vectorised torchvision ``RandomResizedCrop.get_params`` -> (top, left, h, w).

    All ``attempts`` proposals are drawn at once and the first valid one is kept,
    which is the same distribution as torchvision's sequential rejection loop.
    """
    H = W = size
    area = float(H * W)
    log_lo, log_hi = math.log(ratio[0]), math.log(ratio[1])
    target = area * _uniform((attempts, B), scale[0], scale[1], g, device)
    aspect = torch.exp(_uniform((attempts, B), log_lo, log_hi, g, device))
    ww = torch.round(torch.sqrt(target * aspect))
    hh = torch.round(torch.sqrt(target / aspect))
    ok = (ww > 0) & (ww <= W) & (hh > 0) & (hh <= H)
    first, any_ok = _first_success(ok)
    w = ww.gather(0, first.view(1, -1)).squeeze(0)
    h = hh.gather(0, first.view(1, -1)).squeeze(0)
    # samples that never succeeded keep the whole image (torchvision fallback
    # for a square input whose ratio lies inside the allowed range)
    w = torch.where(any_ok, w, torch.full_like(w, float(W)))
    h = torch.where(any_ok, h, torch.full_like(h, float(H)))
    r = _rand((2, B), g, device)
    top = torch.floor(r[0] * (H - h + 1))
    left = torch.floor(r[1] * (W - w + 1))
    return top, left, h, w


def sample_erasing(B, p, scale, ratio, size, g, device, attempts=10):
    """Vectorised torchvision ``RandomErasing.get_params`` -> boolean mask [B,1,H,W]."""
    H = W = size
    area = float(H * W)
    log_lo, log_hi = math.log(ratio[0]), math.log(ratio[1])
    target = area * _uniform((attempts, B), scale[0], scale[1], g, device)
    aspect = torch.exp(_uniform((attempts, B), log_lo, log_hi, g, device))
    hh = torch.round(torch.sqrt(target * aspect))
    ww = torch.round(torch.sqrt(target / aspect))
    ok = (hh < H) & (ww < W)
    first, any_ok = _first_success(ok)
    eh = hh.gather(0, first.view(1, -1)).squeeze(0)
    ew = ww.gather(0, first.view(1, -1)).squeeze(0)
    r = _rand((3, B), g, device)
    top = torch.floor(r[0] * (H - eh + 1))
    left = torch.floor(r[1] * (W - ew + 1))
    apply = (r[2] < p) & any_ok
    ys = torch.arange(H, device=device).view(1, H, 1)
    xs = torch.arange(W, device=device).view(1, 1, W)
    mask = ((ys >= top.view(-1, 1, 1)) & (ys < (top + eh).view(-1, 1, 1))
            & (xs >= left.view(-1, 1, 1)) & (xs < (left + ew).view(-1, 1, 1)))
    return (mask & apply.view(-1, 1, 1)).unsqueeze(1)


# ----------------------------------------------------------------------------
# geometric helpers
# ----------------------------------------------------------------------------

def resized_crop(x, top, left, h, w, flip=None):
    """Crop boxes (top, left, h, w) and resize them back to the input size.

    Bilinear interpolation with half-pixel centres; sampling positions are
    clamped to the crop box, which is what crop-then-resize does in torchvision.
    """
    B, _, H, W = x.shape
    u = torch.arange(W, device=x.device, dtype=x.dtype) + 0.5
    v = torch.arange(H, device=x.device, dtype=x.dtype) + 0.5
    xs = left.view(-1, 1) + u.view(1, -1) * (w / W).view(-1, 1) - 0.5   # source pixel coordinates
    ys = top.view(-1, 1) + v.view(1, -1) * (h / H).view(-1, 1) - 0.5
    xs = torch.minimum(torch.maximum(xs, left.view(-1, 1)), (left + w - 1).view(-1, 1))
    ys = torch.minimum(torch.maximum(ys, top.view(-1, 1)), (top + h - 1).view(-1, 1))
    if flip is not None:
        xs = torch.where(flip.view(-1, 1), xs.flip(1), xs)
    gx = (2 * xs + 1) / W - 1
    gy = (2 * ys + 1) / H - 1
    grid = torch.stack((gx.view(B, 1, W).expand(B, H, W), gy.view(B, H, 1).expand(B, H, W)), dim=-1)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)


def padded_crop(x, pad, flip, g, mode="zeros"):
    """RandomCrop(size, padding=pad) with zero or reflection padding, plus optional flip."""
    B, _, H, W = x.shape
    dev = x.device
    dx = torch.randint(-pad, pad + 1, (B,), generator=g, device=dev).to(x.dtype)
    dy = torch.randint(-pad, pad + 1, (B,), generator=g, device=dev).to(x.dtype)
    theta = torch.zeros(B, 2, 3, device=dev, dtype=x.dtype)
    sx = torch.ones(B, device=dev, dtype=x.dtype)
    if flip is not None:
        sx = torch.where(flip, -sx, sx)
    theta[:, 0, 0] = sx
    theta[:, 1, 1] = 1.0
    theta[:, 0, 2] = 2 * dx / W
    theta[:, 1, 2] = 2 * dy / H
    grid = F.affine_grid(theta, (B, 3, H, W), align_corners=False)
    return F.grid_sample(x, grid, mode="nearest", padding_mode=mode, align_corners=False)


def gaussian_blur3(x, sigma):
    """3x3 Gaussian blur with a per-image sigma and reflect padding."""
    a = torch.exp(-0.5 / sigma ** 2)
    norm = 1 + 2 * a
    side = (a / norm).view(-1, 1, 1, 1)
    mid = (1 / norm).view(-1, 1, 1, 1)
    xp = F.pad(x, (1, 1, 0, 0), mode="reflect")
    x = side * (xp[..., :-2] + xp[..., 2:]) + mid * xp[..., 1:-1]
    xp = F.pad(x, (0, 0, 1, 1), mode="reflect")
    return side * (xp[..., :-2, :] + xp[..., 2:, :]) + mid * xp[..., 1:-1, :]


# ----------------------------------------------------------------------------
# the augmentation module
# ----------------------------------------------------------------------------

class GPUAugment:
    """Applies one pipeline to a uint8 batch [B,3,H,W] and returns normalised floats."""

    def __init__(self, cfg, mean, std, device, generator=None):
        self.cfg = get_pipeline(cfg) if isinstance(cfg, str) else cfg
        self.mean = torch.tensor(mean, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(std, device=device).view(1, 3, 1, 1)
        self.device = device
        self.g = generator

    @torch.no_grad()
    def __call__(self, x_uint8):
        cfg, g, dev = self.cfg, self.g, self.device
        x = x_uint8.float().div_(255.0)
        B, _, H, W = x.shape

        if cfg.rot90:
            k = torch.randint(0, 4, (B,), generator=g, device=dev).view(-1, 1, 1, 1)
            out = x
            for r in (1, 2, 3):
                out = torch.where(k == r, torch.rot90(x, r, dims=(2, 3)), out)
            x = out

        flip = (_rand((B,), g, dev) < 0.5) if cfg.flip else None
        if cfg.crop_scale is not None:
            top, left, h, w = sample_rrc(B, cfg.crop_scale, cfg.crop_ratio, H, g, dev)
            x = resized_crop(x, top, left, h, w, flip)
        elif cfg.pad_crop > 0:
            x = padded_crop(x, cfg.pad_crop, flip, g, cfg.pad_mode)
        elif flip is not None:
            x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)

        if cfg.jitter is not None:
            b, c, s, hue = cfg.jitter
            factors = (
                _uniform((B,), max(0.0, 1 - b), 1 + b, g, dev),
                _uniform((B,), max(0.0, 1 - c), 1 + c, g, dev),
                _uniform((B,), max(0.0, 1 - s), 1 + s, g, dev),
                _uniform((B,), -hue, hue, g, dev),
            )
            apply = _rand((B,), g, dev) < cfg.jitter_p
            order = _rand((B, 4), g, dev).argsort(dim=1)     # random op order per image
            for pos in range(4):
                for op_id, op in enumerate(_JITTER_OPS):
                    sel = ((order[:, pos] == op_id) & apply).nonzero(as_tuple=True)[0]
                    x[sel] = op(x[sel], factors[op_id][sel])

        if cfg.gray_p > 0:
            m = _rand((B,), g, dev) < cfg.gray_p
            x = torch.where(m.view(-1, 1, 1, 1), rgb_to_gray(x).expand(-1, 3, -1, -1), x)

        if cfg.blur:
            sigma = _uniform((B,), cfg.blur_sigma[0], cfg.blur_sigma[1], g, dev)
            x = gaussian_blur3(x, sigma)

        x = (x - self.mean) / self.std

        if cfg.erase_p > 0:
            mask = sample_erasing(B, cfg.erase_p, cfg.erase_scale, cfg.erase_ratio, H, g, dev)
            x = x.masked_fill(mask, 0.0)
        return x

    @torch.no_grad()
    def normalize(self, x_uint8):
        return (x_uint8.float().div_(255.0) - self.mean) / self.std


**`dlt/models.py`** (139 lines) — ResNet-18/50 encoders with the CIFAR stem (and `_mp` variants with max-pooling for 64–96 px images), projection heads, BYOL/SimSiam models, classifier.

In [ ]:
%%writefile dlt/models.py
"""Encoders and heads.

The encoders are torchvision ResNets with the CIFAR stem used throughout this
project (3x3 stride-1 first convolution, no max-pooling; Chen et al. 2020,
App. B.9).
"""

import torch
import torch.nn as nn
import torchvision


def _cifar_resnet(name):
    # "<name>_mp": the same 3x3 stem followed by the standard max-pooling layer, for 64-96 px images
    name, _, variant = name.partition("_")
    if name == "resnet18":
        net = torchvision.models.resnet18(weights=None)
        dim = 512
    elif name == "resnet50":
        net = torchvision.models.resnet50(weights=None)
        dim = 2048
    else:
        raise ValueError(name)
    net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    if variant != "mp":
        net.maxpool = nn.Identity()
    net.fc = nn.Identity()
    return net, dim


class MLPHead(nn.Module):
    """Linear - BN - ReLU - Linear, optionally with BN on the output."""

    def __init__(self, in_dim, hidden_dim, out_dim, last_bn=False, layers=2):
        super().__init__()
        mods = []
        d = in_dim
        for _ in range(layers - 1):
            mods += [nn.Linear(d, hidden_dim, bias=False), nn.BatchNorm1d(hidden_dim), nn.ReLU(inplace=True)]
            d = hidden_dim
        if last_bn:
            mods += [nn.Linear(d, out_dim, bias=False), nn.BatchNorm1d(out_dim)]
        else:
            mods += [nn.Linear(d, out_dim)]
        self.net = nn.Sequential(*mods)

    def forward(self, x):
        return self.net(x)


class ContrastiveModel(nn.Module):
    """Encoder f and projection head g (hidden layer with BN, no BN on the output)."""

    def __init__(self, backbone="resnet18", proj_dim=128):
        super().__init__()
        self.encoder, self.feat_dim = _cifar_resnet(backbone)
        self.head = MLPHead(self.feat_dim, self.feat_dim, proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        return h, self.head(h)


class SimSiamModel(nn.Module):
    """SimSiam (Chen & He 2021), CIFAR variant with a 2-layer projector."""

    def __init__(self, backbone="resnet18", proj_dim=2048, pred_hidden=512):
        super().__init__()
        self.encoder, self.feat_dim = _cifar_resnet(backbone)
        self.head = MLPHead(self.feat_dim, proj_dim, proj_dim, last_bn=True)
        self.predictor = MLPHead(proj_dim, pred_hidden, proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.head(h)
        return h, z, self.predictor(z)


class BYOLModel(nn.Module):
    """BYOL (Grill et al. 2020): online network + predictor; the target is an EMA copy."""

    def __init__(self, backbone="resnet18", proj_hidden=4096, proj_dim=256):
        super().__init__()
        self.encoder, self.feat_dim = _cifar_resnet(backbone)
        self.head = MLPHead(self.feat_dim, proj_hidden, proj_dim)
        self.predictor = MLPHead(proj_dim, proj_hidden, proj_dim)

    def forward(self, x):
        h = self.encoder(x)
        z = self.head(h)
        return h, z, self.predictor(z)


class Classifier(nn.Module):
    """Encoder + linear classifier (supervised and semi-supervised baselines, fine-tuning)."""

    def __init__(self, backbone="resnet18", num_classes=10):
        super().__init__()
        self.encoder, self.feat_dim = _cifar_resnet(backbone)
        self.fc = nn.Linear(self.feat_dim, num_classes)

    def forward(self, x):
        return self.fc(self.encoder(x))


@torch.no_grad()
def extract_features(encoder, x_uint8, normalize_fn, batch_size=2000):
    """Encoder outputs h for un-augmented images (eval mode, mixed precision)."""
    was_training = encoder.training
    encoder.eval()
    out = []
    for i in range(0, x_uint8.shape[0], batch_size):
        xb = normalize_fn(x_uint8[i:i + batch_size]).contiguous(memory_format=torch.channels_last)
        with torch.autocast("cuda", dtype=torch.float16):
            h = encoder(xb)
        out.append(h.float())
    encoder.train(was_training)
    return torch.cat(out)


def load_encoder(path, device="cuda"):
    """Rebuild the encoder of any checkpoint written by dlt.train / dlt.semisup."""
    ckpt = torch.load(path, map_location=device)
    a = ckpt["args"]
    sd = {k: v.float() if v.is_floating_point() else v for k, v in ckpt["model"].items()}
    backbone = a.get("backbone", "resnet18")
    method = a.get("method")
    if method in ("simclr", "supcon"):
        model = ContrastiveModel(backbone)
    elif method == "simsiam":
        model = SimSiamModel(backbone)
    elif method == "byol":
        model = BYOLModel(backbone)
    else:
        n_cls = sd["fc.weight"].shape[0]
        model = Classifier(backbone, n_cls)
    model.load_state_dict(sd)
    enc = model.encoder.to(device).to(memory_format=torch.channels_last).eval()
    return enc, a


**`dlt/losses.py`** (53 lines) — one grouped contrastive loss: SimCLR (no labels), SupCon (all labels) and SupCon+U (some labels).

In [ ]:
%%writefile dlt/losses.py
"""Contrastive losses.

One function covers SimCLR, SupCon and the semi-supervised union of the two.
Every view carries a *group id*; two different views are positives if and only
if they share a group id:

* unlabeled image  -> the group is the image itself (id = C + image index);
* labeled image    -> the group is its class        (id = class label).

With no labels this is NT-Xent (SimCLR). With all labels it is the SupCon loss
(L_out of Khosla et al. 2020). With some labels it is SupCon over the whole
pool in which every unlabeled image is its own class.
"""

import torch
import torch.nn.functional as F


def make_groups(index, labels, labeled_mask, num_classes):
    """Group ids for a batch.

    index        [B] image indices in the training set
    labels       [B] class labels (only read where ``labeled_mask`` is True)
    labeled_mask [B] True where the label may be used
    """
    return torch.where(labeled_mask, labels, num_classes + index)


def grouped_contrastive_loss(z, groups, temperature):
    """Supervised-contrastive loss with positives defined by ``groups``.

    z           [V*B, D] L2-normalised embeddings, the V views concatenated view by view
    groups      [B] group id per image
    temperature float, or a tensor [B] with one temperature per anchor image
    """
    z = z.float()
    n_views = z.shape[0] // groups.shape[0]
    g = groups.repeat(n_views)
    if torch.is_tensor(temperature):
        temperature = temperature.repeat(n_views).view(-1, 1)
    logits = z @ z.t() / temperature
    self_mask = torch.eye(z.shape[0], dtype=torch.bool, device=z.device)
    logits = logits.masked_fill(self_mask, float("-inf"))
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    pos = (g.view(-1, 1) == g.view(1, -1)) & ~self_mask
    # every anchor has at least one positive: its other view
    loss = -(log_prob.masked_fill(~pos, 0.0)).sum(1) / pos.sum(1)
    return loss.mean()


def negative_cosine(p, z):
    """SimSiam / BYOL regression loss between a prediction and a stop-gradient target."""
    return -(F.normalize(p.float(), dim=1) * F.normalize(z.float().detach(), dim=1)).sum(1).mean()


**`dlt/train.py`** (326 lines) — pretraining of one run (SimCLR, SupCon, SupCon+U, BYOL, SimSiam, cross-entropy) and its evaluation; writes `result.json`, `preds.npz` and the checkpoint.

In [ ]:
%%writefile dlt/train.py
"""Pretraining + evaluation of one run.

Methods
-------
simclr   NT-Xent on the pool, labels never used.
supcon   Grouped contrastive loss. Labeled images use their class as positives,
         unlabeled images (if the pool contains any) use only their own second view.
           --pool labeled  -> SupCon on the labeled subset only        ("label-only")
           --pool all      -> SupCon over all images, partially labeled ("union")
byol, simsiam   non-contrastive self-supervised baselines.
ce       supervised cross-entropy on the labeled images.

Budget
------
``--epochs`` is counted in passes over the *full* training set, i.e. the number
of gradient steps is epochs * (N_train // batch_size) no matter how small the
pool is. All methods therefore see the same number of augmented views.

Example
-------
python -m dlt.train --name supcon_L_f0.1_s0 --method supcon --pool labeled \
       --label_fraction 0.1 --seed 0
"""

import argparse
import copy
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from .data import IndexSampler, labeled_subset, load_dataset, stratified_subset
from .evaluate import evaluate_encoder, geometry, knn_predict, to_numpy_preds
from .gpu_aug import GPUAugment, sup_pipeline
from .losses import grouped_contrastive_loss, make_groups, negative_cosine
from .models import BYOLModel, Classifier, ContrastiveModel, SimSiamModel, extract_features

DEFAULTS = {
    # method: (lr at batch 512, weight decay, temperature)
    "simclr": (0.25, 1e-4, 0.5),     # official SupContrast recipe: lr 0.5 @ 1024, tau 0.5
    "supcon": (0.25, 1e-4, 0.1),     # official SupContrast recipe: lr 0.5 @ 1024, tau 0.1
    "simsiam": (0.06, 5e-4, None),   # Chen & He 2021, CIFAR: lr 0.03 * B/256
    "byol": (0.24, 5e-4, None),      # lightly CIFAR-10 benchmark: lr 0.06 * B/128
    "ce": (0.2, 5e-4, None),         # lr 0.1 * B/256
}


def parse_args(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--out", default="results/runs")
    p.add_argument("--ckpt_dir", default=None, help="where to store the final encoder (default: <out>/<name>)")
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10", choices=["cifar10", "cifar100", "stl10", "tinyimagenet"])
    p.add_argument("--method", default="simclr", choices=list(DEFAULTS))
    p.add_argument("--backbone", default="resnet18", choices=["resnet18", "resnet50", "resnet18_mp", "resnet50_mp"])
    p.add_argument("--aug", default="simclr")
    p.add_argument("--pool", default="all", choices=["all", "labeled"])
    p.add_argument("--label_fraction", type=float, default=0.0)
    p.add_argument("--pool_file", default=None, help=".npy with the indices of the labeled images")
    p.add_argument("--labeled_share", type=float, default=None,
                   help="supcon with --pool all: fraction of every batch drawn from the labeled images")
    p.add_argument("--epochs", type=float, default=100.0)
    p.add_argument("--batch_size", type=int, default=512)
    p.add_argument("--lr", type=float, default=None)
    p.add_argument("--wd", type=float, default=None)
    p.add_argument("--temperature", type=float, default=None)
    p.add_argument("--temperature_unlabeled", type=float, default=None,
                   help="supcon with --pool all: temperature of anchors without a label "
                        "(default: same as --temperature)")
    p.add_argument("--holdout", type=int, default=0,
                   help="development mode: hold out this many training images as a validation "
                        "set and report on them instead of the test set")
    p.add_argument("--warmup_frac", type=float, default=0.05)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--split_seed", type=int, default=None)
    p.add_argument("--curve_points", type=int, default=8)
    p.add_argument("--compile", type=int, default=1)
    p.add_argument("--init_run", default=None,
                   help="initialise encoder and head from the checkpoint of this run (two-stage pipelines)")
    p.add_argument("--force", action="store_true")
    p.add_argument("--no_save_ckpt", action="store_true")
    args = p.parse_args(argv)
    lr, wd, temp = DEFAULTS[args.method]
    if args.lr is None:
        args.lr = lr * args.batch_size / 512
    if args.wd is None:
        args.wd = wd
    if args.temperature is None:
        args.temperature = temp
    if args.split_seed is None:
        args.split_seed = args.seed
    return args


def lr_at(step, total, base, warmup):
    if step < warmup:
        return base * (step + 1) / warmup
    return 0.5 * base * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))


def main(argv=None):
    args = parse_args(argv)
    out_dir = os.path.join(args.out, args.name)
    res_path = os.path.join(out_dir, "result.json")
    if os.path.exists(res_path) and not args.force:
        print(f"[skip] {res_path} exists")
        return
    os.makedirs(out_dir, exist_ok=True)
    ckpt_dir = os.path.join(args.ckpt_dir, args.name) if args.ckpt_dir else out_dir
    os.makedirs(ckpt_dir, exist_ok=True)

    dev = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    gen = torch.Generator(device=dev).manual_seed(args.seed)

    data = load_dataset(args.dataset, args.data_dir)
    for k in ("train_x", "train_y", "test_x", "test_y"):
        data[k] = data[k].to(dev)
    if args.holdout > 0:
        # validation protocol used for all design decisions: the test set is not touched
        val_idx = stratified_subset(data["train_y"].cpu().numpy(), args.holdout / data["train_x"].shape[0], 12345)
        keep = np.setdiff1d(np.arange(data["train_x"].shape[0]), val_idx)
        vi, ki = torch.as_tensor(val_idx, device=dev), torch.as_tensor(keep, device=dev)
        data["test_x"], data["test_y"] = data["train_x"][vi], data["train_y"][vi]
        data["train_x"], data["train_y"] = data["train_x"][ki], data["train_y"][ki]
    X, Y, C = data["train_x"], data["train_y"], data["num_classes"]
    N = X.shape[0]

    # ---- which labels may be used, and which images are in the pretraining pool
    if args.pool_file:
        labeled_idx = np.sort(np.load(args.pool_file))
    else:
        labeled_idx = labeled_subset(data, args.label_fraction, args.split_seed)
    labeled_mask = torch.zeros(N, dtype=torch.bool, device=dev)
    labeled_mask[torch.as_tensor(labeled_idx, device=dev, dtype=torch.long)] = True
    pool_idx = np.arange(N) if args.pool == "all" else labeled_idx
    assert len(pool_idx) > 0, "empty pretraining pool"
    uses_labels = args.method in ("supcon", "ce")
    if args.method == "ce":
        assert args.pool == "labeled" or len(labeled_idx) == N, "ce needs --pool labeled"
    # Images with a ground-truth label, used by every evaluation. On STL-10 only the labeled split has
    # labels, so probes, kNN and geometry use those 5,000 images and the test set.
    has_unl = "labeled_idx" in data
    if has_unl:
        assert args.holdout == 0
        ev = torch.as_tensor(data["labeled_idx"], device=dev)
        Xe, Ye = X[ev], Y[ev]
        eval_data = dict(data, train_x=Xe, train_y=Ye)
        eval_pool, eval_lab = np.arange(Xe.shape[0]), None
    else:
        Xe, Ye, eval_data, eval_pool, eval_lab = X, Y, data, pool_idx, labeled_idx

    B = args.batch_size
    total_steps = int(round(args.epochs * (N // B)))
    warmup = max(1, int(args.warmup_frac * total_steps))
    print(f"[run] {args.name}: {args.method} | pool {len(pool_idx)} | labeled {len(labeled_idx)} | "
          f"{total_steps} steps | lr {args.lr} wd {args.wd} tau {args.temperature}", flush=True)

    # ---- samplers
    if args.method == "supcon" and args.pool == "all" and args.labeled_share and 0 < len(labeled_idx) < N:
        n_lab = int(round(args.labeled_share * B))
        unl_idx = np.setdiff1d(np.arange(N), labeled_idx)
        samplers = [(IndexSampler(labeled_idx, n_lab, gen, dev), n_lab),
                    (IndexSampler(unl_idx, B - n_lab, gen, dev), B - n_lab)]
    else:
        samplers = [(IndexSampler(pool_idx, B, gen, dev), B)]

    def next_batch():
        return torch.cat([s.next(n) for s, n in samplers])

    # ---- model
    aug = GPUAugment(sup_pipeline(X.shape[-1]) if args.method == "ce" else args.aug, data["mean"], data["std"], dev, gen)
    if args.method in ("simclr", "supcon"):
        model = ContrastiveModel(args.backbone)
    elif args.method == "simsiam":
        model = SimSiamModel(args.backbone)
    elif args.method == "byol":
        model = BYOLModel(args.backbone)
    else:
        model = Classifier(args.backbone, C)
    if args.init_run:
        src = torch.load(os.path.join(args.ckpt_dir or args.out, args.init_run, "model.pt"), map_location="cpu")["model"]
        own = model.state_dict()
        keep = {k: v.float() if v.is_floating_point() else v for k, v in src.items()
                if k in own and own[k].shape == v.shape}
        missing = [k for k in own if k not in keep]
        model.load_state_dict(keep, strict=False)
        print(f"[init] {args.init_run}: loaded {len(keep)} tensors, {len(missing)} left at initialisation", flush=True)
    model = model.to(dev).to(memory_format=torch.channels_last)
    target = None
    if args.method == "byol":
        target = copy.deepcopy(model)
        for q in target.parameters():
            q.requires_grad_(False)
    net = torch.compile(model) if args.compile else model

    if args.method == "simsiam":   # the predictor keeps a constant learning rate
        groups = [{"params": [q for n, q in model.named_parameters() if not n.startswith("predictor")], "decay": True},
                  {"params": list(model.predictor.parameters()), "decay": False}]
    else:
        groups = [{"params": list(model.parameters()), "decay": True}]
    opt = torch.optim.SGD(groups, lr=args.lr, momentum=0.9, weight_decay=args.wd,
                          nesterov=(args.method == "ce"))
    scaler = torch.amp.GradScaler("cuda")

    curve_at = sorted({int(round(total_steps * (i + 1) / args.curve_points)) for i in range(args.curve_points)})
    curve, loss_log, run_loss, run_n = [], [], 0.0, 0
    geom_curve = []
    pool_t = torch.as_tensor(eval_pool, device=dev)
    lab_t = torch.as_tensor(np.arange(0) if has_unl else labeled_idx, device=dev, dtype=torch.long)

    def geom_point(at, ftr, fte):
        """Class tightness (CDNV) on the pretraining pool, on the labeled images and on the test set."""
        out = {"step": at, "pool_cdnv": geometry(ftr[pool_t], Ye[pool_t], C)[0]["cdnv"],
               "test_cdnv": geometry(fte, data["test_y"], C)[0]["cdnv"]}
        if 0 < lab_t.numel() < N:
            out["labeled_cdnv"] = geometry(ftr[lab_t], Ye[lab_t], C)[0]["cdnv"]
        return out
    t0 = time.time()
    model.train()
    for step in range(total_steps):
        lr = lr_at(step, total_steps, args.lr, warmup)
        for gr in opt.param_groups:
            gr["lr"] = lr if gr["decay"] else args.lr
        idx = next_batch()
        xb = X[idx]
        # forward passes run in mixed precision; every loss is computed in float32
        n = idx.shape[0]
        if args.method == "ce":
            with torch.autocast("cuda", dtype=torch.float16):
                logits = net(aug(xb).contiguous(memory_format=torch.channels_last))
            loss = F.cross_entropy(logits.float(), Y[idx])
        else:
            x = aug(torch.cat([xb, xb])).contiguous(memory_format=torch.channels_last)
            with torch.autocast("cuda", dtype=torch.float16):
                out = net(x)
            if args.method in ("simclr", "supcon"):
                z = F.normalize(out[1].float(), dim=1)
                lab = labeled_mask[idx] if uses_labels else torch.zeros_like(idx, dtype=torch.bool)
                temp = args.temperature
                if args.temperature_unlabeled is not None:
                    temp = torch.where(lab, args.temperature, args.temperature_unlabeled)
                loss = grouped_contrastive_loss(z, make_groups(idx, Y[idx], lab, C), temp)
            elif args.method == "simsiam":
                _, z, pr = out
                loss = 0.5 * (negative_cosine(pr[:n], z[n:]) + negative_cosine(pr[n:], z[:n]))
            else:  # byol
                _, _, pr = out
                with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                    _, zt, _ = target(x)
                loss = negative_cosine(pr[:n], zt[n:]) + negative_cosine(pr[n:], zt[:n])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if target is not None:
            m = 1 - (1 - 0.99) * (math.cos(math.pi * step / total_steps) + 1) / 2
            with torch.no_grad():
                for qt, qo in zip(target.parameters(), model.parameters()):
                    qt.mul_(m).add_(qo.detach(), alpha=1 - m)
                for bt, bo in zip(target.buffers(), model.buffers()):
                    bt.copy_(bo)

        run_loss += loss.detach()
        run_n += 1
        if (step + 1) % 100 == 0 or step + 1 == total_steps:
            avg = (run_loss / run_n).item()
            if not math.isfinite(avg):
                raise RuntimeError(f"non-finite loss at step {step + 1}")
            loss_log.append([step + 1, avg])
            run_loss, run_n = 0.0, 0
        if (step + 1) in curve_at and step + 1 < total_steps:
            ftr = extract_features(model.encoder, Xe, aug.normalize)
            fte = extract_features(model.encoder, data["test_x"], aug.normalize)
            acc = 100 * (knn_predict(ftr, Ye, fte, C) == data["test_y"]).float().mean().item()
            curve.append([step + 1, acc])
            geom_curve.append(geom_point(step + 1, ftr, fte))
            print(f"  step {step + 1:6d}/{total_steps}  loss {loss.item():.4f}  knn {acc:.2f}  "
                  f"({time.time() - t0:.0f}s)", flush=True)
    torch.cuda.synchronize()
    train_time = time.time() - t0

    # ---- evaluation
    model.eval()
    t1 = time.time()
    head = getattr(model, "head", None) if args.method in ("simclr", "supcon") else None
    metrics, preds = evaluate_encoder(model.encoder, eval_data, aug.normalize, eval_lab, eval_pool,
                                      seed=args.seed, head=head, aug=aug)
    curve.append([total_steps, metrics["knn_acc"]])
    if args.method == "ce":
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            pr = torch.cat([model(aug.normalize(data["test_x"][i:i + 2000]).contiguous(memory_format=torch.channels_last)).argmax(1)
                            for i in range(0, data["test_x"].shape[0], 2000)])
        metrics["classifier_acc"] = 100 * (pr == data["test_y"]).float().mean().item()
        preds["classifier"] = pr

    result = {
        "name": args.name, "args": vars(args), "n_pool": int(len(pool_idx)), "n_labeled": int(len(labeled_idx)),
        "steps": total_steps, "train_time_s": train_time, "eval_time_s": time.time() - t1,
        "final_loss": loss_log[-1][1], "loss_log": loss_log, "knn_curve": curve, "geom_curve": geom_curve,
        "metrics": metrics,
        "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
    }
    np.savez_compressed(os.path.join(out_dir, "preds.npz"), **to_numpy_preds(preds),
                        labeled_idx=np.asarray(labeled_idx, dtype=np.int32))
    if not args.no_save_ckpt:
        sd = {k: (v.half() if v.is_floating_point() else v) for k, v in model.state_dict().items()}
        torch.save({"model": sd, "args": vars(args)}, os.path.join(ckpt_dir, "model.pt"))
    with open(res_path, "w") as f:
        json.dump(result, f, indent=1)
    keys = ("linear_acc", "knn_acc", "linear_lab_acc", "classifier_acc")
    print(f"[done] {args.name}: " + "  ".join(f"{k} {metrics[k]:.2f}" for k in keys if k in metrics)
          + f"  | train {train_time / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()


**`dlt/evaluate.py`** (288 lines) — frozen-encoder evaluation: linear probes, weighted kNN, CDNV and simplex-ETF geometry, colour-reliance probes.

In [ ]:
%%writefile dlt/evaluate.py
"""Evaluation of frozen encoders: probes, kNN, representation geometry, colour reliance.

All functions work on cached features that live on the GPU.
"""

import numpy as np
import torch
import torch.nn.functional as F

from .gpu_aug import rgb_to_gray
from .models import extract_features

L2_GRID = (1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1)


# ----------------------------------------------------------------------------
# classifiers on frozen features
# ----------------------------------------------------------------------------

def fit_logreg(x, y, num_classes, l2, max_iter=300, init=None):
    """Multinomial logistic regression (mean cross-entropy + l2/2 ||W||^2), full-batch L-BFGS."""
    d = x.shape[1]
    with torch.enable_grad():
        w = torch.zeros(d, num_classes, device=x.device, requires_grad=True)
        b = torch.zeros(num_classes, device=x.device, requires_grad=True)
        if init is not None:
            w.data.copy_(init[0])
            b.data.copy_(init[1])
        opt = torch.optim.LBFGS([w, b], lr=1.0, max_iter=max_iter, history_size=20,
                                tolerance_grad=1e-7, tolerance_change=1e-10,
                                line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            loss = F.cross_entropy(x @ w + b, y) + 0.5 * l2 * (w ** 2).sum()
            loss.backward()
            return loss

        opt.step(closure)
    return w.detach(), b.detach()


def _val_split(y, num_classes, val_frac, seed):
    """Stratified split of indices into (fit, val)."""
    g = torch.Generator().manual_seed(seed)
    y_cpu = y.cpu()
    fit, val = [], []
    for c in range(num_classes):
        idx = (y_cpu == c).nonzero(as_tuple=True)[0]
        idx = idx[torch.randperm(idx.numel(), generator=g)]
        k = max(1, int(round(val_frac * idx.numel()))) if idx.numel() > 1 else 0
        val.append(idx[:k])
        fit.append(idx[k:])
    return torch.cat(fit).to(y.device), torch.cat(val).to(y.device)


def linear_probe(train_f, train_y, test_f, num_classes, stats=None, seed=0, l2_grid=L2_GRID):
    """Linear evaluation on frozen features.

    Features are standardised, the L2 strength is chosen on a held-out 20% of
    the *training* labels, and the classifier is refit on all training labels.
    The test set is only used for the final prediction.

    Returns (test predictions, dict with chosen l2 and validation accuracy, (W, b, mean, std)).
    """
    if stats is None:
        mean, std = train_f.mean(0, keepdim=True), train_f.std(0, keepdim=True) + 1e-6
    else:
        mean, std = stats
    xtr = (train_f - mean) / std
    xte = (test_f - mean) / std
    fit_idx, val_idx = _val_split(train_y, num_classes, 0.2, seed)
    best = (-1.0, None)
    for l2 in l2_grid:
        w, b = fit_logreg(xtr[fit_idx], train_y[fit_idx], num_classes, l2)
        acc = ((xtr[val_idx] @ w + b).argmax(1) == train_y[val_idx]).float().mean().item()
        if acc > best[0]:
            best = (acc, l2)
    w, b = fit_logreg(xtr, train_y, num_classes, best[1])
    pred = (xte @ w + b).argmax(1)
    return pred, {"l2": best[1], "val_acc": 100 * best[0]}, (w, b, mean, std)


@torch.no_grad()
def knn_predict(train_f, train_y, test_f, num_classes, k=200, temperature=0.1):
    """Weighted k-nearest-neighbour classifier on cosine similarity (Wu et al. 2018)."""
    train_f = F.normalize(train_f, dim=1)
    test_f = F.normalize(test_f, dim=1)
    k = min(k, train_f.shape[0])
    preds = []
    for chunk in test_f.split(2000):
        sim, idx = (chunk @ train_f.t()).topk(k, dim=1)
        votes = torch.zeros(chunk.shape[0], num_classes, device=chunk.device)
        votes.scatter_add_(1, train_y[idx], (sim / temperature).exp())
        preds.append(votes.argmax(1))
    return torch.cat(preds)


# ----------------------------------------------------------------------------
# geometry: neural-collapse / ETF statistics
# ----------------------------------------------------------------------------

@torch.no_grad()
def class_means_vars(f, y, num_classes):
    f = f.double()
    means = torch.zeros(num_classes, f.shape[1], device=f.device, dtype=f.dtype)
    means.index_add_(0, y, f)
    counts = torch.bincount(y, minlength=num_classes).clamp(min=1).to(f.dtype)
    means = means / counts.view(-1, 1)
    sq = ((f - means[y]) ** 2).sum(1)
    var = torch.zeros(num_classes, device=f.device, dtype=f.dtype).index_add_(0, y, sq) / counts
    return means, var


@torch.no_grad()
def geometry(f, y, num_classes, ref_means=None):
    """Neural-collapse statistics of features ``f`` with labels ``y``.

    cdnv            class-distance normalised variance (Galanti et al. 2022); lower = tighter classes
    etf_cos_mean    mean cosine between centred class means (-1/(C-1) for a simplex ETF)
    etf_cos_std     std of those cosines (0 for an ETF)                       -> equiangularity
    etf_norm_cv     coefficient of variation of the class-mean norms (0 for an ETF) -> equinorm
    etf_dev         mean |cos + 1/(C-1)|; distance of the angles from the simplex ETF
    ncc_acc         accuracy of the nearest-class-centre rule, centres taken from ``ref_means``
                    (or from ``f`` itself if not given)
    """
    means, var = class_means_vars(f, y, num_classes)
    dist2 = torch.cdist(means, means) ** 2
    off = ~torch.eye(num_classes, dtype=torch.bool, device=f.device)
    cdnv = ((var.view(-1, 1) + var.view(1, -1)) / (2 * dist2.clamp(min=1e-12)))[off].mean().item()
    centred = means - means.mean(0, keepdim=True)
    norms = centred.norm(dim=1)
    cos = F.normalize(centred, dim=1) @ F.normalize(centred, dim=1).t()
    cos_off = cos[off]
    target = -1.0 / (num_classes - 1)
    centres = means if ref_means is None else ref_means
    pred = torch.cdist(f.double(), centres).argmin(1)
    return {
        "cdnv": cdnv,
        "etf_cos_mean": cos_off.mean().item(),
        "etf_cos_std": cos_off.std().item(),
        "etf_norm_cv": (norms.std() / norms.mean()).item(),
        "etf_dev": (cos_off - target).abs().mean().item(),
        "ncc_acc": 100 * (pred == y).float().mean().item(),
    }, means


@torch.no_grad()
def align_uniform(z1, z2):
    """Alignment and uniformity of L2-normalised embeddings (Wang & Isola 2020)."""
    z1, z2 = F.normalize(z1.float(), dim=1), F.normalize(z2.float(), dim=1)
    align = (z1 - z2).pow(2).sum(1).mean().item()
    sub = z1[:5000]
    d2 = torch.pdist(sub).pow(2)
    uniform = d2.mul(-2).exp().mean().log().item()
    return align, uniform


# ----------------------------------------------------------------------------
# colour reliance
# ----------------------------------------------------------------------------

@torch.no_grad()
def color_histogram(x_uint8, bins=8):
    """Joint RGB histogram (bins^3 values, square-rooted). Contains no spatial information."""
    q = (x_uint8.long() * bins) // 256
    code = (q[:, 0] * bins + q[:, 1]) * bins + q[:, 2]
    code = code.flatten(1)
    hist = torch.zeros(code.shape[0], bins ** 3, device=x_uint8.device)
    hist.scatter_add_(1, code, torch.ones_like(code, dtype=torch.float))
    return (hist / code.shape[1]).sqrt()


@torch.no_grad()
def linear_cka(a, b):
    """Linear centred kernel alignment between two feature matrices with the same rows."""
    a = a.double() - a.double().mean(0, keepdim=True)
    b = b.double() - b.double().mean(0, keepdim=True)
    hsic = (a.t() @ b).pow(2).sum()
    return (hsic / ((a.t() @ a).norm() * (b.t() @ b).norm())).item()


@torch.no_grad()
def ridge_r2(xtr, ytr, xte, yte, lam=1e-2):
    """R^2 of a ridge regression from features to targets (how decodable the targets are)."""
    mu, sd = xtr.mean(0, keepdim=True), xtr.std(0, keepdim=True) + 1e-6
    xtr, xte = ((xtr - mu) / sd).double(), ((xte - mu) / sd).double()
    ymu = ytr.double().mean(0, keepdim=True)
    a = xtr.t() @ xtr + lam * xtr.shape[0] * torch.eye(xtr.shape[1], device=xtr.device, dtype=xtr.dtype)
    w = torch.linalg.solve(a, xtr.t() @ (ytr.double() - ymu))
    pred = xte @ w + ymu
    ss_res = (yte.double() - pred).pow(2).sum()
    ss_tot = (yte.double() - yte.double().mean(0, keepdim=True)).pow(2).sum()
    return (1 - ss_res / ss_tot).item()


def to_gray_uint8(x_uint8):
    g = rgb_to_gray(x_uint8.float() / 255.0).expand(-1, 3, -1, -1)
    return (g * 255.0).round().clamp(0, 255).to(torch.uint8)


# ----------------------------------------------------------------------------
# full evaluation of one encoder
# ----------------------------------------------------------------------------

@torch.no_grad()
def evaluate_encoder(encoder, data, normalize_fn, labeled_idx, pool_idx, seed=0,
                     head=None, aug=None, full=True):
    """Run every evaluation used in the paper on a frozen encoder.

    data         dict with train_x/train_y/test_x/test_y on the GPU
    labeled_idx  indices of training images whose labels the method was allowed to use
    pool_idx     indices of training images the encoder was pretrained on
    Returns (metrics dict, dict of prediction arrays).
    """
    C = data["num_classes"]
    xtr, ytr, xte, yte = data["train_x"], data["train_y"], data["test_x"], data["test_y"]
    ftr = extract_features(encoder, xtr, normalize_fn)
    fte = extract_features(encoder, xte, normalize_fn)
    stats = (ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6)
    m, preds = {}, {}

    # 1. standard linear evaluation: the probe sees all training labels
    pred, info, (w, b, mu, sd) = linear_probe(ftr, ytr, fte, C, stats, seed)
    m["linear_acc"] = 100 * (pred == yte).float().mean().item()
    m["linear_l2"], m["linear_val_acc"] = info["l2"], info["val_acc"]
    preds["linear"] = pred

    # 2. weighted kNN with the full labeled training set as memory bank
    pred = knn_predict(ftr, ytr, fte, C)
    m["knn_acc"] = 100 * (pred == yte).float().mean().item()
    preds["knn"] = pred
    if not full:
        return m, preds

    # 3. label-limited linear probe: the probe sees only the labels the method was given
    if labeled_idx is not None and 0 < len(labeled_idx) < xtr.shape[0]:
        li = torch.as_tensor(labeled_idx, device=xtr.device)
        pred, info, _ = linear_probe(ftr[li], ytr[li], fte, C, stats, seed)
        m["linear_lab_acc"] = 100 * (pred == yte).float().mean().item()
        m["linear_lab_l2"] = info["l2"]
        preds["linear_lab"] = pred

    # 4. geometry on images seen during pretraining vs. the test set
    pi = torch.as_tensor(pool_idx, device=xtr.device)
    g_seen, means_seen = geometry(ftr[pi], ytr[pi], C)
    g_test, _ = geometry(fte, yte, C, ref_means=means_seen)
    m.update({f"seen_{k}": v for k, v in g_seen.items()})
    m.update({f"test_{k}": v for k, v in g_test.items()})
    if labeled_idx is not None and 0 < len(labeled_idx) < len(pool_idx):
        li = torch.as_tensor(labeled_idx, device=xtr.device)
        g_lab, _ = geometry(ftr[li], ytr[li], C)
        m.update({f"labeled_{k}": v for k, v in g_lab.items()})
    held = torch.ones(xtr.shape[0], dtype=torch.bool, device=xtr.device)
    held[pi] = False
    if held.sum() >= 10 * C:
        g_held, _ = geometry(ftr[held], ytr[held], C, ref_means=means_seen)
        m.update({f"heldout_{k}": v for k, v in g_held.items()})

    # 5. colour reliance
    f_gray = extract_features(encoder, to_gray_uint8(xte), normalize_fn)
    m["linear_acc_gray"] = 100 * ((((f_gray - mu) / sd) @ w + b).argmax(1) == yte).float().mean().item()
    f_perm = extract_features(encoder, xte[:, [2, 0, 1]].contiguous(), normalize_fn)
    m["linear_acc_chperm"] = 100 * ((((f_perm - mu) / sd) @ w + b).argmax(1) == yte).float().mean().item()
    hist_te = color_histogram(xte)
    hist_tr = color_histogram(xtr)
    onehot = F.one_hot(yte, C).float()
    m["cka_hist"] = linear_cka(fte, hist_te)
    m["cka_label"] = linear_cka(fte, onehot)
    m["hist_r2"] = ridge_r2(ftr, hist_tr, fte, hist_te)

    # 6. projection-space statistics
    if head is not None and aug is not None:
        sub = xte[:5000]
        v1 = aug(sub).contiguous(memory_format=torch.channels_last)
        v2 = aug(sub).contiguous(memory_format=torch.channels_last)
        was = (encoder.training, head.training)
        encoder.eval(), head.eval()
        with torch.autocast("cuda", dtype=torch.float16):   # in chunks: 5,000 views at 96 px do not fit at once
            z1 = torch.cat([head(encoder(v1[i:i + 500])) for i in range(0, v1.shape[0], 500)])
            z2 = torch.cat([head(encoder(v2[i:i + 500])) for i in range(0, v2.shape[0], 500)])
        encoder.train(was[0]), head.train(was[1])
        m["align"], m["uniform"] = align_uniform(z1, z2)
    return m, preds


def to_numpy_preds(preds):
    return {k: v.cpu().numpy().astype(np.int16) for k, v in preds.items()}


**`dlt/finetune.py`** (129 lines) — fine-tuning on the labeled subset only (linear-probe initialisation, then the whole network); default recipe and the recipes found by the search.

In [ ]:
%%writefile dlt/finetune.py
"""Semi-supervised fine-tuning of a pretrained encoder with the labeled subset only.

Protocol (LP-FT, Kumar et al. 2022): the classifier is initialised with the
label-limited linear probe and the whole network is then fine-tuned on the
labeled images with the standard supervised augmentation (flip + padded crop).

Example
-------
python -m dlt.finetune --name simclr_all_s0__ft --from_run simclr_all_s0 --label_fraction 0.1 --seed 0
"""

import argparse
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from .data import IndexSampler, labeled_subset, load_dataset, stratified_subset
from .evaluate import linear_probe
from .gpu_aug import GPUAugment, sup_pipeline
from .models import Classifier, extract_features


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--from_run", required=True)
    p.add_argument("--out", default="results/runs")
    p.add_argument("--ckpt_dir", default=None)
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--backbone", default="resnet18")
    p.add_argument("--label_fraction", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0, help="must equal the seed of the pretraining run (same split)")
    p.add_argument("--epochs", type=float, default=60.0, help="passes over the labeled subset")
    p.add_argument("--batch_size", type=int, default=128)
    p.add_argument("--lr", type=float, default=0.01)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--holdout", type=int, default=0)
    p.add_argument("--force", action="store_true")
    args = p.parse_args(argv)

    out_dir = os.path.join(args.out, args.name)
    res_path = os.path.join(out_dir, "result.json")
    if os.path.exists(res_path) and not args.force:
        print(f"[skip] {res_path} exists")
        return
    os.makedirs(out_dir, exist_ok=True)
    dev = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.manual_seed(args.seed)
    gen = torch.Generator(device=dev).manual_seed(args.seed)

    data = load_dataset(args.dataset, args.data_dir)
    if args.holdout > 0:
        val_idx = stratified_subset(data["train_y"].numpy(), args.holdout / data["train_x"].shape[0], 12345)
        keep = np.setdiff1d(np.arange(data["train_x"].shape[0]), val_idx)
        data["test_x"], data["test_y"] = data["train_x"][val_idx], data["train_y"][val_idx]
        data["train_x"], data["train_y"] = data["train_x"][keep], data["train_y"][keep]
    for k in ("train_x", "train_y", "test_x", "test_y"):
        data[k] = data[k].to(dev)
    X, Y, C = data["train_x"], data["train_y"], data["num_classes"]
    labeled_idx = labeled_subset(data, args.label_fraction, args.seed)
    li = torch.as_tensor(labeled_idx, device=dev)

    ckpt_root = args.ckpt_dir or args.out
    ckpt = torch.load(os.path.join(ckpt_root, args.from_run, "model.pt"), map_location=dev)
    model = Classifier(args.backbone, C).to(dev)
    enc_sd = {k[len("encoder."):]: v.float() if v.is_floating_point() else v
              for k, v in ckpt["model"].items() if k.startswith("encoder.")}
    model.encoder.load_state_dict(enc_sd)
    model = model.to(memory_format=torch.channels_last)
    aug = GPUAugment(sup_pipeline(X.shape[-1]), data["mean"], data["std"], dev, gen)

    # linear-probe initialisation of the classifier (labels of the subset only)
    if "labeled_idx" in data:      # STL-10: features of the labeled images are enough
        ftr_l = extract_features(model.encoder, X[li], aug.normalize)
        stats = (ftr_l.mean(0, keepdim=True), ftr_l.std(0, keepdim=True) + 1e-6)
    else:
        ftr = extract_features(model.encoder, X, aug.normalize)
        stats = (ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6)
        ftr_l = ftr[li]
    fte = extract_features(model.encoder, data["test_x"], aug.normalize)
    pred, info, (w, b, mu, sd) = linear_probe(ftr_l, Y[li], fte, C, stats, args.seed)
    lp_acc = 100 * (pred == data["test_y"]).float().mean().item()
    with torch.no_grad():
        model.fc.weight.copy_((w / sd.t()).t())
        model.fc.bias.copy_(b - ((mu / sd) @ w).squeeze(0))

    steps = int(round(args.epochs * len(labeled_idx) / args.batch_size))
    warm = max(1, int(0.05 * steps))
    opt = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, nesterov=True, weight_decay=args.wd)
    scaler = torch.amp.GradScaler("cuda")
    sampler = IndexSampler(labeled_idx, args.batch_size, gen, dev)
    t0 = time.time()
    model.train()
    for step in range(steps):
        lr = args.lr * (step + 1) / warm if step < warm else \
            0.5 * args.lr * (1 + math.cos(math.pi * (step - warm) / max(1, steps - warm)))
        for gr in opt.param_groups:
            gr["lr"] = lr
        idx = sampler.next()
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(aug(X[idx]).contiguous(memory_format=torch.channels_last))
        loss = F.cross_entropy(logits.float(), Y[idx])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    model.eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        pred = torch.cat([model(aug.normalize(data["test_x"][i:i + 2000]).contiguous(memory_format=torch.channels_last)).argmax(1)
                          for i in range(0, data["test_x"].shape[0], 2000)])
    acc = 100 * (pred == data["test_y"]).float().mean().item()
    result = {"name": args.name, "args": vars(args), "n_labeled": int(len(labeled_idx)), "steps": steps,
              "train_time_s": time.time() - t0,
              "metrics": {"finetune_acc": acc, "linear_lab_acc": lp_acc}}
    np.savez_compressed(os.path.join(out_dir, "preds.npz"), finetune=pred.cpu().numpy().astype(np.int16))
    with open(res_path, "w") as f:
        json.dump(result, f, indent=1)
    print(f"[done] {args.name}: finetune_acc {acc:.2f} (linear probe on the same labels {lp_acc:.2f})", flush=True)


if __name__ == "__main__":
    main()


**`dlt/semisup.py`** (366 lines) — FixMatch and MixMatch on the same labeled split, from scratch or from a pretrained encoder.

In [ ]:
%%writefile dlt/semisup.py
"""Semi-supervised baselines on the same labeled split: FixMatch and MixMatch.

Both use the ResNet-18 CIFAR encoder of the rest of the study (the original
papers use WRN-28-2) and are evaluated with the EMA model, as in the papers.

FixMatch (Sohn et al. 2020): labeled batch 64, mu = 7, weak = flip + translate,
strong = RandAugment (2 ops, random magnitude) + Cutout, threshold 0.95,
SGD-Nesterov lr 0.03, weight decay 5e-4, cosine schedule cos(7 pi k / 16 K).
MixMatch (Berthelot et al. 2019): batch 64 + 64, K = 2, T = 0.5, alpha = 0.75,
lambda_u = 75 with linear ramp-up, Adam lr 0.002, weight decay 0.02 * lr.

Example
-------
python -m dlt.semisup --name fixmatch_f0.1_s0 --method fixmatch --label_fraction 0.1 --seed 0
"""

import argparse
import copy
import json
import math
import os
import random
import time

import numpy as np
import PIL
import PIL.ImageDraw
import PIL.ImageEnhance
import PIL.ImageOps
import torch
import torch.nn.functional as F
from PIL import Image

from .data import IndexSampler, labeled_subset, load_dataset, stratified_subset
from .evaluate import evaluate_encoder, to_numpy_preds
from .gpu_aug import AugConfig, GPUAugment
from .models import Classifier

# ----------------------------------------------------------------------------
# RandAugment as used by FixMatch (operations and ranges of the paper, Table 12)
# ----------------------------------------------------------------------------
PARAMETER_MAX = 10


def _fp(v, max_v):
    return float(v) * max_v / PARAMETER_MAX


def _ip(v, max_v):
    return int(v * max_v / PARAMETER_MAX)


def _sign(v):
    return -v if random.random() < 0.5 else v


def _autocontrast(img, v, max_v, bias):
    return PIL.ImageOps.autocontrast(img)


def _brightness(img, v, max_v, bias):
    return PIL.ImageEnhance.Brightness(img).enhance(_fp(v, max_v) + bias)


def _color(img, v, max_v, bias):
    return PIL.ImageEnhance.Color(img).enhance(_fp(v, max_v) + bias)


def _contrast(img, v, max_v, bias):
    return PIL.ImageEnhance.Contrast(img).enhance(_fp(v, max_v) + bias)


def _equalize(img, v, max_v, bias):
    return PIL.ImageOps.equalize(img)


def _identity(img, v, max_v, bias):
    return img


def _posterize(img, v, max_v, bias):
    return PIL.ImageOps.posterize(img, _ip(v, max_v) + bias)


def _rotate(img, v, max_v, bias):
    return img.rotate(_sign(_ip(v, max_v) + bias))


def _sharpness(img, v, max_v, bias):
    return PIL.ImageEnhance.Sharpness(img).enhance(_fp(v, max_v) + bias)


def _shear_x(img, v, max_v, bias):
    return img.transform(img.size, Image.AFFINE, (1, _sign(_fp(v, max_v) + bias), 0, 0, 1, 0))


def _shear_y(img, v, max_v, bias):
    return img.transform(img.size, Image.AFFINE, (1, 0, 0, _sign(_fp(v, max_v) + bias), 1, 0))


def _solarize(img, v, max_v, bias):
    return PIL.ImageOps.solarize(img, 256 - (_ip(v, max_v) + bias))


def _translate_x(img, v, max_v, bias):
    return img.transform(img.size, Image.AFFINE, (1, 0, int(_sign(_fp(v, max_v) + bias) * img.size[0]), 0, 1, 0))


def _translate_y(img, v, max_v, bias):
    return img.transform(img.size, Image.AFFINE, (1, 0, 0, 0, 1, int(_sign(_fp(v, max_v) + bias) * img.size[1])))


def _cutout_abs(img, v):
    w, h = img.size
    x0 = int(max(0, np.random.uniform(0, w) - v / 2.0))
    y0 = int(max(0, np.random.uniform(0, h) - v / 2.0))
    img = img.copy()
    PIL.ImageDraw.Draw(img).rectangle((x0, y0, int(min(w, x0 + v)), int(min(h, y0 + v))), (127, 127, 127))
    return img


RA_POOL = [(_autocontrast, None, None), (_brightness, 0.9, 0.05), (_color, 0.9, 0.05), (_contrast, 0.9, 0.05),
           (_equalize, None, None), (_identity, None, None), (_posterize, 4, 4), (_rotate, 30, 0),
           (_sharpness, 0.9, 0.05), (_shear_x, 0.3, 0), (_shear_y, 0.3, 0), (_solarize, 256, 0),
           (_translate_x, 0.3, 0), (_translate_y, 0.3, 0)]


def strong_augment(img, n=2, m=10):
    """flip + reflect-padded crop (pad = size/8) + RandAugment(n, random magnitude < m) + Cutout of size/2."""
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    size = img.size[0]
    pad = size // 8
    arr = np.pad(np.asarray(img), ((pad, pad), (pad, pad), (0, 0)), mode="reflect")
    i, j = np.random.randint(0, 2 * pad + 1), np.random.randint(0, 2 * pad + 1)
    img = Image.fromarray(arr[i:i + size, j:j + size])
    for op, max_v, bias in random.choices(RA_POOL, k=n):
        v = np.random.randint(1, m)
        if random.random() < 0.5:
            img = op(img, v, max_v, bias)
    return _cutout_abs(img, size // 2)


class StrongViews(torch.utils.data.Dataset):
    """Returns (index, strongly augmented uint8 image) for a fixed list of image indices."""

    def __init__(self, x_uint8_nhwc, indices):
        self.x = x_uint8_nhwc
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = int(self.indices[i])
        img = strong_augment(Image.fromarray(self.x[idx]))
        return idx, torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1)


def _worker_init(worker_id):
    seed = torch.utils.data.get_worker_info().seed % (2 ** 31)
    random.seed(seed)
    np.random.seed(seed)


def infinite(loader):
    while True:
        for batch in loader:
            yield batch


@torch.no_grad()
def ema_update(ema, model, decay):
    for pe, pm in zip(ema.parameters(), model.parameters()):
        pe.mul_(decay).add_(pm.detach(), alpha=1 - decay)
    for be, bm in zip(ema.buffers(), model.buffers()):
        if be.dtype.is_floating_point:
            be.mul_(decay).add_(bm, alpha=1 - decay)
        else:
            be.copy_(bm)


@torch.no_grad()
def predict(model, x_uint8, normalize_fn):
    model.eval()
    out = []
    for i in range(0, x_uint8.shape[0], 2000):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(normalize_fn(x_uint8[i:i + 2000]).contiguous(memory_format=torch.channels_last)).argmax(1))
    return torch.cat(out)


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--out", default="results/runs")
    p.add_argument("--ckpt_dir", default=None)
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--method", required=True, choices=["fixmatch", "mixmatch"])
    p.add_argument("--backbone", default="resnet18")
    p.add_argument("--label_fraction", type=float, default=0.1)
    p.add_argument("--steps", type=int, default=None,
                   help="default: the number of steps that processes as many images as a 100-epoch contrastive run")
    p.add_argument("--budget", type=float, default=1.0, help="multiplier on the default number of steps")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--workers", type=int, default=16)
    p.add_argument("--holdout", type=int, default=0)
    p.add_argument("--init_run", default=None,
                   help="start from the encoder of this pretrained run (semi-supervised fine-tuning)")
    p.add_argument("--force", action="store_true")
    args = p.parse_args(argv)

    out_dir = os.path.join(args.out, args.name)
    res_path = os.path.join(out_dir, "result.json")
    if os.path.exists(res_path) and not args.force:
        print(f"[skip] {res_path} exists")
        return
    os.makedirs(out_dir, exist_ok=True)
    dev = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    random.seed(args.seed)
    gen = torch.Generator(device=dev).manual_seed(args.seed)

    data = load_dataset(args.dataset, args.data_dir)
    if args.holdout > 0:
        val_idx = stratified_subset(data["train_y"].numpy(), args.holdout / data["train_x"].shape[0], 12345)
        keep = np.setdiff1d(np.arange(data["train_x"].shape[0]), val_idx)
        data["test_x"], data["test_y"] = data["train_x"][val_idx], data["train_y"][val_idx]
        data["train_x"], data["train_y"] = data["train_x"][keep], data["train_y"][keep]
    x_nhwc = data["train_x"].permute(0, 2, 3, 1).contiguous().numpy()
    for k in ("train_x", "train_y", "test_x", "test_y"):
        data[k] = data[k].to(dev)
    X, Y, C = data["train_x"], data["train_y"], data["num_classes"]
    N = X.shape[0]
    labeled_idx = labeled_subset(data, args.label_fraction, args.seed)

    # image budget of a 100-epoch contrastive run: 100 * (N // 512) steps of 1024 views
    image_budget = 100 * (N // 512) * 1024 * args.budget
    per_step = 64 + 2 * 448 if args.method == "fixmatch" else 64 + 2 * 64
    steps = args.steps or int(round(image_budget / per_step))
    print(f"[run] {args.name}: {args.method} | labeled {len(labeled_idx)} | {steps} steps", flush=True)

    weak = GPUAugment(AugConfig(pad_crop=X.shape[-1] // 8, flip=True, pad_mode="reflection"), data["mean"], data["std"], dev, gen)
    model = Classifier(args.backbone, C)
    if args.init_run:
        src = torch.load(os.path.join(args.ckpt_dir or args.out, args.init_run, "model.pt"), map_location="cpu")["model"]
        enc = {k[len("encoder."):]: (v.float() if v.is_floating_point() else v)
               for k, v in src.items() if k.startswith("encoder.")}
        model.encoder.load_state_dict(enc)
        print(f"[init] encoder from {args.init_run} ({len(enc)} tensors)", flush=True)
    model = model.to(dev).to(memory_format=torch.channels_last)
    ema = copy.deepcopy(model)
    for q in ema.parameters():
        q.requires_grad_(False)
    lab_sampler = IndexSampler(labeled_idx, 64, gen, dev)
    scaler = torch.amp.GradScaler("cuda")
    t0 = time.time()
    log = []

    if args.method == "fixmatch":
        opt = torch.optim.SGD(model.parameters(), lr=0.03, momentum=0.9, nesterov=True, weight_decay=5e-4)
        loader = torch.utils.data.DataLoader(
            StrongViews(x_nhwc, np.arange(N)), batch_size=448, shuffle=True, drop_last=True,
            num_workers=args.workers, worker_init_fn=_worker_init, persistent_workers=True, prefetch_factor=4)
        it = infinite(loader)
        mask_rate = 0.0
        for step in range(steps):
            for gr in opt.param_groups:
                gr["lr"] = 0.03 * math.cos(7 * math.pi * step / (16 * steps))
            li = lab_sampler.next()
            ui, xs = next(it)
            ui, xs = ui.to(dev, non_blocking=True), xs.to(dev, non_blocking=True)
            inputs = torch.cat([weak(X[li]), weak(X[ui]), weak.normalize(xs)]).contiguous(memory_format=torch.channels_last)
            model.train()
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(inputs).float()
            lx, lw, ls = logits[:64], logits[64:64 + 448], logits[64 + 448:]
            loss_x = F.cross_entropy(lx, Y[li])
            probs = torch.softmax(lw.detach(), dim=1)
            conf, pseudo = probs.max(1)
            mask = (conf >= 0.95).float()
            loss_u = (F.cross_entropy(ls, pseudo, reduction="none") * mask).mean()
            loss = loss_x + loss_u
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            ema_update(ema, model, 0.999)
            mask_rate = 0.99 * mask_rate + 0.01 * mask.mean().item() if step % 20 == 0 else mask_rate
            if (step + 1) % max(1, steps // 10) == 0:
                acc = 100 * (predict(ema, data["test_x"], weak.normalize) == data["test_y"]).float().mean().item()
                log.append([step + 1, loss.item(), acc])
                print(f"  step {step + 1:6d}/{steps} loss {loss.item():.3f} ema-acc {acc:.2f} "
                      f"mask {mask.mean().item():.2f} ({time.time() - t0:.0f}s)", flush=True)
    else:  # mixmatch
        lr, alpha, lam_u, temp = 0.002, 0.75, 75.0, 0.5
        opt = torch.optim.Adam(model.parameters(), lr=lr)
        unl_sampler = IndexSampler(np.arange(N), 64, gen, dev)
        beta = torch.distributions.Beta(alpha, alpha)
        for step in range(steps):
            li, ui = lab_sampler.next(), unl_sampler.next()
            xl, u1, u2 = weak(X[li]), weak(X[ui]), weak(X[ui])
            model.train()
            with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
                pu = (torch.softmax(model(u1.contiguous(memory_format=torch.channels_last)).float(), 1)
                      + torch.softmax(model(u2.contiguous(memory_format=torch.channels_last)).float(), 1)) / 2
            pt = pu ** (1 / temp)
            tu = pt / pt.sum(1, keepdim=True)
            all_x = torch.cat([xl, u1, u2])
            all_t = torch.cat([F.one_hot(Y[li], C).float(), tu, tu])
            lam = beta.sample().item()
            lam = max(lam, 1 - lam)
            perm = torch.randperm(all_x.shape[0], generator=gen, device=dev)
            mix_x = lam * all_x + (1 - lam) * all_x[perm]
            mix_t = lam * all_t + (1 - lam) * all_t[perm]
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(mix_x.contiguous(memory_format=torch.channels_last)).float()
            loss_x = -(F.log_softmax(logits[:64], 1) * mix_t[:64]).sum(1).mean()
            loss_u = ((torch.softmax(logits[64:], 1) - mix_t[64:]) ** 2).mean()
            w = lam_u * min(1.0, (step + 1) / steps)
            loss = loss_x + w * loss_u
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            ema_update(ema, model, 0.999)
            with torch.no_grad():   # decoupled weight decay 0.02 * lr, as in the paper
                for q in model.parameters():
                    q.mul_(1 - 0.02 * lr)
            if (step + 1) % max(1, steps // 10) == 0:
                acc = 100 * (predict(ema, data["test_x"], weak.normalize) == data["test_y"]).float().mean().item()
                log.append([step + 1, loss.item(), acc])
                print(f"  step {step + 1:6d}/{steps} loss {loss.item():.3f} ema-acc {acc:.2f} "
                      f"({time.time() - t0:.0f}s)", flush=True)
    train_time = time.time() - t0

    ema.eval()
    pred = predict(ema, data["test_x"], weak.normalize)
    if "labeled_idx" in data:      # STL-10: only the labeled split has ground-truth labels
        ev = torch.as_tensor(data["labeled_idx"], device=dev)
        eval_data, e_lab, e_pool = dict(data, train_x=X[ev], train_y=Y[ev]), None, np.arange(len(ev))
    else:
        eval_data, e_lab, e_pool = data, labeled_idx, np.arange(N)
    metrics, preds = evaluate_encoder(ema.encoder, eval_data, weak.normalize, e_lab, e_pool, seed=args.seed)
    metrics["classifier_acc"] = 100 * (pred == data["test_y"]).float().mean().item()
    preds["classifier"] = pred
    result = {"name": args.name, "args": vars(args), "n_pool": N, "n_labeled": int(len(labeled_idx)),
              "steps": steps, "train_time_s": train_time, "log": log, "metrics": metrics,
              "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0)}
    np.savez_compressed(os.path.join(out_dir, "preds.npz"), **to_numpy_preds(preds),
                        labeled_idx=np.asarray(labeled_idx, dtype=np.int32))
    if args.ckpt_dir:
        os.makedirs(os.path.join(args.ckpt_dir, args.name), exist_ok=True)
        sd = {k: (v.half() if v.is_floating_point() else v) for k, v in ema.state_dict().items()}
        torch.save({"model": sd, "args": vars(args)}, os.path.join(args.ckpt_dir, args.name, "model.pt"))
    with open(res_path, "w") as f:
        json.dump(result, f, indent=1)
    print(f"[done] {args.name}: classifier_acc {metrics['classifier_acc']:.2f}  linear_acc {metrics['linear_acc']:.2f}  "
          f"| train {train_time / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()


**`dlt/distill.py`** (172 lines) — SimCLR v2-style pipeline: fine-tune a teacher, then self-distil into a student on all images.

In [ ]:
%%writefile dlt/distill.py
"""Two-stage pipeline in the style of SimCLR v2 (Chen et al., 2020b): fine-tune, then self-distil.

1. Teacher: the pretrained encoder is fine-tuned on the labeled subset (the default recipe of
   dlt.finetune: linear-probe initialisation, lr 0.01, 60 passes over the labeled images).
2. Student: a network of the same architecture, initialised from the same pretrained encoder,
   is trained on ALL training images (no labels) to match the teacher's class probabilities.

    python -m dlt.distill --name distill_simclr_all_e200_s0 --from_run simclr_all_e200_s0 \
        --label_fraction 0.1 --seed 0 --ckpt_dir <dir>
"""

import argparse
import copy
import json
import math
import os
import time

import numpy as np
import torch
import torch.nn.functional as F

from .data import IndexSampler, labeled_subset, load_dataset
from .evaluate import evaluate_encoder, linear_probe, to_numpy_preds
from .gpu_aug import GPUAugment, sup_pipeline
from .models import Classifier, extract_features


def cosine(step, total, base, warm):
    if step < warm:
        return base * (step + 1) / warm
    return 0.5 * base * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))


@torch.no_grad()
def predict(model, x, normalize):
    model.eval()
    out = []
    for i in range(0, x.shape[0], 2000):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(model(normalize(x[i:i + 2000]).contiguous(memory_format=torch.channels_last)).argmax(1))
    return torch.cat(out)


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--from_run", required=True)
    p.add_argument("--out", default="results/runs")
    p.add_argument("--ckpt_dir", default=None)
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--backbone", default="resnet18")
    p.add_argument("--label_fraction", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--ft_epochs", type=float, default=60.0)
    p.add_argument("--ft_lr", type=float, default=0.01)
    p.add_argument("--epochs", type=float, default=100.0, help="distillation passes over all training images")
    p.add_argument("--batch_size", type=int, default=512)
    p.add_argument("--lr", type=float, default=0.1)
    p.add_argument("--wd", type=float, default=5e-4)
    p.add_argument("--temperature", type=float, default=1.0)
    args = p.parse_args(argv)

    out_dir = os.path.join(args.out, args.name)
    res_path = os.path.join(out_dir, "result.json")
    if os.path.exists(res_path):
        print(f"[skip] {res_path} exists")
        return
    os.makedirs(out_dir, exist_ok=True)
    dev = "cuda"
    torch.backends.cudnn.benchmark = True
    torch.manual_seed(args.seed)
    gen = torch.Generator(device=dev).manual_seed(args.seed)

    data = load_dataset(args.dataset, args.data_dir)
    for k in ("train_x", "train_y", "test_x", "test_y"):
        data[k] = data[k].to(dev)
    X, Y, C = data["train_x"], data["train_y"], data["num_classes"]
    N = X.shape[0]
    labeled_idx = labeled_subset(data, args.label_fraction, args.seed)
    li = torch.as_tensor(labeled_idx, device=dev)
    aug = GPUAugment(sup_pipeline(X.shape[-1]), data["mean"], data["std"], dev, gen)

    ckpt = torch.load(os.path.join(args.ckpt_dir or args.out, args.from_run, "model.pt"), map_location=dev)
    enc_sd = {k[len("encoder."):]: v.float() if v.is_floating_point() else v
              for k, v in ckpt["model"].items() if k.startswith("encoder.")}

    # ---- 1. teacher: LP-initialised fine-tuning on the labeled subset
    teacher = Classifier(args.backbone, C).to(dev)
    teacher.encoder.load_state_dict(enc_sd)
    teacher = teacher.to(memory_format=torch.channels_last)
    ftr = extract_features(teacher.encoder, X[li], aug.normalize)
    stats = (ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6)
    _, _, (w, b, mu, sd) = linear_probe(ftr, Y[li], ftr[:10], C, stats, args.seed)
    with torch.no_grad():
        teacher.fc.weight.copy_((w / sd.t()).t())
        teacher.fc.bias.copy_(b - ((mu / sd) @ w).squeeze(0))
    steps = int(round(args.ft_epochs * len(labeled_idx) / 128))
    opt = torch.optim.SGD(teacher.parameters(), lr=args.ft_lr, momentum=0.9, nesterov=True, weight_decay=5e-4)
    scaler = torch.amp.GradScaler("cuda")
    sampler = IndexSampler(labeled_idx, 128, gen, dev)
    t0 = time.time()
    teacher.train()
    for step in range(steps):
        for gr in opt.param_groups:
            gr["lr"] = cosine(step, steps, args.ft_lr, max(1, int(0.05 * steps)))
        idx = sampler.next()
        with torch.autocast("cuda", dtype=torch.float16):
            logits = teacher(aug(X[idx]).contiguous(memory_format=torch.channels_last))
        loss = F.cross_entropy(logits.float(), Y[idx])
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    teacher_pred = predict(teacher, data["test_x"], aug.normalize)
    teacher_acc = 100 * (teacher_pred == data["test_y"]).float().mean().item()
    print(f"[teacher] fine-tuned on {len(labeled_idx)} labels: {teacher_acc:.2f}", flush=True)
    teacher.eval()
    for q in teacher.parameters():
        q.requires_grad_(False)

    # ---- 2. student: same architecture, same pretrained initialisation, trained on all images without labels
    student = Classifier(args.backbone, C).to(dev)
    student.encoder.load_state_dict(enc_sd)
    student = student.to(memory_format=torch.channels_last)
    B = args.batch_size
    steps = int(round(args.epochs * (N // B)))
    opt = torch.optim.SGD(student.parameters(), lr=args.lr, momentum=0.9, nesterov=True, weight_decay=args.wd)
    scaler = torch.amp.GradScaler("cuda")
    sampler = IndexSampler(np.arange(N), B, gen, dev)
    T = args.temperature
    student.train()
    for step in range(steps):
        for gr in opt.param_groups:
            gr["lr"] = cosine(step, steps, args.lr, max(1, int(0.05 * steps)))
        x = aug(X[sampler.next()]).contiguous(memory_format=torch.channels_last)
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            target = torch.softmax(teacher(x).float() / T, dim=1)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = student(x)
        loss = -(target * torch.log_softmax(logits.float() / T, dim=1)).sum(1).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        if (step + 1) % 2000 == 0:
            print(f"  step {step + 1:6d}/{steps} loss {loss.item():.4f} ({time.time() - t0:.0f}s)", flush=True)
    train_time = time.time() - t0

    pred = predict(student, data["test_x"], aug.normalize)
    if "labeled_idx" in data:
        ev = torch.as_tensor(data["labeled_idx"], device=dev)
        eval_data, e_lab, e_pool = dict(data, train_x=X[ev], train_y=Y[ev]), None, np.arange(len(ev))
    else:
        eval_data, e_lab, e_pool = data, labeled_idx, np.arange(N)
    student.eval()
    metrics, preds = evaluate_encoder(student.encoder, eval_data, aug.normalize, e_lab, e_pool, seed=args.seed, full=False)
    metrics["classifier_acc"] = 100 * (pred == data["test_y"]).float().mean().item()
    metrics["teacher_acc"] = teacher_acc
    preds["classifier"], preds["teacher"] = pred, teacher_pred
    result = {"name": args.name, "args": vars(args), "n_labeled": int(len(labeled_idx)), "steps": steps,
              "train_time_s": train_time, "metrics": metrics}
    np.savez_compressed(os.path.join(out_dir, "preds.npz"), **to_numpy_preds(preds))
    with open(res_path, "w") as f:
        json.dump(result, f, indent=1)
    print(f"[done] {args.name}: student {metrics['classifier_acc']:.2f} (teacher {teacher_acc:.2f}) "
          f"| {train_time / 60:.1f} min", flush=True)


if __name__ == "__main__":
    main()


## 3. Tests

**`tests/test_gpu_aug.py`** (108 lines) — checks the GPU augmentations against torchvision and the loss against reference NT-Xent / SupCon code (`python -m tests.test_gpu_aug`).

In [ ]:
%%writefile tests/test_gpu_aug.py
"""Checks the batched GPU augmentations against torchvision, and the loss against
the reference SupCon / NT-Xent implementations shipped with the original code.

Run:  python -m tests.test_gpu_aug
"""
import importlib.util
import os
import sys

import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt import gpu_aug as A          # noqa: E402
from dlt.losses import grouped_contrastive_loss, make_groups   # noqa: E402

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
x = torch.rand(64, 3, 32, 32, device=dev)
B = x.shape[0]


def close(a, b, tol, name):
    err = (a - b).abs().max().item()
    assert err < tol, f"{name}: max abs err {err:.2e} >= {tol}"
    print(f"  ok  {name:28s} max|err| = {err:.2e}")


print("colour ops vs torchvision.transforms.functional")
f = torch.empty(B, device=dev).uniform_(0.6, 1.4)
hue = torch.empty(B, device=dev).uniform_(-0.1, 0.1)
close(A.adjust_brightness(x.clone(), f), torch.stack([TF.adjust_brightness(x[i], f[i].item()) for i in range(B)]), 1e-5, "brightness")
close(A.adjust_contrast(x.clone(), f), torch.stack([TF.adjust_contrast(x[i], f[i].item()) for i in range(B)]), 1e-5, "contrast")
close(A.adjust_saturation(x.clone(), f), torch.stack([TF.adjust_saturation(x[i], f[i].item()) for i in range(B)]), 1e-5, "saturation")
close(A.adjust_hue(x.clone(), hue), torch.stack([TF.adjust_hue(x[i], hue[i].item()) for i in range(B)]), 1e-4, "hue")
close(A.rgb_to_gray(x), torch.stack([TF.rgb_to_grayscale(x[i]) for i in range(B)]), 1e-5, "grayscale")

print("geometry vs torchvision")
g = torch.Generator(device=dev).manual_seed(1)
top, left, h, w = A.sample_rrc(B, (0.2, 1.0), (3 / 4, 4 / 3), 32, g, dev)
assert ((top >= 0) & (left >= 0) & (top + h <= 32) & (left + w <= 32)).all()
ref = torch.stack([TF.resized_crop(x[i], int(top[i]), int(left[i]), int(h[i]), int(w[i]), [32, 32], antialias=False) for i in range(B)])
close(A.resized_crop(x, top, left, h, w), ref, 1e-4, "resized_crop")
flip = torch.rand(B, device=dev) < 0.5
ref_f = torch.where(flip.view(-1, 1, 1, 1), ref.flip(3), ref)
close(A.resized_crop(x, top, left, h, w, flip), ref_f, 1e-4, "resized_crop + flip")
area = (h * w) / 1024
print(f"  crop area fraction: min {area.min():.3f} mean {area.mean():.3f} max {area.max():.3f}")

sigma = torch.empty(B, device=dev).uniform_(0.1, 2.0)
ref = torch.stack([TF.gaussian_blur(x[i], [3, 3], [sigma[i].item()] * 2) for i in range(B)])
close(A.gaussian_blur3(x, sigma), ref, 1e-5, "gaussian_blur3")

print("random-crop with padding keeps pixels intact")
xi = (x * 255).round().byte()
out = A.padded_crop(xi.float() / 255, 4, None, g)
vals = set(torch.unique((out * 255).round().long()).tolist())
assert vals.issubset(set(range(256)))
zero_frac = (out.sum(1) == 0).float().mean().item()
print(f"  ok  padded_crop                  zero-padded pixel fraction = {zero_frac:.3f} (expected ~0.12)")

print("erasing")
m = A.sample_erasing(4096, 0.7, (0.02, 0.2), (0.3, 3.3), 32, g, dev)
any_ = m.flatten(1).any(1).float().mean().item()
frac = m.flatten(1).float().mean(1)
print(f"  ok  erasing: applied to {any_:.3f} of images (p=0.7), mean erased area {frac[frac > 0].mean():.3f}")
assert abs(any_ - 0.7) < 0.04

print("full pipelines run and are normalised")
mean, std = (0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)
for name in A.PIPELINES:
    aug = A.GPUAugment(name, mean, std, dev, torch.Generator(device=dev).manual_seed(0))
    y = aug(xi)
    assert y.shape == xi.shape and torch.isfinite(y).all(), name
print(f"  ok  {len(A.PIPELINES)} pipelines")

print("loss vs reference implementations")
root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def load(path, name):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


ref_sup = load(os.path.join(root, "src", "Final Work", "supcon", "loss_supcon.py"), "ref_sup").SupConLoss(temperature=0.1)
ref_nt = load(os.path.join(root, "src", "Final Work", "loss.py"), "ref_nt").NTXentLoss(temperature=0.5)
N, C = 96, 10
z1 = F.normalize(torch.randn(N, 128, device=dev), dim=1)
z2 = F.normalize(torch.randn(N, 128, device=dev), dim=1)
labels = torch.randint(0, C, (N,), device=dev)
idx = torch.arange(N, device=dev)
z = torch.cat([z1, z2])
all_lab = torch.ones(N, dtype=torch.bool, device=dev)
no_lab = torch.zeros(N, dtype=torch.bool, device=dev)
close(grouped_contrastive_loss(z, make_groups(idx, labels, all_lab, C), 0.1),
      ref_sup(torch.stack([z1, z2], 1), labels), 1e-4, "SupCon (all labeled)")
close(grouped_contrastive_loss(z, make_groups(idx, labels, no_lab, C), 0.5),
      ref_nt(z1, z2), 1e-4, "NT-Xent (no labels)")
half = torch.arange(N, device=dev) < N // 2
mask = (labels.view(-1, 1) == labels.view(1, -1)) & half.view(-1, 1) & half.view(1, -1)
mask = mask | torch.eye(N, dtype=torch.bool, device=dev)
close(grouped_contrastive_loss(z, make_groups(idx, labels, half, C), 0.1),
      ref_sup(torch.stack([z1, z2], 1), mask=mask.float()), 1e-4, "semi-supervised (half labeled)")
print("all checks passed")


## 4. Datasets and labeled pools

CIFAR-10/100 are downloaded on first use by `dlt/data.py`. STL-10 and Tiny-ImageNet are converted once from their
Hugging Face parquet exports into a single uint8 archive. `make_pools.py` builds the labeled pools of the coverage
experiment.

**`scripts/__init__.py`** (0 lines) — makes `scripts` importable (`python -m scripts.ft_search`).

In [ ]:
%%writefile scripts/__init__.py


**`scripts/prepare_stl10.py`** (52 lines) — STL-10 parquet export → `stl10_uint8.npz` (5,000 labeled, 100,000 unlabeled, 8,000 test images).

In [ ]:
%%writefile scripts/prepare_stl10.py
"""Convert STL-10 (Hugging Face parquet export, repo jxie/stl10) into one uint8 archive.

    python scripts/prepare_stl10.py --src <dir with data/*.parquet> --out <data_dir>

Writes <data_dir>/stl10_uint8.npz with train_x [5000,3,96,96], train_y, test_x [8000,...], test_y,
unlabeled_x [100000,...] and the channel mean / std of the training images (labeled + unlabeled).
"""
import argparse
import glob
import io
import os
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pyarrow.parquet as pq
from PIL import Image


def decode(b):
    return np.asarray(Image.open(io.BytesIO(b)).convert("RGB"), dtype=np.uint8).transpose(2, 0, 1)


def read(files):
    xs, ys = [], []
    with ProcessPoolExecutor(32) as ex:
        for f in sorted(files):
            t = pq.read_table(f).to_pydict()
            xs.extend(ex.map(decode, [d["bytes"] for d in t["image"]], chunksize=256))
            ys.extend(t["label"])
    return np.stack(xs), np.asarray(ys, dtype=np.int64)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--src", required=True)
    p.add_argument("--out", required=True)
    a = p.parse_args()
    g = lambda pat: glob.glob(os.path.join(a.src, "data", pat))   # noqa: E731
    train_x, train_y = read(g("train-*.parquet"))
    test_x, test_y = read(g("test-*.parquet"))
    unl_x, unl_y = read(g("unlabeled-*.parquet"))
    print(train_x.shape, test_x.shape, unl_x.shape, np.bincount(train_y), np.unique(unl_y))
    allx = np.concatenate([train_x, unl_x])
    mean = [float(allx[:, c].mean() / 255) for c in range(3)]
    std = [float((allx[:, c] / 255.0).std()) for c in range(3)]
    print("mean", mean, "std", std)
    np.savez(os.path.join(a.out, "stl10_uint8.npz"), train_x=train_x, train_y=train_y, test_x=test_x,
             test_y=test_y, unlabeled_x=unl_x, mean=np.array(mean), std=np.array(std))


if __name__ == "__main__":
    main()


**`scripts/prepare_tinyimagenet.py`** (49 lines) — Tiny-ImageNet parquet export → `tinyimagenet_uint8.npz` (100,000 training and 10,000 validation images, 64 px).

In [ ]:
%%writefile scripts/prepare_tinyimagenet.py
"""Convert Tiny-ImageNet (Hugging Face parquet export, repo zh-plus/tiny-imagenet) into one uint8 archive.

    python scripts/prepare_tinyimagenet.py --src <dir with data/*.parquet> --out <data_dir>

Writes <data_dir>/tinyimagenet_uint8.npz with train_x [100000,3,64,64], train_y, test_x [10000,...] (the
official validation split, used as the test set), test_y, and the channel mean / std of the training images.
"""
import argparse
import glob
import io
import os
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pyarrow.parquet as pq
from PIL import Image


def decode(b):
    return np.asarray(Image.open(io.BytesIO(b)).convert("RGB"), dtype=np.uint8).transpose(2, 0, 1)


def read(files):
    xs, ys = [], []
    with ProcessPoolExecutor(32) as ex:
        for f in sorted(files):
            t = pq.read_table(f).to_pydict()
            xs.extend(ex.map(decode, [d["bytes"] for d in t["image"]], chunksize=256))
            ys.extend(t["label"])
    return np.stack(xs), np.asarray(ys, dtype=np.int64)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--src", required=True)
    p.add_argument("--out", required=True)
    a = p.parse_args()
    train_x, train_y = read(glob.glob(os.path.join(a.src, "data", "train-*.parquet")))
    test_x, test_y = read(glob.glob(os.path.join(a.src, "data", "valid-*.parquet")))
    print(train_x.shape, test_x.shape, len(np.unique(train_y)), np.bincount(train_y).min(), np.bincount(train_y).max())
    mean = [float(train_x[:, c].mean() / 255) for c in range(3)]
    std = [float((train_x[:, c] / 255.0).std()) for c in range(3)]
    print("mean", mean, "std", std)
    np.savez(os.path.join(a.out, "tinyimagenet_uint8.npz"), train_x=train_x, train_y=train_y, test_x=test_x,
             test_y=test_y, mean=np.array(mean), std=np.array(std))


if __name__ == "__main__":
    main()


**`scripts/make_pools.py`** (127 lines) — labeled pools of fixed size with controlled diversity: redundant groups of near-duplicates, greedy k-center selection, and mixtures with random images.

In [ ]:
%%writefile scripts/make_pools.py
"""Build labeled pools of a fixed size whose *diversity* is controlled.

All pools hold exactly the same number of images per class (and therefore the
same number of labels) as the random stratified subset; only how the images are
spread over each class changes:

  clusterK   n/K randomly chosen seed images per class, each with its K-1 nearest
             same-class neighbours (K = 1 is the random subset). Larger K = more
             redundancy, lower diversity.
  kcenter    greedy farthest-point selection inside each class (maximum coverage).

Neighbours are measured by cosine similarity in the feature space of a
label-free reference encoder (SimCLR trained on all images).

    python scripts/make_pools.py --ref_run simclr_all_e100_s0 --ckpt_dir <dir> --seeds 0 1 2
"""
import argparse
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt.data import load_dataset                       # noqa: E402
from dlt.gpu_aug import GPUAugment                      # noqa: E402
from dlt.models import ContrastiveModel, extract_features   # noqa: E402


def reference_features(ref_run, ckpt_dir, data, dev="cuda"):
    ckpt = torch.load(os.path.join(ckpt_dir, ref_run, "model.pt"), map_location=dev)
    model = ContrastiveModel(ckpt["args"]["backbone"]).to(dev)
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in ckpt["model"].items()})
    model = model.to(memory_format=torch.channels_last)
    aug = GPUAugment("none", data["mean"], data["std"], dev)
    ftr = F.normalize(extract_features(model.encoder, data["train_x"].to(dev), aug.normalize), dim=1)
    fte = F.normalize(extract_features(model.encoder, data["test_x"].to(dev), aug.normalize), dim=1)
    return ftr, fte


def cluster_pool(feat, labels, per_class, k, seed):
    rng = np.random.RandomState(seed)
    out = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        perm = rng.permutation(idx)             # same permutation as the random subset
        f = feat[torch.as_tensor(idx, device=feat.device)]
        sim = f @ f.t()
        pos = {g: i for i, g in enumerate(idx)}
        taken = torch.zeros(len(idx), dtype=torch.bool, device=feat.device)
        chosen = []
        for g in perm:
            if len(chosen) >= per_class:
                break
            i = pos[g]
            if taken[i]:
                continue
            s = sim[i].clone()
            s[taken] = -2.0
            take = s.topk(min(k, per_class - len(chosen))).indices   # includes the seed itself
            taken[take] = True
            chosen.extend(idx[take.cpu().numpy()].tolist())
        out.append(np.array(chosen[:per_class]))
    return np.sort(np.concatenate(out))


def kcenter_pool(feat, labels, per_class, seed):
    rng = np.random.RandomState(seed)
    out = []
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        f = feat[torch.as_tensor(idx, device=feat.device)]
        first = int(np.where(idx == rng.permutation(idx)[0])[0][0])
        chosen = [first]
        mind = 1 - f @ f[first]
        for _ in range(per_class - 1):
            nxt = int(mind.argmax())
            chosen.append(nxt)
            mind = torch.minimum(mind, 1 - f @ f[nxt])
        out.append(idx[np.array(chosen)])
    return np.sort(np.concatenate(out))


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ref_run", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--fraction", type=float, default=0.1)
    p.add_argument("--ks", type=int, nargs="+", default=[10, 50])
    p.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2])
    p.add_argument("--mix", type=float, nargs="*", default=[],
                   help="also build pools in which this share of every class comes from the k-center / "
                        "redundant (k=50) construction and the rest is random")
    p.add_argument("--out", default="results/pools")
    args = p.parse_args()
    os.makedirs(args.out, exist_ok=True)
    data = load_dataset(args.dataset, args.data_dir)
    labels = data["train_y"].numpy()
    ftr, _ = reference_features(args.ref_run, args.ckpt_dir, data)
    per_class = int(round(args.fraction * (labels == 0).sum()))
    for s in args.seeds:
        for k in args.ks:
            pool = cluster_pool(ftr, labels, per_class, k, s)
            np.save(os.path.join(args.out, f"cluster{k}_f{args.fraction}_s{s}.npy"), pool)
            print(f"cluster{k} seed {s}: {len(pool)} images, {np.bincount(labels[pool]).tolist()}")
        pool = kcenter_pool(ftr, labels, per_class, s)
        np.save(os.path.join(args.out, f"kcenter_f{args.fraction}_s{s}.npy"), pool)
        print(f"kcenter seed {s}: {len(pool)} images")
        for share in args.mix:
            m = int(round(share * per_class))
            for kind, part in (("kcenter", kcenter_pool(ftr, labels, m, s)), ("cluster50", cluster_pool(ftr, labels, m, 50, s))):
                rng = np.random.RandomState(1000 + s)
                fill = []
                for c in np.unique(labels):
                    rest = np.setdiff1d(np.where(labels == c)[0], part)
                    fill.append(rng.permutation(rest)[:per_class - m])
                pool = np.sort(np.concatenate([part] + fill))
                assert len(pool) == per_class * len(np.unique(labels)) and len(np.unique(pool)) == len(pool)
                np.save(os.path.join(args.out, f"mix{kind}{int(round(100 * share))}_f{args.fraction}_s{s}.npy"), pool)
                print(f"mix {kind} {share:.2f} seed {s}: {len(pool)} images")


if __name__ == "__main__":
    main()


## 5. The GPU job queue

A job file lists one run per line (the arguments of `python -m <module>`, default `dlt.train`). A run counts as
finished when `results/runs/<name>/result.json` exists, so the queue can be stopped and restarted at any time.
`run_phases.sh` works through the job files of `jobs/phases.txt` in order; the watchdog restarts it if it dies.

**`scripts/run_queue.py`** (86 lines) — runs the jobs of one file on one GPU, a few at a time, skipping finished runs and retrying failures.

In [ ]:
%%writefile scripts/run_queue.py
"""Run a list of jobs on one GPU, a few at a time, skipping finished ones.

Each line of the jobs file is the argument string of ``python -m <module>``
(default module: dlt.train) and must contain ``--name <run name>``. Header lines
``# parallel: N`` and ``# retries: N`` override the command-line settings for that file. A job counts
as finished when ``<out>/<name>/result.json`` exists, so the queue can be
stopped and restarted at any time.

    python scripts/run_queue.py jobs/core.txt --out results/runs --parallel 1
"""
import argparse
import os
import shlex
import subprocess
import sys
import time

p = argparse.ArgumentParser()
p.add_argument("jobs")
p.add_argument("--out", default="results/runs")
p.add_argument("--ckpt_dir", default=None)
p.add_argument("--log_dir", default=None)
p.add_argument("--parallel", type=int, default=1)
p.add_argument("--retries", type=int, default=2)
args = p.parse_args()
# a jobs file may override the queue settings with header lines such as "# parallel: 1" or "# retries: 3"
for line in open(args.jobs):
    if line.startswith("#") and ":" in line:
        key, _, val = line[1:].partition(":")
        if key.strip() in ("parallel", "retries") and val.strip().isdigit():
            setattr(args, key.strip(), int(val.strip()))

log_dir = args.log_dir or os.path.join(args.out, "_logs")
os.makedirs(log_dir, exist_ok=True)
jobs = []
for line in open(args.jobs):
    line = line.split("#")[0].strip()
    if not line:
        continue
    toks = shlex.split(line)
    module = "dlt.train"
    if toks[0] == "-m":
        module, toks = toks[1], toks[2:]
    name = toks[toks.index("--name") + 1]
    jobs.append((name, module, toks))


def done(name):
    return os.path.exists(os.path.join(args.out, name, "result.json"))


pending = [j for j in jobs if not done(j[0])]
print(f"[queue] {len(jobs)} jobs, {len(pending)} pending", flush=True)
running, attempts, failed = [], {}, []
t0 = time.time()
while pending or running:
    while pending and len(running) < args.parallel:
        name, module, toks = pending.pop(0)
        cmd = [sys.executable, "-m", module] + toks + ["--out", args.out]
        if args.ckpt_dir:
            cmd += ["--ckpt_dir", args.ckpt_dir]
        log = open(os.path.join(log_dir, name + ".log"), "a")
        running.append((name, module, toks, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), log, time.time()))
        print(f"[queue] start {name}", flush=True)
    time.sleep(2)
    for item in list(running):
        name, module, toks, proc, log, ts = item
        rc = proc.poll()
        if rc is None:
            continue
        running.remove(item)
        log.close()
        if rc == 0 and done(name):
            left = len(pending) + len(running)
            print(f"[queue] done  {name} ({(time.time() - ts) / 60:.1f} min) | {left} left | "
                  f"elapsed {(time.time() - t0) / 3600:.2f} h", flush=True)
        else:
            attempts[name] = attempts.get(name, 0) + 1
            if attempts[name] <= args.retries:
                print(f"[queue] RETRY {name} (rc={rc})", flush=True)
                pending.insert(0, (name, module, toks))
            else:
                print(f"[queue] FAILED {name} (rc={rc})", flush=True)
                failed.append(name)
print(f"[queue] finished: {len(jobs) - len(failed)} ok, {len(failed)} failed {failed} "
      f"in {(time.time() - t0) / 3600:.2f} h", flush=True)


**`scripts/run_phases.sh`** (20 lines) — runs the job files of a phases file one after another; re-reads the phases file after every phase.

In [ ]:
%%writefile scripts/run_phases.sh
#!/bin/bash
# Run the job files listed in a phases file one after another on one GPU.
# The phases file is read again after every phase, so new phases can be appended while the queue runs.
#   scripts/run_phases.sh jobs/phases.txt <ckpt_dir> <log_dir> [parallel]
PH=$1; CK=$2; LOGS=$3; PAR=${4:-1}
mkdir -p "$LOGS"; DONE="$LOGS/phases_done.txt"; touch "$DONE"
while true; do
  next=""
  while read -r f; do
    [ -z "$f" ] && continue
    case "$f" in \#*) continue;; esac
    grep -qxF "$f" "$DONE" || { next="$f"; break; }
  done < "$PH"
  [ -z "$next" ] && break
  echo "[phases] $(date -u +%F_%T) start $next" >> "$LOGS/phases.log"
  python scripts/run_queue.py "$next" --out results/runs --ckpt_dir "$CK" --parallel "$PAR" >> "$LOGS/$(basename "$next" .txt).log" 2>&1
  echo "$next" >> "$DONE"
  echo "[phases] $(date -u +%F_%T) end   $next" >> "$LOGS/phases.log"
done
echo "[phases] $(date -u +%F_%T) all phases finished" >> "$LOGS/phases.log"


**`scripts/resume_queue.sh`** (21 lines) — starts the queue and its watchdog (safe to run again; nothing starts twice).

In [ ]:
%%writefile scripts/resume_queue.sh
#!/bin/bash
# Start (or restart) the GPU queue and its watchdog. Safe to run at any time and more than once:
# finished runs are skipped, and nothing is started twice.
#   bash scripts/resume_queue.sh
# Checkpoints, datasets and queue logs live in $STORE (default /home/jovyan/simclr_store, a persistent volume).
STORE=${STORE:-/home/jovyan/simclr_store}
cd "$(dirname "$0")/.." || exit 1
export STORE DLT_DATA=$STORE/data
mkdir -p "$STORE/logs"
if ps -ww -eo args | grep -v grep | grep -q "scripts/run_phases.sh jobs/phases.txt"; then
  echo "queue: already running"
else
  setsid nohup scripts/run_phases.sh jobs/phases.txt "$STORE/ckpt" "$STORE/logs" 2 > /dev/null 2>&1 < /dev/null &
  echo "queue: started (logs in $STORE/logs)"
fi
if ps -ww -eo args | grep -v grep | grep -q "scripts/queue_watchdog.sh"; then
  echo "watchdog: already running"
else
  setsid nohup scripts/queue_watchdog.sh > /dev/null 2>&1 < /dev/null &
  echo "watchdog: started"
fi


**`scripts/queue_watchdog.sh`** (34 lines) — every 5 minutes: restarts the queue if it died while phases are left and copies the logs to the persistent store.

In [ ]:
%%writefile scripts/queue_watchdog.sh
#!/bin/bash
# Every 5 minutes: restart the GPU queue if it died while phases are left (e.g. a killed process), and keep a
# copy of the queue logs in the persistent store. Only one watchdog runs at a time.
# Started by scripts/resume_queue.sh. A server restart kills it too: run scripts/resume_queue.sh again.
STORE=${STORE:-/home/jovyan/simclr_store}
cd "$(dirname "$0")/.." || exit 1
mkdir -p "$STORE/logs"
exec 9> "$STORE/watchdog.lock"
flock -n 9 || exit 0
LOG="$STORE/logs/watchdog.log"
echo "$(date -u +%F_%T) watchdog started" >> "$LOG"
while true; do
  # a queue started from another log directory (e.g. a Claude session scratchpad): mirror its logs
  if [ -n "$EXTRA_LOG_DIR" ] && [ -d "$EXTRA_LOG_DIR" ] && \
     ps -ww -eo args | grep -v grep | grep -q "run_phases.sh jobs/phases.txt .* $EXTRA_LOG_DIR"; then
    cp -a "$EXTRA_LOG_DIR"/. "$STORE/logs/" 2> /dev/null
  fi
  if ! ps -ww -eo args | grep -v grep | grep -q "scripts/run_phases.sh jobs/phases.txt"; then
    left=0
    while read -r f; do
      [ -z "$f" ] && continue
      case "$f" in \#*) continue;; esac
      grep -qxF "$f" "$STORE/logs/phases_done.txt" 2> /dev/null || left=$((left + 1))
    done < jobs/phases.txt
    if [ "$left" -eq 0 ]; then
      echo "$(date -u +%F_%T) all phases finished; watchdog exits" >> "$LOG"
      exit 0
    fi
    echo "$(date -u +%F_%T) queue not running, $left phases left: restarting" >> "$LOG"
    DLT_DATA="$STORE/data" setsid nohup scripts/run_phases.sh jobs/phases.txt "$STORE/ckpt" "$STORE/logs" 2 \
      > /dev/null 2>&1 < /dev/null &
  fi
  sleep 300 9>&-      # the child must not keep the lock
done


**`scripts/queue_status.sh`** (26 lines) — one-screen status: is the queue running, which jobs are on the GPU, finished runs per phase, failed jobs.

In [ ]:
%%writefile scripts/queue_status.sh
#!/bin/bash
# One-screen status of the GPU queue:   bash scripts/queue_status.sh
STORE=${STORE:-/home/jovyan/simclr_store}
cd "$(dirname "$0")/.." || exit 1
date -u +"now: %F %H:%M UTC"
if ps -ww -eo args | grep -v grep | grep -q "scripts/run_phases.sh jobs/phases.txt"; then echo "queue: RUNNING"; else echo "queue: NOT RUNNING  ->  bash scripts/resume_queue.sh"; fi
if ps -ww -eo args | grep -v grep | grep -q "scripts/queue_watchdog.sh"; then echo "watchdog: running"; else echo "watchdog: not running"; fi
echo "jobs on the GPU now:"; ps -eo etime,args | grep -E "python .*-m (dlt|scripts)\." | grep -v grep | sed -E 's/.*--name ([^ ]+).*/  \1/' | sort -u 
nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader 2> /dev/null | sed 's/^/GPU: /'
python - <<'PY'
import os, shlex
tot = done = 0; rows = []
for ph in open("jobs/phases.txt").read().split():
    n = d = 0
    for line in open(ph):
        s = line.split("#")[0].strip()
        if not s: continue
        t = shlex.split(s); name = t[t.index("--name") + 1]; n += 1
        d += os.path.exists(f"results/runs/{name}/result.json")
    tot += n; done += d
    rows.append(f"  {os.path.basename(ph)[4:-4]:24s} {d:3d}/{n}")
print(f"runs finished: {done}/{tot}"); print("\n".join(rows))
PY
# failures whose run has not been completed since (a later retry may have succeeded)
grep -h "FAILED" "$STORE"/logs/rev_*.log 2> /dev/null | sed -E 's/.*FAILED ([^ ]+).*/\1/' | sort -u | while read -r n; do
  [ -f "results/runs/$n/result.json" ] || echo "failed (not yet rerun): $n"; done


## 6. Experiment definitions

Every run of the study, as job files. `00`–`04` are the first study, `05`–`06` the first revision, and
`jobs/phases.txt` lists the phases of the second revision (`rev_p*`) in the order the queue runs them.

**`jobs/00_pilot.txt`** (12 lines) — 10 runs.

In [ ]:
%%writefile jobs/00_pilot.txt
# Design decisions are made here, on a 5,000-image validation split of the training set
# (--holdout 5000, seed 100, 50 epochs). The test set is not used.
--name pilot_simclr_t0.5        --method simclr --holdout 5000 --seed 100 --epochs 50
--name pilot_supconL_t0.1       --method supcon --pool labeled --label_fraction 0.1 --holdout 5000 --seed 100 --epochs 50
--name pilot_union_mixed        --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --holdout 5000 --seed 100 --epochs 50
--name pilot_union_mixed_sh0.5  --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.5 --holdout 5000 --seed 100 --epochs 50
--name pilot_union_t0.1         --method supcon --pool all --label_fraction 0.1 --holdout 5000 --seed 100 --epochs 50
--name pilot_union_t0.5         --method supcon --pool all --label_fraction 0.1 --temperature 0.5 --holdout 5000 --seed 100 --epochs 50
--name pilot_union_mixed_sh0.25 --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --holdout 5000 --seed 100 --epochs 50
--name pilot_simclr_t0.1        --method simclr --temperature 0.1 --holdout 5000 --seed 100 --epochs 50
--name pilot_simclr_t0.2        --method simclr --temperature 0.2 --holdout 5000 --seed 100 --epochs 50
--name pilot_supconL_t0.5       --method supcon --pool labeled --label_fraction 0.1 --temperature 0.5 --holdout 5000 --seed 100 --epochs 50


**`jobs/01_main.txt`** (99 lines) — 81 runs.

In [ ]:
%%writefile jobs/01_main.txt
# Main study. ResNet-18, CIFAR-10, epochs = passes over the FULL training set (same number of steps for every method).
# Union variant (tau_lab 0.1, tau_unl 0.5, 25% labeled per batch) was chosen on the validation split (jobs/00_pilot.txt).
# 1) core 2x2 + union, 3 seeds
--name simclr_all_e100_s0 --method simclr --pool all --epochs 100 --seed 0
--name supcon_lab_f0.1_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name simclr_sub_f0.1_e100_s0 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name union_f0.1_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 0
--name simclr_all_e100_s1 --method simclr --pool all --epochs 100 --seed 1
--name supcon_lab_f0.1_e100_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name simclr_sub_f0.1_e100_s1 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name union_f0.1_e100_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 1
--name simclr_all_e100_s2 --method simclr --pool all --epochs 100 --seed 2
--name supcon_lab_f0.1_e100_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name simclr_sub_f0.1_e100_s2 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name union_f0.1_e100_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 2
# 2) the original report's budget: SupCon for 100 epochs over the 5,000 images = 10 full-data epochs
--name supcon_lab_f0.1_e10_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 0
--name simclr_all_e10_s0 --method simclr --pool all --epochs 10 --seed 0
--name supcon_lab_f0.1_e10_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 1
--name simclr_all_e10_s1 --method simclr --pool all --epochs 10 --seed 1
--name supcon_lab_f0.1_e10_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 2
--name simclr_all_e10_s2 --method simclr --pool all --epochs 10 --seed 2
--name supcon_lab_f0.1_e25_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --seed 0
--name simclr_all_e25_s0 --method simclr --pool all --epochs 25 --seed 0
--name union_f0.1_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 25 --seed 0
--name supcon_lab_f0.1_e50_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 50 --seed 0
--name simclr_all_e50_s0 --method simclr --pool all --epochs 50 --seed 0
--name union_f0.1_e50_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 50 --seed 0
# 2b) the original report used a ResNet-50: same short budget with ResNet-50 (report lr 0.05 and our lr 0.25)
--name r50_supcon_lab_f0.1_e10_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 0
--name r50_supcon_lab_f0.1_e10_lr0.05_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --lr 0.05 --seed 0
--name r50_simclr_all_e10_s0 --backbone resnet50 --method simclr --pool all --epochs 10 --seed 0
# 2c) longer budget (200 epochs = 19,400 steps): does label-only SupCon saturate?
--name supcon_lab_f0.1_e200_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
--name simclr_all_e200_s0 --method simclr --pool all --epochs 200 --seed 0
--name union_f0.1_e200_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 0
# 2c') union without oversampling the labeled images (uniform batches), to test whether oversampling causes the long-budget drop
--name unionunif_f0.1_e200_s0 --method supcon --pool all --temperature_unlabeled 0.5 --label_fraction 0.1 --epochs 200 --seed 0
--name unionunif_f0.1_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --label_fraction 0.1 --epochs 100 --seed 0
# 2d) more seeds for the 200-epoch budget
--name supcon_lab_f0.1_e200_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name simclr_all_e200_s1 --method simclr --pool all --epochs 200 --seed 1
--name union_f0.1_e200_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 1
--name supcon_lab_f0.1_e200_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name simclr_all_e200_s2 --method simclr --pool all --epochs 200 --seed 2
--name union_f0.1_e200_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 2
# 3) supervised cross-entropy
--name ce_f0.1_e100_s0 --method ce --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name ce_f1.0_e100_s0 --method ce --pool labeled --label_fraction 1.0 --epochs 100 --seed 0
# 4) label-fraction sweep (seed 0)
--name supcon_lab_f0.01_e100_s0 --method supcon --pool labeled --label_fraction 0.01 --epochs 100 --seed 0
--name union_f0.01_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.01 --epochs 100 --seed 0
--name simclr_sub_f0.01_e100_s0 --method simclr --pool labeled --label_fraction 0.01 --epochs 100 --seed 0
--name supcon_lab_f0.05_e100_s0 --method supcon --pool labeled --label_fraction 0.05 --epochs 100 --seed 0
--name union_f0.05_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.05 --epochs 100 --seed 0
--name simclr_sub_f0.05_e100_s0 --method simclr --pool labeled --label_fraction 0.05 --epochs 100 --seed 0
--name supcon_lab_f0.2_e100_s0 --method supcon --pool labeled --label_fraction 0.2 --epochs 100 --seed 0
--name union_f0.2_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.2 --epochs 100 --seed 0
--name simclr_sub_f0.2_e100_s0 --method simclr --pool labeled --label_fraction 0.2 --epochs 100 --seed 0
# 4b) fixed-size labeled pools (5,000 images, 500/class) with controlled coverage (built by scripts/make_pools.py)
--name supcon_lab_cluster10_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/cluster10_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_cluster50_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/cluster50_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_kcenter_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/kcenter_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_cluster50_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/cluster50_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_kcenter_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/kcenter_f0.1_s0.npy --epochs 100 --seed 0
# 4c) fine-tuning every 10%-label encoder on its 5,000 labeled images (LP-FT, 60 epochs over the labeled set)
-m dlt.finetune --name ft_simclr_all_e100_s0 --from_run simclr_all_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s0 --from_run supcon_lab_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s0 --from_run simclr_sub_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e100_s0 --from_run union_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_s1 --from_run simclr_all_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s1 --from_run supcon_lab_f0.1_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s1 --from_run simclr_sub_f0.1_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e100_s1 --from_run union_f0.1_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e100_s2 --from_run simclr_all_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s2 --from_run supcon_lab_f0.1_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s2 --from_run simclr_sub_f0.1_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e100_s2 --from_run union_f0.1_e100_s2 --label_fraction 0.1 --seed 2
# 5) other self-supervised methods
--name byol_all_e100_s0 --method byol --pool all --epochs 100 --seed 0
--name simsiam_all_e100_s0 --method simsiam --pool all --epochs 100 --seed 0
# 6) colour shortcut / augmentation tiers (SimCLR, all images)
--name simclr_exp2_e100_s0 --method simclr --pool all --aug exp2 --epochs 100 --seed 0
--name simclr_exp5_e100_s0 --method simclr --pool all --aug exp5 --epochs 100 --seed 0
--name simclr_exp6_e100_s0 --method simclr --pool all --aug exp6 --epochs 100 --seed 0
--name supcon_lab_f0.1_exp6_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --aug exp6 --epochs 100 --seed 0
# 7) CIFAR-100
--name c100_simclr_all_e100_s0 --dataset cifar100 --method simclr --pool all --epochs 100 --seed 0
--name c100_supcon_lab_f0.1_e100_s0 --dataset cifar100 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name c100_union_f0.1_e100_s0 --dataset cifar100 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 0
--name c100_simclr_sub_f0.1_e100_s0 --dataset cifar100 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
# 8) semi-supervised baselines on the same split (same number of processed images)
# 9) ResNet-50
--name r50_simclr_all_e100_s0 --backbone resnet50 --method simclr --pool all --epochs 100 --seed 0
--name r50_supcon_lab_f0.1_e100_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name r50_union_f0.1_e100_s0 --backbone resnet50 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 0
# 10) fine-tuning of the other encoders
-m dlt.finetune --name ft_supcon_lab_cluster50_f0.1_e100_s0 --from_run supcon_lab_cluster50_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_kcenter_f0.1_e100_s0 --from_run supcon_lab_kcenter_f0.1_e100_s0 --label_fraction 0.1 --seed 0


**`jobs/02_semisup.txt`** (3 lines) — 2 runs.

In [ ]:
%%writefile jobs/02_semisup.txt
# FixMatch / MixMatch on the same split; they are light on the GPU and ran in parallel with jobs/01_main.txt
-m dlt.semisup --name fixmatch_f0.1_s0 --method fixmatch --label_fraction 0.1 --seed 0
-m dlt.semisup --name mixmatch_f0.1_s0 --method mixmatch --label_fraction 0.1 --seed 0


**`jobs/03_finetune.txt`** (22 lines) — 19 runs.

In [ ]:
%%writefile jobs/03_finetune.txt
# fine-tuning of the 200-epoch encoders and of the label-fraction sweep (ran in parallel with jobs/01_main.txt)
# 11) fine-tuning of the 200-epoch encoders
-m dlt.finetune --name ft_simclr_all_e200_s0 --from_run simclr_all_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s0 --from_run supcon_lab_f0.1_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e200_s0 --from_run union_f0.1_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e200_s1 --from_run simclr_all_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s1 --from_run supcon_lab_f0.1_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e200_s1 --from_run union_f0.1_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e200_s2 --from_run simclr_all_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s2 --from_run supcon_lab_f0.1_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e200_s2 --from_run union_f0.1_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_unionunif_f0.1_e200_s0 --from_run unionunif_f0.1_e200_s0 --label_fraction 0.1 --seed 0
# 12) fine-tuning with other label fractions (seed 0)
-m dlt.finetune --name ft_simclr_all_e100_f0.01_s0 --from_run simclr_all_e100_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.01_e100_s0 --from_run supcon_lab_f0.01_e100_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_union_f0.01_e100_s0 --from_run union_f0.01_e100_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.05_s0 --from_run simclr_all_e100_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.05_e100_s0 --from_run supcon_lab_f0.05_e100_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_union_f0.05_e100_s0 --from_run union_f0.05_e100_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.2_s0 --from_run simclr_all_e100_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.2_e100_s0 --from_run supcon_lab_f0.2_e100_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_union_f0.2_e100_s0 --from_run union_f0.2_e100_s0 --label_fraction 0.2 --seed 0


**`jobs/04_extra_finetune.txt`** (10 lines) — 9 runs.

In [ ]:
%%writefile jobs/04_extra_finetune.txt
# fine-tuning on CIFAR-100 (10% labels) and of BYOL/SimSiam/pool encoders
-m dlt.finetune --name ft_c100_simclr_all_e100_s0 --from_run c100_simclr_all_e100_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_c100_supcon_lab_f0.1_e100_s0 --from_run c100_supcon_lab_f0.1_e100_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_c100_union_f0.1_e100_s0 --from_run c100_union_f0.1_e100_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_byol_all_e100_s0 --from_run byol_all_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simsiam_all_e100_s0 --from_run simsiam_all_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_unionunif_f0.1_e100_s0 --from_run unionunif_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_r50_simclr_all_e100_s0 --from_run r50_simclr_all_e100_s0 --backbone resnet50 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_r50_supcon_lab_f0.1_e100_s0 --from_run r50_supcon_lab_f0.1_e100_s0 --backbone resnet50 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_r50_union_f0.1_e100_s0 --from_run r50_union_f0.1_e100_s0 --backbone resnet50 --label_fraction 0.1 --seed 0


**`jobs/05_revision_main.txt`** (131 lines) — 123 runs.

In [ ]:
%%writefile jobs/05_revision_main.txt
# Revision round 2, part 1 (runs that need no new code). Order = priority.
# two-stage: SimCLR on all images, then SupCon on the 5k labeled images (labels last, still contrastive)
--name twostage_simclr100_supcon5_s0 --init_run simclr_all_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --seed 0
--name twostage_simclr100_supcon25_s0 --init_run simclr_all_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.02 --seed 0
--name twostage_simclr100_supcon100_s0 --init_run simclr_all_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --lr 0.02 --seed 0
--name twostage_simclr200_supcon5_s0 --init_run simclr_all_e200_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --seed 0
--name twostage_simclr200_supcon25_s0 --init_run simclr_all_e200_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.02 --seed 0
--name twostage_simclr100_supcon5_s1 --init_run simclr_all_e100_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --seed 1
--name twostage_simclr100_supcon25_s1 --init_run simclr_all_e100_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.02 --seed 1
--name twostage_simclr100_supcon100_s1 --init_run simclr_all_e100_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --lr 0.02 --seed 1
--name twostage_simclr200_supcon5_s1 --init_run simclr_all_e200_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --seed 1
--name twostage_simclr200_supcon25_s1 --init_run simclr_all_e200_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.02 --seed 1
--name twostage_simclr100_supcon5_s2 --init_run simclr_all_e100_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --seed 2
--name twostage_simclr100_supcon25_s2 --init_run simclr_all_e100_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.02 --seed 2
--name twostage_simclr100_supcon100_s2 --init_run simclr_all_e100_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --lr 0.02 --seed 2
--name twostage_simclr200_supcon5_s2 --init_run simclr_all_e200_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --seed 2
--name twostage_simclr200_supcon25_s2 --init_run simclr_all_e200_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.02 --seed 2
--name twostage_simclr100_supcon25_lr0.005_s0 --init_run simclr_all_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.005 --seed 0
--name twostage_simclr100_supcon25_lr0.25_s0 --init_run simclr_all_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --lr 0.25 --seed 0
# five seeds for the main arms at 9,700 and 19,400 steps
--name simclr_all_e100_s3 --method simclr --pool all --epochs 100 --seed 3
--name supcon_lab_f0.1_e100_s3 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 3
--name union_f0.1_e100_s3 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 3
--name simclr_sub_f0.1_e100_s3 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 3
--name simclr_all_e100_s4 --method simclr --pool all --epochs 100 --seed 4
--name supcon_lab_f0.1_e100_s4 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 4
--name union_f0.1_e100_s4 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 4
--name simclr_sub_f0.1_e100_s4 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 4
--name simclr_all_e200_s0 --method simclr --pool all --epochs 200 --seed 0
--name supcon_lab_f0.1_e200_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
--name union_f0.1_e200_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 0
--name simclr_sub_f0.1_e200_s0 --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
--name simclr_all_e200_s1 --method simclr --pool all --epochs 200 --seed 1
--name supcon_lab_f0.1_e200_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name union_f0.1_e200_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 1
--name simclr_sub_f0.1_e200_s1 --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name simclr_all_e200_s2 --method simclr --pool all --epochs 200 --seed 2
--name supcon_lab_f0.1_e200_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name union_f0.1_e200_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 2
--name simclr_sub_f0.1_e200_s2 --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name simclr_all_e200_s3 --method simclr --pool all --epochs 200 --seed 3
--name supcon_lab_f0.1_e200_s3 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 3
--name union_f0.1_e200_s3 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 3
--name simclr_sub_f0.1_e200_s3 --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 3
--name simclr_all_e200_s4 --method simclr --pool all --epochs 200 --seed 4
--name supcon_lab_f0.1_e200_s4 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 4
--name union_f0.1_e200_s4 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 4
--name simclr_sub_f0.1_e200_s4 --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 4
# three seeds at every point of the step curve; 900 steps = the report's exact SupCon budget
--name simclr_all_e10_s0 --method simclr --pool all --epochs 10 --seed 0
--name supcon_lab_f0.1_e10_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 0
--name union_f0.1_e10_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 10 --seed 0
--name simclr_all_e25_s0 --method simclr --pool all --epochs 25 --seed 0
--name supcon_lab_f0.1_e25_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --seed 0
--name union_f0.1_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 25 --seed 0
--name simclr_all_e50_s0 --method simclr --pool all --epochs 50 --seed 0
--name supcon_lab_f0.1_e50_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 50 --seed 0
--name union_f0.1_e50_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 50 --seed 0
--name supcon_lab_f0.1_steps900_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --seed 0
--name simclr_all_e10_s1 --method simclr --pool all --epochs 10 --seed 1
--name supcon_lab_f0.1_e10_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 1
--name union_f0.1_e10_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 10 --seed 1
--name simclr_all_e25_s1 --method simclr --pool all --epochs 25 --seed 1
--name supcon_lab_f0.1_e25_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --seed 1
--name union_f0.1_e25_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 25 --seed 1
--name simclr_all_e50_s1 --method simclr --pool all --epochs 50 --seed 1
--name supcon_lab_f0.1_e50_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 50 --seed 1
--name union_f0.1_e50_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 50 --seed 1
--name supcon_lab_f0.1_steps900_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --seed 1
--name simclr_all_e10_s2 --method simclr --pool all --epochs 10 --seed 2
--name supcon_lab_f0.1_e10_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 10 --seed 2
--name union_f0.1_e10_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 10 --seed 2
--name simclr_all_e25_s2 --method simclr --pool all --epochs 25 --seed 2
--name supcon_lab_f0.1_e25_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 25 --seed 2
--name union_f0.1_e25_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 25 --seed 2
--name simclr_all_e50_s2 --method simclr --pool all --epochs 50 --seed 2
--name supcon_lab_f0.1_e50_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 50 --seed 2
--name union_f0.1_e50_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 50 --seed 2
--name supcon_lab_f0.1_steps900_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --seed 2
# supervised CE and BYOL at matched budgets; temperature swap; missing pool cell
--name ce_f0.1_e100_s0 --method ce --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name byol_all_e100_s0 --method byol --pool all --epochs 100 --seed 0
--name ce_f0.1_e200_s0 --method ce --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
--name byol_all_e200_s0 --method byol --pool all --epochs 200 --seed 0
--name ce_f0.1_e100_s1 --method ce --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name byol_all_e100_s1 --method byol --pool all --epochs 100 --seed 1
--name ce_f0.1_e200_s1 --method ce --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name byol_all_e200_s1 --method byol --pool all --epochs 200 --seed 1
--name ce_f0.1_e100_s2 --method ce --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name byol_all_e100_s2 --method byol --pool all --epochs 100 --seed 2
--name ce_f0.1_e200_s2 --method ce --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name byol_all_e200_s2 --method byol --pool all --epochs 200 --seed 2
--name simclr_all_t0.1_e100_s0 --method simclr --pool all --temperature 0.1 --epochs 100 --seed 0
--name supcon_lab_f0.1_t0.5_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --temperature 0.5 --epochs 100 --seed 0
--name simclr_sub_cluster10_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/cluster10_f0.1_s0.npy --epochs 100 --seed 0
# ResNet-50: the report's budgets (900 vs 19,400 steps) and the crossover
--name r50_supcon_lab_f0.1_steps900_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --seed 0
--name r50_supcon_lab_f0.1_steps900_lr0.05_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --lr 0.05 --seed 0
--name r50_simclr_all_e200_s0 --backbone resnet50 --method simclr --pool all --epochs 200 --seed 0
--name r50_supcon_lab_f0.1_e200_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
# long budgets: 400 and 1,000 epoch-equivalents (38,800 and 97,000 steps), seed by seed
--name simclr_all_e400_s0 --method simclr --pool all --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name union_f0.1_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name simclr_sub_f0.1_e400_s0 --method simclr --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name simclr_all_e1000_s0 --method simclr --pool all --epochs 1000 --curve_points 20 --seed 0
--name supcon_lab_f0.1_e1000_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 0
--name union_f0.1_e1000_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 0
--name simclr_sub_f0.1_e1000_s0 --method simclr --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 0
--name union_unif_f0.1_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name simclr_all_e400_s1 --method simclr --pool all --epochs 400 --curve_points 16 --seed 1
--name supcon_lab_f0.1_e400_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 1
--name union_f0.1_e400_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 1
--name simclr_sub_f0.1_e400_s1 --method simclr --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 1
--name simclr_all_e1000_s1 --method simclr --pool all --epochs 1000 --curve_points 20 --seed 1
--name supcon_lab_f0.1_e1000_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 1
--name union_f0.1_e1000_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 1
--name simclr_sub_f0.1_e1000_s1 --method simclr --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 1
--name simclr_all_e400_s2 --method simclr --pool all --epochs 400 --curve_points 16 --seed 2
--name supcon_lab_f0.1_e400_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 2
--name union_f0.1_e400_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 2
--name simclr_sub_f0.1_e400_s2 --method simclr --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 2
--name simclr_all_e1000_s2 --method simclr --pool all --epochs 1000 --curve_points 20 --seed 2
--name supcon_lab_f0.1_e1000_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 2
--name union_f0.1_e1000_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 2
--name simclr_sub_f0.1_e1000_s2 --method simclr --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 2
# ResNet-50 seeds
--name r50_simclr_all_e200_s1 --backbone resnet50 --method simclr --pool all --epochs 200 --seed 1
--name r50_supcon_lab_f0.1_e200_s1 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name r50_simclr_all_e200_s2 --backbone resnet50 --method simclr --pool all --epochs 200 --seed 2
--name r50_supcon_lab_f0.1_e200_s2 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 2


**`jobs/06_revision_followup.txt`** (232 lines) — 225 runs.

In [ ]:
%%writefile jobs/06_revision_followup.txt
# Revision round 2, part 2. Runs after jobs/05_revision_main.txt (uses its checkpoints).
# fine-tuning hyper-parameter search on the 19,400-step encoders (selection on a held-out 20% of the 5k labels)
-m scripts.ft_search --name ftsearch_simclr_all_e200_s0 --from_run simclr_all_e200_s0 --seed 0
-m scripts.ft_search --name ftsearch_supcon_lab_f0.1_e200_s0 --from_run supcon_lab_f0.1_e200_s0 --seed 0
-m scripts.ft_search --name ftsearch_union_f0.1_e200_s0 --from_run union_f0.1_e200_s0 --seed 0
-m scripts.ft_search --name ftsearch_simclr_all_e200_s1 --from_run simclr_all_e200_s1 --seed 1 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name ftsearch_simclr_all_e200_s2 --from_run simclr_all_e200_s2 --seed 2 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name ftsearch_supcon_lab_f0.1_e200_s1 --from_run supcon_lab_f0.1_e200_s1 --seed 1 --config_from ftsearch_supcon_lab_f0.1_e200_s0
-m scripts.ft_search --name ftsearch_supcon_lab_f0.1_e200_s2 --from_run supcon_lab_f0.1_e200_s2 --seed 2 --config_from ftsearch_supcon_lab_f0.1_e200_s0
-m scripts.ft_search --name ftsearch_union_f0.1_e200_s1 --from_run union_f0.1_e200_s1 --seed 1 --config_from ftsearch_union_f0.1_e200_s0
-m scripts.ft_search --name ftsearch_union_f0.1_e200_s2 --from_run union_f0.1_e200_s2 --seed 2 --config_from ftsearch_union_f0.1_e200_s0
# FixMatch started from each pretrained encoder; FixMatch / MixMatch from scratch with more seeds and 3x budget
-m dlt.semisup --name fixmatch_from_simclr_all_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run simclr_all_e200_s0
-m dlt.semisup --name fixmatch_from_supcon_lab_f0.1_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run supcon_lab_f0.1_e200_s0
-m dlt.semisup --name fixmatch_from_union_f0.1_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run union_f0.1_e200_s0
-m dlt.semisup --name fixmatch_from_byol_all_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run byol_all_e200_s0
-m dlt.semisup --name fixmatch_from_twostage_simclr200_supcon25_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run twostage_simclr200_supcon25_s0
-m dlt.semisup --name fixmatch_from_simclr_all_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run simclr_all_e200_s1
-m dlt.semisup --name fixmatch_from_supcon_lab_f0.1_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run supcon_lab_f0.1_e200_s1
-m dlt.semisup --name fixmatch_from_union_f0.1_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run union_f0.1_e200_s1
-m dlt.semisup --name fixmatch_from_byol_all_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run byol_all_e200_s1
-m dlt.semisup --name fixmatch_from_twostage_simclr200_supcon25_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run twostage_simclr200_supcon25_s1
-m dlt.semisup --name fixmatch_from_simclr_all_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run simclr_all_e200_s2
-m dlt.semisup --name fixmatch_from_supcon_lab_f0.1_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run supcon_lab_f0.1_e200_s2
-m dlt.semisup --name fixmatch_from_union_f0.1_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run union_f0.1_e200_s2
-m dlt.semisup --name fixmatch_from_byol_all_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run byol_all_e200_s2
-m dlt.semisup --name fixmatch_from_twostage_simclr200_supcon25_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run twostage_simclr200_supcon25_s2
-m dlt.semisup --name fixmatch_f0.1_s1 --method fixmatch --label_fraction 0.1 --seed 1
-m dlt.semisup --name mixmatch_f0.1_s1 --method mixmatch --label_fraction 0.1 --seed 1
-m dlt.semisup --name fixmatch_f0.1_s2 --method fixmatch --label_fraction 0.1 --seed 2
-m dlt.semisup --name mixmatch_f0.1_s2 --method mixmatch --label_fraction 0.1 --seed 2
-m dlt.semisup --name fixmatch_b3_f0.1_s0 --method fixmatch --label_fraction 0.1 --budget 3 --seed 0
-m dlt.semisup --name fixmatch_b3_f0.1_s1 --method fixmatch --label_fraction 0.1 --budget 3 --seed 1
-m dlt.semisup --name fixmatch_b3_f0.1_s2 --method fixmatch --label_fraction 0.1 --budget 3 --seed 2
# default-recipe fine-tuning of the new encoders
-m dlt.finetune --name ft_simclr_all_e100_s0 --from_run simclr_all_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e200_s0 --from_run simclr_all_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s0 --from_run supcon_lab_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s0 --from_run supcon_lab_f0.1_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e100_s0 --from_run union_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e200_s0 --from_run union_f0.1_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s0 --from_run simclr_sub_f0.1_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s0 --from_run simclr_sub_f0.1_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_s1 --from_run simclr_all_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e200_s1 --from_run simclr_all_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s1 --from_run supcon_lab_f0.1_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s1 --from_run supcon_lab_f0.1_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e100_s1 --from_run union_f0.1_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e200_s1 --from_run union_f0.1_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s1 --from_run simclr_sub_f0.1_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s1 --from_run simclr_sub_f0.1_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e100_s2 --from_run simclr_all_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_all_e200_s2 --from_run simclr_all_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s2 --from_run supcon_lab_f0.1_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s2 --from_run supcon_lab_f0.1_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e100_s2 --from_run union_f0.1_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e200_s2 --from_run union_f0.1_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s2 --from_run simclr_sub_f0.1_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s2 --from_run simclr_sub_f0.1_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_all_e100_s3 --from_run simclr_all_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_all_e200_s3 --from_run simclr_all_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s3 --from_run supcon_lab_f0.1_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s3 --from_run supcon_lab_f0.1_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_union_f0.1_e100_s3 --from_run union_f0.1_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_union_f0.1_e200_s3 --from_run union_f0.1_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s3 --from_run simclr_sub_f0.1_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s3 --from_run simclr_sub_f0.1_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_all_e100_s4 --from_run simclr_all_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_all_e200_s4 --from_run simclr_all_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s4 --from_run supcon_lab_f0.1_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s4 --from_run supcon_lab_f0.1_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_union_f0.1_e100_s4 --from_run union_f0.1_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_union_f0.1_e200_s4 --from_run union_f0.1_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s4 --from_run simclr_sub_f0.1_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s4 --from_run simclr_sub_f0.1_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_all_e400_s0 --from_run simclr_all_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e1000_s0 --from_run simclr_all_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e400_s0 --from_run supcon_lab_f0.1_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e1000_s0 --from_run supcon_lab_f0.1_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e400_s0 --from_run union_f0.1_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e1000_s0 --from_run union_f0.1_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e400_s0 --from_run simclr_sub_f0.1_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e1000_s0 --from_run simclr_sub_f0.1_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr100_supcon5_s0 --from_run twostage_simclr100_supcon5_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr100_supcon25_s0 --from_run twostage_simclr100_supcon25_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr100_supcon100_s0 --from_run twostage_simclr100_supcon100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr200_supcon5_s0 --from_run twostage_simclr200_supcon5_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr200_supcon25_s0 --from_run twostage_simclr200_supcon25_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_byol_all_e100_s0 --from_run byol_all_e100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_byol_all_e200_s0 --from_run byol_all_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_s1 --from_run simclr_all_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e1000_s1 --from_run simclr_all_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e400_s1 --from_run supcon_lab_f0.1_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e1000_s1 --from_run supcon_lab_f0.1_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e400_s1 --from_run union_f0.1_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e1000_s1 --from_run union_f0.1_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e400_s1 --from_run simclr_sub_f0.1_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e1000_s1 --from_run simclr_sub_f0.1_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr100_supcon5_s1 --from_run twostage_simclr100_supcon5_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr100_supcon25_s1 --from_run twostage_simclr100_supcon25_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr100_supcon100_s1 --from_run twostage_simclr100_supcon100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr200_supcon5_s1 --from_run twostage_simclr200_supcon5_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr200_supcon25_s1 --from_run twostage_simclr200_supcon25_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_byol_all_e100_s1 --from_run byol_all_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_byol_all_e200_s1 --from_run byol_all_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e400_s2 --from_run simclr_all_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_all_e1000_s2 --from_run simclr_all_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e400_s2 --from_run supcon_lab_f0.1_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e1000_s2 --from_run supcon_lab_f0.1_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e400_s2 --from_run union_f0.1_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e1000_s2 --from_run union_f0.1_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e400_s2 --from_run simclr_sub_f0.1_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e1000_s2 --from_run simclr_sub_f0.1_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr100_supcon5_s2 --from_run twostage_simclr100_supcon5_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr100_supcon25_s2 --from_run twostage_simclr100_supcon25_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr100_supcon100_s2 --from_run twostage_simclr100_supcon100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr200_supcon5_s2 --from_run twostage_simclr200_supcon5_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr200_supcon25_s2 --from_run twostage_simclr200_supcon25_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_byol_all_e100_s2 --from_run byol_all_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_byol_all_e200_s2 --from_run byol_all_e200_s2 --label_fraction 0.1 --seed 2
# coverage intervention: pools of exactly 5,000 labeled images with a controlled share of k-center / redundant images
--name supcon_lab_mixkcenter25_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixkcenter25_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixkcenter50_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixkcenter50_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixkcenter75_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixkcenter75_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixcluster5025_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixcluster5025_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixcluster5050_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixcluster5050_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixcluster5075_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixcluster5075_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_kcenter_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/kcenter_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_kcenter_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/kcenter_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_cluster50_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/cluster50_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_cluster50_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/cluster50_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixkcenter25_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixkcenter25_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixkcenter50_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixkcenter50_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixkcenter75_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixkcenter75_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixcluster5025_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixcluster5025_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixcluster5050_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixcluster5050_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixcluster5075_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixcluster5075_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_kcenter_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/kcenter_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_kcenter_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/kcenter_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_cluster50_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/cluster50_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_cluster50_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/cluster50_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixkcenter25_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixkcenter25_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixkcenter50_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixkcenter50_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixkcenter75_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixkcenter75_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixcluster5025_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixcluster5025_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixcluster5050_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixcluster5050_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixcluster5075_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixcluster5075_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_kcenter_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/kcenter_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_kcenter_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/kcenter_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_cluster50_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/cluster50_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_cluster50_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/cluster50_f0.1_s2.npy --epochs 100 --seed 2
# label fraction x budget grid (one seed): where does each method win?
--name supcon_lab_f0.5_e100_s0 --method supcon --pool labeled --label_fraction 0.5 --epochs 100 --seed 0
--name supcon_lab_f1.0_e100_s0 --method supcon --pool labeled --label_fraction 1.0 --epochs 100 --seed 0
--name union_f0.5_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.5 --epochs 100 --seed 0
--name supcon_lab_f0.01_e25_s0 --method supcon --pool labeled --label_fraction 0.01 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.05_e25_s0 --method supcon --pool labeled --label_fraction 0.05 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.2_e25_s0 --method supcon --pool labeled --label_fraction 0.2 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.5_e25_s0 --method supcon --pool labeled --label_fraction 0.5 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f1.0_e25_s0 --method supcon --pool labeled --label_fraction 1.0 --epochs 25 --curve_points 8 --seed 0
--name union_f0.01_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.01 --epochs 25 --curve_points 8 --seed 0
--name union_f0.05_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.05 --epochs 25 --curve_points 8 --seed 0
--name union_f0.2_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.2 --epochs 25 --curve_points 8 --seed 0
--name union_f0.5_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.5 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.01_e400_s0 --method supcon --pool labeled --label_fraction 0.01 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.05_e400_s0 --method supcon --pool labeled --label_fraction 0.05 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.2_e400_s0 --method supcon --pool labeled --label_fraction 0.2 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.5_e400_s0 --method supcon --pool labeled --label_fraction 0.5 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f1.0_e400_s0 --method supcon --pool labeled --label_fraction 1.0 --epochs 400 --curve_points 16 --seed 0
--name union_f0.01_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.01 --epochs 400 --curve_points 16 --seed 0
--name union_f0.05_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.05 --epochs 400 --curve_points 16 --seed 0
--name union_f0.2_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.2 --epochs 400 --curve_points 16 --seed 0
--name union_f0.5_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.5 --epochs 400 --curve_points 16 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.01_e25_s0 --from_run supcon_lab_f0.01_e25_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_union_f0.01_e25_s0 --from_run union_f0.01_e25_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.01_s0 --from_run simclr_all_e25_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.01_e100_s0 --from_run supcon_lab_f0.01_e100_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_union_f0.01_e100_s0 --from_run union_f0.01_e100_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.01_s0 --from_run simclr_all_e100_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.01_e400_s0 --from_run supcon_lab_f0.01_e400_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_union_f0.01_e400_s0 --from_run union_f0.01_e400_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.01_s0 --from_run simclr_all_e400_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.05_e25_s0 --from_run supcon_lab_f0.05_e25_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_union_f0.05_e25_s0 --from_run union_f0.05_e25_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.05_s0 --from_run simclr_all_e25_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.05_e100_s0 --from_run supcon_lab_f0.05_e100_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_union_f0.05_e100_s0 --from_run union_f0.05_e100_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.05_s0 --from_run simclr_all_e100_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.05_e400_s0 --from_run supcon_lab_f0.05_e400_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_union_f0.05_e400_s0 --from_run union_f0.05_e400_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.05_s0 --from_run simclr_all_e400_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.2_e25_s0 --from_run supcon_lab_f0.2_e25_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_union_f0.2_e25_s0 --from_run union_f0.2_e25_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.2_s0 --from_run simclr_all_e25_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.2_e100_s0 --from_run supcon_lab_f0.2_e100_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_union_f0.2_e100_s0 --from_run union_f0.2_e100_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.2_s0 --from_run simclr_all_e100_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.2_e400_s0 --from_run supcon_lab_f0.2_e400_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_union_f0.2_e400_s0 --from_run union_f0.2_e400_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.2_s0 --from_run simclr_all_e400_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.5_e25_s0 --from_run supcon_lab_f0.5_e25_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_union_f0.5_e25_s0 --from_run union_f0.5_e25_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.5_s0 --from_run simclr_all_e25_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.5_e100_s0 --from_run supcon_lab_f0.5_e100_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_union_f0.5_e100_s0 --from_run union_f0.5_e100_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.5_s0 --from_run simclr_all_e100_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.5_e400_s0 --from_run supcon_lab_f0.5_e400_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_union_f0.5_e400_s0 --from_run union_f0.5_e400_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.5_s0 --from_run simclr_all_e400_s0 --label_fraction 0.5 --seed 0
# is the SupCon plateau a regularisation failure? weight decay and stronger augmentation at 38,800 steps
--name supcon_lab_f0.1_wd0.0005_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --wd 0.0005 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_wd0.001_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --wd 0.001 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_strongaug_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --aug simclr_strong --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_wd0.0005_strongaug_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --wd 0.0005 --aug simclr_strong --epochs 400 --curve_points 16 --seed 0


**`jobs/phases.txt`** (30 lines) — order of the second-revision phases.

In [ ]:
%%writefile jobs/phases.txt
jobs/rev_p01_short.txt
jobs/rev_p02_ftsearch_s0.txt
jobs/rev_p03_ftsearch_s12.txt
jobs/rev_p04_fixmatch_from.txt
jobs/rev_p04b_ftoracle_s0.txt
jobs/rev_p04c_ftoracle_s12.txt
jobs/rev_p05_ft_ready.txt
jobs/rev_p05b_extras.txt
jobs/rev_p05c_extras_post.txt
jobs/rev_p06_e400.txt
jobs/rev_p07_ft_e400.txt
jobs/rev_p08a_stl_e100.txt
jobs/rev_p08b_stl_e100_post.txt
jobs/rev_p08r_stl_retry.txt
jobs/rev_p10_e1000.txt
jobs/rev_p11_ft_e1000.txt
jobs/rev_p08c_stl_e200.txt
jobs/rev_p08d_stl_e200_post.txt
jobs/rev_p08e_stl_ftsearch_s12.txt
jobs/rev_p09_coverage.txt
jobs/rev_p12_grid.txt
jobs/rev_p13_ft_grid.txt
jobs/rev_p14_reg_r50.txt
jobs/rev_p14b_ft_grid_fill.txt
jobs/rev_p15_c100_e200.txt
jobs/rev_p16_c100_r50_post.txt
jobs/rev_p17_tin_s0.txt
jobs/rev_p18_tin_s0_post.txt
jobs/rev_p19_tin_s12.txt
jobs/rev_p20_tin_s12_post.txt


**`jobs/rev_p01_short.txt`** (16 lines) — 16 runs.

In [ ]:
%%writefile jobs/rev_p01_short.txt
--name byol_all_e200_s0 --method byol --pool all --epochs 200 --seed 0
--name ce_f0.1_e100_s1 --method ce --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name byol_all_e100_s1 --method byol --pool all --epochs 100 --seed 1
--name ce_f0.1_e200_s1 --method ce --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name byol_all_e200_s1 --method byol --pool all --epochs 200 --seed 1
--name ce_f0.1_e100_s2 --method ce --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name byol_all_e100_s2 --method byol --pool all --epochs 100 --seed 2
--name ce_f0.1_e200_s2 --method ce --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name byol_all_e200_s2 --method byol --pool all --epochs 200 --seed 2
--name simclr_all_t0.1_e100_s0 --method simclr --pool all --temperature 0.1 --epochs 100 --seed 0
--name supcon_lab_f0.1_t0.5_e100_s0 --method supcon --pool labeled --label_fraction 0.1 --temperature 0.5 --epochs 100 --seed 0
--name simclr_sub_cluster10_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/cluster10_f0.1_s0.npy --epochs 100 --seed 0
--name r50_supcon_lab_f0.1_steps900_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --seed 0
--name r50_supcon_lab_f0.1_steps900_lr0.05_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 9.28 --lr 0.05 --seed 0
--name r50_simclr_all_e200_s0 --backbone resnet50 --method simclr --pool all --epochs 200 --seed 0
--name r50_supcon_lab_f0.1_e200_s0 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 0


**`jobs/rev_p02_ftsearch_s0.txt`** (3 lines) — 3 runs.

In [ ]:
%%writefile jobs/rev_p02_ftsearch_s0.txt
-m scripts.ft_search --name ftsearch_simclr_all_e200_s0 --from_run simclr_all_e200_s0 --seed 0
-m scripts.ft_search --name ftsearch_supcon_lab_f0.1_e200_s0 --from_run supcon_lab_f0.1_e200_s0 --seed 0
-m scripts.ft_search --name ftsearch_union_f0.1_e200_s0 --from_run union_f0.1_e200_s0 --seed 0


**`jobs/rev_p03_ftsearch_s12.txt`** (6 lines) — 6 runs.

In [ ]:
%%writefile jobs/rev_p03_ftsearch_s12.txt
-m scripts.ft_search --name ftsearch_simclr_all_e200_s1 --from_run simclr_all_e200_s1 --seed 1 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name ftsearch_simclr_all_e200_s2 --from_run simclr_all_e200_s2 --seed 2 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name ftsearch_supcon_lab_f0.1_e200_s1 --from_run supcon_lab_f0.1_e200_s1 --seed 1 --config_from ftsearch_supcon_lab_f0.1_e200_s0
-m scripts.ft_search --name ftsearch_supcon_lab_f0.1_e200_s2 --from_run supcon_lab_f0.1_e200_s2 --seed 2 --config_from ftsearch_supcon_lab_f0.1_e200_s0
-m scripts.ft_search --name ftsearch_union_f0.1_e200_s1 --from_run union_f0.1_e200_s1 --seed 1 --config_from ftsearch_union_f0.1_e200_s0
-m scripts.ft_search --name ftsearch_union_f0.1_e200_s2 --from_run union_f0.1_e200_s2 --seed 2 --config_from ftsearch_union_f0.1_e200_s0


**`jobs/rev_p04_fixmatch_from.txt`** (22 lines) — 22 runs.

In [ ]:
%%writefile jobs/rev_p04_fixmatch_from.txt
-m dlt.semisup --name fixmatch_from_simclr_all_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run simclr_all_e200_s0
-m dlt.semisup --name fixmatch_from_supcon_lab_f0.1_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run supcon_lab_f0.1_e200_s0
-m dlt.semisup --name fixmatch_from_union_f0.1_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run union_f0.1_e200_s0
-m dlt.semisup --name fixmatch_from_byol_all_e200_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run byol_all_e200_s0
-m dlt.semisup --name fixmatch_from_twostage_simclr200_supcon25_s0 --method fixmatch --label_fraction 0.1 --seed 0 --init_run twostage_simclr200_supcon25_s0
-m dlt.semisup --name fixmatch_from_simclr_all_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run simclr_all_e200_s1
-m dlt.semisup --name fixmatch_from_supcon_lab_f0.1_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run supcon_lab_f0.1_e200_s1
-m dlt.semisup --name fixmatch_from_union_f0.1_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run union_f0.1_e200_s1
-m dlt.semisup --name fixmatch_from_byol_all_e200_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run byol_all_e200_s1
-m dlt.semisup --name fixmatch_from_twostage_simclr200_supcon25_s1 --method fixmatch --label_fraction 0.1 --seed 1 --init_run twostage_simclr200_supcon25_s1
-m dlt.semisup --name fixmatch_from_simclr_all_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run simclr_all_e200_s2
-m dlt.semisup --name fixmatch_from_supcon_lab_f0.1_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run supcon_lab_f0.1_e200_s2
-m dlt.semisup --name fixmatch_from_union_f0.1_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run union_f0.1_e200_s2
-m dlt.semisup --name fixmatch_from_byol_all_e200_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run byol_all_e200_s2
-m dlt.semisup --name fixmatch_from_twostage_simclr200_supcon25_s2 --method fixmatch --label_fraction 0.1 --seed 2 --init_run twostage_simclr200_supcon25_s2
-m dlt.semisup --name fixmatch_f0.1_s1 --method fixmatch --label_fraction 0.1 --seed 1
-m dlt.semisup --name mixmatch_f0.1_s1 --method mixmatch --label_fraction 0.1 --seed 1
-m dlt.semisup --name fixmatch_f0.1_s2 --method fixmatch --label_fraction 0.1 --seed 2
-m dlt.semisup --name mixmatch_f0.1_s2 --method mixmatch --label_fraction 0.1 --seed 2
-m dlt.semisup --name fixmatch_b3_f0.1_s0 --method fixmatch --label_fraction 0.1 --budget 3 --seed 0
-m dlt.semisup --name fixmatch_b3_f0.1_s1 --method fixmatch --label_fraction 0.1 --budget 3 --seed 1
-m dlt.semisup --name fixmatch_b3_f0.1_s2 --method fixmatch --label_fraction 0.1 --budget 3 --seed 2


**`jobs/rev_p04b_ftoracle_s0.txt`** (6 lines) — 6 runs.

In [ ]:
%%writefile jobs/rev_p04b_ftoracle_s0.txt
-m scripts.ft_search --name ftoracle_simclr_all_e200_s0 --from_run simclr_all_e200_s0 --label_fraction 0.1 --seed 0 --oracle
-m scripts.ft_search --name ftoracle_supcon_lab_f0.1_e200_s0 --from_run supcon_lab_f0.1_e200_s0 --label_fraction 0.1 --seed 0 --oracle
-m scripts.ft_search --name ftoracle_union_f0.1_e200_s0 --from_run union_f0.1_e200_s0 --label_fraction 0.1 --seed 0 --oracle
-m scripts.ft_search --name ftoracle_twostage_simclr200_supcon5_s0 --from_run twostage_simclr200_supcon5_s0 --label_fraction 0.1 --seed 0 --oracle
-m scripts.ft_search --name ftoracle_byol_all_e200_s0 --from_run byol_all_e200_s0 --label_fraction 0.1 --seed 0 --oracle
-m scripts.ft_search --name ftsearch_byol_all_e200_s0 --from_run byol_all_e200_s0 --label_fraction 0.1 --seed 0


**`jobs/rev_p04c_ftoracle_s12.txt`** (21 lines) — 21 runs.

In [ ]:
%%writefile jobs/rev_p04c_ftoracle_s12.txt
-m scripts.ft_search --name ftoracle_simclr_all_e200_s1 --from_run simclr_all_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftoracle_simclr_all_e200_s0
-m scripts.ft_search --name ftoracle_simclr_all_e200_s2 --from_run simclr_all_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftoracle_simclr_all_e200_s0
-m scripts.ft_search --name ftoracle_supcon_lab_f0.1_e200_s1 --from_run supcon_lab_f0.1_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftoracle_supcon_lab_f0.1_e200_s0
-m scripts.ft_search --name ftoracle_supcon_lab_f0.1_e200_s2 --from_run supcon_lab_f0.1_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftoracle_supcon_lab_f0.1_e200_s0
-m scripts.ft_search --name ftoracle_union_f0.1_e200_s1 --from_run union_f0.1_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftoracle_union_f0.1_e200_s0
-m scripts.ft_search --name ftoracle_union_f0.1_e200_s2 --from_run union_f0.1_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftoracle_union_f0.1_e200_s0
-m scripts.ft_search --name ftoracle_twostage_simclr200_supcon5_s1 --from_run twostage_simclr200_supcon5_s1 --label_fraction 0.1 --seed 1 --config_from ftoracle_twostage_simclr200_supcon5_s0
-m scripts.ft_search --name ftoracle_twostage_simclr200_supcon5_s2 --from_run twostage_simclr200_supcon5_s2 --label_fraction 0.1 --seed 2 --config_from ftoracle_twostage_simclr200_supcon5_s0
-m scripts.ft_search --name ftoracle_byol_all_e200_s1 --from_run byol_all_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftoracle_byol_all_e200_s0
-m scripts.ft_search --name ftoracle_byol_all_e200_s2 --from_run byol_all_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftoracle_byol_all_e200_s0
-m scripts.ft_search --name ftsearch_byol_all_e200_s1 --from_run byol_all_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftsearch_byol_all_e200_s0
-m scripts.ft_search --name ftsearch_byol_all_e200_s2 --from_run byol_all_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftsearch_byol_all_e200_s0
-m scripts.ft_search --name fttransfer_supcon_lab_f0.1_e200_s0 --from_run supcon_lab_f0.1_e200_s0 --label_fraction 0.1 --seed 0 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_supcon_lab_f0.1_e200_s1 --from_run supcon_lab_f0.1_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_supcon_lab_f0.1_e200_s2 --from_run supcon_lab_f0.1_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_union_f0.1_e200_s0 --from_run union_f0.1_e200_s0 --label_fraction 0.1 --seed 0 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_union_f0.1_e200_s1 --from_run union_f0.1_e200_s1 --label_fraction 0.1 --seed 1 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_union_f0.1_e200_s2 --from_run union_f0.1_e200_s2 --label_fraction 0.1 --seed 2 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_twostage_simclr200_supcon5_s0 --from_run twostage_simclr200_supcon5_s0 --label_fraction 0.1 --seed 0 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_twostage_simclr200_supcon5_s1 --from_run twostage_simclr200_supcon5_s1 --label_fraction 0.1 --seed 1 --config_from ftsearch_simclr_all_e200_s0
-m scripts.ft_search --name fttransfer_twostage_simclr200_supcon5_s2 --from_run twostage_simclr200_supcon5_s2 --label_fraction 0.1 --seed 2 --config_from ftsearch_simclr_all_e200_s0


**`jobs/rev_p05_ft_ready.txt`** (39 lines) — 39 runs.

In [ ]:
%%writefile jobs/rev_p05_ft_ready.txt
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s0 --from_run simclr_sub_f0.1_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s1 --from_run simclr_sub_f0.1_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s2 --from_run simclr_sub_f0.1_e200_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_all_e100_s3 --from_run simclr_all_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_all_e200_s3 --from_run simclr_all_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s3 --from_run supcon_lab_f0.1_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s3 --from_run supcon_lab_f0.1_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_union_f0.1_e100_s3 --from_run union_f0.1_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_union_f0.1_e200_s3 --from_run union_f0.1_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s3 --from_run simclr_sub_f0.1_e100_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s3 --from_run simclr_sub_f0.1_e200_s3 --label_fraction 0.1 --seed 3
-m dlt.finetune --name ft_simclr_all_e100_s4 --from_run simclr_all_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_all_e200_s4 --from_run simclr_all_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_supcon_lab_f0.1_e100_s4 --from_run supcon_lab_f0.1_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_supcon_lab_f0.1_e200_s4 --from_run supcon_lab_f0.1_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_union_f0.1_e100_s4 --from_run union_f0.1_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_union_f0.1_e200_s4 --from_run union_f0.1_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_sub_f0.1_e100_s4 --from_run simclr_sub_f0.1_e100_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_simclr_sub_f0.1_e200_s4 --from_run simclr_sub_f0.1_e200_s4 --label_fraction 0.1 --seed 4
-m dlt.finetune --name ft_twostage_simclr100_supcon5_s0 --from_run twostage_simclr100_supcon5_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr100_supcon25_s0 --from_run twostage_simclr100_supcon25_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr100_supcon100_s0 --from_run twostage_simclr100_supcon100_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr200_supcon5_s0 --from_run twostage_simclr200_supcon5_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr200_supcon25_s0 --from_run twostage_simclr200_supcon25_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_byol_all_e200_s0 --from_run byol_all_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_twostage_simclr100_supcon5_s1 --from_run twostage_simclr100_supcon5_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr100_supcon25_s1 --from_run twostage_simclr100_supcon25_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr100_supcon100_s1 --from_run twostage_simclr100_supcon100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr200_supcon5_s1 --from_run twostage_simclr200_supcon5_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr200_supcon25_s1 --from_run twostage_simclr200_supcon25_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_byol_all_e100_s1 --from_run byol_all_e100_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_byol_all_e200_s1 --from_run byol_all_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_twostage_simclr100_supcon5_s2 --from_run twostage_simclr100_supcon5_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr100_supcon25_s2 --from_run twostage_simclr100_supcon25_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr100_supcon100_s2 --from_run twostage_simclr100_supcon100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr200_supcon5_s2 --from_run twostage_simclr200_supcon5_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_twostage_simclr200_supcon25_s2 --from_run twostage_simclr200_supcon25_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_byol_all_e100_s2 --from_run byol_all_e100_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_byol_all_e200_s2 --from_run byol_all_e200_s2 --label_fraction 0.1 --seed 2


**`jobs/rev_p05b_extras.txt`** (18 lines) — 18 runs.

In [ ]:
%%writefile jobs/rev_p05b_extras.txt
--name simclr_exp6_e100_s1 --method simclr --pool all --aug exp6 --epochs 100 --seed 1
--name simclr_exp6_e100_s2 --method simclr --pool all --aug exp6 --epochs 100 --seed 2
--name supcon_lab_f0.1_exp6_e100_s1 --method supcon --pool labeled --label_fraction 0.1 --aug exp6 --epochs 100 --seed 1
--name supcon_lab_f0.1_exp6_e100_s2 --method supcon --pool labeled --label_fraction 0.1 --aug exp6 --epochs 100 --seed 2
--name union_f0.1_exp6_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --aug exp6 --epochs 100 --seed 0
--name union_f0.1_exp6_e100_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --aug exp6 --epochs 100 --seed 1
--name union_f0.1_exp6_e100_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --aug exp6 --epochs 100 --seed 2
-m dlt.distill --name distill_simclr_all_e200_s0 --from_run simclr_all_e200_s0 --label_fraction 0.1 --seed 0
-m dlt.distill --name distill_simclr_all_e200_s1 --from_run simclr_all_e200_s1 --label_fraction 0.1 --seed 1
-m dlt.distill --name distill_simclr_all_e200_s2 --from_run simclr_all_e200_s2 --label_fraction 0.1 --seed 2
--name c100_simclr_all_e100_s1 --dataset cifar100 --method simclr --pool all --epochs 100 --seed 1
--name c100_supcon_lab_f0.1_e100_s1 --dataset cifar100 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name c100_union_f0.1_e100_s1 --dataset cifar100 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 1
--name c100_simclr_sub_f0.1_e100_s1 --dataset cifar100 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name c100_simclr_all_e100_s2 --dataset cifar100 --method simclr --pool all --epochs 100 --seed 2
--name c100_supcon_lab_f0.1_e100_s2 --dataset cifar100 --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name c100_union_f0.1_e100_s2 --dataset cifar100 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 100 --seed 2
--name c100_simclr_sub_f0.1_e100_s2 --dataset cifar100 --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 2


**`jobs/rev_p05c_extras_post.txt`** (9 lines) — 9 runs.

In [ ]:
%%writefile jobs/rev_p05c_extras_post.txt
-m dlt.finetune --name ft_c100_simclr_sub_f0.1_e100_s0 --from_run c100_simclr_sub_f0.1_e100_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_c100_simclr_all_e100_s1 --from_run c100_simclr_all_e100_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_supcon_lab_f0.1_e100_s1 --from_run c100_supcon_lab_f0.1_e100_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_union_f0.1_e100_s1 --from_run c100_union_f0.1_e100_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_simclr_sub_f0.1_e100_s1 --from_run c100_simclr_sub_f0.1_e100_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_simclr_all_e100_s2 --from_run c100_simclr_all_e100_s2 --dataset cifar100 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_c100_supcon_lab_f0.1_e100_s2 --from_run c100_supcon_lab_f0.1_e100_s2 --dataset cifar100 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_c100_union_f0.1_e100_s2 --from_run c100_union_f0.1_e100_s2 --dataset cifar100 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_c100_simclr_sub_f0.1_e100_s2 --from_run c100_simclr_sub_f0.1_e100_s2 --dataset cifar100 --label_fraction 0.1 --seed 2


**`jobs/rev_p06_e400.txt`** (13 lines) — 13 runs.

In [ ]:
%%writefile jobs/rev_p06_e400.txt
--name simclr_all_e400_s0 --method simclr --pool all --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name union_f0.1_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name simclr_sub_f0.1_e400_s0 --method simclr --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name union_unif_f0.1_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 0
--name simclr_all_e400_s1 --method simclr --pool all --epochs 400 --curve_points 16 --seed 1
--name supcon_lab_f0.1_e400_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 1
--name union_f0.1_e400_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 1
--name simclr_sub_f0.1_e400_s1 --method simclr --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 1
--name simclr_all_e400_s2 --method simclr --pool all --epochs 400 --curve_points 16 --seed 2
--name supcon_lab_f0.1_e400_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 2
--name union_f0.1_e400_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 2
--name simclr_sub_f0.1_e400_s2 --method simclr --pool labeled --label_fraction 0.1 --epochs 400 --curve_points 16 --seed 2


**`jobs/rev_p07_ft_e400.txt`** (12 lines) — 12 runs.

In [ ]:
%%writefile jobs/rev_p07_ft_e400.txt
-m dlt.finetune --name ft_simclr_all_e400_s0 --from_run simclr_all_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e400_s0 --from_run supcon_lab_f0.1_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e400_s0 --from_run union_f0.1_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e400_s0 --from_run simclr_sub_f0.1_e400_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_s1 --from_run simclr_all_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e400_s1 --from_run supcon_lab_f0.1_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e400_s1 --from_run union_f0.1_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e400_s1 --from_run simclr_sub_f0.1_e400_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e400_s2 --from_run simclr_all_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e400_s2 --from_run supcon_lab_f0.1_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e400_s2 --from_run union_f0.1_e400_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e400_s2 --from_run simclr_sub_f0.1_e400_s2 --label_fraction 0.1 --seed 2


**`jobs/rev_p08a_stl_e100.txt`** (15 lines) — 15 runs.

In [ ]:
%%writefile jobs/rev_p08a_stl_e100.txt
--name stl_simclr_all_e100_s0 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 0 --curve_points 8
--name stl_supcon_lab_e100_s0 --dataset stl10 --backbone resnet18_mp --method supcon --pool labeled --label_fraction 1 --epochs 100 --seed 0 --curve_points 8
--name stl_union_e100_s0 --dataset stl10 --backbone resnet18_mp --method supcon --pool all --label_fraction 1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 100 --seed 0 --curve_points 8
--name stl_simclr_sub_e100_s0 --dataset stl10 --backbone resnet18_mp --method simclr --pool labeled --label_fraction 1 --epochs 100 --seed 0 --curve_points 8
--name stl_simclr_all_e100_s1 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 1 --curve_points 8
--name stl_supcon_lab_e100_s1 --dataset stl10 --backbone resnet18_mp --method supcon --pool labeled --label_fraction 1 --epochs 100 --seed 1 --curve_points 8
--name stl_union_e100_s1 --dataset stl10 --backbone resnet18_mp --method supcon --pool all --label_fraction 1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 100 --seed 1 --curve_points 8
--name stl_simclr_sub_e100_s1 --dataset stl10 --backbone resnet18_mp --method simclr --pool labeled --label_fraction 1 --epochs 100 --seed 1 --curve_points 8
--name stl_simclr_all_e100_s2 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 2 --curve_points 8
--name stl_supcon_lab_e100_s2 --dataset stl10 --backbone resnet18_mp --method supcon --pool labeled --label_fraction 1 --epochs 100 --seed 2 --curve_points 8
--name stl_union_e100_s2 --dataset stl10 --backbone resnet18_mp --method supcon --pool all --label_fraction 1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 100 --seed 2 --curve_points 8
--name stl_simclr_sub_e100_s2 --dataset stl10 --backbone resnet18_mp --method simclr --pool labeled --label_fraction 1 --epochs 100 --seed 2 --curve_points 8
--name stl_ce_e100_s0 --dataset stl10 --backbone resnet18_mp --method ce --pool labeled --label_fraction 1 --epochs 100 --seed 0
--name stl_ce_e100_s1 --dataset stl10 --backbone resnet18_mp --method ce --pool labeled --label_fraction 1 --epochs 100 --seed 1
--name stl_ce_e100_s2 --dataset stl10 --backbone resnet18_mp --method ce --pool labeled --label_fraction 1 --epochs 100 --seed 2


**`jobs/rev_p08b_stl_e100_post.txt`** (17 lines) — 15 runs.

In [ ]:
%%writefile jobs/rev_p08b_stl_e100_post.txt
# parallel: 1
# (96-px runs: two at once can exceed the memory of the GPU)
-m dlt.finetune --name ft_stl_simclr_all_e100_s0 --from_run stl_simclr_all_e100_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_supcon_lab_e100_s0 --from_run stl_supcon_lab_e100_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_union_e100_s0 --from_run stl_union_e100_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_simclr_sub_e100_s0 --from_run stl_simclr_sub_e100_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_simclr_all_e100_s1 --from_run stl_simclr_all_e100_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_supcon_lab_e100_s1 --from_run stl_supcon_lab_e100_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_union_e100_s1 --from_run stl_union_e100_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_simclr_sub_e100_s1 --from_run stl_simclr_sub_e100_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_simclr_all_e100_s2 --from_run stl_simclr_all_e100_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.finetune --name ft_stl_supcon_lab_e100_s2 --from_run stl_supcon_lab_e100_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.finetune --name ft_stl_union_e100_s2 --from_run stl_union_e100_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.finetune --name ft_stl_simclr_sub_e100_s2 --from_run stl_simclr_sub_e100_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.semisup --name stl_fixmatch_s0 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.semisup --name stl_fixmatch_s1 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.semisup --name stl_fixmatch_s2 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2


**`jobs/rev_p08r_stl_retry.txt`** (4 lines) — 2 runs.

In [ ]:
%%writefile jobs/rev_p08r_stl_retry.txt
# parallel: 1
# retry of the STL-10 SimCLR seed-0 run, which ran out of GPU memory in its final evaluation twice
--name stl_simclr_all_e100_s0 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 0 --curve_points 8
-m dlt.finetune --name ft_stl_simclr_all_e100_s0 --from_run stl_simclr_all_e100_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0


**`jobs/rev_p10_e1000.txt`** (12 lines) — 12 runs.

In [ ]:
%%writefile jobs/rev_p10_e1000.txt
--name simclr_all_e1000_s0 --method simclr --pool all --epochs 1000 --curve_points 20 --seed 0
--name supcon_lab_f0.1_e1000_s0 --method supcon --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 0
--name union_f0.1_e1000_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 0
--name simclr_sub_f0.1_e1000_s0 --method simclr --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 0
--name simclr_all_e1000_s1 --method simclr --pool all --epochs 1000 --curve_points 20 --seed 1
--name supcon_lab_f0.1_e1000_s1 --method supcon --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 1
--name union_f0.1_e1000_s1 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 1
--name simclr_sub_f0.1_e1000_s1 --method simclr --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 1
--name simclr_all_e1000_s2 --method simclr --pool all --epochs 1000 --curve_points 20 --seed 2
--name supcon_lab_f0.1_e1000_s2 --method supcon --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 2
--name union_f0.1_e1000_s2 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 2
--name simclr_sub_f0.1_e1000_s2 --method simclr --pool labeled --label_fraction 0.1 --epochs 1000 --curve_points 20 --seed 2


**`jobs/rev_p11_ft_e1000.txt`** (12 lines) — 12 runs.

In [ ]:
%%writefile jobs/rev_p11_ft_e1000.txt
-m dlt.finetune --name ft_simclr_all_e1000_s0 --from_run simclr_all_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.1_e1000_s0 --from_run supcon_lab_f0.1_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e1000_s0 --from_run union_f0.1_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e1000_s1 --from_run simclr_all_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_supcon_lab_f0.1_e1000_s1 --from_run supcon_lab_f0.1_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_union_f0.1_e1000_s1 --from_run union_f0.1_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_all_e1000_s2 --from_run simclr_all_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_supcon_lab_f0.1_e1000_s2 --from_run supcon_lab_f0.1_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_union_f0.1_e1000_s2 --from_run union_f0.1_e1000_s2 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_simclr_sub_f0.1_e1000_s0 --from_run simclr_sub_f0.1_e1000_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_sub_f0.1_e1000_s1 --from_run simclr_sub_f0.1_e1000_s1 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_simclr_sub_f0.1_e1000_s2 --from_run simclr_sub_f0.1_e1000_s2 --label_fraction 0.1 --seed 2


**`jobs/rev_p08c_stl_e200.txt`** (14 lines) — 12 runs.

In [ ]:
%%writefile jobs/rev_p08c_stl_e200.txt
# parallel: 1
# (96-px runs: two at once can exceed the memory of the GPU)
--name stl_simclr_all_e200_s0 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 200 --seed 0 --curve_points 8
--name stl_supcon_lab_e200_s0 --dataset stl10 --backbone resnet18_mp --method supcon --pool labeled --label_fraction 1 --epochs 200 --seed 0 --curve_points 8
--name stl_union_e200_s0 --dataset stl10 --backbone resnet18_mp --method supcon --pool all --label_fraction 1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 200 --seed 0 --curve_points 8
--name stl_simclr_sub_e200_s0 --dataset stl10 --backbone resnet18_mp --method simclr --pool labeled --label_fraction 1 --epochs 200 --seed 0 --curve_points 8
--name stl_simclr_all_e200_s1 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 200 --seed 1 --curve_points 8
--name stl_supcon_lab_e200_s1 --dataset stl10 --backbone resnet18_mp --method supcon --pool labeled --label_fraction 1 --epochs 200 --seed 1 --curve_points 8
--name stl_union_e200_s1 --dataset stl10 --backbone resnet18_mp --method supcon --pool all --label_fraction 1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 200 --seed 1 --curve_points 8
--name stl_simclr_sub_e200_s1 --dataset stl10 --backbone resnet18_mp --method simclr --pool labeled --label_fraction 1 --epochs 200 --seed 1 --curve_points 8
--name stl_simclr_all_e200_s2 --dataset stl10 --backbone resnet18_mp --method simclr --pool all --epochs 200 --seed 2 --curve_points 8
--name stl_supcon_lab_e200_s2 --dataset stl10 --backbone resnet18_mp --method supcon --pool labeled --label_fraction 1 --epochs 200 --seed 2 --curve_points 8
--name stl_union_e200_s2 --dataset stl10 --backbone resnet18_mp --method supcon --pool all --label_fraction 1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 200 --seed 2 --curve_points 8
--name stl_simclr_sub_e200_s2 --dataset stl10 --backbone resnet18_mp --method simclr --pool labeled --label_fraction 1 --epochs 200 --seed 2 --curve_points 8


**`jobs/rev_p08d_stl_e200_post.txt`** (26 lines) — 24 runs.

In [ ]:
%%writefile jobs/rev_p08d_stl_e200_post.txt
# parallel: 1
# (96-px runs: two at once can exceed the memory of the GPU)
-m dlt.finetune --name ft_stl_simclr_all_e200_s0 --from_run stl_simclr_all_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_supcon_lab_e200_s0 --from_run stl_supcon_lab_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_union_e200_s0 --from_run stl_union_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_simclr_sub_e200_s0 --from_run stl_simclr_sub_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0
-m dlt.finetune --name ft_stl_simclr_all_e200_s1 --from_run stl_simclr_all_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_supcon_lab_e200_s1 --from_run stl_supcon_lab_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_union_e200_s1 --from_run stl_union_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_simclr_sub_e200_s1 --from_run stl_simclr_sub_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1
-m dlt.finetune --name ft_stl_simclr_all_e200_s2 --from_run stl_simclr_all_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.finetune --name ft_stl_supcon_lab_e200_s2 --from_run stl_supcon_lab_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.finetune --name ft_stl_union_e200_s2 --from_run stl_union_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.finetune --name ft_stl_simclr_sub_e200_s2 --from_run stl_simclr_sub_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2
-m dlt.semisup --name stl_fixmatch_from_simclr_all_e200_s0 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0 --init_run stl_simclr_all_e200_s0
-m dlt.semisup --name stl_fixmatch_from_supcon_lab_e200_s0 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0 --init_run stl_supcon_lab_e200_s0
-m dlt.semisup --name stl_fixmatch_from_union_e200_s0 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0 --init_run stl_union_e200_s0
-m dlt.semisup --name stl_fixmatch_from_simclr_all_e200_s1 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1 --init_run stl_simclr_all_e200_s1
-m dlt.semisup --name stl_fixmatch_from_supcon_lab_e200_s1 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1 --init_run stl_supcon_lab_e200_s1
-m dlt.semisup --name stl_fixmatch_from_union_e200_s1 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1 --init_run stl_union_e200_s1
-m dlt.semisup --name stl_fixmatch_from_simclr_all_e200_s2 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2 --init_run stl_simclr_all_e200_s2
-m dlt.semisup --name stl_fixmatch_from_supcon_lab_e200_s2 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2 --init_run stl_supcon_lab_e200_s2
-m dlt.semisup --name stl_fixmatch_from_union_e200_s2 --method fixmatch --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2 --init_run stl_union_e200_s2
-m scripts.ft_search --name ftoracle_stl_simclr_all_e200_s0 --from_run stl_simclr_all_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0 --oracle --grid small
-m scripts.ft_search --name ftoracle_stl_supcon_lab_e200_s0 --from_run stl_supcon_lab_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0 --oracle --grid small
-m scripts.ft_search --name ftoracle_stl_union_e200_s0 --from_run stl_union_e200_s0 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 0 --oracle --grid small


**`jobs/rev_p08e_stl_ftsearch_s12.txt`** (8 lines) — 6 runs.

In [ ]:
%%writefile jobs/rev_p08e_stl_ftsearch_s12.txt
# parallel: 1
# (96-px runs: two at once can exceed the memory of the GPU)
-m scripts.ft_search --name ftoracle_stl_simclr_all_e200_s1 --from_run stl_simclr_all_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1 --config_from ftoracle_stl_simclr_all_e200_s0
-m scripts.ft_search --name ftoracle_stl_simclr_all_e200_s2 --from_run stl_simclr_all_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2 --config_from ftoracle_stl_simclr_all_e200_s0
-m scripts.ft_search --name ftoracle_stl_supcon_lab_e200_s1 --from_run stl_supcon_lab_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1 --config_from ftoracle_stl_supcon_lab_e200_s0
-m scripts.ft_search --name ftoracle_stl_supcon_lab_e200_s2 --from_run stl_supcon_lab_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2 --config_from ftoracle_stl_supcon_lab_e200_s0
-m scripts.ft_search --name ftoracle_stl_union_e200_s1 --from_run stl_union_e200_s1 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 1 --config_from ftoracle_stl_union_e200_s0
-m scripts.ft_search --name ftoracle_stl_union_e200_s2 --from_run stl_union_e200_s2 --dataset stl10 --backbone resnet18_mp --label_fraction 1 --seed 2 --config_from ftoracle_stl_union_e200_s0


**`jobs/rev_p09_coverage.txt`** (44 lines) — 44 runs.

In [ ]:
%%writefile jobs/rev_p09_coverage.txt
--name supcon_lab_mixkcenter25_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixkcenter25_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixkcenter50_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixkcenter50_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixkcenter75_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixkcenter75_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixcluster5025_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixcluster5025_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixcluster5050_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixcluster5050_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixcluster5075_f0.1_e100_s0 --method supcon --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s0.npy --epochs 100 --seed 0
--name simclr_sub_mixcluster5075_f0.1_e100_s0 --method simclr --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s0.npy --epochs 100 --seed 0
--name supcon_lab_mixkcenter25_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixkcenter25_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixkcenter50_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixkcenter50_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixkcenter75_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixkcenter75_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixcluster5025_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixcluster5025_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixcluster5050_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixcluster5050_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixcluster5075_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_mixcluster5075_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_kcenter_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/kcenter_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_kcenter_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/kcenter_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_cluster50_f0.1_e100_s1 --method supcon --pool labeled --pool_file results/pools/cluster50_f0.1_s1.npy --epochs 100 --seed 1
--name simclr_sub_cluster50_f0.1_e100_s1 --method simclr --pool labeled --pool_file results/pools/cluster50_f0.1_s1.npy --epochs 100 --seed 1
--name supcon_lab_mixkcenter25_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixkcenter25_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixkcenter25_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixkcenter50_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixkcenter50_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixkcenter50_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixkcenter75_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixkcenter75_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixkcenter75_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixcluster5025_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixcluster5025_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixcluster5025_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixcluster5050_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixcluster5050_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixcluster5050_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_mixcluster5075_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_mixcluster5075_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/mixcluster5075_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_kcenter_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/kcenter_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_kcenter_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/kcenter_f0.1_s2.npy --epochs 100 --seed 2
--name supcon_lab_cluster50_f0.1_e100_s2 --method supcon --pool labeled --pool_file results/pools/cluster50_f0.1_s2.npy --epochs 100 --seed 2
--name simclr_sub_cluster50_f0.1_e100_s2 --method simclr --pool labeled --pool_file results/pools/cluster50_f0.1_s2.npy --epochs 100 --seed 2


**`jobs/rev_p12_grid.txt`** (21 lines) — 21 runs.

In [ ]:
%%writefile jobs/rev_p12_grid.txt
--name supcon_lab_f0.5_e100_s0 --method supcon --pool labeled --label_fraction 0.5 --epochs 100 --seed 0
--name supcon_lab_f1.0_e100_s0 --method supcon --pool labeled --label_fraction 1.0 --epochs 100 --seed 0
--name union_f0.5_e100_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.5 --epochs 100 --seed 0
--name supcon_lab_f0.01_e25_s0 --method supcon --pool labeled --label_fraction 0.01 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.05_e25_s0 --method supcon --pool labeled --label_fraction 0.05 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.2_e25_s0 --method supcon --pool labeled --label_fraction 0.2 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.5_e25_s0 --method supcon --pool labeled --label_fraction 0.5 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f1.0_e25_s0 --method supcon --pool labeled --label_fraction 1.0 --epochs 25 --curve_points 8 --seed 0
--name union_f0.01_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.01 --epochs 25 --curve_points 8 --seed 0
--name union_f0.05_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.05 --epochs 25 --curve_points 8 --seed 0
--name union_f0.2_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.2 --epochs 25 --curve_points 8 --seed 0
--name union_f0.5_e25_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.5 --epochs 25 --curve_points 8 --seed 0
--name supcon_lab_f0.01_e400_s0 --method supcon --pool labeled --label_fraction 0.01 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.05_e400_s0 --method supcon --pool labeled --label_fraction 0.05 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.2_e400_s0 --method supcon --pool labeled --label_fraction 0.2 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.5_e400_s0 --method supcon --pool labeled --label_fraction 0.5 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f1.0_e400_s0 --method supcon --pool labeled --label_fraction 1.0 --epochs 400 --curve_points 16 --seed 0
--name union_f0.01_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.01 --epochs 400 --curve_points 16 --seed 0
--name union_f0.05_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.05 --epochs 400 --curve_points 16 --seed 0
--name union_f0.2_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.2 --epochs 400 --curve_points 16 --seed 0
--name union_f0.5_e400_s0 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.5 --epochs 400 --curve_points 16 --seed 0


**`jobs/rev_p13_ft_grid.txt`** (27 lines) — 27 runs.

In [ ]:
%%writefile jobs/rev_p13_ft_grid.txt
-m dlt.finetune --name ft_supcon_lab_f0.01_e25_s0 --from_run supcon_lab_f0.01_e25_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_union_f0.01_e25_s0 --from_run union_f0.01_e25_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.01_s0 --from_run simclr_all_e25_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.01_e400_s0 --from_run supcon_lab_f0.01_e400_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_union_f0.01_e400_s0 --from_run union_f0.01_e400_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.01_s0 --from_run simclr_all_e400_s0 --label_fraction 0.01 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.05_e25_s0 --from_run supcon_lab_f0.05_e25_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_union_f0.05_e25_s0 --from_run union_f0.05_e25_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.05_s0 --from_run simclr_all_e25_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.05_e400_s0 --from_run supcon_lab_f0.05_e400_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_union_f0.05_e400_s0 --from_run union_f0.05_e400_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.05_s0 --from_run simclr_all_e400_s0 --label_fraction 0.05 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.2_e25_s0 --from_run supcon_lab_f0.2_e25_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_union_f0.2_e25_s0 --from_run union_f0.2_e25_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.2_s0 --from_run simclr_all_e25_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.2_e400_s0 --from_run supcon_lab_f0.2_e400_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_union_f0.2_e400_s0 --from_run union_f0.2_e400_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.2_s0 --from_run simclr_all_e400_s0 --label_fraction 0.2 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.5_e25_s0 --from_run supcon_lab_f0.5_e25_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_union_f0.5_e25_s0 --from_run union_f0.5_e25_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_f0.5_s0 --from_run simclr_all_e25_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.5_e100_s0 --from_run supcon_lab_f0.5_e100_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_union_f0.5_e100_s0 --from_run union_f0.5_e100_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_simclr_all_e100_f0.5_s0 --from_run simclr_all_e100_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_supcon_lab_f0.5_e400_s0 --from_run supcon_lab_f0.5_e400_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_union_f0.5_e400_s0 --from_run union_f0.5_e400_s0 --label_fraction 0.5 --seed 0
-m dlt.finetune --name ft_simclr_all_e400_f0.5_s0 --from_run simclr_all_e400_s0 --label_fraction 0.5 --seed 0


**`jobs/rev_p14_reg_r50.txt`** (8 lines) — 8 runs.

In [ ]:
%%writefile jobs/rev_p14_reg_r50.txt
--name r50_simclr_all_e200_s1 --backbone resnet50 --method simclr --pool all --epochs 200 --seed 1
--name r50_supcon_lab_f0.1_e200_s1 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name r50_simclr_all_e200_s2 --backbone resnet50 --method simclr --pool all --epochs 200 --seed 2
--name r50_supcon_lab_f0.1_e200_s2 --backbone resnet50 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name supcon_lab_f0.1_wd0.0005_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --wd 0.0005 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_wd0.001_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --wd 0.001 --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_strongaug_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --aug simclr_strong --epochs 400 --curve_points 16 --seed 0
--name supcon_lab_f0.1_wd0.0005_strongaug_e400_s0 --method supcon --pool labeled --label_fraction 0.1 --wd 0.0005 --aug simclr_strong --epochs 400 --curve_points 16 --seed 0


**`jobs/rev_p14b_ft_grid_fill.txt`** (4 lines) — 3 runs.

In [ ]:
%%writefile jobs/rev_p14b_ft_grid_fill.txt
# fine-tuning cell of the label-fraction grid at f=10%, 2,425 steps (missing from p13)
-m dlt.finetune --name ft_supcon_lab_f0.1_e25_s0 --from_run supcon_lab_f0.1_e25_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_union_f0.1_e25_s0 --from_run union_f0.1_e25_s0 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_simclr_all_e25_s0 --from_run simclr_all_e25_s0 --label_fraction 0.1 --seed 0


**`jobs/rev_p15_c100_e200.txt`** (9 lines) — 9 runs.

In [ ]:
%%writefile jobs/rev_p15_c100_e200.txt
--name c100_simclr_all_e200_s0 --dataset cifar100 --method simclr --pool all --epochs 200 --seed 0
--name c100_supcon_lab_f0.1_e200_s0 --dataset cifar100 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
--name c100_union_f0.1_e200_s0 --dataset cifar100 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 0
--name c100_simclr_all_e200_s1 --dataset cifar100 --method simclr --pool all --epochs 200 --seed 1
--name c100_supcon_lab_f0.1_e200_s1 --dataset cifar100 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name c100_union_f0.1_e200_s1 --dataset cifar100 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 1
--name c100_simclr_all_e200_s2 --dataset cifar100 --method simclr --pool all --epochs 200 --seed 2
--name c100_supcon_lab_f0.1_e200_s2 --dataset cifar100 --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name c100_union_f0.1_e200_s2 --dataset cifar100 --method supcon --pool all --temperature_unlabeled 0.5 --labeled_share 0.25 --label_fraction 0.1 --epochs 200 --seed 2


**`jobs/rev_p16_c100_r50_post.txt`** (15 lines) — 15 runs.

In [ ]:
%%writefile jobs/rev_p16_c100_r50_post.txt
-m dlt.finetune --name ft_c100_simclr_all_e200_s0 --from_run c100_simclr_all_e200_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_c100_supcon_lab_f0.1_e200_s0 --from_run c100_supcon_lab_f0.1_e200_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_c100_union_f0.1_e200_s0 --from_run c100_union_f0.1_e200_s0 --dataset cifar100 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_c100_simclr_all_e200_s1 --from_run c100_simclr_all_e200_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_supcon_lab_f0.1_e200_s1 --from_run c100_supcon_lab_f0.1_e200_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_union_f0.1_e200_s1 --from_run c100_union_f0.1_e200_s1 --dataset cifar100 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_c100_simclr_all_e200_s2 --from_run c100_simclr_all_e200_s2 --dataset cifar100 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_c100_supcon_lab_f0.1_e200_s2 --from_run c100_supcon_lab_f0.1_e200_s2 --dataset cifar100 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_c100_union_f0.1_e200_s2 --from_run c100_union_f0.1_e200_s2 --dataset cifar100 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_r50_simclr_all_e200_s0 --from_run r50_simclr_all_e200_s0 --backbone resnet50 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_r50_supcon_lab_f0.1_e200_s0 --from_run r50_supcon_lab_f0.1_e200_s0 --backbone resnet50 --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_r50_simclr_all_e200_s1 --from_run r50_simclr_all_e200_s1 --backbone resnet50 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_r50_supcon_lab_f0.1_e200_s1 --from_run r50_supcon_lab_f0.1_e200_s1 --backbone resnet50 --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_r50_simclr_all_e200_s2 --from_run r50_simclr_all_e200_s2 --backbone resnet50 --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_r50_supcon_lab_f0.1_e200_s2 --from_run r50_supcon_lab_f0.1_e200_s2 --backbone resnet50 --label_fraction 0.1 --seed 2


**`jobs/rev_p17_tin_s0.txt`** (8 lines) — 8 runs.

In [ ]:
%%writefile jobs/rev_p17_tin_s0.txt
--name tin_simclr_all_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 0
--name tin_supcon_lab_f0.1_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name tin_union_f0.1_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 100 --seed 0
--name tin_simclr_sub_f0.1_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 0
--name tin_simclr_all_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool all --epochs 200 --seed 0
--name tin_supcon_lab_f0.1_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 0
--name tin_union_f0.1_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 200 --seed 0
--name tin_simclr_sub_f0.1_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 0


**`jobs/rev_p18_tin_s0_post.txt`** (8 lines) — 8 runs.

In [ ]:
%%writefile jobs/rev_p18_tin_s0_post.txt
-m dlt.finetune --name ft_tin_simclr_all_e100_s0 --from_run tin_simclr_all_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_supcon_lab_f0.1_e100_s0 --from_run tin_supcon_lab_f0.1_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_union_f0.1_e100_s0 --from_run tin_union_f0.1_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_simclr_sub_f0.1_e100_s0 --from_run tin_simclr_sub_f0.1_e100_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_simclr_all_e200_s0 --from_run tin_simclr_all_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_supcon_lab_f0.1_e200_s0 --from_run tin_supcon_lab_f0.1_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_union_f0.1_e200_s0 --from_run tin_union_f0.1_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0
-m dlt.finetune --name ft_tin_simclr_sub_f0.1_e200_s0 --from_run tin_simclr_sub_f0.1_e200_s0 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 0


**`jobs/rev_p19_tin_s12.txt`** (16 lines) — 16 runs.

In [ ]:
%%writefile jobs/rev_p19_tin_s12.txt
--name tin_simclr_all_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 1
--name tin_supcon_lab_f0.1_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name tin_union_f0.1_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 100 --seed 1
--name tin_simclr_sub_f0.1_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 1
--name tin_simclr_all_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool all --epochs 100 --seed 2
--name tin_supcon_lab_f0.1_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name tin_union_f0.1_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 100 --seed 2
--name tin_simclr_sub_f0.1_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool labeled --label_fraction 0.1 --epochs 100 --seed 2
--name tin_simclr_all_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool all --epochs 200 --seed 1
--name tin_supcon_lab_f0.1_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name tin_union_f0.1_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 200 --seed 1
--name tin_simclr_sub_f0.1_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 1
--name tin_simclr_all_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool all --epochs 200 --seed 2
--name tin_supcon_lab_f0.1_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool labeled --label_fraction 0.1 --epochs 200 --seed 2
--name tin_union_f0.1_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --method supcon --pool all --label_fraction 0.1 --temperature_unlabeled 0.5 --labeled_share 0.25 --epochs 200 --seed 2
--name tin_simclr_sub_f0.1_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --method simclr --pool labeled --label_fraction 0.1 --epochs 200 --seed 2


**`jobs/rev_p20_tin_s12_post.txt`** (16 lines) — 16 runs.

In [ ]:
%%writefile jobs/rev_p20_tin_s12_post.txt
-m dlt.finetune --name ft_tin_simclr_all_e100_s1 --from_run tin_simclr_all_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_supcon_lab_f0.1_e100_s1 --from_run tin_supcon_lab_f0.1_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_union_f0.1_e100_s1 --from_run tin_union_f0.1_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_simclr_sub_f0.1_e100_s1 --from_run tin_simclr_sub_f0.1_e100_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_simclr_all_e100_s2 --from_run tin_simclr_all_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_supcon_lab_f0.1_e100_s2 --from_run tin_supcon_lab_f0.1_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_union_f0.1_e100_s2 --from_run tin_union_f0.1_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_simclr_sub_f0.1_e100_s2 --from_run tin_simclr_sub_f0.1_e100_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_simclr_all_e200_s1 --from_run tin_simclr_all_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_supcon_lab_f0.1_e200_s1 --from_run tin_supcon_lab_f0.1_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_union_f0.1_e200_s1 --from_run tin_union_f0.1_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_simclr_sub_f0.1_e200_s1 --from_run tin_simclr_sub_f0.1_e200_s1 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 1
-m dlt.finetune --name ft_tin_simclr_all_e200_s2 --from_run tin_simclr_all_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_supcon_lab_f0.1_e200_s2 --from_run tin_supcon_lab_f0.1_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_union_f0.1_e200_s2 --from_run tin_union_f0.1_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2
-m dlt.finetune --name ft_tin_simclr_sub_f0.1_e200_s2 --from_run tin_simclr_sub_f0.1_e200_s2 --dataset tinyimagenet --backbone resnet18_mp --label_fraction 0.1 --seed 2


## 7. Analysis

After the runs: label-limited probes and the CDNV split for every saved encoder (`reeval.py`), statistics of the
pools (`diversity.py`), the fine-tuning recipe search (`ft_search.py`, launched from the job files), the
hierarchical bootstrap (`stats.py`), and `make_assets.py`, which writes every number, table and figure of the paper.

**`scripts/reeval.py`** (91 lines) — linear probes and kNN with 1–50% of the labels, probe fit on its own images, CDNV on labeled / other / test images; writes `probes.json`.

In [ ]:
%%writefile scripts/reeval.py
"""Label-limited evaluation of every saved encoder.

For each run, linear probes and kNN classifiers are fitted with only 1%, 5%, 10% and 20%
of the training labels (the nested stratified subsets of the run's seed, i.e. the same
images whose labels the semi-supervised methods were given). Results are written to
results/runs/<name>/probes.json and the 10%-probe predictions to probes_preds.npz.

    python scripts/reeval.py --ckpt_dir <dir>
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt.data import load_dataset, stratified_subset   # noqa: E402
from dlt.evaluate import geometry, knn_predict, linear_probe   # noqa: E402
from dlt.gpu_aug import GPUAugment                       # noqa: E402
from dlt.models import extract_features, load_encoder    # noqa: E402

FRACTIONS = (0.01, 0.05, 0.1, 0.2, 0.5)

p = argparse.ArgumentParser()
p.add_argument("--ckpt_dir", required=True)
p.add_argument("--root", default="results/runs")
p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
p.add_argument("--force", action="store_true")
p.add_argument("--filter", default=None)
args = p.parse_args()

cache = {}
for res_path in sorted(glob.glob(os.path.join(args.root, "*", "result.json"))):
    d = os.path.dirname(res_path)
    name = os.path.basename(d)
    if name.startswith("pilot") or (args.filter and args.filter not in name):
        continue
    ck = os.path.join(args.ckpt_dir, name, "model.pt")
    out = os.path.join(d, "probes.json")
    if not os.path.exists(ck):
        continue
    if os.path.exists(out) and not args.force and all(k in json.load(open(out)) for k in ("cdnv_rest_0.1", "linear_0.5")):
        continue
    enc, a = load_encoder(ck)
    ds = a.get("dataset", "cifar10")
    if ds == "stl10":      # only the labeled split has labels; dlt.train already reports everything
        continue
    if ds not in cache:
        data = load_dataset(ds, args.data_dir)
        for k in ("train_x", "train_y", "test_x", "test_y"):
            data[k] = data[k].cuda()
        cache[ds] = data
    data = cache[ds]
    C = data["num_classes"]
    norm = GPUAugment("none", data["mean"], data["std"], "cuda").normalize
    ftr = extract_features(enc, data["train_x"], norm)
    fte = extract_features(enc, data["test_x"], norm)
    stats = (ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6)
    seed = a.get("split_seed", a.get("seed", 0))
    seed = a.get("seed", 0) if seed is None else seed
    res, preds = {}, {}
    for f in FRACTIONS:
        li = torch.as_tensor(stratified_subset(data["train_y"].cpu().numpy(), f, seed), device="cuda")
        pr, info, _ = linear_probe(ftr[li], data["train_y"][li], fte, C, stats, seed)
        res[f"linear_{f}"] = 100 * (pr == data["test_y"]).float().mean().item()
        pk = knn_predict(ftr[li], data["train_y"][li], fte, C, k=min(200, len(li) // C))
        res[f"knn_{f}"] = 100 * (pk == data["test_y"]).float().mean().item()
        if f == 0.1:
            preds["linear_0.1"] = pr.cpu().numpy().astype(np.int16)
            # how "used up" are these labels? fit of the probe on its own training images,
            # and class tightness of the labeled images vs. unseen test images
            w, b, mu, sd = _
            tr = ((((ftr[li] - mu) / sd) @ w + b).argmax(1) == data["train_y"][li]).float().mean().item()
            res["linear_0.1_train"] = 100 * tr
            g_lab, means_lab = geometry(ftr[li], data["train_y"][li], C)
            g_te, _m = geometry(fte, data["test_y"], C, ref_means=means_lab)
            res["cdnv_labeled_0.1"] = g_lab["cdnv"]
            res["cdnv_test_0.1"] = g_te["cdnv"]
            # the training images outside the labeled subset (for methods trained on all images these
            # were seen without a label; for methods trained on the labeled subset they were never seen)
            rest = torch.ones(ftr.shape[0], dtype=torch.bool, device="cuda")
            rest[li] = False
            res["cdnv_rest_0.1"] = geometry(ftr[rest], data["train_y"][rest], C, ref_means=means_lab)[0]["cdnv"]
            res["etf_dev_labeled_0.1"], res["etf_dev_test_0.1"] = g_lab["etf_dev"], g_te["etf_dev"]
            res["ncc_0.1"] = g_te["ncc_acc"]
    json.dump(res, open(out, "w"), indent=1)
    np.savez_compressed(os.path.join(d, "probes_preds.npz"), **preds)
    print(f"{name:36s} " + " ".join(f"{k} {v:.2f}" for k, v in res.items()), flush=True)


**`scripts/diversity.py`** (154 lines) — pool statistics in a fixed reference space: spread, Vendi score, coverage radius.

In [ ]:
%%writefile scripts/diversity.py
"""Diversity statistics of pretraining pools, measured in a fixed reference feature space.

For every pool (a set of training images) and every class we compute

  spread    mean pairwise cosine distance between images of the class;
  spread_aug  the same between one augmented view of every image
              ("average pairwise augmentation distance within a class");
  vendi     Vendi score = exp(entropy of the eigenvalues of the cosine kernel / n),
            the effective number of distinct directions (Friedman & Dieng 2023);
  radius    coverage radius: mean cosine distance from a *test* image of the class to its
            nearest pool image of the class (how far unseen images are from what was seen);
  coverage  share of test images whose nearest same-class pool image is closer than r0, the
            median such distance for the full training set (so the full set has coverage 0.5).

All numbers are averaged over classes. ``n_eff`` converts ``radius`` into the size of a
random pool with the same coverage radius (log-linear interpolation of the random pools).

    python scripts/diversity.py --ref_run simclr_all_e100_s0 --ckpt_dir <dir>
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt.data import load_dataset, stratified_subset   # noqa: E402
from dlt.gpu_aug import GPUAugment                      # noqa: E402
from dlt.models import ContrastiveModel, extract_features, load_encoder   # noqa: E402


@torch.no_grad()
def pool_stats(f_pool, y_pool, f_aug, f_test, y_test, C, r0=None):
    out = {"spread": [], "spread_aug": [], "vendi": [], "radius": [], "nn": []}
    for c in range(C):
        a = f_pool[y_pool == c].double()
        n = a.shape[0]
        s = a.sum(0)
        out["spread"].append((1 - ((s @ s) - n) / (n * (n - 1))).item())
        b = f_aug[y_pool == c].double()
        sb = b.sum(0)
        out["spread_aug"].append((1 - ((sb @ sb) - n) / (n * (n - 1))).item())
        ev = torch.linalg.eigvalsh(a.t() @ a / n).clamp(min=1e-12)
        ev = ev / ev.sum()
        out["vendi"].append(torch.exp(-(ev * ev.log()).sum()).item())
        t = f_test[y_test == c].double()
        d = (1 - t @ a.t()).min(1).values
        out["radius"].append(d.mean().item())
        out["nn"].append(d)
    res = {k: float(np.mean(v)) for k, v in out.items() if k != "nn"}
    nn = torch.cat(out["nn"])
    res["radius_median"] = nn.median().item()
    if r0 is not None:
        res["coverage"] = (nn <= r0).float().mean().item()
    return res


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ref_run", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--fractions", type=float, nargs="+", default=[0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1.0])
    p.add_argument("--seeds", type=int, nargs="+", default=[0, 1, 2, 3, 4])
    p.add_argument("--pools", default="results/pools")
    p.add_argument("--extra_refs", nargs="*", default=[], help="tag=run_name of further reference encoders")
    p.add_argument("--out", default="results/diversity.json")
    args = p.parse_args()
    dev = "cuda"
    data = load_dataset(args.dataset, args.data_dir)
    C = data["num_classes"]
    ckpt = torch.load(os.path.join(args.ckpt_dir, args.ref_run, "model.pt"), map_location=dev)
    model = ContrastiveModel(ckpt["args"]["backbone"]).to(dev)
    model.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in ckpt["model"].items()})
    model = model.to(memory_format=torch.channels_last).eval()
    X, Y = data["train_x"].to(dev), data["train_y"].to(dev)
    Xt, Yt = data["test_x"].to(dev), data["test_y"].to(dev)
    g = torch.Generator(device=dev).manual_seed(0)
    aug = GPUAugment("simclr", data["mean"], data["std"], dev, g)
    spaces = {}
    ftr = F.normalize(extract_features(model.encoder, X, aug.normalize), dim=1)
    fte = F.normalize(extract_features(model.encoder, Xt, aug.normalize), dim=1)
    faug = []
    with torch.no_grad():
        for i in range(0, X.shape[0], 2000):
            v = aug(X[i:i + 2000]).contiguous(memory_format=torch.channels_last)
            with torch.autocast("cuda", dtype=torch.float16):
                faug.append(model.encoder(v).float())
    faug = torch.cat(faug)
    spaces["simclr"] = (ftr, F.normalize(faug, dim=1), fte)
    # further reference encoders (e.g. a supervised network), to check that the result does not
    # depend on measuring distances in a SimCLR feature space
    for tag_ref in args.extra_refs:
        tag, ref = tag_ref.split("=")
        enc, _ = load_encoder(os.path.join(args.ckpt_dir, ref, "model.pt"))
        fa = []
        with torch.no_grad():
            for i in range(0, X.shape[0], 2000):
                v = aug(X[i:i + 2000]).contiguous(memory_format=torch.channels_last)
                with torch.autocast("cuda", dtype=torch.float16):
                    fa.append(enc(v).float())
        spaces[tag] = (F.normalize(extract_features(enc, X, aug.normalize), dim=1), F.normalize(torch.cat(fa), dim=1),
                       F.normalize(extract_features(enc, Xt, aug.normalize), dim=1))
    # a model-free space: mean-centred pixels
    px = aug.normalize(X).flatten(1)
    pxa = aug(X).flatten(1)
    pxt = aug.normalize(Xt).flatten(1)
    spaces["pixel"] = (F.normalize(px, dim=1), F.normalize(pxa, dim=1), F.normalize(pxt, dim=1))

    pools = {}
    labels = Y.cpu().numpy()
    for f in args.fractions:
        for s in (args.seeds if f < 1 else [0]):
            pools[f"random_f{f}_s{s}"] = (np.arange(len(labels)) if f >= 1 else stratified_subset(labels, f, s),
                                           {"kind": "random", "fraction": f, "seed": s})
    for path in sorted(glob.glob(os.path.join(args.pools, "*.npy"))):
        name = os.path.basename(path)[:-4]
        kind, frac, seed = name.split("_")
        pools[name] = (np.load(path), {"kind": kind, "fraction": float(frac[1:]), "seed": int(seed[1:])})

    results = {}
    for space, (f_tr, f_aug, f_te) in spaces.items():
        full = pool_stats(f_tr, Y, f_aug, f_te, Yt, C)
        r0 = full["radius_median"]
        rows = {}
        for name, (idx, meta) in pools.items():
            ii = torch.as_tensor(idx, device=dev)
            st = pool_stats(f_tr[ii], Y[ii], f_aug[ii], f_te, Yt, C, r0)
            st.update(meta)
            st["n"] = int(len(idx))
            rows[name] = st
        # calibration: coverage radius of random pools as a function of pool size
        sizes = sorted({v["n"] for v in rows.values() if v["kind"] == "random"})
        rad = [float(np.mean([v["radius"] for v in rows.values() if v["kind"] == "random" and v["n"] == n])) for n in sizes]
        for v in rows.values():
            # radius decreases with n: interpolate log(n) as a function of radius
            v["n_eff"] = float(np.exp(np.interp(-v["radius"], [-r for r in rad], np.log(sizes))))
        results[space] = {"pools": rows, "calibration": {"n": sizes, "radius": rad}}
        print(f"== space: {space}")
        for name, v in rows.items():
            if v["seed"] == 0:
                print(f"  {name:26s} n={v['n']:6d} spread {v['spread']:.4f} spread_aug {v['spread_aug']:.4f} "
                      f"vendi {v['vendi']:.1f} radius {v['radius']:.4f} cov {v['coverage']:.3f} n_eff {v['n_eff']:.0f}")
    with open(args.out, "w") as f:
        json.dump(results, f, indent=1)


if __name__ == "__main__":
    main()


**`scripts/ft_search.py`** (167 lines) — fine-tuning recipe search on a split of the labeled images (72 recipes), plus the oracle grid scored on the test set.

In [ ]:
%%writefile scripts/ft_search.py
"""Is the fine-tuning result a property of one recipe?  Hyper-parameter search for label-budget fine-tuning.

For one pretrained encoder, every configuration of the grid
    head init   {linear-probe, random}
    lr          {0.003, 0.01, 0.03, 0.1}
    epochs      {30, 60, 120}         (passes over the labeled images)
    regulariser {none, label smoothing 0.1, mixup alpha 0.2}
is trained on 80% of the labeled images and scored on the other 20% -- the test set and the
unlabeled labels are never used for selection. The best configuration is then retrained on all
labeled images and evaluated on the test set (the test accuracy of every configuration is
also stored, for transparency, but is not used to choose).

    python scripts/ft_search.py --from_run simclr_all_e200_s0 --ckpt_dir <dir> --label_fraction 0.1 --seed 0
"""
import argparse
import itertools
import json
import math
import os
import sys
import time

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt.data import IndexSampler, labeled_subset, load_dataset, stratified_subset   # noqa: E402
from dlt.evaluate import linear_probe                                  # noqa: E402
from dlt.gpu_aug import GPUAugment, sup_pipeline                       # noqa: E402
from dlt.models import Classifier, extract_features                    # noqa: E402

GRID = {"head": ["lp", "random"], "lr": [0.003, 0.01, 0.03, 0.1], "epochs": [30, 60, 120],
        "reg": ["none", "ls", "mixup"]}


def finetune(enc_sd, backbone, data, fit_idx, eval_x, eval_y, cfg, seed, dev="cuda", batch=128):
    """Fine-tune on the images ``fit_idx`` and return accuracy on (eval_x, eval_y) and predictions."""
    torch.manual_seed(seed)
    gen = torch.Generator(device=dev).manual_seed(seed)
    X, Y, C = data["train_x"], data["train_y"], data["num_classes"]
    model = Classifier(backbone, C).to(dev)
    model.encoder.load_state_dict(enc_sd)
    model = model.to(memory_format=torch.channels_last)
    aug = GPUAugment(sup_pipeline(X.shape[-1]), data["mean"], data["std"], dev, gen)
    fi = torch.as_tensor(fit_idx, device=dev)
    if cfg["head"] == "lp":
        ftr = extract_features(model.encoder, X[fi], aug.normalize)
        stats = (ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6)
        _, _, (w, b, mu, sd) = linear_probe(ftr, Y[fi], ftr[:10], C, stats, seed)
        with torch.no_grad():
            model.fc.weight.copy_((w / sd.t()).t())
            model.fc.bias.copy_(b - ((mu / sd) @ w).squeeze(0))
    steps = int(round(cfg["epochs"] * len(fit_idx) / batch))
    warm = max(1, int(0.05 * steps))
    opt = torch.optim.SGD(model.parameters(), lr=cfg["lr"], momentum=0.9, nesterov=True, weight_decay=5e-4)
    scaler = torch.amp.GradScaler("cuda")
    sampler = IndexSampler(fit_idx, batch, gen, dev)
    smooth = 0.1 if cfg["reg"] == "ls" else 0.0
    model.train()
    for step in range(steps):
        lr = cfg["lr"] * (step + 1) / warm if step < warm else \
            0.5 * cfg["lr"] * (1 + math.cos(math.pi * (step - warm) / max(1, steps - warm)))
        for gr in opt.param_groups:
            gr["lr"] = lr
        idx = sampler.next()
        x, y = aug(X[idx]), Y[idx]
        if cfg["reg"] == "mixup":
            lam = float(np.random.beta(0.2, 0.2))
            perm = torch.randperm(x.shape[0], device=dev, generator=gen)
            x = lam * x + (1 - lam) * x[perm]
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(x.contiguous(memory_format=torch.channels_last)).float()
        if cfg["reg"] == "mixup":
            loss = lam * F.cross_entropy(logits, y) + (1 - lam) * F.cross_entropy(logits, y[perm])
        else:
            loss = F.cross_entropy(logits, y, label_smoothing=smooth)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    model.eval()
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        pred = torch.cat([model(aug.normalize(eval_x[i:i + 2000]).contiguous(memory_format=torch.channels_last)).argmax(1)
                          for i in range(0, eval_x.shape[0], 2000)])
    return 100 * (pred == eval_y).float().mean().item(), pred


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--name", required=True)
    p.add_argument("--from_run", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--out", default="results/runs")
    p.add_argument("--config_from", default=None,
                   help="name of a finished search: reuse its selected configuration instead of searching")
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--backbone", default="resnet18")
    p.add_argument("--label_fraction", type=float, default=0.1)
    p.add_argument("--seed", type=int, default=0, help="seed of the pretraining run (defines the labeled subset)")
    p.add_argument("--final_only", default=None, help="JSON of a configuration: skip the search and only run it")
    p.add_argument("--no_test_per_config", action="store_true")
    p.add_argument("--oracle", action="store_true",
                   help="score every recipe on the TEST set after training on all labeled images. This is not a "
                        "legitimate selection rule; it gives an upper bound on what recipe selection can achieve")
    p.add_argument("--grid", default="full", choices=["full", "small"],
                   help="small: 2 heads x lr {0.01, 0.03, 0.1} x {30, 60} epochs x {none, mixup}")
    args = p.parse_args()
    out_dir = os.path.join(args.out, args.name)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, "result.json")
    if args.config_from:
        prev = json.load(open(os.path.join(args.out, args.config_from, "result.json")))
        args.final_only = json.dumps(prev["best"])
    if os.path.exists(out_path):
        print("[skip]", out_path)
        return
    dev = "cuda"
    torch.backends.cudnn.benchmark = True
    np.random.seed(args.seed)
    data = load_dataset(args.dataset, args.data_dir)
    for k in ("train_x", "train_y", "test_x", "test_y"):
        data[k] = data[k].to(dev)
    labeled = labeled_subset(data, args.label_fraction, args.seed)
    # split the LABELED images 80/20 (stratified); selection uses nothing else
    val = stratified_subset(data["train_y"].cpu().numpy()[labeled], 0.2, 777)
    val_idx = labeled[val]
    fit_idx = np.setdiff1d(labeled, val_idx)
    vi = torch.as_tensor(val_idx, device=dev)
    ck = torch.load(os.path.join(args.ckpt_dir, args.from_run, "model.pt"), map_location=dev)["model"]
    enc_sd = {k[len("encoder."):]: (v.float() if v.is_floating_point() else v) for k, v in ck.items() if k.startswith("encoder.")}

    rows = []
    t0 = time.time()
    if args.final_only:
        best = json.loads(args.final_only)
    else:
        grid = GRID if args.grid == "full" else {"head": ["lp", "random"], "lr": [0.01, 0.03, 0.1], "epochs": [30, 60],
                                                 "reg": ["none", "mixup"]}
        for head, lr, ep, reg in itertools.product(grid["head"], grid["lr"], grid["epochs"], grid["reg"]):
            cfg = {"head": head, "lr": lr, "epochs": ep, "reg": reg}
            if args.oracle:
                acc, _ = finetune(enc_sd, args.backbone, data, labeled, data["test_x"], data["test_y"], cfg, args.seed)
                rows.append({**cfg, "test_acc": acc, "val_acc": acc})
                print(f"{args.from_run} {cfg} test {acc:.2f} ({time.time() - t0:.0f}s)", flush=True)
                continue
            val_acc, _ = finetune(enc_sd, args.backbone, data, fit_idx, data["train_x"][vi], data["train_y"][vi], cfg, args.seed)
            rows.append({**cfg, "val_acc": val_acc})
            print(f"{args.from_run} {cfg} val {val_acc:.2f} ({time.time() - t0:.0f}s)", flush=True)
        best = max(rows, key=lambda r: r["val_acc"])
        best = {k: best[k] for k in ("head", "lr", "epochs", "reg")}
    test_acc, pred = finetune(enc_sd, args.backbone, data, labeled, data["test_x"], data["test_y"], best, args.seed)
    default = {"head": "lp", "lr": 0.01, "epochs": 60, "reg": "none"}
    default_acc, _ = finetune(enc_sd, args.backbone, data, labeled, data["test_x"], data["test_y"], default, args.seed)
    res = {"name": args.name, "args": vars(args), "from_run": args.from_run, "n_labeled": int(len(labeled)),
           "grid": rows, "best": best, "default": default, "train_time_s": time.time() - t0,
           "oracle": bool(args.oracle),
           "metrics": {("finetune_oracle_acc" if args.oracle else "finetune_tuned_acc"): test_acc,
                       "finetune_default_acc": default_acc}}
    np.savez_compressed(os.path.join(out_dir, "preds.npz"), finetune_tuned=pred.cpu().numpy().astype(np.int16))
    json.dump(res, open(out_path, "w"), indent=1)
    print(f"[done] {args.from_run}: best {best} -> test {test_acc:.2f} (default recipe {default_acc:.2f})", flush=True)


if __name__ == "__main__":
    main()


**`scripts/stats.py`** (149 lines) — hierarchical bootstrap over seeds and test images, Holm correction, TOST equivalence test.

In [ ]:
%%writefile scripts/stats.py
"""Statistical comparison of two configurations (sets of runs that differ only in the seed).

Hierarchical (two-level) bootstrap
    One bootstrap replicate (i) resamples the seeds of each configuration with replacement and
    (ii) resamples the test images with replacement, using the SAME resampled images for both
    configurations (paired at the image level). When the two configurations were run with the same
    seeds, the seeds are resampled jointly, because runs with the same seed share the labeled
    subset. The statistic is the difference of mean test accuracy. We report the percentile 95%
    interval and a two-sided p-value 2*min(P(d<=0), P(d>=0)), which cannot be smaller than
    1/n_boot and is reported as "< 0.001" below that.

Equivalence (TOST)
    For "no difference" claims we test H0: |d| >= margin with two one-sided tests on the same
    bootstrap distribution: p_tost = max(P(d <= -margin), P(d >= margin)). Equivalence within the
    margin is declared when p_tost < 0.05 (the 90% interval lies inside +-margin).

Multiple comparisons
    holm() applies the Holm-Bonferroni step-down correction to a family of p-values.

    python scripts/stats.py simclr_all_e100 supcon_lab_f0.1_e100 --key linear
"""
import argparse
import glob
import os
import re

import numpy as np
from scipy import stats as sps

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def config_runs(config, root="results/runs"):
    runs = {}
    for d in sorted(glob.glob(os.path.join(root, glob.escape(config) + "_s*"))):
        m = re.fullmatch(re.escape(config) + r"_s(\d+)", os.path.basename(d))
        if m:
            runs[int(m.group(1))] = d
    return runs


def load_correct(config, key, y, root="results/runs", files=("preds.npz", "probes_preds.npz")):
    """Dict seed -> boolean vector over test images (was the image classified correctly?)."""
    out = {}
    for seed, d in config_runs(config, root).items():
        for fn in files:
            p = os.path.join(d, fn)
            if os.path.exists(p):
                z = np.load(p)
                if key in z:
                    out[seed] = (z[key] == y).astype(np.float32)
                    break
    return out


def hierarchical_bootstrap(ca, cb, n_boot=10000, seed=0, chunk=500, margin=1.0):
    """ca, cb: dicts seed -> correctness vector on the same n test images."""
    rng = np.random.default_rng(seed)
    sa, sb = sorted(ca), sorted(cb)
    A, B = np.stack([ca[s] for s in sa]), np.stack([cb[s] for s in sb])
    paired = sa == sb
    n = A.shape[1]
    diffs = np.empty(n_boot, dtype=np.float64)
    for start in range(0, n_boot, chunk):
        m = min(chunk, n_boot - start)
        w = rng.multinomial(n, np.full(n, 1.0 / n), size=m).astype(np.float32) / n   # [m, n] image weights
        acc_a, acc_b = w @ A.T, w @ B.T                                              # [m, seeds]
        ia = rng.integers(0, A.shape[0], size=(m, A.shape[0]))
        ib = ia if paired else rng.integers(0, B.shape[0], size=(m, B.shape[0]))
        diffs[start:start + m] = (np.take_along_axis(acc_a, ia, 1).mean(1)
                                  - np.take_along_axis(acc_b, ib, 1).mean(1))
    diffs *= 100
    obs = 100 * (A.mean() - B.mean())
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    p_tost = max((diffs <= -margin).mean(), (diffs >= margin).mean())
    acc_a, acc_b = 100 * A.mean(1), 100 * B.mean(1)
    res = {"diff": obs, "ci_lo": lo, "ci_hi": hi, "p": float(min(1.0, max(p, 1.0 / n_boot))),
           "p_tost": float(max(p_tost, 1.0 / n_boot)), "margin": margin, "paired_seeds": paired,
           "n_a": len(sa), "n_b": len(sb), "mean_a": acc_a.mean(), "mean_b": acc_b.mean(),
           "std_a": acc_a.std(ddof=1) if len(sa) > 1 else float("nan"),
           "std_b": acc_b.std(ddof=1) if len(sb) > 1 else float("nan")}
    if len(sa) > 1 and len(sb) > 1:
        res["p_welch"] = float(sps.ttest_ind(acc_a, acc_b, equal_var=False).pvalue)
    return res


def holm(pvals):
    """Holm-Bonferroni adjusted p-values (same order as the input)."""
    p = np.asarray(pvals, dtype=float)
    order = np.argsort(p)
    adj = np.empty_like(p)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(p) - rank) * p[i])
        adj[i] = min(1.0, running)
    return adj


def mean_ci(values, n_boot=10000, seed=0):
    """Mean and percentile 95% interval of a handful of per-seed numbers."""
    v = np.asarray(values, dtype=float)
    if len(v) < 2:
        return float(v.mean()), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    b = v[rng.integers(0, len(v), size=(n_boot, len(v)))].mean(1)
    return float(v.mean()), float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


def test_labels(dataset="cifar10", data_dir=None):
    """Test labels, cached in results/test_labels_<dataset>.npy."""
    cache = os.path.join(ROOT, "results", f"test_labels_{dataset}.npy")
    if os.path.exists(cache):
        return np.load(cache)
    import sys
    sys.path.insert(0, ROOT)
    from dlt.data import load_dataset
    y = load_dataset(dataset, data_dir or os.environ.get("DLT_DATA", "./data"))["test_y"].numpy()
    np.save(cache, y)
    return y


def compare(a, b, y, key="linear", key_b=None, root="results/runs", n_boot=10000, margin=1.0):
    ca, cb = load_correct(a, key, y, root), load_correct(b, key_b or key, y, root)
    if not ca or not cb:
        raise FileNotFoundError(f"missing predictions for {a} [{key}] or {b} [{key_b or key}]")
    res = hierarchical_bootstrap(ca, cb, n_boot, margin=margin)
    res.update({"a": a, "b": b, "key": key})
    return res


def main():
    p = argparse.ArgumentParser()
    p.add_argument("a")
    p.add_argument("b")
    p.add_argument("--key", default="linear")
    p.add_argument("--dataset", default="cifar10")
    p.add_argument("--root", default="results/runs")
    p.add_argument("--margin", type=float, default=1.0)
    args = p.parse_args()
    r = compare(args.a, args.b, test_labels(args.dataset), args.key, root=args.root, margin=args.margin)
    print(f"{r['a']}: {r['mean_a']:.2f} ± {r['std_a']:.2f} (n={r['n_a']})")
    print(f"{r['b']}: {r['mean_b']:.2f} ± {r['std_b']:.2f} (n={r['n_b']})")
    print(f"difference {r['diff']:+.2f} points, 95% CI [{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}], p = {r['p']:.4g}, "
          f"TOST (±{r['margin']}) p = {r['p_tost']:.4g}" + (f", Welch p = {r['p_welch']:.3g}" if "p_welch" in r else ""))


if __name__ == "__main__":
    main()


**`scripts/summarize.py`** (75 lines) — collects every `result.json` into one table with group means.

In [ ]:
%%writefile scripts/summarize.py
"""Collect every results/runs/*/result.json into one table and print group means.

    python scripts/summarize.py                 # all runs, grouped by configuration
    python scripts/summarize.py --filter pilot  # only runs whose name contains "pilot"
    python scripts/summarize.py --csv results/all_runs.csv
"""
import argparse
import glob
import json
import os
import re

import numpy as np
import pandas as pd

KEYS = ["linear_acc", "knn_acc", "linear_lab_acc", "classifier_acc", "finetune_acc"]


def load_runs(root="results/runs"):
    rows = []
    for path in sorted(glob.glob(os.path.join(root, "*", "result.json"))):
        r = json.load(open(path))
        a = r["args"]
        row = {
            "name": r["name"],
            "config": re.sub(r"_s\d+(?=(__|$))", "", r["name"]),
            "seed": a.get("seed"),
            "method": a.get("method", "finetune"),
            "dataset": a.get("dataset", "cifar10"),
            "backbone": a.get("backbone", "resnet18"),
            "aug": a.get("aug"),
            "pool": a.get("pool"),
            "fraction": a.get("label_fraction"),
            "epochs": a.get("epochs"),
            "n_pool": r.get("n_pool"),
            "n_labeled": r.get("n_labeled"),
            "steps": r.get("steps"),
            "train_min": r.get("train_time_s", 0) / 60,
        }
        row.update(r["metrics"])
        rows.append(row)
    return pd.DataFrame(rows)


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--root", default="results/runs")
    p.add_argument("--filter", default=None)
    p.add_argument("--csv", default=None)
    p.add_argument("--keys", default=",".join(KEYS))
    args = p.parse_args()
    df = load_runs(args.root)
    if df.empty:
        print("no runs found")
        return
    if args.filter:
        df = df[df["name"].str.contains(args.filter)]
    if args.csv:
        df.to_csv(args.csv, index=False)
    keys = [k for k in args.keys.split(",") if k in df.columns]
    pd.set_option("display.width", 250)
    pd.set_option("display.max_rows", 500)
    out = []
    for cfg, g in df.groupby("config", sort=False):
        row = {"config": cfg, "n": len(g), "min": g["train_min"].mean()}
        for k in keys:
            v = g[k].dropna()
            if len(v):
                row[k] = f"{v.mean():.2f}" + (f" ± {v.std(ddof=1):.2f}" if len(v) > 1 else "")
        out.append(row)
    print(pd.DataFrame(out).fillna("").to_string(index=False))


if __name__ == "__main__":
    main()


**`scripts/hist_probe.py`** (26 lines) — trivial colour baseline: a linear probe on the RGB histogram of the raw images.

In [ ]:
%%writefile scripts/hist_probe.py
"""The trivial colour baseline: a linear probe on the joint RGB histogram of the raw images.

    python scripts/hist_probe.py      -> results/hist_probe.json
"""
import json
import os
import sys

import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt.data import load_dataset                 # noqa: E402
from dlt.evaluate import color_histogram, linear_probe   # noqa: E402

out = {}
for name in ("cifar10", "cifar100"):
    d = load_dataset(name, os.environ.get("DLT_DATA", "./data"))
    xtr, ytr = d["train_x"].cuda(), d["train_y"].cuda()
    xte, yte = d["test_x"].cuda(), d["test_y"].cuda()
    for bins in (4, 8):
        htr, hte = color_histogram(xtr, bins), color_histogram(xte, bins)
        pred, info, _ = linear_probe(htr, ytr, hte, d["num_classes"])
        acc = 100 * (pred == yte).float().mean().item()
        out[f"{name}_bins{bins}"] = {"acc": acc, "l2": info["l2"], "dim": bins ** 3}
        print(name, bins, f"{acc:.2f}")
json.dump(out, open("results/hist_probe.json", "w"), indent=1)


**`scripts/ood_resolution.py`** (174 lines) — resolution shift: ImageNet dogs (Imagewoof) shrunk to 32×32, probe recall and retrieval precision.

In [ ]:
%%writefile scripts/ood_resolution.py
"""Resolution shift: high-resolution ImageNet dogs shrunk to 32x32.

Queries are the validation images of Imagewoof (ten dog breeds from ImageNet, >= 320 px).
Every image is centre-cropped to a square and downsampled to 32x32 (bicubic, antialiased),
which is what the retrieval demo of this project does with an uploaded photo.

For every CIFAR-10 encoder we report, for these out-of-distribution dogs and for the
in-distribution CIFAR-10 test dogs:
  * dog recall of the linear probe (share of images classified as "dog");
  * retrieval precision@5: share of the 5 nearest CIFAR-10 test images that are dogs.

CLIP ViT-B/32 (zero-shot, 10 CIFAR-10 class prompts) is evaluated on the same images at
several resolutions as an external reference. It is not a competitor: it was trained on
400M image-text pairs. It tells us how much of the failure is information lost by the
downsampling itself and how much is the encoder.

    python scripts/ood_resolution.py --runs simclr_all_e100_s0 ... --ckpt_dir <dir> --ood_dir <imagewoof2-320/val>
"""
import argparse
import glob
import json
import os
import sys

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from dlt.data import CIFAR10_CLASSES, load_dataset     # noqa: E402
from dlt.evaluate import linear_probe                    # noqa: E402
from dlt.gpu_aug import GPUAugment                       # noqa: E402
from dlt.models import Classifier, ContrastiveModel, extract_features   # noqa: E402

DOG = CIFAR10_CLASSES.index("dog")


def load_squares(folder, size=224, limit=None):
    """Centre-cropped square images, resized to ``size`` (uint8 [N,3,size,size])."""
    files = sorted(glob.glob(os.path.join(folder, "*", "*.JPEG")))
    if limit:
        files = files[:limit]
    out = []
    for f in files:
        im = Image.open(f).convert("RGB")
        w, h = im.size
        s = min(w, h)
        im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((size, size), Image.BICUBIC)
        out.append(np.asarray(im))
    return torch.from_numpy(np.stack(out)).permute(0, 3, 1, 2).contiguous(), files


def downsample(x_uint8, size):
    x = F.interpolate(x_uint8.float(), size=(size, size), mode="bicubic", antialias=True, align_corners=False)
    return x.round().clamp(0, 255).to(torch.uint8)


def boot_ci(v, n_boot=2000, seed=0):
    rng = np.random.default_rng(seed)
    v = np.asarray(v, dtype=np.float64)
    m = [v[rng.integers(0, len(v), len(v))].mean() for _ in range(n_boot)]
    return [100 * float(np.percentile(m, 2.5)), 100 * float(np.percentile(m, 97.5))]


@torch.no_grad()
def eval_encoder(run, ckpt_dir, data, ood32, dev="cuda"):
    ckpt = torch.load(os.path.join(ckpt_dir, run, "model.pt"), map_location=dev)
    sd = {k: v.float() if v.is_floating_point() else v for k, v in ckpt["model"].items()}
    backbone = ckpt["args"].get("backbone", "resnet18")
    model = Classifier(backbone, 10) if any(k.startswith("fc.") for k in sd) else ContrastiveModel(backbone)
    model.load_state_dict(sd)
    enc = model.encoder.to(dev).to(memory_format=torch.channels_last).eval()
    norm = GPUAugment("none", data["mean"], data["std"], dev).normalize
    ftr = extract_features(enc, data["train_x"], norm)
    fte = extract_features(enc, data["test_x"], norm)
    food = extract_features(enc, ood32, norm)
    stats = (ftr.mean(0, keepdim=True), ftr.std(0, keepdim=True) + 1e-6)
    pred_te, _, (w, b, mu, sd_) = linear_probe(ftr, data["train_y"], fte, 10, stats, 0)
    pred_ood = (((food - mu) / sd_) @ w + b).argmax(1)
    yte = data["test_y"]
    dogs = yte == DOG
    # retrieval in the CIFAR-10 test index, cosine similarity on encoder features
    index = F.normalize(fte, dim=1)
    sim_ood = F.normalize(food, dim=1) @ index.t()
    nn_ood = yte[sim_ood.topk(5, dim=1).indices]
    sim_id = index[dogs] @ index.t()
    sim_id[torch.arange(int(dogs.sum())), dogs.nonzero(as_tuple=True)[0]] = -2.0   # drop the query itself
    nn_id = yte[sim_id.topk(5, dim=1).indices]
    hist = torch.bincount(pred_ood, minlength=10).float() / len(pred_ood)
    return {
        "test_acc": 100 * (pred_te == yte).float().mean().item(),
        "dog_recall_cifar": 100 * (pred_te[dogs] == DOG).float().mean().item(),
        "dog_recall_ood": 100 * (pred_ood == DOG).float().mean().item(),
        "dog_recall_ood_ci": boot_ci((pred_ood == DOG).float().cpu().numpy()),
        "p_at_5_cifar": 100 * (nn_id == DOG).float().mean().item(),
        "p_at_5_ood": 100 * (nn_ood == DOG).float().mean().item(),
        "top1_cifar": 100 * (nn_id[:, 0] == DOG).float().mean().item(),
        "top1_ood": 100 * (nn_ood[:, 0] == DOG).float().mean().item(),
        "ood_prediction_hist": {CIFAR10_CLASSES[i]: round(100 * hist[i].item(), 2) for i in range(10)},
    }


@torch.no_grad()
def eval_clip(data, ood224, resolutions, dev="cuda", name="openai/clip-vit-base-patch32"):
    from transformers import CLIPModel, CLIPTokenizer
    model = CLIPModel.from_pretrained(name).to(dev).eval()
    tok = CLIPTokenizer.from_pretrained(name)
    text = tok([f"a photo of a {c}" for c in CIFAR10_CLASSES], padding=True, return_tensors="pt").to(dev)
    tfeat = model.get_text_features(**text)
    tfeat = F.normalize(tfeat if torch.is_tensor(tfeat) else tfeat.pooler_output, dim=-1)
    mean = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=dev).view(1, 3, 1, 1)
    std = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=dev).view(1, 3, 1, 1)

    def predict(x_uint8):
        preds = []
        for i in range(0, x_uint8.shape[0], 500):
            xb = x_uint8[i:i + 500].to(dev).float()
            if xb.shape[-1] != 224:
                xb = F.interpolate(xb, size=(224, 224), mode="bicubic", align_corners=False).clamp(0, 255)
            f = model.get_image_features(pixel_values=(xb / 255.0 - mean) / std)
            f = F.normalize(f if torch.is_tensor(f) else f.pooler_output, dim=-1)
            preds.append((f @ tfeat.t()).argmax(1))
        return torch.cat(preds)

    out = {}
    pte = predict(data["test_x"])
    yte = data["test_y"]
    out["test_acc"] = 100 * (pte == yte).float().mean().item()
    out["dog_recall_cifar"] = 100 * (pte[yte == DOG] == DOG).float().mean().item()
    out["dog_recall_ood_by_resolution"] = {}
    for r in resolutions:
        x = ood224 if r == 224 else downsample(ood224.to(dev), r)
        pr = predict(x)
        out["dog_recall_ood_by_resolution"][str(r)] = 100 * (pr == DOG).float().mean().item()
        if r == 32:
            hist = torch.bincount(pr, minlength=10).float() / len(pr)
            out["ood_prediction_hist"] = {CIFAR10_CLASSES[i]: round(100 * hist[i].item(), 2) for i in range(10)}
    return out


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--runs", nargs="+", required=True)
    p.add_argument("--ckpt_dir", required=True)
    p.add_argument("--ood_dir", required=True)
    p.add_argument("--data_dir", default=os.environ.get("DLT_DATA", "./data"))
    p.add_argument("--out", default="results/ood_resolution.json")
    p.add_argument("--no_clip", action="store_true")
    args = p.parse_args()
    dev = "cuda"
    data = load_dataset("cifar10", args.data_dir)
    for k in ("train_x", "train_y", "test_x", "test_y"):
        data[k] = data[k].to(dev)
    ood224, files = load_squares(args.ood_dir)
    ood32 = downsample(ood224.to(dev), 32)
    res = {"n_ood": len(files), "encoders": {}}
    for run in args.runs:
        if not os.path.exists(os.path.join(args.ckpt_dir, run, "model.pt")):
            print(f"[skip] no checkpoint for {run}")
            continue
        res["encoders"][run] = r = eval_encoder(run, args.ckpt_dir, data, ood32)
        print(f"{run:34s} test {r['test_acc']:.2f} | dog recall: CIFAR {r['dog_recall_cifar']:.1f} -> OOD {r['dog_recall_ood']:.1f} "
              f"| P@5: CIFAR {r['p_at_5_cifar']:.1f} -> OOD {r['p_at_5_ood']:.1f}", flush=True)
    if not args.no_clip:
        res["clip"] = c = eval_clip(data, ood224, [16, 32, 64, 128, 224])
        print(f"CLIP ViT-B/32 zero-shot: test {c['test_acc']:.2f} | dog recall CIFAR {c['dog_recall_cifar']:.1f} | "
              f"OOD by resolution {c['dog_recall_ood_by_resolution']}")
    with open(args.out, "w") as f:
        json.dump(res, f, indent=1)


if __name__ == "__main__":
    main()


**`scripts/make_assets.py`** (1022 lines) — every number (`\R{...}` macros), table and figure of the paper from `results/runs/`.

In [ ]:
%%writefile scripts/make_assets.py
r"""Build every number, table and figure of the paper from results/.

    python scripts/make_assets.py            # writes paper/generated/*.tex and paper/figures/*.pdf

Nothing in the paper is typed by hand: paper/generated/numbers.tex defines \R{<config>:<protocol>} for
every configuration, and the tables are written as tabular bodies. A cell whose runs are not finished
is printed as "--".

Protocols (columns)
    lp50   linear probe trained with all training labels (the oracle-label probe)
    knn    weighted kNN with all training labels as memory bank
    lp5    linear probe trained with the labeled subset only
    ft     fine-tuning on the labeled subset, default recipe (dlt.finetune)
    ftt    fine-tuning on the labeled subset, recipe selected on a split of the labeled subset
    ftx    the recipe that the labeled subset selects for SimCLR, applied to another encoder
    fts    "selected recipe": ftt for label-free encoders, ftx for encoders that were trained with the labels
           (their own labeled images cannot serve as a validation set)
    fto    recipe chosen on the TEST set of seed 0 (an upper bound), evaluated on the other seeds
    fm     FixMatch initialised from the encoder
    dist   SimCLR v2-style fine-tune + self-distillation
    clf    accuracy of the method's own classifier (cross-entropy, FixMatch, MixMatch)
"""
import glob
import json
import os
import re
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt     # noqa: E402
import numpy as np                  # noqa: E402
from scipy import stats as sps      # noqa: E402

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, os.path.join(ROOT, "scripts"))
import stats as st                  # noqa: E402

RUNS = os.path.join(ROOT, "results", "runs")
GEN = os.path.join(ROOT, "paper", "generated")
FIG = os.path.join(ROOT, "paper", "figures")
os.makedirs(GEN, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({"font.family": "serif", "font.size": 8, "axes.labelsize": 8, "axes.titlesize": 8.5,
                     "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.linewidth": 0.6, "lines.linewidth": 1.4,
                     "lines.markersize": 4, "pdf.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
                     "axes.grid": True, "grid.color": "#e3e3e3", "grid.linewidth": 0.5})
# Okabe-Ito colours; every series also has its own marker and line style
STYLE = {"simclr": ("#0072B2", "o", "-", "SimCLR (all images, no labels)"),
         "sub": ("#56B4E9", "s", "--", "SimCLR (labeled images only)"),
         "supcon": ("#D55E00", "^", "-", "SupCon (labeled images only)"),
         "union": ("#CC79A7", "D", "-.", "SupCon+U (all images)")}
ARMS = {"simclr": "simclr_all_e{E}", "sub": "simclr_sub_f0.1_e{E}", "supcon": "supcon_lab_f0.1_e{E}",
        "union": "union_f0.1_e{E}"}
ARM_TEX = {"simclr": r"\simclr", "sub": r"\simclrsub", "supcon": r"\supcon", "union": r"\union"}

# ----------------------------------------------------------------------------
# loading
# ----------------------------------------------------------------------------
RUN = {}
for path in glob.glob(os.path.join(RUNS, "*", "result.json")):
    d = os.path.dirname(path)
    name = os.path.basename(d)
    r = json.load(open(path))
    pj = os.path.join(d, "probes.json")
    r["probes"] = json.load(open(pj)) if os.path.exists(pj) else {}
    r["dir"] = d
    RUN[name] = r
CONFIGS = sorted({re.sub(r"_s\d+$", "", n) for n in RUN if re.search(r"_s\d+$", n)})


def seeds(cfg):
    return sorted(int(m.group(1)) for n in RUN for m in [re.fullmatch(re.escape(cfg) + r"_s(\d+)", n)] if m)


def companion(name, kind):
    """Name of the run that post-processes encoder ``name`` (fine-tuning, FixMatch, ...)."""
    if kind == "fm" and name.startswith("stl_"):
        return "stl_fixmatch_from_" + name[4:]
    return {"ft": "ft_", "ftt": "ftsearch_", "ftd": "ftsearch_", "fm": "fixmatch_from_", "dist": "distill_",
            "fto": "ftoracle_", "fto0": "ftoracle_", "ftx": "fttransfer_"}[kind] + name


def label_free(name):
    return name.startswith(("simclr_all", "byol_all", "simsiam_all", "stl_simclr_all"))


def metric(name, proto):
    r = RUN.get(name)
    if r is None:
        return None
    m, p = r["metrics"], r["probes"]
    stl = name.startswith("stl_")
    if proto == "lp50":
        return None if stl else m.get("linear_acc")
    if proto == "knn":
        return m.get("knn_acc")
    if proto == "lp5":
        return m.get("linear_acc") if stl else m.get("linear_lab_acc", p.get("linear_0.1"))
    if proto.startswith("lpf:"):
        return p.get("linear_" + proto[4:])
    if proto == "fts":
        proto = "ftt" if label_free(name) else "ftx"
    if proto in ("ft", "ftt", "ftd", "fm", "dist", "fto", "fto0", "ftx"):
        c = RUN.get(companion(name, proto))
        key = {"ft": "finetune_acc", "ftt": "finetune_tuned_acc", "ftd": "finetune_default_acc",
               "fm": "classifier_acc", "dist": "classifier_acc", "fto": "finetune_tuned_acc",
               "fto0": "finetune_oracle_acc", "ftx": "finetune_tuned_acc"}[proto]
        return None if c is None else c["metrics"].get(key)
    if proto == "clf":
        return m.get("classifier_acc")
    if proto == "cdnv_lab":
        return m.get("seen_cdnv") if stl else p.get("cdnv_labeled_0.1")
    if proto == "cdnv_rest":
        return p.get("cdnv_rest_0.1")
    if proto == "cdnv_test":
        return m.get("test_cdnv") if stl else p.get("cdnv_test_0.1")
    if proto == "lp5_train":
        return p.get("linear_0.1_train")
    if proto == "etf_lab":
        return p.get("etf_dev_labeled_0.1")
    if proto == "time":
        return r.get("train_time_s", 0) / 60
    return m.get(proto, p.get(proto))


def V(cfg, proto, only=None):
    out = [metric(f"{cfg}_s{s}", proto) for s in seeds(cfg) if only is None or s in only]
    return np.array([v for v in out if v is not None], dtype=float)


def mean(cfg, proto):
    v = V(cfg, proto)
    return float(v.mean()) if len(v) else float("nan")


def MS(cfg, proto, d=1, bold=False):
    v = V(cfg, proto)
    if len(v) == 0:
        return "--"
    if d == 2 and abs(v.mean()) < 0.05:
        d = 3
    s = f"{v.mean():.{d}f}"
    if bold:
        s = r"\textbf{" + s + "}"
    if len(v) > 1:
        s += rf"{{\scriptsize$\,\pm\,${v.std(ddof=1):.{d}f}}}"
    return s


def nseeds(cfg, proto="knn"):
    return len(V(cfg, proto))


def fmt(x, d=1, sign=False):
    if x is None or not np.isfinite(x):
        return "--"
    return f"{x:+.{d}f}" if sign else f"{x:.{d}f}"


_HYPHEN_NUM = re.compile(r"(?<![\w\-$\\])-(?=\d)")


def minus(text):
    """Typeset the sign of negative numbers as a minus sign, not a hyphen ('-0.4' -> '\\ensuremath{-}0.4').
    Ranges ('1--5'), hyphenated words ('LP-5k') and math such as '$-$' are left alone."""
    return _HYPHEN_NUM.sub(r"\\ensuremath{-}", text)


def write(name, text):
    if re.search(r"(?<!\\)%", text):     # an unescaped % would silently comment out the rest of a table row
        raise ValueError(f"unescaped % in generated/{name}")
    with open(os.path.join(GEN, name), "w") as f:
        f.write("% generated by scripts/make_assets.py -- do not edit\n" + minus(text.rstrip()) + "\n")


M = {}   # macros: key -> text, used in the paper as \R{key}


def best_of(cfgs, proto):
    vals = {c: mean(c, proto) for c in cfgs}
    vals = {c: v for c, v in vals.items() if np.isfinite(v)}
    return max(vals, key=vals.get) if vals else None


# ----------------------------------------------------------------------------
# macros for every configuration and protocol
# ----------------------------------------------------------------------------
PROTOS = ["lp50", "knn", "lp5", "ft", "ftt", "ftd", "fts", "ftx", "fto", "fto0", "fm", "dist", "clf", "cdnv_lab", "cdnv_rest", "cdnv_test",
          "lp5_train", "seen_cdnv", "heldout_cdnv", "test_cdnv", "linear_acc_gray", "cka_hist", "cka_label"]
for cfg in CONFIGS:
    for proto in PROTOS:
        v = V(cfg, proto)
        if len(v) == 0:
            continue
        d = 2 if "cdnv" in proto or "cka" in proto else 1
        if d == 2 and abs(v.mean()) < 0.05:
            d = 3
        M[f"{cfg}:{proto}"] = f"{v.mean():.{d}f}"
        M[f"{cfg}:{proto}:pm"] = f"{v.mean():.{d}f}" + (rf"\,$\pm$\,{v.std(ddof=1):.{d}f}" if len(v) > 1 else "")
        M[f"{cfg}:{proto}:n"] = str(len(v))
    t = V(cfg, "time")
    if len(t):
        M[f"{cfg}:time"] = f"{t.min():.0f}"


def diffmacro(tag, a, pa, b, pb, d=1):
    x, y = mean(a, pa), mean(b, pb)
    if np.isfinite(x) and np.isfinite(y):
        M[f"d:{tag}"] = f"{abs(x - y):.{d}f}"
        M[f"ds:{tag}"] = f"{x - y:+.{d}f}"


# ----------------------------------------------------------------------------
# statistics: hierarchical bootstrap, Holm correction, TOST
# ----------------------------------------------------------------------------
PRED = {"lp50": [("{c}", "linear")], "knn": [("{c}", "knn")], "lp5": [("{c}", "linear_lab"), ("{c}", "linear_0.1")],
        "ft": [("ft_{c}", "finetune")], "ftt": [("ftsearch_{c}", "finetune_tuned")],
        "ftx": [("fttransfer_{c}", "finetune_tuned")],
        "fm": [("fixmatch_from_{c}", "classifier")], "clf": [("{c}", "classifier")]}
_Y = {}


def correct(cfg, proto, dataset="cifar10"):
    if dataset not in _Y:
        _Y[dataset] = st.test_labels(dataset)
    if proto == "fts":
        proto = "ftt" if label_free(cfg) else "ftx"
    if proto == "fto":      # seed 0 chose the recipe on its test set and is left out
        c = st.load_correct(f"ftoracle_{cfg}", "finetune_tuned", _Y[dataset], RUNS)
        return {k: v for k, v in c.items() if k != 0}
    for tmpl, key in PRED[proto]:
        if proto == "fm" and cfg.startswith("stl_"):
            tmpl = "stl_fixmatch_from_" + cfg[4:]
        c = st.load_correct(tmpl.format(c=cfg), key, _Y[dataset], RUNS)
        if c:
            return c
    return {}


N_BOOT = 100000
CACHE_PATH = os.path.join(ROOT, "results", "stats_cache.json")
CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}


def cmp(a, pa, b, pb=None, dataset="cifar10"):
    ca, cb = correct(a, pa, dataset), correct(b, pb or pa, dataset)
    if len(ca) < 2 or len(cb) < 2:
        return None
    key = f"{a}|{pa}|{sorted(ca)}|{b}|{pb or pa}|{sorted(cb)}|{N_BOOT}"
    if key not in CACHE:
        CACHE[key] = {k: (float(v) if not isinstance(v, (bool, int)) else v)
                      for k, v in st.hierarchical_bootstrap(ca, cb, n_boot=N_BOOT).items()}
    return dict(CACHE[key])


BUDGETS = [(100, "9,700"), (200, "19,400"), (400, "38,800"), (1000, "97,000")]
FAMILY = []   # (tag, group label, text of the comparison, a, pa, b, pb)
for E, steps in BUDGETS:
    s, c, u, sub = (ARMS[k].format(E=E) for k in ("simclr", "supcon", "union", "sub"))
    for proto, pname in (("lp50", "LP-50k"), ("lp5", "LP-5k"), ("ft", "FT-5k")):
        FAMILY.append((f"simclr-supcon:{proto}:e{E}", f"{pname}, {steps} steps", r"\simclr $-$ \supcon", s, proto, c, proto))
        FAMILY.append((f"simclr-union:{proto}:e{E}", f"{pname}, {steps} steps", r"\simclr $-$ \union", s, proto, u, proto))
    FAMILY.append((f"union-supcon:lp50:e{E}", f"LP-50k, {steps} steps", r"\union $-$ \supcon", u, "lp50", c, "lp50"))
    FAMILY.append((f"simclr-sub:lp50:e{E}", f"LP-50k, {steps} steps", r"\simclr $-$ \simclrsub", s, "lp50", sub, "lp50"))
    FAMILY.append((f"supcon-sub:lp50:e{E}", f"LP-50k, {steps} steps", r"\supcon $-$ \simclrsub", c, "lp50", sub, "lp50"))
for E, steps in ((10, "970"), (25, "2,425"), (50, "4,850")):
    FAMILY.append((f"simclr-supcon:lp50:e{E}", f"LP-50k, {steps} steps", r"\simclr $-$ \supcon",
                   ARMS["simclr"].format(E=E), "lp50", ARMS["supcon"].format(E=E), "lp50"))
for _p, _n in (("fts", "FT-5k (selected recipe)"), ("fto", "FT-5k (oracle recipe)")):
    FAMILY.append((f"simclr-supcon:{_p}:e200", f"{_n}, 19,400 steps", r"\simclr $-$ \supcon", "simclr_all_e200", _p,
                   "supcon_lab_f0.1_e200", _p))
    FAMILY.append((f"simclr-union:{_p}:e200", f"{_n}, 19,400 steps", r"\simclr $-$ \union", "simclr_all_e200", _p,
                   "union_f0.1_e200", _p))
FAMILY.append(("simclr-supcon:fm:e200", "FixMatch from encoder, 19,400 steps", r"\simclr $-$ \supcon", "simclr_all_e200", "fm",
               "supcon_lab_f0.1_e200", "fm"))
FAMILY.append(("simclr-union:fm:e200", "FixMatch from encoder, 19,400 steps", r"\simclr $-$ \union", "simclr_all_e200", "fm",
               "union_f0.1_e200", "fm"))
FAMILY.append(("fmb3-fmsimclr", "same total compute (about 3x the reference budget)", r"FixMatch from scratch $-$ \simclr{}+FixMatch",
               "fixmatch_b3_f0.1", "clf", "simclr_all_e200", "fm"))
FAMILY.append(("twostage-simclr:lp5:e200", "LP-5k, 19,400 steps", r"\simclr{}$\to$\supcon $-$ \simclr",
               "twostage_simclr200_supcon5", "lp5", "simclr_all_e200", "lp5"))
FAMILY.append(("simclrft-ce:e200", "label-budget pipelines, 19,400 steps", r"\simclr{}+FT $-$ CE on 5k", "simclr_all_e200", "ft",
               "ce_f0.1_e200", "clf"))
FAMILY.append(("supconft-ce:e200", "label-budget pipelines, 19,400 steps", r"\supcon{}+FT $-$ CE on 5k", "supcon_lab_f0.1_e200", "ft",
               "ce_f0.1_e200", "clf"))
# STL-10 has only the 5,000 labels: its "linear" predictions are the LP-5k probe
for E, steps in ((100, "20,500"), (200, "41,000")):
    for proto, pname in (("lp50", "LP-5k"), ("ft", "FT-5k")):
        FAMILY.append((f"stl:simclr-supcon:{proto}:e{E}", f"STL-10, {pname}, {steps} steps", r"\simclr $-$ \supcon",
                       f"stl_simclr_all_e{E}", proto, f"stl_supcon_lab_e{E}", proto, "stl10"))
        FAMILY.append((f"stl:simclr-union:{proto}:e{E}", f"STL-10, {pname}, {steps} steps", r"\simclr $-$ \union",
                       f"stl_simclr_all_e{E}", proto, f"stl_union_e{E}", proto, "stl10"))
    FAMILY.append((f"stl:simclr-sub:lp50:e{E}", f"STL-10, LP-5k, {steps} steps", r"\simclr $-$ \simclrsub",
                   f"stl_simclr_all_e{E}", "lp50", f"stl_simclr_sub_e{E}", "lp50", "stl10"))
for proto, pname in (("fm", "FixMatch from encoder"), ("fto", "FT-5k (oracle recipe)")):
    FAMILY.append((f"stl:simclr-supcon:{proto}:e200", f"STL-10, {pname}, 41,000 steps", r"\simclr $-$ \supcon",
                   "stl_simclr_all_e200", proto, "stl_supcon_lab_e200", proto, "stl10"))
    FAMILY.append((f"stl:simclr-union:{proto}:e200", f"STL-10, {pname}, 41,000 steps", r"\simclr $-$ \union",
                   "stl_simclr_all_e200", proto, "stl_union_e200", proto, "stl10"))
STATS = []
for tag, group, text, a, pa, b, pb, *ds in FAMILY:
    r = cmp(a, pa, b, pb, dataset=ds[0] if ds else "cifar10")
    if r is not None:
        STATS.append((tag, group, text, r))
json.dump(CACHE, open(CACHE_PATH, "w"), indent=0)
_first = {}
for i, (_t, g, *_r) in enumerate(STATS):
    _first.setdefault(g, i)
STATS.sort(key=lambda x: _first[x[1]])
if STATS:
    adj = st.holm([r["p"] for *_x, r in STATS])
    for (tag, group, text, r), pa_ in zip(STATS, adj):
        r["p_holm"] = float(pa_)

        def pf(p):
            return "$<$0.001" if p < 0.001 else f"{p:.3f}"
        M[f"cmp:{tag}:diff"] = f"{abs(r['diff']):.1f}"
        M[f"cmp:{tag}:sdiff"] = f"{r['diff']:+.1f}"
        M[f"cmp:{tag}:ci"] = f"[{r['ci_lo']:+.1f}, {r['ci_hi']:+.1f}]"
        M[f"cmp:{tag}:p"] = pf(r["p_holm"])
        M[f"cmp:{tag}:tost"] = pf(r["p_tost"])
        # complete expressions for running text: "$p<0.001$" or "$p=0.022$"
        M[f"cmp:{tag}:p:expr"] = "$p<0.001$" if r["p_holm"] < 0.001 else f"$p={r['p_holm']:.3f}$"
        M[f"cmp:{tag}:tost:expr"] = "$p<0.001$" if r["p_tost"] < 0.001 else f"$p={r['p_tost']:.3f}$"
    rows, last = [], None
    for tag, group, text, r in STATS:
        if group != last:
            rows.append(rf"\multicolumn{{6}}{{l}}{{\emph{{{group}}}}} \\")
            last = group
        eq = "yes" if r["p_tost"] < 0.05 else "no"
        rows.append(f"\\quad {text} & {r['n_a']}/{r['n_b']} & {r['diff']:+.1f} & [{r['ci_lo']:+.1f}, {r['ci_hi']:+.1f}] & "
                    f"{M[f'cmp:{tag}:p']} & {eq} \\\\")
    # rows only: the table is longer than a page, so appendix.tex wraps them in a longtable with repeated headers
    write("tab_stats_rows.tex", "\n".join(rows))
    M["stats:ntests"] = str(len(STATS))

# ----------------------------------------------------------------------------
# Table: main results on CIFAR-10 (four arms x budgets x protocols)
# ----------------------------------------------------------------------------
rows = []
for E, steps in BUDGETS:
    cfgs = {k: ARMS[k].format(E=E) for k in ("sub", "simclr", "supcon", "union")}
    if not any(nseeds(c) for c in cfgs.values()):
        continue
    rows.append(rf"\multicolumn{{6}}{{l}}{{\emph{{{steps} steps}}}} \\")
    best = {p: best_of(cfgs.values(), p) for p in ("lp50", "knn", "lp5", "ft")}
    for k, c in cfgs.items():
        if not nseeds(c):
            continue
        rows.append(f"\\quad {ARM_TEX[k]} & {nseeds(c)} & " + " & ".join(MS(c, p, bold=(best[p] == c)) for p in ("lp50", "knn", "lp5", "ft")) + r" \\")
write("tab_main.tex", "\\begin{tabular}{lccccc}\n\\toprule\n & & \\multicolumn{2}{c}{oracle labels (50k)} & "
      "\\multicolumn{2}{c}{label budget (5k)} \\\\\n\\cmidrule(lr){3-4}\\cmidrule(lr){5-6}\n"
      "method & seeds & LP-50k & $k$NN-50k & LP-5k & FT-5k \\\\\n\\midrule\n" + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")

# all budgets (appendix)
rows = []
for E in (10, 25, 50, 100, 200, 400, 1000):
    line = [f"{97 * E:,}"]
    for k in ("simclr", "sub", "supcon", "union"):
        c = ARMS[k].format(E=E)
        line += [MS(c, "lp50"), MS(c, "lp5")]
    rows.append(" & ".join(line) + r" \\")
write("tab_budgets.tex", "\\begin{tabular}{rcccccccc}\n\\toprule\n & \\multicolumn{2}{c}{\\simclr} & \\multicolumn{2}{c}{\\simclrsub} & "
      "\\multicolumn{2}{c}{\\supcon} & \\multicolumn{2}{c}{\\union} \\\\\n"
      "\\cmidrule(lr){2-3}\\cmidrule(lr){4-5}\\cmidrule(lr){6-7}\\cmidrule(lr){8-9}\n"
      "steps & LP-50k & LP-5k & LP-50k & LP-5k & LP-50k & LP-5k & LP-50k & LP-5k \\\\\n\\midrule\n" + "\n".join(rows)
      + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Figure 1: accuracy against the number of steps, three protocols
# ----------------------------------------------------------------------------
ES = (10, 25, 50, 100, 200, 400, 1000)
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.3), sharey=True)
for ax, (proto, title) in zip(axes, (("lp50", "(a) LP-50k (oracle labels)"), ("lp5", "(b) LP-5k"), ("ft", "(c) FT-5k"))):
    for k, tmpl in ARMS.items():
        xs, ys, es = [], [], []
        for E in ES:
            v = V(tmpl.format(E=E), proto)
            if len(v):
                xs.append(97 * E), ys.append(v.mean()), es.append(v.std(ddof=1) if len(v) > 1 else 0)
        if xs:
            col, mk, ls, lab = STYLE[k]
            ax.errorbar(xs, ys, yerr=es, color=col, marker=mk, ls=ls, label=lab, capsize=2, elinewidth=0.8)
    ax.set_xscale("log")
    ax.set_xticks([970, 2425, 4850, 9700, 19400, 38800, 97000])
    ax.set_xticklabels(["970", "2.4k", "4.9k", "9.7k", "19.4k", "38.8k", "97k"], rotation=40)
    ax.minorticks_off()
    ax.set_xlabel("gradient steps")
    ax.set_title(title, loc="left")
axes[0].set_ylabel("CIFAR-10 test accuracy (%)")
axes[0].set_ylim(60, 93)
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=4, frameon=False, bbox_to_anchor=(0.5, -0.22), columnspacing=1.2, handlelength=2.6)
fig.savefig(os.path.join(FIG, "fig_budget.pdf"))
plt.close(fig)

# ----------------------------------------------------------------------------
# Figure 2: which method wins, as a function of steps and label fraction
# ----------------------------------------------------------------------------
FRS = (0.01, 0.05, 0.1, 0.2, 0.5, 1.0)
GES = (25, 100, 400)


def frac_cfg(kind, f, E):
    if kind == "simclr":
        return f"simclr_all_e{E}"
    return f"{'supcon_lab' if kind == 'supcon' else 'union'}_f{f}_e{E}"


def frac_ft(kind, f, E):
    """Fine-tuning on the f labels (FT-f), mean over the seeds that have a fine-tuned run."""
    if f >= 1.0:
        return float("nan")      # fine-tuning with all labels was not run
    if f == 0.1:
        return mean(frac_cfg(kind, f, E), "ft")
    name = f"ft_simclr_all_e{E}_f{f}_s0" if kind == "simclr" else f"ft_{frac_cfg(kind, f, E)}_s0"
    r = RUN.get(name)
    return float(r["metrics"]["finetune_acc"]) if r else float("nan")


def frac_val(kind, f, E, proto):
    """proto: 'oracle' (LP with all labels), 'budget' (LP with the f labels) or 'ft' (fine-tuning on the f labels)."""
    cfg = frac_cfg(kind, f, E)
    if proto == "ft":
        return frac_ft(kind, f, E)
    if proto == "oracle" or f >= 1.0:
        return mean(cfg, "lp50")
    v = V(cfg, "linear_lab_acc") if kind != "simclr" else V(cfg, f"lpf:{f}")
    if len(v) == 0:
        v = V(cfg, f"lpf:{f}")
    return float(v.mean()) if len(v) else float("nan")


def frac_diff(kind, f, E, proto):
    if kind == "union" and f >= 1.0:
        return float("nan")      # with all labels SupCon+U is SupCon
    return frac_val(kind, f, E, proto) - frac_val("simclr", f, E, proto)


def break_even(kind, E, proto):
    """Where the label-using method starts to stay ahead of SimCLR, as a phrase: 'between a\\% and b\\%' (the
    two grid fractions around the last sign change), or a bound when there is no change inside the grid."""
    fr = [f for f in FRS if np.isfinite(frac_diff(kind, f, E, proto))]
    d = [frac_diff(kind, f, E, proto) for f in fr]
    if all(x > 0 for x in d):
        return f"below {100 * fr[0]:g}\\%"
    j = max(i for i, x in enumerate(d) if x <= 0)
    if j + 1 == len(fr):
        return f"above {100 * fr[-1]:g}\\%"
    return f"between {100 * fr[j]:g}\\% and {100 * fr[j + 1]:g}\\%"


fig, axes = plt.subplots(2, 3, figsize=(7.2, 3.3), sharey=True, sharex=True, gridspec_kw={"hspace": 0.34, "wspace": 0.05})
panels = [("supcon", "oracle", r"(a) SupCon $-$ SimCLR, LP-all"),
          ("supcon", "budget", r"(b) SupCon $-$ SimCLR, LP-$f$"),
          ("supcon", "ft", r"(c) SupCon $-$ SimCLR, FT-$f$"),
          ("union", "oracle", r"(d) SupCon+U $-$ SimCLR, LP-all"),
          ("union", "budget", r"(e) SupCon+U $-$ SimCLR, LP-$f$"),
          ("union", "ft", r"(f) SupCon+U $-$ SimCLR, FT-$f$")]
rows_frac = []
for ax, (kind, proto, title) in zip(axes.ravel(), panels):
    G = np.full((len(GES), len(FRS)), np.nan)
    for i, E in enumerate(GES):
        for j, f in enumerate(FRS):
            G[i, j] = frac_diff(kind, f, E, proto)
    im = ax.imshow(G, cmap="RdBu_r", vmin=-12, vmax=12, aspect="auto", origin="lower")
    for i in range(len(GES)):
        for j in range(len(FRS)):
            if np.isfinite(G[i, j]):
                ax.text(j, i, f"{G[i, j]:+.1f}".replace("-", "\u2212"), ha="center", va="center", fontsize=5.8,
                        color="white" if abs(G[i, j]) > 7 else "black")
    ax.set_xticks(range(len(FRS)))
    ax.set_xticklabels([f"{100 * f:g}" for f in FRS])
    ax.set_yticks(range(len(GES)))
    ax.set_yticklabels([f"{97 * E:,}" for E in GES])
    ax.set_title(title, loc="left", fontsize=7)
    ax.tick_params(labelsize=6.5)
    ax.grid(False)
for ax in axes[1]:
    ax.set_xlabel("labeled fraction $f$ (%)")
for ax in axes[:, 0]:
    ax.set_ylabel("gradient steps")
cb = fig.colorbar(im, ax=axes, fraction=0.02, pad=0.015, extend="both")
cb.set_label("difference (points)")
fig.savefig(os.path.join(FIG, "fig_wins.pdf"))
plt.close(fig)
rows = []
for E in GES:
    for f in FRS:
        line = [f"{97 * E:,}", f"{100 * f:g}\\%"]
        for kind in ("simclr", "supcon", "union"):
            for proto in ("oracle", "budget", "ft"):
                x = float("nan") if (kind == "union" and f >= 1) else frac_val(kind, f, E, proto)
                line.append(fmt(x))
        rows.append(" & ".join(line) + r" \\")
    rows.append(r"\addlinespace")
write("tab_fraction.tex", "\\begin{tabular}{rrccccccccc}\n\\toprule\n & & \\multicolumn{3}{c}{\\simclr} & \\multicolumn{3}{c}{\\supcon} & "
      "\\multicolumn{3}{c}{\\union} \\\\\n\\cmidrule(lr){3-5}\\cmidrule(lr){6-8}\\cmidrule(lr){9-11}\n"
      "steps & $f$ & LP-all & LP-$f$ & FT-$f$ & LP-all & LP-$f$ & FT-$f$ & LP-all & LP-$f$ & FT-$f$ \\\\\n\\midrule\n"
      + "\n".join(rows[:-1]) + "\n\\bottomrule\n\\end{tabular}")
# macros for the text: differences (label-using minus SimCLR) and break-even fractions
for kind in ("supcon", "union"):
    for E in GES:
        for proto, tag in (("oracle", "lpall"), ("budget", "lpf"), ("ft", "ftf")):
            M[f"grid:{kind}:{tag}:e{E}:be"] = break_even(kind, E, proto)
            for f in FRS:
                x = frac_diff(kind, f, E, proto)
                if np.isfinite(x):
                    M[f"grid:{kind}:{tag}:e{E}:f{f:g}"] = f"{x:+.1f}"
                    M[f"grid:{kind}:{tag}:e{E}:f{f:g}:abs"] = f"{abs(x):.1f}"
            xs = [frac_diff(kind, f, E, proto) for f in FRS if 0.05 <= f < 1.0]
            xs = [x for x in xs if np.isfinite(x)]
            if xs:      # largest lead of the label-using method from 5% to 50% of the labels
                M[f"grid:{kind}:{tag}:e{E}:max"] = f"{max(xs):.1f}"
        for f in FRS:
            for proto, tag in (("oracle", "lpall"), ("budget", "lpf"), ("ft", "ftf")):
                x = frac_val(kind, f, E, proto)
                if np.isfinite(x):
                    M[f"grid:{kind}:{tag}:e{E}:f{f:g}:acc"] = f"{x:.1f}"
for E in GES:
    for f in FRS:
        for proto, tag in (("oracle", "lpall"), ("budget", "lpf"), ("ft", "ftf")):
            x = frac_val("simclr", f, E, proto)
            if np.isfinite(x):
                M[f"grid:simclr:{tag}:e{E}:f{f:g}:acc"] = f"{x:.1f}"

# ----------------------------------------------------------------------------
# Table: other datasets and architectures
# ----------------------------------------------------------------------------
BLOCKS = [("CIFAR-100, 10\\% labels (5,000), ResNet-18", "c100_", "_f0.1", [(100, "9,700"), (200, "19,400")]),
          ("STL-10, 5,000 labels + 100,000 unlabeled images, 96 px, ResNet-18", "stl_", "", [(100, "20,500"), (200, "41,000")]),
          ("Tiny-ImageNet, 10\\% labels (10,000), 64 px, ResNet-18", "tin_", "_f0.1", [(100, "19,500"), (200, "39,000")]),
          ("CIFAR-10, 10\\% labels, ResNet-50", "r50_", "_f0.1", [(100, "9,700"), (200, "19,400")])]
rows = []
for title, pre, suf, buds in BLOCKS:
    if not any(nseeds(f"{pre}{a}{suf if a != 'simclr_all' else ''}_e{E}") for E, _ in buds for a in ("simclr_all", "supcon_lab", "union")):
        continue      # no finished run for this dataset yet
    rows.append(rf"\multicolumn{{6}}{{l}}{{\emph{{{title}}}}} \\")
    for E, steps in buds:
        cfgs = {"sub": f"{pre}simclr_sub{suf}_e{E}", "simclr": f"{pre}simclr_all_e{E}", "supcon": f"{pre}supcon_lab{suf}_e{E}",
                "union": f"{pre}union{suf}_e{E}"}
        best = {p: best_of(cfgs.values(), p) for p in ("lp50", "lp5", "ft")}
        for k, c in cfgs.items():
            if nseeds(c):
                rows.append(f"\\quad {ARM_TEX[k]} & {steps} & {nseeds(c)} & "
                            + " & ".join(MS(c, p, bold=(best[p] == c)) for p in ("lp50", "lp5", "ft")) + r" \\")
    rows.append(r"\addlinespace")
write("tab_datasets.tex", "\\begin{tabular}{lrcccc}\n\\toprule\nmethod & steps & seeds & LP-all & LP-budget & FT-budget \\\\\n\\midrule\n"
      + "\n".join(rows[:-1]) + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Table: the protocol decides the ranking (encoders at 19,400 steps)
# ----------------------------------------------------------------------------
ENC = [(r"\simclr", "simclr_all_e200"), ("BYOL", "byol_all_e200"), (r"\simclrsub", "simclr_sub_f0.1_e200"),
       (r"\supcon", "supcon_lab_f0.1_e200"), (r"\union", "union_f0.1_e200"),
       (r"\simclr{}$\to$\supcon", "twostage_simclr200_supcon5")]
PCOLS = ("lp50", "knn", "lp5", "ft", "fts", "fto", "fm")


def ranks(cfgs, proto):
    vals = {c: mean(c, proto) for c in cfgs}
    order = sorted([c for c in vals if np.isfinite(vals[c])], key=lambda c: -vals[c])
    return {c: i + 1 for i, c in enumerate(order)}


RK = {p: ranks([c for _, c in ENC], p) for p in PCOLS}
rows = []
for name, c in ENC:
    cells = []
    for p in PCOLS:
        s = MS(c, p)
        if c in RK[p]:
            s += rf"\,\textsuperscript{{({RK[p][c]})}}"
        cells.append(s)
    rows.append(f"{name} & " + " & ".join(cells) + r" \\")
write("tab_protocol.tex", "\\begin{tabular}{lccccccc}\n\\toprule\n & \\multicolumn{2}{c}{oracle labels} & \\multicolumn{5}{c}{label budget (5,000 labels)} \\\\\n"
      "\\cmidrule(lr){2-3}\\cmidrule(lr){4-8}\n & & & & \\multicolumn{3}{c}{FT-5k, by recipe} & \\\\\n\\cmidrule(lr){5-7}\n"
      "encoder & LP-50k & $k$NN-50k & LP-5k & default & selected & oracle & FixMatch-init \\\\\n\\midrule\n"
      + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")
# rank agreement between the oracle probe and each label-budget protocol
main_enc = [c for _, c in ENC[:5]]
for p in ("lp5", "ft", "fts", "fto", "fm"):
    a = [mean(c, "lp50") for c in main_enc]
    b = [mean(c, p) for c in main_enc]
    ok = [i for i in range(len(a)) if np.isfinite(a[i]) and np.isfinite(b[i])]
    if len(ok) >= 4:
        M[f"kendall:{p}"] = f"{sps.kendalltau([a[i] for i in ok], [b[i] for i in ok]).statistic:.2f}"

# fine-tuning search: label-split selection and oracle selection side by side
REGN = {"none": "none", "ls": "label smoothing", "mixup": "mixup"}


def recipe(b):
    return f"{'LP' if b['head'] == 'lp' else 'random'} head, lr {b['lr']:g}, {b['epochs']} passes, {REGN[b['reg']]}"


rows = []
for name, c in ENC:
    a, o = RUN.get(f"ftsearch_{c}_s0"), RUN.get(f"ftoracle_{c}_s0")
    if a and a.get("grid"):
        g = a["grid"]
        vals = [x["val_acc"] for x in g]
        M[f"ftgrid:{c}:valmin"], M[f"ftgrid:{c}:valmax"] = f"{min(vals):.1f}", f"{max(vals):.1f}"
        M[f"ftgrid:{c}:valrange"] = f"{max(vals) - min(vals):.1f}"
        rows.append(f"{name} & label split & {recipe(a['best'])} & {min(vals):.1f}--{max(vals):.1f} & {MS(c, 'ftt')} \\\\")
    if o and o.get("grid"):
        g = o["grid"]
        tv = [x["test_acc"] for x in g]
        M[f"ftgrid:{c}:testmin"], M[f"ftgrid:{c}:testmax"] = f"{min(tv):.1f}", f"{max(tv):.1f}"
        for head in ("lp", "random"):
            M[f"ftgrid:{c}:{head}"] = f"{max(x['test_acc'] for x in g if x['head'] == head):.1f}"
        for reg in ("none", "ls", "mixup"):
            M[f"ftgrid:{c}:{reg}"] = f"{max(x['test_acc'] for x in g if x['reg'] == reg):.1f}"
        M[f"ftgrid:{c}:recipe"] = recipe(o["best"])
        rows.append(f"{name} & test set (oracle) & {recipe(o['best'])} & {min(tv):.1f}--{max(tv):.1f} & {MS(c, 'fto')} \\\\")
    if nseeds("fttransfer_" + c, "finetune_tuned_acc") or len(V(c, "ftx")):
        rows.append(f"{name} & recipe of \\simclr & -- & -- & {MS(c, 'ftx')} \\\\")
    rows.append(r"\addlinespace")
write("tab_ftsearch.tex", "\\begin{tabular}{lllcc}\n\\toprule\nencoder & recipe chosen on & chosen recipe & range of the selection score over the grid & "
      "test accuracy \\\\\n\\midrule\n" + ("\n".join(rows[:-1]) if rows else r"\multicolumn{5}{c}{--} \\")
      + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Table: where the labels should go (pipelines, CIFAR-10, 5,000 labels)
# ----------------------------------------------------------------------------
PIPE = [
    ("\\emph{labels inside the contrastive loss from the first step}", None, None, None),
    (r"\supcon, then FT-5k", "inside the loss", "supcon_lab_f0.1_e200", "ft"),
    (r"\supcon, then FT-5k (oracle recipe)", "inside the loss", "supcon_lab_f0.1_e200", "fto"),
    (r"\union, then FT-5k", "inside the loss", "union_f0.1_e200", "ft"),
    (r"\union, then FT-5k (oracle recipe)", "inside the loss", "union_f0.1_e200", "fto"),
    ("\\emph{labels after label-free pretraining}", None, None, None),
    (r"\simclr, then LP-5k", "after", "simclr_all_e200", "lp5"),
    (r"\simclr, then FT-5k", "after", "simclr_all_e200", "ft"),
    (r"\simclr, then FT-5k (selected recipe)", "after", "simclr_all_e200", "ftt"),
    (r"\simclr, then FT-5k (oracle recipe)", "after", "simclr_all_e200", "fto"),
    (r"BYOL, then FT-5k", "after", "byol_all_e200", "ft"),
    (r"\simclr, then \supcon on the 5k (485 steps), then LP-5k", "after, contrastive", "twostage_simclr200_supcon5", "lp5"),
    (r"\simclr, then \supcon on the 5k (485 steps), then FT-5k", "after, contrastive", "twostage_simclr200_supcon5", "ft"),
    (r"\simclr, then FT-5k, then self-distillation", "after", "simclr_all_e200", "dist"),
    ("\\emph{labels in a semi-supervised objective}", None, None, None),
    (r"FixMatch from scratch", "pseudo-labels", "fixmatch_f0.1", "clf"),
    (r"FixMatch from scratch, 3$\times$ budget", "pseudo-labels", "fixmatch_b3_f0.1", "clf"),
    (r"MixMatch from scratch", "pseudo-labels", "mixmatch_f0.1", "clf"),
    (r"\simclr, then FixMatch", "after, pseudo-labels", "simclr_all_e200", "fm"),
    (r"BYOL, then FixMatch", "after, pseudo-labels", "byol_all_e200", "fm"),
    (r"\supcon, then FixMatch", "both", "supcon_lab_f0.1_e200", "fm"),
    (r"\union, then FixMatch", "both", "union_f0.1_e200", "fm"),
    (r"\simclr, then \supcon (2,425 steps), then FixMatch", "both", "twostage_simclr200_supcon25", "fm"),
    ("\\emph{references}", None, None, None),
    (r"cross-entropy on the 5k, 9,700 steps", "supervised only", "ce_f0.1_e100", "clf"),
    (r"cross-entropy on the 5k, 19,400 steps", "supervised only", "ce_f0.1_e200", "clf"),
    (r"cross-entropy on all 50k labels", "all labels", "ce_f1.0_e100", "clf"),
]
rows = []
for name, where, c, p in PIPE:
    if c is None:
        rows.append(rf"\multicolumn{{4}}{{l}}{{{name}}} \\")
    else:
        rows.append(f"\\quad {name} & {where} & {len(V(c, p)) or '--'} & {MS(c, p)} \\\\")
write("tab_where.tex", "\\begin{tabular}{llcc}\n\\toprule\npipeline & where the labels enter & seeds & test accuracy \\\\\n\\midrule\n"
      + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Table + figure: geometry on the labeled images, the other training images and the test set
# ----------------------------------------------------------------------------
GEO = [(r"\simclr", "simclr_all_e200", "seen, no label"), (r"\simclrsub", "simclr_sub_f0.1_e200", "not seen"),
       (r"\supcon", "supcon_lab_f0.1_e200", "not seen"), (r"\union", "union_f0.1_e200", "seen, no label"),
       ("cross-entropy on 5k", "ce_f0.1_e200", "not seen"), (r"\simclr{}$\to$\supcon", "twostage_simclr200_supcon5", "seen, no label")]
rows = []
for name, c, rest in GEO:
    a, b, t = (V(c, k) for k in ("cdnv_lab", "cdnv_rest", "cdnv_test"))
    ratio = f"{(t / a).mean():.1f}" if len(a) and len(a) == len(t) and (a > 1e-6).all() else ("$>$100" if len(a) and len(t) else "--")
    rows.append(f"{name} & {MS(c, 'cdnv_lab', 2)} & {MS(c, 'cdnv_rest', 2)} & {MS(c, 'cdnv_test', 2)} & {ratio} & "
                f"{MS(c, 'lp5_train')} & {MS(c, 'lp5')} \\\\")
    if ratio not in ("--",):
        M[f"{c}:cdnv_ratio"] = ratio
write("tab_geometry.tex", "\\begin{tabular}{lcccccc}\n\\toprule\n & \\multicolumn{3}{c}{CDNV ($\\downarrow$ tighter classes)} & & \\multicolumn{2}{c}{LP-5k accuracy} \\\\\n"
      "\\cmidrule(lr){2-4}\\cmidrule(lr){6-7}\nencoder (19,400 steps) & labeled 5k & other 45k & test 10k & test / labeled & on its 5k & test \\\\\n\\midrule\n"
      + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.5), gridspec_kw={"width_ratios": [1.35, 1], "wspace": 0.28})
ax = axes[0]
labs = ["SimCLR", "SimCLR\n-sub", "SupCon", "SupCon\n+U", "CE\non 5k", "SimCLR\n→SupCon"]
w = 0.26
for j, (key, lab, col, hatch) in enumerate((("cdnv_lab", "labeled 5k", "#555555", None), ("cdnv_rest", "other 45k training images", "#aaaaaa", "//"),
                                            ("cdnv_test", "test images", "#e6e6e6", None))):
    ms_ = [(V(c, key).mean() if len(V(c, key)) else np.nan) for _, c, _ in GEO]
    sd_ = [(V(c, key).std(ddof=1) if len(V(c, key)) > 1 else 0) for _, c, _ in GEO]
    ms_ = [max(m_, 2e-3) if np.isfinite(m_) else m_ for m_ in ms_]
    ax.bar(np.arange(len(GEO)) + (j - 1) * w, ms_, w, yerr=sd_, label=lab, color=col, edgecolor="black", linewidth=0.4,
           hatch=hatch, error_kw={"elinewidth": 0.7, "capsize": 1.5})
ax.set_yscale("log")
ax.set_xticks(range(len(GEO)))
ax.set_xticklabels(labs, fontsize=6.5)
ax.set_ylabel("CDNV (log scale)")
ax.set_title("(a) CDNV on three sets of images", loc="left")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=6.5, columnspacing=1.0)
ax.grid(axis="x", visible=False)
ax = axes[1]
for k, cands, a_key in (("supcon", ["supcon_lab_f0.1_e1000", "supcon_lab_f0.1_e400", "supcon_lab_f0.1_e200"], "pool_cdnv"),
                        ("union", ["union_f0.1_e1000", "union_f0.1_e400", "union_f0.1_e200"], "labeled_cdnv"),
                        ("simclr", ["simclr_all_e1000", "simclr_all_e400", "simclr_all_e200"], "pool_cdnv")):
    for c in cands:
        curves = [RUN[f"{c}_s{s}"].get("geom_curve") for s in seeds(c)]
        curves = [g for g in curves if g and a_key in g[0]]
        if curves:
            xs = [p["step"] for p in curves[0]]
            R = np.array([[p["test_cdnv"] / max(p[a_key], 1e-9) for p in g] for g in curves if len(g) == len(xs)])
            col, mk, ls, lab = STYLE[k]
            ax.errorbar(xs, R.mean(0), yerr=R.std(0, ddof=1) if len(R) > 1 else None, color=col, marker=mk, ls=ls,
                        label=lab.split(" (")[0], capsize=2, elinewidth=0.8, markersize=3)
            M[f"geomcurve:{k}"] = c
            M[f"geomcurve:{k}:first"], M[f"geomcurve:{k}:last"] = f"{R.mean(0)[0]:.1f}", f"{R.mean(0)[-1]:.1f}"
            M[f"geomcurve:{k}:firststep"], M[f"geomcurve:{k}:laststep"] = f"{xs[0]:,}", f"{xs[-1]:,}"
            M[f"geomcurve:{k}:n"] = str(len(R))
            break
ax.set_xscale("log")
ax.minorticks_off()
tk = [t for t in (2425, 4850, 9700, 19400, 38800, 97000) if ax.get_xlim()[0] <= t <= ax.get_xlim()[1]]
ax.set_xticks(tk)
ax.set_xticklabels([f"{t / 1000:.1f}k".replace(".0k", "k") for t in tk])
ax.axhline(1, color="black", lw=0.6, ls=":")
ax.set_xlabel("gradient steps")
ax.set_ylabel("CDNV(test) / CDNV(labeled)")
ax.set_title("(b) the gap during training", loc="left")
ax.legend(frameon=False, fontsize=6.5)
fig.savefig(os.path.join(FIG, "fig_geometry.pdf"))
plt.close(fig)

# ----------------------------------------------------------------------------
# Coverage intervention: pools of fixed size whose coverage is varied
# ----------------------------------------------------------------------------
DIV = {}
dj = os.path.join(ROOT, "results", "diversity.json")
if os.path.exists(dj):
    DIV = json.load(open(dj))
KINDS = [("random", "random", ""), ("mixkcenter25", "25% k-center", "mixkcenter25_"), ("mixkcenter50", "50% k-center", "mixkcenter50_"),
         ("mixkcenter75", "75% k-center", "mixkcenter75_"), ("kcenter", "k-center", "kcenter_"),
         ("mixcluster5025", "25% redundant", "mixcluster5025_"), ("mixcluster5050", "50% redundant", "mixcluster5050_"),
         ("mixcluster5075", "75% redundant", "mixcluster5075_"), ("cluster10", "redundant (k=10)", "cluster10_"),
         ("cluster50", "redundant (k=50)", "cluster50_")]


def pool_stat(space, kind, seed, key):
    try:
        return DIV[space]["pools"][f"{kind}_f0.1_s{seed}"][key]
    except KeyError:
        return None


PTS = []   # one point per (kind, seed): pool statistics and accuracies
for kind, label, infix in KINDS:
    for s in range(5):
        row = {"kind": kind, "label": label, "seed": s}
        for space in ("simclr", "supervised", "pixel"):
            for key in ("radius", "spread"):
                row[f"{key}_{space}"] = pool_stat(space, kind, s, key)
        row["supcon"] = metric(f"supcon_lab_{infix}f0.1_e100_s{s}", "lp50")
        row["supcon_lp5"] = metric(f"supcon_lab_{infix}f0.1_e100_s{s}", "lp5")
        row["sub"] = metric(f"simclr_sub_{infix}f0.1_e100_s{s}", "lp50")
        if row["radius_simclr"] is not None and (row["supcon"] is not None or row["sub"] is not None):
            PTS.append(row)


def agg(kind, key):
    v = [p[key] for p in PTS if p["kind"] == kind and p.get(key) is not None]
    return (float(np.mean(v)), float(np.std(v, ddof=1)) if len(v) > 1 else 0.0, len(v)) if v else None


rows = []
for kind, label, _ in KINDS:
    cells = []
    for key, d in (("radius_simclr", 3), ("radius_supervised", 3), ("spread_simclr", 3), ("supcon", 1), ("supcon_lp5", 1), ("sub", 1)):
        a = agg(kind, key)
        cells.append("--" if a is None else (f"{a[0]:.{d}f}" + (rf"{{\scriptsize$\,\pm\,${a[1]:.{d}f}}}" if a[2] > 1 and d == 1 else "")))
    n = agg(kind, "supcon")
    if any(c != "--" for c in cells[3:]):
        rows.append(f"{label.replace('%', chr(92) + '%')} & {n[2] if n else '--'} & " + " & ".join(cells) + r" \\")
write("tab_coverage.tex", "\\begin{tabular}{lccccccc}\n\\toprule\n & & \\multicolumn{2}{c}{coverage radius $\\downarrow$} & spread & "
      "\\multicolumn{2}{c}{\\supcon on the pool} & \\simclr on the pool \\\\\n\\cmidrule(lr){3-4}\\cmidrule(lr){6-7}\n"
      "pool (5,000 images) & seeds & \\simclr space & supervised space & & LP-50k & LP-5k & LP-50k \\\\\n\\midrule\n"
      + ("\n".join(rows) if rows else r"\multicolumn{8}{c}{--} \\") + "\n\\bottomrule\n\\end{tabular}")


def spearman(xk, yk, pts):
    x = [p[xk] for p in pts if p.get(xk) is not None and p.get(yk) is not None]
    y = [p[yk] for p in pts if p.get(xk) is not None and p.get(yk) is not None]
    if len(x) < 5:
        return None
    r = sps.spearmanr(x, y)
    return r.statistic, r.pvalue, len(x)


for subset, pts in (("all", PTS), ("nokc", [p for p in PTS if "kcenter" not in p["kind"]]),
                    ("red", [p for p in PTS if "kcenter" not in p["kind"]]),
                    ("kc", [p for p in PTS if "cluster" not in p["kind"]])):
    for xk in ("radius_simclr", "radius_supervised", "spread_simclr"):
        for yk in ("supcon", "sub", "supcon_lp5"):
            r = spearman(xk, yk, pts)
            if r:
                M[f"rho:{subset}:{xk}:{yk}"] = f"{r[0]:+.2f}"
                M[f"rho:{subset}:{xk}:{yk}:p"] = "$<$0.001" if r[1] < 0.001 else f"{r[1]:.3f}"
                M[f"rho:{subset}:{xk}:{yk}:n"] = str(r[2])

# partial rank correlation between coverage radius and accuracy across pool SIZES, controlling for log n
if DIV:
    xs, ys, ns = [], [], []
    for f in (0.01, 0.05, 0.1, 0.2):
        for s in range(5):
            rad = None
            try:
                rad = DIV["simclr"]["pools"][f"random_f{f}_s{s}"]["radius"]
            except KeyError:
                pass
            acc = metric(f"supcon_lab_f{f}_e100_s{s}", "lp50")
            if rad is not None and acc is not None:
                xs.append(rad), ys.append(acc), ns.append(np.log(f))
    for p in PTS:
        if p["kind"] != "random" and p["supcon"] is not None:
            xs.append(p["radius_simclr"]), ys.append(p["supcon"]), ns.append(np.log(0.1))
    if len(xs) >= 8:
        rx, ry, rn = (sps.rankdata(v) for v in (xs, ys, ns))
        cxy, cxn, cyn = np.corrcoef(rx, ry)[0, 1], np.corrcoef(rx, rn)[0, 1], np.corrcoef(ry, rn)[0, 1]
        M["rho:partial"] = f"{(cxy - cxn * cyn) / np.sqrt((1 - cxn ** 2) * (1 - cyn ** 2)):+.2f}"
        M["rho:partial:raw"] = f"{cxy:+.2f}"
        M["rho:partial:n"] = str(len(xs))

fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.6), sharex=True)
fam = {"random": ("#000000", "o"), "kcenter": ("#009E73", "^"), "cluster": ("#D55E00", "s")}
for ax, (yk, title) in zip(axes, (("supcon", "(a) SupCon trained on the pool"), ("sub", "(b) SimCLR trained on the pool"))):
    for kind, label, _ in KINDS:
        a, r = agg(kind, yk), agg(kind, "radius_simclr")
        if a is None or r is None:
            continue
        col, mk = fam["kcenter" if "kcenter" in kind else "cluster" if "cluster" in kind else "random"]
        ax.errorbar(r[0], a[0], yerr=a[1], xerr=r[1], color=col, marker=mk, capsize=2, elinewidth=0.8, ls="none")
        ax.annotate(label, (r[0], a[0]), textcoords="offset points", xytext=(4, 3), fontsize=6, color=col)
    ax.set_xlabel("coverage radius of the pool (larger = worse coverage)")
    ax.set_title(title, loc="left")
axes[0].set_ylabel("LP-50k accuracy (%)")
fig.savefig(os.path.join(FIG, "fig_coverage.pdf"))
plt.close(fig)

# ----------------------------------------------------------------------------
# Table: labels make colour jitter less necessary
# ----------------------------------------------------------------------------
rows = []
for name, full, nojit in ((r"\simclr", "simclr_all_e100", "simclr_exp6_e100"), (r"\supcon", "supcon_lab_f0.1_e100", "supcon_lab_f0.1_exp6_e100"),
                          (r"\union", "union_f0.1_e100", "union_f0.1_exp6_e100")):
    common = sorted(set(seeds(full)) & set(seeds(nojit)))
    a, b = V(full, "lp50", common), V(nojit, "lp50", common)
    if len(a) and len(a) == len(b):
        dlt_ = b - a
        M[f"jitter:{full}"] = f"{abs(dlt_.mean()):.1f}"
        M[f"jitter:{full}:n"] = str(len(a))
        sd = lambda v: (rf"{{\scriptsize$\,\pm\,${v.std(ddof=1):.1f}}}" if len(v) > 1 else "")   # noqa: E731
        rows.append(f"{name} & {len(a)} & {a.mean():.1f}{sd(a)} & {b.mean():.1f}{sd(b)} & {dlt_.mean():+.1f}{sd(dlt_)} \\\\")
write("tab_jitter.tex", "\\begin{tabular}{lcccc}\n\\toprule\nmethod & seeds & full augmentation & without colour jitter & change \\\\\n\\midrule\n"
      + ("\n".join(rows) if rows else r"\multicolumn{5}{c}{--} \\") + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Table: is the SupCon plateau a regularisation failure? (38,800 steps)
# ----------------------------------------------------------------------------
REG = [("default (weight decay $10^{-4}$)", "supcon_lab_f0.1_e400"), ("weight decay $5\\times10^{-4}$", "supcon_lab_f0.1_wd0.0005_e400"),
       ("weight decay $10^{-3}$", "supcon_lab_f0.1_wd0.001_e400"), ("stronger augmentation", "supcon_lab_f0.1_strongaug_e400"),
       ("weight decay $5\\times10^{-4}$ + stronger augmentation", "supcon_lab_f0.1_wd0.0005_strongaug_e400")]
rows = []
for name, c in REG:
    if not nseeds(c):
        continue
    best_knn = [max(a for _, a in RUN[f"{c}_s{s}"]["knn_curve"]) for s in seeds(c)]
    M[f"{c}:bestknn"] = f"{np.mean(best_knn):.1f}"
    rows.append(f"{name} & {nseeds(c)} & {MS(c, 'lp50')} & {MS(c, 'lp5')} & {MS(c, 'knn')} & {np.mean(best_knn):.1f} \\\\")
write("tab_reg.tex", "\\begin{tabular}{lccccc}\n\\toprule\n\\supcon, 38,800 steps & seeds & LP-50k & LP-5k & $k$NN-50k (final) & $k$NN-50k (best checkpoint) \\\\\n\\midrule\n"
      + ("\n".join(rows) if rows else r"\multicolumn{6}{c}{--} \\") + "\n\\bottomrule\n\\end{tabular}")
for k, tmpl in ARMS.items():      # best checkpoint along the longest run of every arm (an upper bound on early stopping)
    for E in (1000, 400, 200):
        c = tmpl.format(E=E)
        if nseeds(c):
            M[f"bestknn:{k}"] = f"{np.mean([max(a for _, a in RUN[f'{c}_s{s}']['knn_curve']) for s in seeds(c)]):.1f}"
            M[f"bestknn:{k}:E"] = f"{97 * E:,}"
            break

# ----------------------------------------------------------------------------
# Table: steps, architecture and learning rate (ResNet-18 vs ResNet-50)
# ----------------------------------------------------------------------------
R50 = [("ResNet-18", r"\supcon", "900", "0.25", "supcon_lab_f0.1_steps900"), ("ResNet-18", r"\supcon", "970", "0.25", "supcon_lab_f0.1_e10"),
       ("ResNet-18", r"\supcon", "9,700", "0.25", "supcon_lab_f0.1_e100"), ("ResNet-18", r"\supcon", "19,400", "0.25", "supcon_lab_f0.1_e200"),
       ("ResNet-18", r"\simclr", "970", "0.25", "simclr_all_e10"), ("ResNet-18", r"\simclr", "9,700", "0.25", "simclr_all_e100"),
       ("ResNet-18", r"\simclr", "19,400", "0.25", "simclr_all_e200"),
       ("ResNet-50", r"\supcon", "900", "0.25", "r50_supcon_lab_f0.1_steps900"), ("ResNet-50", r"\supcon", "900", "0.05", "r50_supcon_lab_f0.1_steps900_lr0.05"),
       ("ResNet-50", r"\supcon", "9,700", "0.25", "r50_supcon_lab_f0.1_e100"), ("ResNet-50", r"\supcon", "19,400", "0.25", "r50_supcon_lab_f0.1_e200"),
       ("ResNet-50", r"\simclr", "970", "0.25", "r50_simclr_all_e10"), ("ResNet-50", r"\simclr", "9,700", "0.25", "r50_simclr_all_e100"),
       ("ResNet-50", r"\simclr", "19,400", "0.25", "r50_simclr_all_e200")]
rows = [f"{a} & {m} & {s} & {lr} & {nseeds(c)} & {MS(c, 'lp50')} & {MS(c, 'lp5')} \\\\" for a, m, s, lr, c in R50 if nseeds(c)]
write("tab_r50.tex", "\\begin{tabular}{llrcccc}\n\\toprule\nencoder & method & steps & learning rate & seeds & LP-50k & LP-5k \\\\\n\\midrule\n"
      + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Table: compute accounting (per step and for the reference budget)
# ----------------------------------------------------------------------------
COMP = [(r"\simclr, \supcon, \union", "1,024", "1,024", "1,024", "9,700", "simclr_all_e100"),
        ("BYOL", "1,024", "2,048", "1,024", "9,700", "byol_all_e100"),
        ("SimSiam", "1,024", "1,024", "1,024", "9,700", "simsiam_all_e100"),
        ("cross-entropy", "512", "512", "512", "9,700", "ce_f0.1_e100"),
        ("FixMatch", "64 + 448 + 448", "960", "960", "10,347", "fixmatch_f0.1"),
        ("MixMatch", "64 + 2$\\times$64", "320", "192", "51,733", "mixmatch_f0.1"),
        ("fine-tuning on the 5k (FT-5k)", "128", "128", "128", "2,344", "ft_simclr_all_e100")]
rows = []
for name, views, fw, bw, steps, c in COMP:
    t = V(c, "time")
    tot = float(fw.replace(",", "")) * float(steps.replace(",", "")) / 1e6
    rows.append(f"{name} & {views} & {fw} & {bw} & {steps} & {tot:.1f} & {t.min():.0f} \\\\" if len(t) else
                f"{name} & {views} & {fw} & {bw} & {steps} & {tot:.1f} & -- \\\\")
write("tab_compute.tex", "\\begin{tabular}{lcccrcc}\n\\toprule\n & \\multicolumn{3}{c}{per step} & & \\multicolumn{2}{c}{reference budget} \\\\\n"
      "\\cmidrule(lr){2-4}\\cmidrule(lr){6-7}\nmethod & augmented views & forward passes & backward passes & steps & forward passes ($10^6$) & minutes \\\\\n"
      "\\midrule\n" + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# Appendix tables: colour reliance of SimCLR, and the validation-split choice of the SupCon+U variant
# ----------------------------------------------------------------------------
COL = [("crop + flip", "simclr_exp2_e100"), ("crop + flip + colour jitter", "simclr_exp5_e100"),
       ("crop + flip + grayscale", "simclr_exp6_e100"), ("crop + flip + colour jitter + grayscale (default)", "simclr_all_e100")]
rows = []
for name, c in COL:
    if nseeds(c):
        rows.append(f"{name} & {nseeds(c)} & {MS(c, 'lp50')} & {MS(c, 'linear_acc_gray')} & {fmt(mean(c, 'cka_hist'), 2)} & "
                    f"{fmt(mean(c, 'cka_label'), 2)} & {fmt(mean(c, 'hist_r2'), 2)} \\\\")
hp = os.path.join(ROOT, "results", "hist_probe.json")
if os.path.exists(hp):
    h = json.load(open(hp))
    if "cifar10_bins8" in h:
        M["histprobe"] = f"{h['cifar10_bins8']['acc']:.1f}"
write("tab_color.tex", "\\begin{tabular}{lcccccc}\n\\toprule\n & & \\multicolumn{2}{c}{LP-50k accuracy} & \\multicolumn{3}{c}{colour reliance} \\\\\n"
      "\\cmidrule(lr){3-4}\\cmidrule(lr){5-7}\n\\simclr augmentation & seeds & colour test & gray test & CKA(histogram) & CKA(label) & histogram $R^2$ \\\\\n\\midrule\n"
      + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")
PIL = [(r"\simclr, $\tau=0.5$", "pilot_simclr_t0.5"), (r"\simclr, $\tau=0.2$", "pilot_simclr_t0.2"), (r"\simclr, $\tau=0.1$", "pilot_simclr_t0.1"),
       (r"\supcon, $\tau=0.1$", "pilot_supconL_t0.1"), (r"\supcon, $\tau=0.5$", "pilot_supconL_t0.5"),
       (r"\union, $\tau=0.1$ for all anchors", "pilot_union_t0.1"), (r"\union, $\tau=0.5$ for all anchors", "pilot_union_t0.5"),
       (r"\union, $\tau_\ell=0.1$, $\tau_u=0.5$, uniform batches", "pilot_union_mixed"),
       (r"\union, $\tau_\ell=0.1$, $\tau_u=0.5$, 50\% labeled per batch", "pilot_union_mixed_sh0.5"),
       (r"\union, $\tau_\ell=0.1$, $\tau_u=0.5$, 25\% labeled per batch (chosen)", "pilot_union_mixed_sh0.25")]
rows = []
for name, c in PIL:
    r = RUN.get(c) or RUN.get(c + "_s0")
    if r:
        m = r["metrics"]
        rows.append(f"{name} & {m['linear_acc']:.1f} & {m['knn_acc']:.1f} & {fmt(m.get('linear_lab_acc', float('nan')))} \\\\")
write("tab_pilot.tex", "\\begin{tabular}{lccc}\n\\toprule\nconfiguration (validation split, 4,850 steps) & LP (all labels) & $k$NN & LP (10\\% labels) \\\\\n\\midrule\n"
      + "\n".join(rows) + "\n\\bottomrule\n\\end{tabular}")

# ----------------------------------------------------------------------------
# a few derived numbers used in the text
# ----------------------------------------------------------------------------
for E in (10, 25, 50, 100, 200, 400, 1000):
    s, c, u, sub = (ARMS[k].format(E=E) for k in ("simclr", "supcon", "union", "sub"))
    for p in ("lp50", "lp5", "ft", "fts", "fto", "fm", "knn"):
        diffmacro(f"simclr-supcon:{p}:e{E}", s, p, c, p)
        diffmacro(f"simclr-union:{p}:e{E}", s, p, u, p)
        diffmacro(f"union-supcon:{p}:e{E}", u, p, c, p)
        diffmacro(f"simclr-sub:{p}:e{E}", s, p, sub, p)
        diffmacro(f"supcon-sub:{p}:e{E}", c, p, sub, p)
for c in ("simclr_all_e200", "supcon_lab_f0.1_e200", "union_f0.1_e200", "byol_all_e200"):
    diffmacro(f"ftt-ft:{c}", c, "ftt", c, "ft")
    diffmacro(f"ftt-ftd:{c}", c, "ftt", c, "ftd")
    diffmacro(f"fto-ft:{c}", c, "fto", c, "ft")
    diffmacro(f"ftx-ft:{c}", c, "ftx", c, "ft")
    diffmacro(f"fm-ft:{c}", c, "fm", c, "ft")
diffmacro("twostage-simclr:lp5", "twostage_simclr200_supcon5", "lp5", "simclr_all_e200", "lp5")
diffmacro("twostage-simclr:lp50", "twostage_simclr200_supcon5", "lp50", "simclr_all_e200", "lp50")
diffmacro("twostage-simclr:ft", "twostage_simclr200_supcon5", "ft", "simclr_all_e200", "ft")
for pre, tag in (("c100_", "c100"), ("r50_", "r50"), ("tin_", "tin")):
    for E in (100, 200):
        for p in ("lp50", "lp5", "ft"):
            diffmacro(f"{tag}:simclr-supcon:{p}:e{E}", f"{pre}simclr_all_e{E}", p, f"{pre}supcon_lab_f0.1_e{E}", p)
            diffmacro(f"{tag}:simclr-union:{p}:e{E}", f"{pre}simclr_all_e{E}", p, f"{pre}union_f0.1_e{E}", p)
for E in (100, 200):
    for p in ("lp5", "ft", "knn", "fm", "fto"):
        diffmacro(f"stl:simclr-supcon:{p}:e{E}", f"stl_simclr_all_e{E}", p, f"stl_supcon_lab_e{E}", p)
        diffmacro(f"stl:simclr-union:{p}:e{E}", f"stl_simclr_all_e{E}", p, f"stl_union_e{E}", p)
        diffmacro(f"stl:simclr-sub:{p}:e{E}", f"stl_simclr_all_e{E}", p, f"stl_simclr_sub_e{E}", p)
diffmacro("steps900", "supcon_lab_f0.1_e100", "lp50", "supcon_lab_f0.1_steps900", "lp50")
diffmacro("twostage-supcon:lp5", "twostage_simclr200_supcon5", "lp5", "supcon_lab_f0.1_e200", "lp5")
# the longest budget with at least two seeds for both SimCLR and SupCon
for E in (1000, 400, 200):
    if nseeds(ARMS["simclr"].format(E=E)) >= 2 and nseeds(ARMS["supcon"].format(E=E)) >= 2:
        M["long:steps"] = f"{97 * E:,}"
        for k in ARMS:
            if nseeds(ARMS[k].format(E=E)):
                M[f"long:{k}"] = f"{mean(ARMS[k].format(E=E), 'lp50'):.1f}"
        for p in ("lp50", "lp5", "ft"):
            x, y = mean(ARMS["simclr"].format(E=E), p), mean(ARMS["supcon"].format(E=E), p)
            if np.isfinite(x) and np.isfinite(y):
                M["long:d:simclr-supcon" + ("" if p == "lp50" else ":" + p)] = f"{abs(x - y):.1f}"
        break
regs = [(mean(c, "lp50"), name, c) for name, c in REG[1:] if nseeds(c)]
if regs:
    best = max(regs)
    M["reg:best"] = f"{best[0]:.1f}"
    M["reg:best:name"] = best[1]
    M["reg:best:d"] = f"{best[0] - mean(REG[0][1], 'lp50'):.1f}"
    M["reg:best:lp5"] = f"{mean(best[2], 'lp5'):.1f}"
    M["reg:range"] = f"{min(r[0] for r in regs):.1f}--{max(r[0] for r in regs):.1f}"
    # largest drop from the best checkpoint to the end of the run, over all regularised and default runs
    M["reg:knndrop"] = f"{max(max(a for _, a in RUN[n]['knn_curve']) - RUN[n]['metrics']['knn_acc'] for _, c in REG for n in (f'{c}_s{s}' for s in seeds(c))):.1f}"
diffmacro("dist-ft:simclr", "simclr_all_e200", "dist", "simclr_all_e200", "ft")
diffmacro("fm-b3:simclr", "simclr_all_e200", "fm", "fixmatch_b3_f0.1", "clf")
diffmacro("fm-b3:supcon", "supcon_lab_f0.1_e200", "fm", "fixmatch_b3_f0.1", "clf")
diffmacro("fm-scratch:simclr", "simclr_all_e200", "fm", "fixmatch_f0.1", "clf")
diffmacro("fm-ft:simclr", "simclr_all_e200", "fm", "simclr_all_e200", "ft")
M["nruns"] = str(len(RUN))
M["gpuhours"] = f"{sum(r.get('train_time_s', 0) for r in RUN.values()) / 3600:.0f}"

lines = [r"\newcommand{\R}[1]{\ifcsname R@#1\endcsname\csname R@#1\endcsname\else\textbf{??}\fi}"]
for k in sorted(M):
    lines.append(rf"\expandafter\def\csname R@{k}\endcsname{{{minus(M[k])}}}")
write("numbers.tex", "\n".join(lines))
print(f"{len(RUN)} runs, {len(CONFIGS)} configurations, {len(M)} macros, {len(STATS)} tests")


## 8. Building the paper

The LaTeX sources live in `paper/` (`arxiv.tex`, `submission.tex`, `highlights.tex`, the `sec_*.tex` files);
they are text, not code, and are not repeated here. The two scripts below build the PDFs and the arXiv package.

**`paper/build.sh`** (12 lines) — builds `arxiv.pdf`, `submission.pdf` and `highlights.pdf` and reports errors and undefined references.

In [ ]:
%%writefile paper/build.sh
#!/bin/bash
# Builds arxiv.pdf (preprint), submission.pdf (journal review layout) and highlights.pdf.   Usage: bash build.sh
set -e
cd "$(dirname "$0")"
for f in arxiv submission; do
  pdflatex -interaction=nonstopmode $f.tex > /dev/null || true
  bibtex $f > /dev/null || true
  pdflatex -interaction=nonstopmode $f.tex > /dev/null || true
  pdflatex -interaction=nonstopmode $f.tex > /dev/null || true
  echo "$f: $(grep -c '^!' $f.log) errors, $(grep -c 'undefined' $f.log) undefined refs/cites, $(grep -c '??' $f.log || true) log '??', pages $(grep -o 'Output written on .*' $f.log | grep -o '([0-9]* pages' | tr -d '(')"
done
pdflatex -interaction=nonstopmode highlights.tex > /dev/null || true


**`paper/make_arxiv_package.sh`** (16 lines) — packs the sources, the `.bbl` file and the figures for arXiv and checks that the package compiles on its own.

In [ ]:
%%writefile paper/make_arxiv_package.sh
#!/bin/bash
# Creates arxiv_submission.tar.gz, ready to upload to arXiv (main file: arxiv.tex).
set -e
cd "$(dirname "$0")"
bash build.sh > /dev/null
rm -rf _arxiv && mkdir -p _arxiv/generated _arxiv/figures
cp arxiv.tex preamble.tex frontmatter.tex abstract.tex body.tex sec_*.tex statements.tex appendix.tex \
   elsarticle.cls elsarticle-num-names.bst references.bib arxiv.bbl _arxiv/
cp generated/numbers.tex generated/tab_*.tex _arxiv/generated/
cp figures/fig_budget.pdf figures/fig_wins.pdf figures/fig_geometry.pdf figures/fig_coverage.pdf _arxiv/figures/
# arXiv compiles with its own TeX Live; check that the package builds on its own
(cd _arxiv && pdflatex -interaction=nonstopmode arxiv.tex > /dev/null; pdflatex -interaction=nonstopmode arxiv.tex > /dev/null; \
   echo "standalone build: $(grep -c '^!' arxiv.log) errors"; rm -f arxiv.aux arxiv.log arxiv.out arxiv.pdf arxiv.spl)
tar -czf arxiv_submission.tar.gz -C _arxiv .
rm -rf _arxiv
echo "wrote paper/arxiv_submission.tar.gz ($(du -h arxiv_submission.tar.gz | cut -f1))"


## 9. Running the whole study

These commands are **not** executed by this notebook. Run them in a terminal from the repository root.

**Set-up and checks**
```bash
conda env create -f environment.yml && conda activate spend-labels-last   # or: docker build -t eic .
export DLT_DATA=./data                       # CIFAR-10/100 are downloaded here on first use
python -m tests.test_gpu_aug                 # GPU augmentations and loss against reference code

# STL-10 and Tiny-ImageNet: download the parquet exports, then build one uint8 archive each
#   huggingface-cli download jxie/stl10 --repo-type dataset --local-dir $DLT_DATA/stl10_hf
python scripts/prepare_stl10.py --src $DLT_DATA/stl10_hf --out $DLT_DATA
#   huggingface-cli download zh-plus/tiny-imagenet --repo-type dataset --local-dir $DLT_DATA/tin_hf
python scripts/prepare_tinyimagenet.py --src $DLT_DATA/tin_hf --out $DLT_DATA
```

**1. First study**
```bash
for f in jobs/00_pilot.txt jobs/01_main.txt jobs/02_semisup.txt jobs/03_finetune.txt jobs/04_extra_finetune.txt; do
  python scripts/run_queue.py $f --out results/runs; done
python scripts/make_pools.py --ref_run simclr_all_e100_s0 --ckpt_dir results/runs --mix 0.25 0.5 0.75
```

**2. Everything else, in the order of `jobs/phases.txt`** (finished runs are skipped; safe to stop and restart)
```bash
scripts/run_phases.sh jobs/phases.txt results/runs results/_queue_logs 1
# on the shared server: checkpoints and logs on the persistent volume, two runs at a time, with a watchdog
bash scripts/resume_queue.sh
bash scripts/queue_status.sh
```

**3. Analysis and paper**
```bash
python scripts/reeval.py    --ckpt_dir results/runs      # label-limited probes, CDNV on labeled / other / test images
python scripts/diversity.py --ref_run simclr_all_e100_s0 --ckpt_dir results/runs --extra_refs supervised=ce_f1.0_e100_s0
python scripts/make_assets.py                # every table, figure and number -> paper/generated, paper/figures
python scripts/stats.py simclr_all_e200 supcon_lab_f0.1_e200 --key linear   # any pairwise hierarchical-bootstrap test
cd paper && bash build.sh && bash make_arxiv_package.sh
```

**Single runs by hand** (CIFAR-10, 10% labels, 9,700 steps)
```bash
python -m dlt.train --name simclr     --method simclr --pool all     --epochs 100                          # SimCLR, all images
python -m dlt.train --name simclr_sub --method simclr --pool labeled --label_fraction 0.1 --epochs 100     # SimCLR, labeled subset
python -m dlt.train --name supcon     --method supcon --pool labeled --label_fraction 0.1 --epochs 100     # SupCon, labeled subset
python -m dlt.train --name supcon_u   --method supcon --pool all     --label_fraction 0.1 --epochs 100 \
       --temperature_unlabeled 0.5 --labeled_share 0.25                                                     # SupCon+U, all images
python -m dlt.finetune  --name ft_simclr --from_run simclr --label_fraction 0.1                           # FT-5k (default recipe)
python -m scripts.ft_search --name fts_simclr --from_run simclr --ckpt_dir results/runs --label_fraction 0.1  # recipe search
python -m dlt.semisup   --name fm_simclr --method fixmatch --label_fraction 0.1 --init_run simclr          # FixMatch from the encoder
python -m dlt.train     --name two_stage --method supcon --pool labeled --label_fraction 0.1 --epochs 5 --lr 0.02 --init_run simclr
python -m dlt.distill   --name distill   --from_run simclr --label_fraction 0.1                           # fine-tune + self-distil
```

## Appendix A: the original course project

CISC 867 (Group 20, Queen's University): SimCLR and SupCon on CIFAR-10 with augmentation ablations, before the
compute-matched study. Two snapshots, `src/Midterm Work` and `src/Final Work`, plus the ablation launcher at the
repository root. The scripts read `configs/augmentation.yaml` relative to their own folder.

**`configs/augmentation.yaml`** (6 lines) — augmentation settings of the course project (repository root).

In [ ]:
%%writefile configs/augmentation.yaml
normalization:
  cifar_mean: [0.4914, 0.4822, 0.4465]
  cifar_std: [0.2023, 0.1994, 0.2010]

color_distortion:
  s: 0.5


### Midterm version

**`src/Midterm Work/configs/augmentation.yaml`** (6 lines) — augmentation settings.

In [ ]:
%%writefile "src/Midterm Work/configs/augmentation.yaml"
normalization:
  cifar_mean: [0.4914, 0.4822, 0.4465]
  cifar_std: [0.2023, 0.1994, 0.2010]

color_distortion:
  s: 0.5


**`src/Midterm Work/augmentations.py`** (243 lines) — augmentation pipelines built from the YAML settings.

In [ ]:
%%writefile "src/Midterm Work/augmentations.py"

# IMPORT PART
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
import torch
import random
import yaml
import os

# Load configs - use absolute path relative to this file
config_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'configs', 'augmentation.yaml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

CIFAR_MEAN = tuple(config['normalization']['cifar_mean'])
CIFAR_STD = tuple(config['normalization']['cifar_std'])
s = config['color_distortion']['s']

# SIMCLR VIEW GENERATOR CLASS 
class SimCLRViewGenerator(object):
    """
    A custom wrapper to generate multiple augmented views of a single image,
    following the SimCLR framework for contrastive learning.
    """
    def __init__(self, base_transform, n_views=2):
        # The core stochastic augmentation pipeline to be applied to each input.
        self.base_transform = base_transform
        # Number of augmented views (typically 2 for creating positive pairs in SimCLR).
        self.n_views = n_views

    def __call__(self, x):
        """
        Executes the transformation pipeline multiple times to produce different 
        stochastic realizations of the same input image 'x'.
        """
        # Returns a list of 'n_views' variations, facilitating the calculation of contrastive loss.
        return [self.base_transform(x) for i in range(self.n_views)]


color_jitter = T.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.2*s)
# Applying jitter with 0.8 probability as per Chen et al. (2020)
rnd_color_jitter = T.RandomApply([color_jitter], p=0.8)


# EXPERIMENT 1 — Random Resized Crop (Crop + Resize)
transform_exp1 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)), 
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 2 — Crop + Flip
transform_exp2 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 3 — Crop + Color
transform_exp3 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    rnd_color_jitter,
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 4 — Crop + Grayscale
transform_exp4 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 5 — Crop + Flip + Color
transform_exp5 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    rnd_color_jitter,
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 6 — Crop + Flip + Grayscale
transform_exp6 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 7 — Crop + Color + Grayscale
transform_exp7 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    rnd_color_jitter,
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 8 — Crop + Flip + Color + Grayscale
transform_exp8 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    rnd_color_jitter,
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# ============================================================================
# NATALIE'S 28 EXPERIMENTS (from final augmentation.py)
# ============================================================================

class SobelTransform:
    def __call__(self, img):
        if not torch.is_tensor(img):
            img = TF.to_tensor(img)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float().view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float().view(1, 1, 3, 3)
        gray = img.mean(dim=0, keepdim=True).unsqueeze(0)
        gx = F.conv2d(gray, sobel_x, padding=1)
        gy = F.conv2d(gray, sobel_y, padding=1)
        edge = torch.sqrt(gx**2 + gy**2).squeeze(0)
        return edge.repeat(3, 1, 1)

class GaussianNoise:
    def __init__(self, std=0.1): 
        self.std = std
    def __call__(self, img):
        if not torch.is_tensor(img): 
            img = TF.to_tensor(img)
        noise = torch.randn_like(img) * self.std
        return (img + noise).clamp(0, 1)

class DiscreteRotation:
    def __call__(self, img):
        angles = [0, 90, 180, 270]
        return TF.rotate(img, random.choice(angles))

nat_transform_exp1 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp2 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.7, scale=(0.02, 0.2), ratio=(0.3, 3.3))])
nat_transform_exp3 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp4 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp5 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp6 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp7 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp8 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp9 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp10 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp11 = T.Compose([T.RandomResizedCrop(32), rnd_color_jitter, T.ToTensor(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp12 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp13 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp14 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp15 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp16 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp17 = T.Compose([T.RandomResizedCrop(32), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp18 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp19 = T.Compose([T.RandomResizedCrop(32), T.ToTensor(), SobelTransform(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp20 = T.Compose([T.RandomResizedCrop(32), T.GaussianBlur(3), T.ToTensor(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp21 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp22 = T.Compose([T.RandomResizedCrop(32), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.9, scale=(0.05, 0.4), ratio=(0.3, 3.3))])

nat_transform_exp23 = T.Compose([T.RandomResizedCrop(32, scale=(0.08, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp24 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp25 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.ToTensor(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp26 = T.Compose([T.RandomResizedCrop(32), T.RandomGrayscale(p=1.0), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp27 = T.Compose([T.RandomResizedCrop(32, scale=(0.8, 1.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp28 = T.Compose([DiscreteRotation(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

# EXP 29 (N29) — THE ULTIMATE BEAST
# Chen et al. (2020): Full color-disruption + structural cutout.
# Rationale: ColorJitter forces the encoder to ignore color shortcuts.
# RandomErasing (Cutout) forces it to learn spatial structure.
# Together, these synergistically prevent ALL forms of shortcut learning.
nat_transform_exp29 = T.Compose([
    # 1. Base spatial: aggressive crop forces invariance to scale/position
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    # 2. Horizontal flip: invariance to left-right mirroring
    T.RandomHorizontalFlip(p=0.5),
    # 3. Color jitter (p=0.8): prevents color-histogram shortcut learning
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    # 4. Grayscale (p=0.2): forces encoder to occasionally rely on luminance only
    T.RandomGrayscale(p=0.2),
    # 5. Must convert to tensor BEFORE RandomErasing (requires tensor input)
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD),
    # 6. Cutout / Random Erasing: destroys local patches -> spatial robustness
    T.RandomErasing(p=0.5, scale=(0.05, 0.2)),
])

# ============================================================================
# NEW JITTERED EXPERIMENTS (Added 19 May 2026) - For 5 Top Models
# ============================================================================
# 1. Exp 38: Pure Rotation + Jitter (Base: Exp 36)
nat_transform_exp30 = T.Compose([
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    DiscreteRotation(), 
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 2. Exp 39: Weak Baseline + Jitter (Base: Exp 35)
nat_transform_exp31 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.8, 1.0)), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 3. Exp 40: Crop + Blur + Jitter (Base: Exp 9)
nat_transform_exp32 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), 
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 4. Exp 41: Crop + Flip + Blur + Jitter (Base: Exp 13)
nat_transform_exp33 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)), 
    T.RandomHorizontalFlip(), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), 
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# 5. Exp 42: Crop + Cutout + Jitter (Base: Exp 10)
nat_transform_exp34 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.6, 1.0)), 
    T.RandomApply([T.ColorJitter(0.8, 0.8, 0.8, 0.2)], p=0.8),
    T.ToTensor(), 
    T.Normalize(CIFAR_MEAN, CIFAR_STD), 
    T.RandomErasing(p=0.7, scale=(0.02, 0.2), ratio=(0.3, 3.3))
])


**`src/Midterm Work/dataset.py`** (453 lines) — dataset and augmentation registry, SimCLR and SupCon loaders.

In [ ]:
%%writefile "src/Midterm Work/dataset.py"
"""
dataset.py — Modular Dataset & Augmentation Registry
Group 20, CISC 867, Queen's University, Spring 2026
Author (Student A): Natalie Nashed  — augmentation definitions
Author (Student B): Mahmoud Alyosify — registry architecture, SupCon loader

TWO REGISTRIES
──────────────
1. EXP_REGISTRY        — the 8 midterm ablation configs (exp_id 1–8)
2. EXTENDED_REGISTRY   — Natalie's 15–20 new augmentations (names as strings)
                          She fills in the transforms; the loader just calls
                          get_train_dataloader(..., aug_name='randaugment')

SUPCON SUPPORT
──────────────
get_supcon_dataloader() returns a labeled DataLoader for Bonus #4:
    - Stratified subsampling to `label_fraction` of the training set
    - Returns (view1, view2), label  so SupConLoss can use class info
"""

import os
import numpy as np
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T

# ── Import from Natalie's augmentation module ───────────────────────────────
from augmentations import (
    SimCLRViewGenerator, CIFAR_MEAN, CIFAR_STD,
    transform_exp1, transform_exp2, transform_exp3, transform_exp4,
    transform_exp5, transform_exp6, transform_exp7, transform_exp8,
    nat_transform_exp1, nat_transform_exp2, nat_transform_exp3, nat_transform_exp4,
    nat_transform_exp5, nat_transform_exp6, nat_transform_exp7, nat_transform_exp8,
    nat_transform_exp9, nat_transform_exp10, nat_transform_exp11, nat_transform_exp12,
    nat_transform_exp13, nat_transform_exp14, nat_transform_exp15, nat_transform_exp16,
    nat_transform_exp17, nat_transform_exp18, nat_transform_exp19, nat_transform_exp20,
    nat_transform_exp21, nat_transform_exp22, nat_transform_exp23, nat_transform_exp24,
    nat_transform_exp25, nat_transform_exp26, nat_transform_exp27, nat_transform_exp28,
    nat_transform_exp29, nat_transform_exp30, nat_transform_exp31, nat_transform_exp32,
    nat_transform_exp33, nat_transform_exp34
)


# ══════════════════════════════════════════════════════════════════════════
# 1. REGISTRY 1 — Midterm 8-Experiment Ablation Configs
# ══════════════════════════════════════════════════════════════════════════

EXP_REGISTRY = {
    1: {
        "transform":   transform_exp1,
        "name":        "Exp1 — Crop only",
        "description": "Spatial only: RandomResizedCrop.",
    },
    2: {
        "transform":   transform_exp2,
        "name":        "Exp2 — Crop + Flip",
        "description": "Spatial: RandomResizedCrop + HorizontalFlip.",
    },
    3: {
        "transform":   transform_exp3,
        "name":        "Exp3 — Crop + ColorJitter",
        "description": "Hybrid: RandomResizedCrop + ColorJitter.",
    },
    4: {
        "transform":   transform_exp4,
        "name":        "Exp4 — Crop + Grayscale",
        "description": "Hybrid: RandomResizedCrop + RandomGrayscale.",
    },
    5: {
        "transform":   transform_exp5,
        "name":        "Exp5 — Crop + Flip + Color",
        "description": "Hybrid: RandomResizedCrop + Flip + ColorJitter.",
    },
    6: {
        "transform":   transform_exp6,
        "name":        "Exp6 — Crop + Flip + Grayscale",
        "description": "Hybrid: RandomResizedCrop + Flip + Grayscale.",
    },
    7: {
        "transform":   transform_exp7,
        "name":        "Exp7 — Crop + Color + Grayscale",
        "description": "Hybrid: RandomResizedCrop + ColorJitter + Grayscale.",
    },
    8: {
        "transform":   transform_exp8,
        "name":        "Exp8 — Full SimCLR",
        "description": "Full pipeline: Crop + Flip + ColorJitter + Grayscale.",
    },
    # NATALIE'S 28 EXPERIMENTS (Offset by 8)
    9:  {"transform": nat_transform_exp1,  "name": "Exp9(N1) — Crop + Blur", "description": "Single: Crop + Gaussian Blur."},
    10: {"transform": nat_transform_exp2,  "name": "Exp10(N2) — Crop + Cutout", "description": "Single: Crop + Random Erasing."},
    11: {"transform": nat_transform_exp3,  "name": "Exp11(N3) — Crop + Sobel", "description": "Single: Crop + Sobel Edge Extraction."},
    12: {"transform": nat_transform_exp4,  "name": "Exp12(N4) — Crop + Noise", "description": "Single: Crop + Gaussian Noise."},
    13: {"transform": nat_transform_exp5,  "name": "Exp13(N5) — Crop + Flip + Blur", "description": "Dual: Spatial + Blur."},
    14: {"transform": nat_transform_exp6,  "name": "Exp14(N6) — Crop + Flip + Cutout", "description": "Dual: Spatial + Corruption."},
    15: {"transform": nat_transform_exp7,  "name": "Exp15(N7) — Crop + Flip + Sobel", "description": "Dual: Spatial + Structural."},
    16: {"transform": nat_transform_exp8,  "name": "Exp16(N8) — Crop + Flip + Noise", "description": "Dual: Spatial + Noise."},
    17: {"transform": nat_transform_exp9,  "name": "Exp17(N9) — Crop + Color + Blur", "description": "Dual: Photometric + Blur."},
    18: {"transform": nat_transform_exp10, "name": "Exp18(N10) — Crop + Color + Cutout", "description": "Dual: Photometric + Corruption."},
    19: {"transform": nat_transform_exp11, "name": "Exp19(N11) — Crop + Color + Sobel", "description": "Dual: Photometric + Structural."},
    20: {"transform": nat_transform_exp12, "name": "Exp20(N12) — Crop + Color + Noise", "description": "Dual: Photometric + Noise."},
    21: {"transform": nat_transform_exp13, "name": "Exp21(N13) — Crop + Gray + Blur", "description": "Dual: Grayscale + Blur."},
    22: {"transform": nat_transform_exp14, "name": "Exp22(N14) — Crop + Gray + Cutout", "description": "Dual: Grayscale + Corruption."},
    23: {"transform": nat_transform_exp15, "name": "Exp23(N15) — Crop + Gray + Sobel", "description": "Dual: Grayscale + Structural."},
    24: {"transform": nat_transform_exp16, "name": "Exp24(N16) — Crop + Gray + Noise", "description": "Dual: Grayscale + Noise."},
    25: {"transform": nat_transform_exp17, "name": "Exp25(N17) — Crop + Blur + Sobel", "description": "Conflict: Smooth vs Sharp Edge."},
    26: {"transform": nat_transform_exp18, "name": "Exp26(N18) — Crop + Flip + Blur + Cutout", "description": "Conflict: Content Erasure with Blur."},
    27: {"transform": nat_transform_exp19, "name": "Exp27(N19) — Crop + Sobel + Noise", "description": "Conflict: Edge Extraction vs Pixel Noise."},
    28: {"transform": nat_transform_exp20, "name": "Exp28(N20) — Crop + Blur + Noise", "description": "Conflict: High-frequency Distortion (JPEG-like)."},
    29: {"transform": nat_transform_exp21, "name": "Exp29(N21) — Crop + Flip + Color + Erasing", "description": "Conflict: Color Distortion with Patch Erasure."},
    30: {"transform": nat_transform_exp22, "name": "Exp30(N22) — Crop + Heavy Cutout Grid", "description": "Extreme Spatial Corruption Baseline."},
    31: {"transform": nat_transform_exp23, "name": "Exp31(N23) — SimCLR Classic Strong", "description": "Standard SimCLR Pipeline (Chen et al. 2020)."},
    32: {"transform": nat_transform_exp24, "name": "Exp32(N24) — SimCLR + Rotation Task", "description": "Pipeline for E-SSL Joint-Task Pretraining."},
    33: {"transform": nat_transform_exp25, "name": "Exp33(N25) — Full Heatmap Suite", "description": "All Invariance Transforms Active Simultaneously."},
    34: {"transform": nat_transform_exp26, "name": "Exp34(N26) — Color Destruction", "description": "Strict Color Elimination: Grayscale p=1.0 + ColorJitter."},
    35: {"transform": nat_transform_exp27, "name": "Exp35(N27) — Weak Baseline", "description": "Minimal Spatial Invariance Only."},
    36: {"transform": nat_transform_exp28, "name": "Exp36(N28) — Pure Discrete Rotation", "description": "Isolated Structural Rotation Alignment."},
    # ── Exp 29 (N29) — The Ultimate Beast (added 18 May 2026) ──────────────────
    37: {
        "transform":   nat_transform_exp29,
        "name":        "Exp37(N29) — The Ultimate Beast",
        "description": (
            "Full pipeline: Crop + Flip + ColorJitter(p=0.8) + Grayscale(p=0.2) "
            "+ RandomErasing/Cutout(p=0.5, scale=5-20%). "
            "Designed to eliminate ALL shortcut-learning pathways simultaneously."
        ),
    },
    # ── Re-runs with Color Jitter (Added 19 May 2026) ──────────────────────────
    38: {"transform": nat_transform_exp30, "name": "Exp38 — Pure Rotation + Jitter (Base: Exp 36)", "description": "Exp 36 + Color Jitter"},
    39: {"transform": nat_transform_exp31, "name": "Exp39 — Weak Baseline + Jitter (Base: Exp 35)", "description": "Exp 35 + Color Jitter"},
    40: {"transform": nat_transform_exp32, "name": "Exp40 — Crop + Blur + Jitter (Base: Exp 9)", "description": "Exp 9 + Color Jitter"},
    41: {"transform": nat_transform_exp33, "name": "Exp41 — Crop + Flip + Blur + Jitter (Base: Exp 13)", "description": "Exp 13 + Color Jitter"},
    42: {"transform": nat_transform_exp34, "name": "Exp42 — Crop + Cutout + Jitter (Base: Exp 10)", "description": "Exp 10 + Color Jitter"},
}


# ══════════════════════════════════════════════════════════════════════════
# 2. REGISTRY 2 — Extended Augmentation Registry (Natalie fills these in)
#
#    Each entry key is the --aug_name CLI argument.
#    Natalie: add your transforms below following the same pattern.
#    Mahmoud's train_master.py will pick them up automatically.
# ══════════════════════════════════════════════════════════════════════════

def _base_crop():
    """All new augs build on top of the base SimCLR crop."""
    return T.RandomResizedCrop(32, scale=(0.2, 1.0))

# ── Placeholder entries — Natalie fills in the transform pipelines ─────────
# For each entry, replace T.Compose([_base_crop(), T.ToTensor(), ...])
# with your actual pipeline.

EXTENDED_REGISTRY = {

    # ── Natalie: fill these in ────────────────────────────────────────

    "randaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.RandAugment(num_ops=2, magnitude=9),   # NATALIE: tune params
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "RandAugment",
        "description": "AutoML-style random augmentation policy search.",
    },

    "autoaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.AutoAugment(policy=T.AutoAugmentPolicy.CIFAR10),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "AutoAugment (CIFAR-10 policy)",
        "description": "Learned CIFAR-10 augmentation policy (Cubuk et al.).",
    },

    "trivialaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.TrivialAugmentWide(),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "TrivialAugmentWide",
        "description": "Parameter-free augmentation (Müller & Hutter, 2021).",
    },

    "solarize": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomSolarize(threshold=128, p=0.2),   # NATALIE: tune threshold
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Solarize",
        "description": "Exp8 + RandomSolarize (inspired by SimCLRv2).",
    },

    "equalize": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomEqualize(p=0.2),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Equalize",
        "description": "Exp8 + RandomEqualize histogram normalization.",
    },

    "sharpness": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Sharpness",
        "description": "Exp8 + RandomAdjustSharpness.",
    },

    "rotation": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomRotation(degrees=15),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Rotation",
        "description": "Exp8 + RandomRotation ±15°.",
    },

    "perspective": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomPerspective(distortion_scale=0.2, p=0.3),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Perspective",
        "description": "Exp8 + RandomPerspective distortion.",
    },

    # NOTE: Mixup and CutMix require batch-level operations — they are
    # implemented in the training loop via torchvision.transforms.v2.
    # Natalie: implement these as collate_fn wrappers in augmentations.py
    # and register them here by referencing your collate_fn.
    # Mahmoud's DataLoader accepts an optional collate_fn parameter.
    "mixup": {
        "transform": transform_exp8,   # NATALIE: replace with Mixup-aware transform
        "name":        "Mixup (placeholder)",
        "description": "Batch-level Mixup — requires custom collate_fn.",
        "collate_fn":  None,           # NATALIE: set to your mixup_collate_fn
    },

    "cutmix": {
        "transform": transform_exp8,   # NATALIE: replace with CutMix-aware transform
        "name":        "CutMix (placeholder)",
        "description": "Batch-level CutMix — requires custom collate_fn.",
        "collate_fn":  None,           # NATALIE: set to your cutmix_collate_fn
    },
}


# ══════════════════════════════════════════════════════════════════════════
# 3. AugmentedDataset — returns (view1, view2), label
# ══════════════════════════════════════════════════════════════════════════

class AugmentedDataset(Dataset):
    """
    Wraps any torchvision dataset to return two augmented views.
    Compatible with both NT-Xent (labels ignored) and SupCon (labels used).
    """
    def __init__(self, dataset, transform):
        self.dataset   = dataset
        self.transform = SimCLRViewGenerator(transform, n_views=2)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        views = self.transform(img)      # [view_1, view_2]
        return (views[0], views[1]), label


# ══════════════════════════════════════════════════════════════════════════
# 4. DATALOADER FACTORIES
# ══════════════════════════════════════════════════════════════════════════

def _resolve_transform(exp_id: int, aug_name: str):
    """
    Resolves which transform to use.
    Priority: aug_name (extended registry) > exp_id (midterm registry)
    """
    if aug_name is not None:
        aug_name = aug_name.lower().strip()
        if aug_name not in EXTENDED_REGISTRY:
            available = list(EXTENDED_REGISTRY.keys())
            raise ValueError(
                f"Unknown aug_name='{aug_name}'. "
                f"Available: {available}"
            )
        info = EXTENDED_REGISTRY[aug_name]
        print(f"  [Dataset] Augmentation (extended): {info['name']}")
        print(f"            {info['description']}")
        return info["transform"], info.get("collate_fn", None)

    # Fallback to midterm exp_id
    if exp_id not in EXP_REGISTRY:
        raise ValueError(
            f"Invalid exp_id={exp_id}. Choose from {list(EXP_REGISTRY.keys())}."
        )
    info = EXP_REGISTRY[exp_id]
    print(f"  [Dataset] Augmentation (midterm): {info['name']}")
    print(f"            {info['description']}")
    return info["transform"], None


def get_train_dataloader(
    data_dir: str,
    batch_size: int   = 128,
    num_workers: int  = 4,
    shuffle: bool     = True,
    exp_id: int       = 8,
    aug_name: str     = None,
) -> DataLoader:
    """
    Returns a training DataLoader yielding ((view1, view2), label).

    Args:
        data_dir    : root directory for CIFAR-10 download
        batch_size  : training batch size
        num_workers : DataLoader worker processes
        shuffle     : shuffle training data
        exp_id      : midterm ablation config 1–8 (overridden by aug_name)
        aug_name    : named augmentation from EXTENDED_REGISTRY
    """
    transform, collate_fn = _resolve_transform(exp_id, aug_name)

    os.makedirs(data_dir, exist_ok=True)
    base_dataset = CIFAR10(root=data_dir, train=True, download=True)
    augmented    = AugmentedDataset(base_dataset, transform)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,      # ensures consistent batch size for NT-Xent
        persistent_workers=(num_workers > 0),
    )
    if collate_fn is not None:
        loader_kwargs["collate_fn"] = collate_fn

    return DataLoader(augmented, **loader_kwargs)


def get_supcon_dataloader(
    data_dir: str,
    batch_size: int     = 256,
    num_workers: int    = 4,
    label_fraction: float = 0.1,
    exp_id: int         = 8,
    aug_name: str       = None,
    seed: int           = 42,
) -> DataLoader:
    """
    Returns a labeled DataLoader for Bonus #4 (SupCon training).
    Subsamples `label_fraction` of the CIFAR-10 training set, stratified
    per class so class balance is maintained.

    Args:
        label_fraction: 0.1 = 10% = 5,000 images (500 per class)
    """
    transform, collate_fn = _resolve_transform(exp_id, aug_name)
    rng = np.random.RandomState(seed)

    os.makedirs(data_dir, exist_ok=True)
    base_dataset  = CIFAR10(root=data_dir, train=True, download=True)
    labels_array  = np.array(base_dataset.targets)

    # Stratified subsampling
    selected = []
    for cls in range(10):
        cls_idx = np.where(labels_array == cls)[0].tolist()
        n_keep  = max(1, int(len(cls_idx) * label_fraction))
        chosen  = rng.choice(cls_idx, size=n_keep, replace=False).tolist()
        selected.extend(chosen)

    print(f"  [SupCon] {len(selected):,} labeled images "
          f"({label_fraction*100:.0f}% of 50k, stratified per class)")

    subset    = Subset(base_dataset, selected)
    augmented = AugmentedDataset(subset, transform)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(num_workers > 0),
    )
    if collate_fn is not None:
        loader_kwargs["collate_fn"] = collate_fn

    return DataLoader(augmented, **loader_kwargs)


def get_eval_dataloader(
    data_dir: str,
    train: bool      = False,
    batch_size: int  = 256,
    num_workers: int = 4,
) -> DataLoader:
    """
    Returns an evaluation DataLoader (no augmentation, no shuffling).
    Used for: t-SNE generation, linear probe caching, FAISS index building.
    """
    os.makedirs(data_dir, exist_ok=True)

    eval_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    dataset = CIFAR10(
        root=data_dir, train=train, download=True, transform=eval_transform
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
    )


**`src/Midterm Work/loss.py`** (314 lines) — NT-Xent and SupCon losses.

In [ ]:
%%writefile "src/Midterm Work/loss.py"
"""
loss.py — NT-Xent + SupConLoss
Group 20, CISC 867, Queen's University, Spring 2026
Author (Student B): Mahmoud Alyosify

Two loss functions in one file:
  • NTXentLoss   — used for ALL self-supervised (unlabeled) pretraining
  • SupConLoss   — Bonus #4, Khosla et al. (NeurIPS 2020)
                   used for the hybrid 10%-label supervised-contrastive run

SupCon paper: https://arxiv.org/abs/2004.11362

Key design difference vs NT-Xent
────────────────────────────────
NT-Xent defines ONE positive per anchor (its augmented twin).
SupCon defines ALL same-class embeddings in the batch as positives.
When labels are present, the model can exploit the richer supervision
signal while still using unlabeled negatives for discrimination.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════════════
# 1. NT-Xent Loss  (unchanged from midterm, kept here for single import)
# ══════════════════════════════════════════════════════════════════════

class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy loss.
    For a batch of N images → 2N views (two augmented copies each).
    Each view's positive is its twin; all 2(N-1) other views are negatives.

    Args:
        temperature (float): τ scaling factor. Default 0.5 (paper default).
    """

    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z1, z2: projection embeddings, shape (N, D). NOT yet normalized.
        Returns:
            Scalar NT-Xent loss averaged over both views.
        """
        N = z1.shape[0]
        device = z1.device

        # ℓ₂-normalize so cosine sim = dot product
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        # Stack into (2N, D) — [z1_0, z1_1, …, z2_0, z2_1, …]
        z = torch.cat([z1, z2], dim=0)                         # (2N, D)

        # Full similarity matrix scaled by temperature
        sim = torch.mm(z, z.T) / self.temperature             # (2N, 2N)

        # Mask out self-similarity (diagonal)
        diag_mask = torch.eye(2 * N, dtype=torch.bool, device=device)
        sim = sim.masked_fill(diag_mask, float("-inf"))

        # Positive targets: view i pairs with view i+N (and vice versa)
        targets = torch.cat([
            torch.arange(N, 2 * N, device=device),
            torch.arange(0, N, device=device),
        ])

        return F.cross_entropy(sim, targets)


# ══════════════════════════════════════════════════════════════════════
# 2. SupCon Loss  (Bonus #4)  — Khosla et al., NeurIPS 2020
# ══════════════════════════════════════════════════════════════════════

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., NeurIPS 2020).

    Supports two modes controlled by `labels`:
      • labels=None  → self-supervised mode (same as NT-Xent with two views)
      • labels given → supervised mode: all same-class embeddings are positives

    For Bonus #4 we use SUPERVISED mode with 10% of CIFAR-10 labels.

    Mathematical formulation (paper Eq. 2):
    ┌──────────────────────────────────────────────────────────────────┐
    │  L = Σᵢ  -1/|P(i)| · Σₚ∈P(i)  log  exp(zᵢ·zₚ/τ)             │
    │                                      ──────────────────────     │
    │                                      Σₐ∈A(i) exp(zᵢ·zₐ/τ)     │
    │                                                                  │
    │  P(i) = set of positives for anchor i (same class, not self)    │
    │  A(i) = all indices except i (the denominator negatives)        │
    └──────────────────────────────────────────────────────────────────┘

    Args:
        temperature (float): τ. Recommended 0.07–0.5 for SupCon.
        contrast_mode (str): 'all'  — use all views as anchors (default)
                             'one'  — use only first view as anchor
        base_temperature (float): τ_base for loss re-scaling. Default 0.07
                                   as in the original paper.
    """

    def __init__(
        self,
        temperature: float = 0.07,
        contrast_mode: str = "all",
        base_temperature: float = 0.07,
    ):
        super().__init__()
        assert temperature > 0, "Temperature must be positive."
        assert contrast_mode in ("all", "one"), \
            "contrast_mode must be 'all' or 'one'."

        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(
        self,
        features: torch.Tensor,
        labels: torch.Tensor = None,
        mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Args:
            features: shape (N, n_views, D) — ℓ₂-normalized embeddings.
                      For two augmented views: n_views = 2.
                      Each slice features[:, v, :] is one view of the batch.
            labels:   shape (N,) — class labels for the N images.
                      Pass None for self-supervised mode (treats each sample
                      as its own class, recovers standard contrastive loss).
            mask:     shape (N, N) — binary mask where mask[i,j]=1 means i
                      and j form a positive pair. Overrides labels if given.
        Returns:
            Scalar supervised contrastive loss.
        """
        device = features.device

        # ── Input validation ─────────────────────────────────────────
        if features.ndim == 2:
            # Caller passed (N, D) — assume single view, unsqueeze
            features = features.unsqueeze(1)

        batch_size, n_views, feat_dim = features.shape

        # ── Build positive-pair mask ──────────────────────────────────
        if mask is not None:
            # Explicit mask provided — trust the caller
            mask = mask.float().to(device)

        elif labels is not None:
            # Supervised: mask[i,j] = 1 iff label[i] == label[j]
            labels = labels.contiguous().view(-1, 1)          # (N, 1)
            mask = torch.eq(labels, labels.T).float().to(device)  # (N, N)

        else:
            # Self-supervised fallback: each sample is its own class
            mask = torch.eye(batch_size, dtype=torch.float32, device=device)

        # ── Flatten views into one big (N*n_views, D) tensor ─────────
        # Anchor features — determined by contrast_mode
        if self.contrast_mode == "one":
            anchor_feat = features[:, 0, :]                   # (N, D)
            anchor_count = 1
        else:  # "all"
            # Stack all views: each becomes an anchor
            anchor_feat = features.view(
                batch_size * n_views, feat_dim
            )                                                  # (N*v, D)
            anchor_count = n_views

        # Contrast features = ALL views of ALL samples
        contrast_feat = features.view(
            batch_size * n_views, feat_dim
        )                                                      # (N*v, D)
        contrast_count = n_views

        # ── Similarity matrix ─────────────────────────────────────────
        # anchor_dot_contrast: (N*anchor_count, N*contrast_count)
        anchor_dot_contrast = (
            torch.mm(anchor_feat, contrast_feat.T) / self.temperature
        )

        # For numerical stability: subtract row-wise max
        logits_max, _ = anchor_dot_contrast.max(dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # ── Expand mask to cover all view combinations ─────────────────
        # Original mask is (N, N). We tile it to (N*anchor_count, N*contrast_count)
        # so that every view of sample i treats every view of a same-class
        # sample j as a positive.
        mask_tiled = mask.repeat(anchor_count, contrast_count)   # (N*a, N*c)

        # Remove self-contrast (an anchor must not compare with itself)
        self_contrast_mask = torch.scatter(
            torch.ones_like(mask_tiled),
            1,
            torch.arange(batch_size * anchor_count, device=device).view(-1, 1),
            0,
        )
        mask_tiled = mask_tiled * self_contrast_mask

        # ── Log-softmax over negatives ────────────────────────────────
        exp_logits = torch.exp(logits) * self_contrast_mask    # zero out self
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

        # ── Mean over positives ───────────────────────────────────────
        # For each anchor, average log-prob over its positives
        mean_log_prob_pos = (mask_tiled * log_prob).sum(1) / (
            mask_tiled.sum(1) + 1e-9
        )

        # ── Final loss (temperature re-scaling as in paper Eq. 2) ─────
        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss


# ══════════════════════════════════════════════════════════════════════
# 3. Loss Factory  — used by train_master.py and train_supcon.py
# ══════════════════════════════════════════════════════════════════════

def get_loss(loss_type: str, temperature: float = 0.5) -> nn.Module:
    """
    Factory function so all training scripts import losses by name.

    Args:
        loss_type   : 'ntxent' | 'supcon'
        temperature : τ value (same for both losses)
    Returns:
        Instantiated loss module.
    """
    if loss_type == "ntxent":
        return NTXentLoss(temperature=temperature)

    if loss_type == "supcon":
        # SupCon paper uses τ=0.07 by default; we allow override
        return SupConLoss(
            temperature=temperature,
            contrast_mode="all",
            base_temperature=0.07,
        )

    raise ValueError(
        f"Unknown loss '{loss_type}'. Choose 'ntxent' or 'supcon'."
    )


# ══════════════════════════════════════════════════════════════════════
# 4. Smoke Tests
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import math

    print("=" * 60)
    print("Smoke Test — NTXentLoss")
    print("=" * 60)

    crit = NTXentLoss(temperature=0.5)
    N, D = 32, 128

    z1 = torch.randn(N, D)
    z2 = torch.randn(N, D)
    loss_rand = crit(z1, z2)
    loss_expected = math.log(2 * N - 1)

    print(f"Random embeddings loss : {loss_rand.item():.4f}")
    print(f"Expected (uniform)     : {loss_expected:.4f}")

    loss_perfect = crit(z1, z1.clone())
    print(f"Identical embeddings   : {loss_perfect.item():.6f}  (should be ~0)")
    assert loss_rand.shape == torch.Size([]), "Not a scalar!"
    print("PASSED ✓\n")

    # ──────────────────────────────────────────────────────────────────
    print("=" * 60)
    print("Smoke Test — SupConLoss (supervised mode, Bonus #4)")
    print("=" * 60)

    supcon = SupConLoss(temperature=0.07)
    N_sup = 16
    n_views = 2
    D_sup = 128
    n_classes = 10

    # Simulate two augmented views: features shape (N, n_views, D)
    feats = F.normalize(
        torch.randn(N_sup, n_views, D_sup), dim=2
    )
    labels = torch.randint(0, n_classes, (N_sup,))

    loss_sup = supcon(feats, labels=labels)
    print(f"SupCon loss (supervised, 2 views) : {loss_sup.item():.4f}")
    assert loss_sup.shape == torch.Size([]), "Not a scalar!"
    assert not torch.isnan(loss_sup), "NaN detected!"
    print("PASSED ✓\n")

    # Self-supervised fallback (no labels)
    loss_self = supcon(feats, labels=None)
    print(f"SupCon loss (self-supervised mode) : {loss_self.item():.4f}")
    assert not torch.isnan(loss_self), "NaN detected!"
    print("PASSED ✓\n")

    print("=" * 60)
    print("All loss tests passed.")
    print("=" * 60)


**`src/Midterm Work/model.py`** (182 lines) — ResNet encoder with projection head.

In [ ]:
%%writefile "src/Midterm Work/model.py"
import torch
import torch.nn as nn
import torchvision.models as models


class ProjectionHead(nn.Module):

    def __init__(
        self,
        input_dim=512,
        hidden_dim=512,
        output_dim=128
    ):

        super().__init__()

        self.layers = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
                bias=False
            ),

            nn.BatchNorm1d(hidden_dim),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                output_dim
            )
        )

    def forward(self, x):

        return self.layers(x)


class SimCLRResNet18(nn.Module):

    def __init__(self, projection_dim=128):

        super().__init__()

        backbone = models.resnet18(weights=None)

        # adjust first layer for CIFAR-10 images
        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove original maxpool
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,

            backbone.avgpool
        )

        self.feature_dim = 512

        self.projector = ProjectionHead(
            input_dim=self.feature_dim,
            hidden_dim=self.feature_dim,
            output_dim=projection_dim
        )

    def forward(self, x):

        features = self.encoder(x)

        features = torch.flatten(
            features,
            start_dim=1
        )

        projections = self.projector(features)

        return features, projections


class SimCLRResNet50(nn.Module):
    """
    ResNet-50 backbone for SimCLR with CIFAR-10 stem adjustment.
    Output: (h, z) where h is the 2048-dim hidden representation
    and z is the 128-dim projection (or custom projection_dim).
    """

    def __init__(self, projection_dim=128):

        super().__init__()

        backbone = models.resnet50(weights=None)

        # adjust first layer for CIFAR-10 images (32x32)
        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove original maxpool
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        # global average pooling
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # projection head: ResNet50 outputs 2048-dim features
        self.projection = ProjectionHead(
            input_dim=2048,
            hidden_dim=2048,
            output_dim=projection_dim
        )

    def forward(self, x):
        """
        Args:
            x: (N, 3, 32, 32) CIFAR-10 batch
        Returns:
            h: (N, 2048) encoder output
            z: (N, projection_dim) projection output
        """
        x = self.encoder(x)            # (N, 2048, 1, 1)
        x = self.pool(x)               # (N, 2048, 1, 1)
        h = x.flatten(start_dim=1)     # (N, 2048)
        z = self.projection(h)         # (N, projection_dim)
        return h, z


def get_simclr_model(
    backbone="resnet18",
    projection_dim=128
):

    if backbone == "resnet18":

        return SimCLRResNet18(
            projection_dim=projection_dim
        )

    elif backbone == "resnet50":

        return SimCLRResNet50(
            projection_dim=projection_dim
        )

    raise ValueError(
        f"Backbone '{backbone}' is not supported. "
        f"Supported: resnet18, resnet50"
    )

**`src/Midterm Work/train_master.py`** (713 lines) — SimCLR training script with logging and checkpoints.

In [ ]:
%%writefile "src/Midterm Work/train_master.py"
"""
train_master.py — SimCLR Final-Phase Master Training Script
Group 20, CISC 867, Queen's University, Spring 2026
Author (Student B): Mahmoud Alyosify

PURPOSE
───────
This is THE production training script for the final 200-epoch ResNet-50
SimCLR run on the RTX 5000 Ada (32 GB VRAM). It also supports:
  • Quick hyperparameter sweeps (τ, batch size, LR)
  • The 8 midterm ablation experiments (--exp_id 1-8)
  • SupCon Bonus #4 (--loss supcon --label_fraction 0.1)

PERFORMANCE FEATURES ENABLED
─────────────────────────────
1. torch.amp.autocast    — FP16 activations, halves memory bandwidth
2. torch.compile()       — TorchInductor/Triton fusion, ~25-35% speedup
3. cudnn.benchmark=True  — fastest conv algorithm auto-select
4. pin_memory + non_blocking — overlap CPU→GPU transfers with compute
5. drop_last=True        — ensures every batch has exactly N samples
                           (critical for NT-Xent similarity matrix)

USAGE EXAMPLES
──────────────
# The 200-epoch master run (wait for Natalie's best aug combo):
python train_master.py --epochs 200 --batch_size 1024 --backbone resnet50

# Quick 8-experiment ablation (all 8 midterm configs in sequence):
python train_master.py --run_all_ablations --epochs 20 --batch_size 128

# Temperature sweep:
python train_master.py --temperature 0.1 --exp_id 8 --epochs 50
python train_master.py --temperature 0.5 --exp_id 8 --epochs 50
python train_master.py --temperature 1.0 --exp_id 8 --epochs 50

# Batch size sweep:
python train_master.py --batch_size 256 --exp_id 8 --epochs 50
python train_master.py --batch_size 512 --exp_id 8 --epochs 50
python train_master.py --batch_size 1024 --exp_id 8 --epochs 50

# Bonus #4 — SupCon with 10% labels:
python train_master.py --loss supcon --label_fraction 0.1 --epochs 200

HAND-OFFS
─────────
→ Mirna : checkpoints at outputs/<run_id>/checkpoints/  (.pth files)
           training CSV log at outputs/<run_id>/logs/training_log.csv
→ Natalie: training CSV log to verify convergence with her augmentations
"""

import os
import sys
import time

# Windows console encoding fix (tau character support)
sys.stdout.reconfigure(encoding='utf-8')
sys.stderr.reconfigure(encoding='utf-8')
import math
import json
import argparse
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.cuda.amp import GradScaler
from sklearn.manifold import TSNE

# ── Local imports ──────────────────────────────────────────────────────────
sys.path.insert(0, str(Path(__file__).parent))
from model import get_simclr_model
from loss import get_loss
from dataset import get_train_dataloader, get_eval_dataloader


# ══════════════════════════════════════════════════════════════════════════
# 1. ARGUMENT PARSER
# ══════════════════════════════════════════════════════════════════════════

def parse_args():
    p = argparse.ArgumentParser(
        description="SimCLR Final-Phase Training — Group 20"
    )

    # ── Data ──────────────────────────────────────────────────────────
    p.add_argument("--data_dir",      type=str,   default="./data")
    p.add_argument("--output_dir",    type=str,   default="./outputs")
    p.add_argument("--exp_id",        type=int,   default=8,
        help="Augmentation config 1–8 (midterm ablations). "
             "Ignored if --aug_name is set.")
    p.add_argument("--aug_name",      type=str,   default=None,
        help="Named augmentation from Natalie's extended registry "
             "(e.g. 'randaugment', 'cutmix'). Overrides --exp_id.")

    # ── Model ─────────────────────────────────────────────────────────
    p.add_argument("--backbone",      type=str,   default="resnet50",
        choices=["resnet18", "resnet50"])
    p.add_argument("--projection_dim", type=int,  default=128)

    # ── Loss ──────────────────────────────────────────────────────────
    p.add_argument("--loss",          type=str,   default="ntxent",
        choices=["ntxent", "supcon"],
        help="'ntxent' for standard SimCLR. "
             "'supcon' for Bonus #4 hybrid training.")
    p.add_argument("--temperature",   type=float, default=0.5,
        help="NT-Xent τ. Sweep {0.1, 0.5, 1.0}.")
    p.add_argument("--label_fraction", type=float, default=0.1,
        help="Fraction of labels used for SupCon. Only active when "
             "--loss supcon. Default 0.10 = 10%%.")

    # ── Training ──────────────────────────────────────────────────────
    p.add_argument("--epochs",        type=int,   default=200)
    p.add_argument("--batch_size",    type=int,   default=1024)
    p.add_argument("--lr",            type=float, default=0.03,
        help="Peak LR for cosine schedule. "
             "Linear scaling rule: lr = 0.03 * batch_size / 256")
    p.add_argument("--auto_lr",       action="store_true", default=True,
        help="Apply linear LR scaling rule automatically.")
    p.add_argument("--weight_decay",  type=float, default=1e-4)
    p.add_argument("--warmup_epochs", type=int,   default=10,
        help="Linear warmup. Paper uses 10 epochs for 200-epoch runs.")
    p.add_argument("--optimizer",     type=str,   default="adamw",
        choices=["adamw", "sgd", "lars"],
        help="'lars' recommended for very large batches (BS >= 4096). "
             "'adamw' works well for BS 1024 on CIFAR-10.")

    # ── Checkpointing ─────────────────────────────────────────────────
    p.add_argument("--save_every",    type=int,   default=50,
        help="Save checkpoint every N epochs. "
             "Always saves at 50/100/150/200 regardless.")
    p.add_argument("--resume",        type=str,   default=None,
        help="Path to checkpoint .pth to resume from.")

    # ── Compute ───────────────────────────────────────────────────────
    p.add_argument("--num_workers",   type=int,   default=4)
    p.add_argument("--seed",          type=int,   default=42)
    p.add_argument("--no_compile",    action="store_true", default=False,
        help="Disable torch.compile (useful for debugging).")

    # ── Convenience ───────────────────────────────────────────────────
    p.add_argument("--run_all_ablations", action="store_true", default=False,
        help="Run all 8 midterm experiments sequentially in subprocesses.")
    p.add_argument("--tsne_final_only",   action="store_true", default=True)

    return p.parse_args()


# ══════════════════════════════════════════════════════════════════════════
# 2. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════

def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        cudnn.deterministic = True


# ══════════════════════════════════════════════════════════════════════════
# 3. LEARNING RATE SCHEDULE
# ══════════════════════════════════════════════════════════════════════════

def build_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    """
    Linear warmup followed by cosine annealing (step-level granularity).
    Using step-level (not epoch-level) ensures smooth LR even for large batches
    where each epoch has very few steps.
    """
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            # Linear ramp from 0 → 1
            return float(step) / max(warmup_steps, 1)
        # Cosine annealing from 1 → 0
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ══════════════════════════════════════════════════════════════════════════
# 4. OPTIMIZER FACTORY
# ══════════════════════════════════════════════════════════════════════════

def build_optimizer(model, args):
    """
    Three optimizer options:
      adamw — best for BS 512–1024, stable, our primary choice
      sgd   — used in the original SimCLR paper with LARS for large batches
      lars  — wraps SGD with layer-adaptive rate scaling; needed for BS >=4096
    """
    params = model.parameters()

    if args.optimizer == "adamw":
        return optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)

    if args.optimizer == "sgd":
        return optim.SGD(
            params, lr=args.lr,
            momentum=0.9, weight_decay=args.weight_decay, nesterov=True
        )

    if args.optimizer == "lars":
        # LARS is not in core PyTorch — use a pure-Python wrapper
        # that applies layer-wise adaptive rate scaling on top of SGD.
        try:
            from torch.optim import SGD

            class LARS(optim.Optimizer):
                """Simplified LARS (Layerwise Adaptive Rate Scaling)."""
                def __init__(self, params, lr, momentum=0.9,
                             weight_decay=1e-6, eta=1e-3):
                    defaults = dict(lr=lr, momentum=momentum,
                                    weight_decay=weight_decay, eta=eta)
                    super().__init__(params, defaults)

                @torch.no_grad()
                def step(self, closure=None):
                    loss = None
                    if closure is not None:
                        with torch.enable_grad():
                            loss = closure()
                    for group in self.param_groups:
                        for p in group["params"]:
                            if p.grad is None:
                                continue
                            param_norm = p.data.norm(2)
                            grad_norm  = p.grad.norm(2)
                            if param_norm > 0 and grad_norm > 0:
                                adaptive_lr = (
                                    group["eta"] * param_norm / (
                                        grad_norm
                                        + group["weight_decay"] * param_norm
                                        + 1e-9
                                    )
                                )
                                adaptive_lr = min(adaptive_lr, group["lr"])
                            else:
                                adaptive_lr = group["lr"]
                            d_p = p.grad + group["weight_decay"] * p.data
                            if "momentum_buffer" not in self.state[p]:
                                self.state[p]["momentum_buffer"] = torch.zeros_like(p.data)
                            buf = self.state[p]["momentum_buffer"]
                            buf.mul_(group["momentum"]).add_(d_p)
                            p.data.add_(buf, alpha=-adaptive_lr)
                    return loss

            return LARS(params, lr=args.lr, weight_decay=args.weight_decay)

        except Exception as e:
            print(f"[WARN] LARS build failed ({e}), falling back to AdamW.")
            return optim.AdamW(params, lr=args.lr,
                               weight_decay=args.weight_decay)

    raise ValueError(f"Unknown optimizer '{args.optimizer}'.")


# ══════════════════════════════════════════════════════════════════════════
# 5. CHECKPOINT UTILITIES
# ══════════════════════════════════════════════════════════════════════════

def save_checkpoint(state: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)
    print(f"  [Checkpoint] Saved → {path}")


def load_checkpoint(path: str, model, optimizer, scheduler, scaler, device):
    print(f"  [Resume] Loading checkpoint from {path}")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_loss   = ckpt.get("best_loss", float("inf"))
    loss_history = ckpt.get("loss_history", [])
    print(f"  [Resume] Resuming from epoch {start_epoch}, best loss {best_loss:.4f}")
    return start_epoch, best_loss, loss_history


# ══════════════════════════════════════════════════════════════════════════
# 6. t-SNE VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def generate_tsne(model, device, data_dir, epoch, output_dir,
                  num_workers=4, run_id="run"):
    print(f"\n  [t-SNE] Generating for epoch {epoch}...")
    model.eval()

    loader = get_eval_dataloader(
        data_dir, train=False, batch_size=512, num_workers=num_workers
    )
    all_feats, all_labels = [], []
    for imgs, lbls in loader:
        feats, _ = model(imgs.to(device))
        all_feats.append(feats.cpu().numpy())
        all_labels.append(lbls.numpy())
        if sum(x.shape[0] for x in all_feats) >= 2000:
            break

    feats  = np.concatenate(all_feats)[:2000]
    labels = np.concatenate(all_labels)[:2000]
    
    if np.isnan(feats).any():
        print("[t-SNE] Skipped because of NaN features")
        return

    emb = TSNE(
        n_components=2,
        random_state=42,
        perplexity=30
    ).fit_transform(feats)

    class_names = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck"
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10",
                    alpha=0.6, s=8)
    cb = plt.colorbar(sc, ax=ax, ticks=range(10))
    cb.ax.set_yticklabels(class_names)
    ax.set_title(f"t-SNE — Epoch {epoch}  ({run_id})", fontsize=13,
                 fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])

    save_path = os.path.join(output_dir, "plots", f"tsne_epoch_{epoch:03d}.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [t-SNE] Saved → {save_path}")
    model.train()


# ══════════════════════════════════════════════════════════════════════════
# 7. LOSS CURVE PLOT
# ══════════════════════════════════════════════════════════════════════════

def plot_loss_curve(loss_history, best_loss, output_dir, run_id):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, len(loss_history) + 1), loss_history,
            lw=2, marker="o", markersize=4)
    ax.axhline(y=best_loss, ls="--", alpha=0.6,
               label=f"Best loss = {best_loss:.4f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NT-Xent / SupCon Loss")
    ax.set_title(f"Training Loss — {run_id}", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    path = os.path.join(output_dir, "plots", "training_loss_curve.png")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [Plot] Loss curve saved → {path}")


# ══════════════════════════════════════════════════════════════════════════
# 8. DATASET HELPERS FOR SUPCON (10% LABEL FILTERING)
# ══════════════════════════════════════════════════════════════════════════

def get_supcon_dataloader(data_dir, batch_size, num_workers,
                          label_fraction=0.1, exp_id=8, seed=42):
    """
    For Bonus #4: returns a DataLoader that yields
        (view1, view2), label
    but only for `label_fraction` of CIFAR-10 training images
    (stratified per class so class balance is maintained).
    """
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader, Subset
    from dataset import AugmentedDataset, EXP_REGISTRY
    import random as _random

    rng = np.random.RandomState(seed)

    base = CIFAR10(root=data_dir, train=True, download=True)
    labels_array = np.array(base.targets)

    # Stratified sampling — keep `label_fraction` per class
    selected_indices = []
    for cls in range(10):
        cls_idx = np.where(labels_array == cls)[0].tolist()
        n_keep  = max(1, int(len(cls_idx) * label_fraction))
        chosen  = rng.choice(cls_idx, size=n_keep, replace=False).tolist()
        selected_indices.extend(chosen)

    subset    = Subset(base, selected_indices)
    transform = EXP_REGISTRY[exp_id]["transform"]
    augmented = AugmentedDataset(subset, transform)

    print(f"  [SupCon] Using {len(selected_indices):,} images "
          f"({label_fraction*100:.0f}% of CIFAR-10)")

    return DataLoader(
        augmented,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )


# ══════════════════════════════════════════════════════════════════════════
# 9. RUN ALL ABLATIONS (convenience wrapper)
# ══════════════════════════════════════════════════════════════════════════

def run_all_ablations(args):
    """
    Spawns 8 subprocesses — one per midterm augmentation config.
    Each uses --exp_id 1..8 and the epoch/batch settings you passed.
    Useful for regenerating all 8 t-SNE plots and loss curves at once.
    """
    print("\n" + "=" * 60)
    print("Running all 8 ablation experiments sequentially")
    print("=" * 60)

    for exp in range(1, 9):
        print(f"\n{'─'*50}")
        print(f"  Ablation Exp {exp}/8")
        print(f"{'─'*50}")

        cmd = [
            sys.executable, __file__,
            "--exp_id",       str(exp),
            "--epochs",       str(args.epochs),
            "--batch_size",   str(args.batch_size),
            "--backbone",     args.backbone,
            "--loss",         "ntxent",
            "--temperature",  str(args.temperature),
            "--output_dir",   os.path.join(args.output_dir, f"exp_{exp}"),
            "--data_dir",     args.data_dir,
            "--num_workers",  str(args.num_workers),
            "--seed",         str(args.seed),
        ]
        if args.no_compile:
            cmd.append("--no_compile")

        try:
            subprocess.run(cmd, check=True)
            print(f"  [OK] Experiment {exp} finished.")
        except subprocess.CalledProcessError as e:
            print(f"  [ERROR] Experiment {exp} crashed — code {e.returncode}")

    print("\n" + "=" * 60)
    print("All 8 ablations done.")
    print("=" * 60)


# ══════════════════════════════════════════════════════════════════════════
# 10. MAIN TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════

def train(args):

    # ── Seed & cuDNN ────────────────────────────────────────────────
    set_seed(args.seed)
    cudnn.benchmark = True    # fastest conv algo selection

    # ── Run ID for organized output dirs ────────────────────────────
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    aug_tag = args.aug_name if args.aug_name else f"exp{args.exp_id}"
    run_id  = (
        f"{args.backbone}_{aug_tag}_"
        f"{args.loss}_τ{args.temperature}_"
        f"bs{args.batch_size}_ep{args.epochs}_{ts}"
    )
    output_dir = os.path.join(args.output_dir, run_id)

    for sub in ("checkpoints", "plots", "logs"):
        os.makedirs(os.path.join(output_dir, sub), exist_ok=True)

    # ── Save run config to JSON (for reproducibility) ───────────────
    config_path = os.path.join(output_dir, "run_config.json")
    with open(config_path, "w") as f:
        json.dump(vars(args), f, indent=2)
    print(f"\n  [Config] Saved run config → {config_path}")

    # ── Device ──────────────────────────────────────────────────────
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"  [Device] {device}  |  AMP={use_amp}")
    if device.type == "cuda":
        print(f"  [GPU]    {torch.cuda.get_device_name(0)}")
        print(f"  [VRAM]   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    # ── Auto LR scaling ─────────────────────────────────────────────
    # Linear scaling rule: lr = base_lr × batch_size / 256
    # Reference: Goyal et al. (2017) "Accurate, Large Minibatch SGD"
    if args.auto_lr:
        args.lr = 0.03 * args.batch_size / 256
        print(f"  [LR]     Auto-scaled to {args.lr:.4f} "
              f"(0.03 × {args.batch_size} / 256)")

    # ── Data ────────────────────────────────────────────────────────
    if args.loss == "supcon":
        train_loader = get_supcon_dataloader(
            args.data_dir,
            args.batch_size,
            args.num_workers,
            label_fraction=args.label_fraction,
            exp_id=args.exp_id,
            seed=args.seed,
        )
    else:
        train_loader = get_train_dataloader(
            args.data_dir,
            batch_size=args.batch_size,
            num_workers=args.num_workers,
            exp_id=args.exp_id,
            aug_name=args.aug_name,   # None → falls back to exp_id
        )

    steps_per_epoch = len(train_loader)
    total_samples   = len(train_loader.dataset)

    print(f"\n  [Data]   {total_samples:,} training samples")
    print(f"           {steps_per_epoch} steps/epoch  |  BS={args.batch_size}")

    # ── Model ───────────────────────────────────────────────────────
    model = get_simclr_model(
        backbone=args.backbone,
        projection_dim=args.projection_dim,
    ).to(device)

    # torch.compile: TorchInductor fuses kernels → ~25-35% speedup on Ada
    # Disabled in first-run debugging with --no_compile
    if not args.no_compile and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
            print("  [Compile] torch.compile() enabled (TorchInductor backend)")
        except Exception as e:
            print(f"  [Compile] torch.compile skipped: {e}")
    else:
        print("  [Compile] torch.compile disabled")

    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  [Model]  {args.backbone} — {total_params:.2f}M parameters")

    # ── Loss ────────────────────────────────────────────────────────
    criterion = get_loss(args.loss, args.temperature).to(device)
    print(f"  [Loss]   {args.loss.upper()}  τ={args.temperature}")

    # ── Optimizer + Scheduler + Scaler ──────────────────────────────
    optimizer = build_optimizer(model, args)
    scheduler = build_scheduler(
        optimizer, args.warmup_epochs, args.epochs, steps_per_epoch
    )
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # ── Resume from checkpoint ──────────────────────────────────────
    start_epoch  = 1
    best_loss    = float("inf")
    loss_history = []

    if args.resume:
        start_epoch, best_loss, loss_history = load_checkpoint(
            args.resume, model, optimizer, scheduler, scaler, device
        )

    # ── Training loop ───────────────────────────────────────────────
    log_path = os.path.join(output_dir, "logs", "training_log.csv")
    with open(log_path, "w") as f:
        f.write("epoch,loss,lr,time_s\n")

    print(f"\n{'─'*55}")
    print(f"  {'Epoch':>6}  {'Loss':>10}  {'LR':>12}  {'Time':>8}")
    print(f"{'─'*55}")

    # Fixed checkpoint milestone epochs for Mirna's linear probe
    MILESTONE_EPOCHS = {50, 100, 150, 200}

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        running_loss = 0.0
        t0 = time.time()

        for batch in train_loader:
            (x1, x2), labels_batch = batch

            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                _, z1 = model(x1)
                _, z2 = model(x2)

                if args.loss == "ntxent":
                    loss = criterion(z1, z2)
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"[WARN] NaN loss at epoch {epoch}")
                    continue
                elif args.loss == "supcon":
                    # SupCon expects (N, n_views, D)
                    import torch.nn.functional as F
                    z1n = F.normalize(z1, dim=1)
                    z2n = F.normalize(z2, dim=1)
                    features = torch.stack([z1n, z2n], dim=1)  # (N, 2, D)
                    lbl = labels_batch.to(device, non_blocking=True)
                    loss = criterion(features, labels=lbl)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() == scale_before:
                scheduler.step()

            running_loss += loss.item()

        avg_loss   = running_loss / steps_per_epoch
        current_lr = optimizer.param_groups[0]["lr"]
        elapsed    = time.time() - t0

        loss_history.append(avg_loss)

        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss

        print(f"  {epoch:>6}  {avg_loss:>10.4f}  {current_lr:>12.6f}  {elapsed:>7.1f}s"
              f"{'  ★' if is_best else ''}")

        with open(log_path, "a") as f:
            f.write(f"{epoch},{avg_loss:.6f},{current_lr:.8f},{elapsed:.2f}\n")

        # ── Save checkpoints ──────────────────────────────────────────
        # Save at: milestone epochs, best loss, user-defined interval
        should_save = (
            epoch in MILESTONE_EPOCHS
            or epoch % args.save_every == 0
            or epoch == args.epochs
            or is_best
        )

        if should_save:
            tag    = "BEST" if is_best else f"ep{epoch:03d}"
            ckpt_path = os.path.join(
                output_dir, "checkpoints",
                f"simclr_{args.backbone}_{aug_tag}_{tag}.pth"
            )
            save_checkpoint(
                {
                    "epoch": epoch,
                    "model_state_dict":     model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict":    scaler.state_dict(),
                    "loss": avg_loss,
                    "best_loss": best_loss,
                    "loss_history": loss_history,
                    "args": vars(args),
                    "run_id": run_id,
                },
                ckpt_path,
            )

        # ── t-SNE (final epoch only by default) ──────────────────────
        if not args.tsne_final_only or epoch == args.epochs:
            generate_tsne(
                model, device, args.data_dir, epoch,
                output_dir, args.num_workers, run_id
            )

    # ── Final loss curve ────────────────────────────────────────────
    plot_loss_curve(loss_history, best_loss, output_dir, run_id)

    # ── Save encoder-only weights (for Mirna + ONNX export) ─────────
    encoder_path = os.path.join(output_dir, "simclr_encoder_final.pth")
    torch.save(model.state_dict(), encoder_path)
    print(f"\n  [Export] Encoder weights → {encoder_path}")

    # ── Summary ─────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print(f"  Run complete: {run_id}")
    print(f"  Best loss  : {best_loss:.4f}")
    print(f"  Epochs     : {args.epochs}")
    print(f"  Log CSV    : {log_path}")
    print(f"  Checkpoints: {output_dir}/checkpoints/")
    print("=" * 55 + "\n")


# ══════════════════════════════════════════════════════════════════════════
# 11. ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    args = parse_args()

    if args.run_all_ablations:
        run_all_ablations(args)
    else:
        train(args)


### Final version

**`src/Final Work/configs/augmentation.yaml`** (6 lines) — augmentation settings.

In [ ]:
%%writefile "src/Final Work/configs/augmentation.yaml"
normalization:
  cifar_mean: [0.4914, 0.4822, 0.4465]
  cifar_std: [0.2023, 0.1994, 0.2010]

color_distortion:
  s: 0.5


**`src/Final Work/run_config.json`** (25 lines) — configuration of the final runs.

In [ ]:
%%writefile "src/Final Work/run_config.json"
{
  "data_dir": "./data",
  "output_dir": "./outputs",
  "exp_id": 36,
  "aug_name": null,
  "backbone": "resnet50",
  "projection_dim": 128,
  "loss": "ntxent",
  "temperature": 0.5,
  "label_fraction": 0.1,
  "epochs": 200,
  "batch_size": 512,
  "lr": 0.03,
  "auto_lr": true,
  "weight_decay": 0.0001,
  "warmup_epochs": 10,
  "optimizer": "adamw",
  "save_every": 50,
  "resume": null,
  "num_workers": 4,
  "seed": 42,
  "no_compile": true,
  "run_all_ablations": false,
  "tsne_final_only": true
}

**`src/Final Work/augmentations.py`** (175 lines) — augmentation pipelines built from the YAML settings.

In [ ]:
%%writefile "src/Final Work/augmentations.py"

# IMPORT PART
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
import torch
import random
import yaml
import os

# Load configs - use absolute path relative to this file
config_path = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'configs', 'augmentation.yaml')
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

CIFAR_MEAN = tuple(config['normalization']['cifar_mean'])
CIFAR_STD = tuple(config['normalization']['cifar_std'])
s = config['color_distortion']['s']

# SIMCLR VIEW GENERATOR CLASS 
class SimCLRViewGenerator(object):
    """
    A custom wrapper to generate multiple augmented views of a single image,
    following the SimCLR framework for contrastive learning.
    """
    def __init__(self, base_transform, n_views=2):
        # The core stochastic augmentation pipeline to be applied to each input.
        self.base_transform = base_transform
        # Number of augmented views (typically 2 for creating positive pairs in SimCLR).
        self.n_views = n_views

    def __call__(self, x):
        """
        Executes the transformation pipeline multiple times to produce different 
        stochastic realizations of the same input image 'x'.
        """
        # Returns a list of 'n_views' variations, facilitating the calculation of contrastive loss.
        return [self.base_transform(x) for i in range(self.n_views)]


color_jitter = T.ColorJitter(0.8*s, 0.8*s, 0.8*s, 0.2*s)
# Applying jitter with 0.8 probability as per Chen et al. (2020)
rnd_color_jitter = T.RandomApply([color_jitter], p=0.8)


# EXPERIMENT 1 — Random Resized Crop (Crop + Resize)
transform_exp1 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)), 
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 2 — Crop + Flip
transform_exp2 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 3 — Crop + Color
transform_exp3 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    rnd_color_jitter,
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 4 — Crop + Grayscale
transform_exp4 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 5 — Crop + Flip + Color
transform_exp5 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    rnd_color_jitter,
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 6 — Crop + Flip + Grayscale
transform_exp6 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 7 — Crop + Color + Grayscale
transform_exp7 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    rnd_color_jitter,
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# EXP 8 — Crop + Flip + Color + Grayscale
transform_exp8 = T.Compose([
    T.RandomResizedCrop(32, scale=(0.2, 1.0)),
    T.RandomHorizontalFlip(),
    rnd_color_jitter,
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(CIFAR_MEAN, CIFAR_STD)
])

# ============================================================================
# NATALIE'S 28 EXPERIMENTS (from final augmentation.py)
# ============================================================================

class SobelTransform:
    def __call__(self, img):
        if not torch.is_tensor(img):
            img = TF.to_tensor(img)
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float().view(1, 1, 3, 3)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float().view(1, 1, 3, 3)
        gray = img.mean(dim=0, keepdim=True).unsqueeze(0)
        gx = F.conv2d(gray, sobel_x, padding=1)
        gy = F.conv2d(gray, sobel_y, padding=1)
        edge = torch.sqrt(gx**2 + gy**2).squeeze(0)
        return edge.repeat(3, 1, 1)

class GaussianNoise:
    def __init__(self, std=0.1): 
        self.std = std
    def __call__(self, img):
        if not torch.is_tensor(img): 
            img = TF.to_tensor(img)
        noise = torch.randn_like(img) * self.std
        return (img + noise).clamp(0, 1)

class DiscreteRotation:
    def __call__(self, img):
        angles = [0, 90, 180, 270]
        return TF.rotate(img, random.choice(angles))

nat_transform_exp1 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp2 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.7, scale=(0.02, 0.2), ratio=(0.3, 3.3))])
nat_transform_exp3 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp4 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp5 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp6 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp7 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp8 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp9 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp10 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp11 = T.Compose([T.RandomResizedCrop(32), rnd_color_jitter, T.ToTensor(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp12 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), rnd_color_jitter, GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp13 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp14 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp15 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp16 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomGrayscale(p=0.2), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp17 = T.Compose([T.RandomResizedCrop(32), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), SobelTransform(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp18 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), T.ToTensor(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp19 = T.Compose([T.RandomResizedCrop(32), T.ToTensor(), SobelTransform(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp20 = T.Compose([T.RandomResizedCrop(32), T.GaussianBlur(3), T.ToTensor(), GaussianNoise(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp21 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp22 = T.Compose([T.RandomResizedCrop(32), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.9, scale=(0.05, 0.4), ratio=(0.3, 3.3))])

nat_transform_exp23 = T.Compose([T.RandomResizedCrop(32, scale=(0.08, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.RandomApply([T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0))], p=0.5), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp24 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp25 = T.Compose([T.RandomResizedCrop(32, scale=(0.6, 1.0)), T.RandomHorizontalFlip(), rnd_color_jitter, T.RandomGrayscale(p=0.2), T.ToTensor(), T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), T.Normalize(CIFAR_MEAN, CIFAR_STD), T.RandomErasing(p=0.5)])
nat_transform_exp26 = T.Compose([T.RandomResizedCrop(32), T.RandomGrayscale(p=1.0), rnd_color_jitter, T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

nat_transform_exp27 = T.Compose([T.RandomResizedCrop(32, scale=(0.8, 1.0)), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
nat_transform_exp28 = T.Compose([DiscreteRotation(), T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])


**`src/Final Work/dataset.py`** (435 lines) — dataset and augmentation registry, SimCLR and SupCon loaders.

In [ ]:
%%writefile "src/Final Work/dataset.py"
"""
dataset.py — Modular Dataset & Augmentation Registry
Group 20, CISC 867, Queen's University, Spring 2026
Author (Student A): Natalie Nashed  — augmentation definitions
Author (Student B): Mahmoud Alyosify — registry architecture, SupCon loader

TWO REGISTRIES
──────────────
1. EXP_REGISTRY        — the 8 midterm ablation configs (exp_id 1–8)
2. EXTENDED_REGISTRY   — Natalie's 15–20 new augmentations (names as strings)
                          She fills in the transforms; the loader just calls
                          get_train_dataloader(..., aug_name='randaugment')

SUPCON SUPPORT
──────────────
get_supcon_dataloader() returns a labeled DataLoader for Bonus #4:
    - Stratified subsampling to `label_fraction` of the training set
    - Returns (view1, view2), label  so SupConLoss can use class info
"""

import os
import numpy as np
from torchvision.datasets import CIFAR10
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T

# ── Import from Natalie's augmentation module ───────────────────────────────
from augmentations import (
    SimCLRViewGenerator, CIFAR_MEAN, CIFAR_STD,
    transform_exp1, transform_exp2, transform_exp3, transform_exp4,
    transform_exp5, transform_exp6, transform_exp7, transform_exp8,
    nat_transform_exp1, nat_transform_exp2, nat_transform_exp3, nat_transform_exp4,
    nat_transform_exp5, nat_transform_exp6, nat_transform_exp7, nat_transform_exp8,
    nat_transform_exp9, nat_transform_exp10, nat_transform_exp11, nat_transform_exp12,
    nat_transform_exp13, nat_transform_exp14, nat_transform_exp15, nat_transform_exp16,
    nat_transform_exp17, nat_transform_exp18, nat_transform_exp19, nat_transform_exp20,
    nat_transform_exp21, nat_transform_exp22, nat_transform_exp23, nat_transform_exp24,
    nat_transform_exp25, nat_transform_exp26, nat_transform_exp27, nat_transform_exp28
)


# ══════════════════════════════════════════════════════════════════════════
# 1. REGISTRY 1 — Midterm 8-Experiment Ablation Configs
# ══════════════════════════════════════════════════════════════════════════

EXP_REGISTRY = {
    1: {
        "transform":   transform_exp1,
        "name":        "Exp1 — Crop only",
        "description": "Spatial only: RandomResizedCrop.",
    },
    2: {
        "transform":   transform_exp2,
        "name":        "Exp2 — Crop + Flip",
        "description": "Spatial: RandomResizedCrop + HorizontalFlip.",
    },
    3: {
        "transform":   transform_exp3,
        "name":        "Exp3 — Crop + ColorJitter",
        "description": "Hybrid: RandomResizedCrop + ColorJitter.",
    },
    4: {
        "transform":   transform_exp4,
        "name":        "Exp4 — Crop + Grayscale",
        "description": "Hybrid: RandomResizedCrop + RandomGrayscale.",
    },
    5: {
        "transform":   transform_exp5,
        "name":        "Exp5 — Crop + Flip + Color",
        "description": "Hybrid: RandomResizedCrop + Flip + ColorJitter.",
    },
    6: {
        "transform":   transform_exp6,
        "name":        "Exp6 — Crop + Flip + Grayscale",
        "description": "Hybrid: RandomResizedCrop + Flip + Grayscale.",
    },
    7: {
        "transform":   transform_exp7,
        "name":        "Exp7 — Crop + Color + Grayscale",
        "description": "Hybrid: RandomResizedCrop + ColorJitter + Grayscale.",
    },
    8: {
        "transform":   transform_exp8,
        "name":        "Exp8 — Full SimCLR",
        "description": "Full pipeline: Crop + Flip + ColorJitter + Grayscale.",
    },
    # NATALIE'S 28 EXPERIMENTS (Offset by 8)
    9:  {"transform": nat_transform_exp1,  "name": "Exp9(N1) — Crop + Blur", "description": "Single: Crop + Gaussian Blur."},
    10: {"transform": nat_transform_exp2,  "name": "Exp10(N2) — Crop + Cutout", "description": "Single: Crop + Random Erasing."},
    11: {"transform": nat_transform_exp3,  "name": "Exp11(N3) — Crop + Sobel", "description": "Single: Crop + Sobel Edge Extraction."},
    12: {"transform": nat_transform_exp4,  "name": "Exp12(N4) — Crop + Noise", "description": "Single: Crop + Gaussian Noise."},
    13: {"transform": nat_transform_exp5,  "name": "Exp13(N5) — Crop + Flip + Blur", "description": "Dual: Spatial + Blur."},
    14: {"transform": nat_transform_exp6,  "name": "Exp14(N6) — Crop + Flip + Cutout", "description": "Dual: Spatial + Corruption."},
    15: {"transform": nat_transform_exp7,  "name": "Exp15(N7) — Crop + Flip + Sobel", "description": "Dual: Spatial + Structural."},
    16: {"transform": nat_transform_exp8,  "name": "Exp16(N8) — Crop + Flip + Noise", "description": "Dual: Spatial + Noise."},
    17: {"transform": nat_transform_exp9,  "name": "Exp17(N9) — Crop + Color + Blur", "description": "Dual: Photometric + Blur."},
    18: {"transform": nat_transform_exp10, "name": "Exp18(N10) — Crop + Color + Cutout", "description": "Dual: Photometric + Corruption."},
    19: {"transform": nat_transform_exp11, "name": "Exp19(N11) — Crop + Color + Sobel", "description": "Dual: Photometric + Structural."},
    20: {"transform": nat_transform_exp12, "name": "Exp20(N12) — Crop + Color + Noise", "description": "Dual: Photometric + Noise."},
    21: {"transform": nat_transform_exp13, "name": "Exp21(N13) — Crop + Gray + Blur", "description": "Dual: Grayscale + Blur."},
    22: {"transform": nat_transform_exp14, "name": "Exp22(N14) — Crop + Gray + Cutout", "description": "Dual: Grayscale + Corruption."},
    23: {"transform": nat_transform_exp15, "name": "Exp23(N15) — Crop + Gray + Sobel", "description": "Dual: Grayscale + Structural."},
    24: {"transform": nat_transform_exp16, "name": "Exp24(N16) — Crop + Gray + Noise", "description": "Dual: Grayscale + Noise."},
    25: {"transform": nat_transform_exp17, "name": "Exp25(N17) — Crop + Blur + Sobel", "description": "Conflict: Smooth vs Sharp Edge."},
    26: {"transform": nat_transform_exp18, "name": "Exp26(N18) — Crop + Flip + Blur + Cutout", "description": "Conflict: Content Erasure with Blur."},
    27: {"transform": nat_transform_exp19, "name": "Exp27(N19) — Crop + Sobel + Noise", "description": "Conflict: Edge Extraction vs Pixel Noise."},
    28: {"transform": nat_transform_exp20, "name": "Exp28(N20) — Crop + Blur + Noise", "description": "Conflict: High-frequency Distortion (JPEG-like)."},
    29: {"transform": nat_transform_exp21, "name": "Exp29(N21) — Crop + Flip + Color + Erasing", "description": "Conflict: Color Distortion with Patch Erasure."},
    30: {"transform": nat_transform_exp22, "name": "Exp30(N22) — Crop + Heavy Cutout Grid", "description": "Extreme Spatial Corruption Baseline."},
    31: {"transform": nat_transform_exp23, "name": "Exp31(N23) — SimCLR Classic Strong", "description": "Standard SimCLR Pipeline (Chen et al. 2020)."},
    32: {"transform": nat_transform_exp24, "name": "Exp32(N24) — SimCLR + Rotation Task", "description": "Pipeline for E-SSL Joint-Task Pretraining."},
    33: {"transform": nat_transform_exp25, "name": "Exp33(N25) — Full Heatmap Suite", "description": "All Invariance Transforms Active Simultaneously."},
    34: {"transform": nat_transform_exp26, "name": "Exp34(N26) — Color Destruction", "description": "Strict Color Elimination: Grayscale p=1.0 + ColorJitter."},
    35: {"transform": nat_transform_exp27, "name": "Exp35(N27) — Weak Baseline", "description": "Minimal Spatial Invariance Only."},
    36: {"transform": nat_transform_exp28, "name": "Exp36(N28) — Pure Discrete Rotation", "description": "Isolated Structural Rotation Alignment."},
}


# ══════════════════════════════════════════════════════════════════════════
# 2. REGISTRY 2 — Extended Augmentation Registry (Natalie fills these in)
#
#    Each entry key is the --aug_name CLI argument.
#    Natalie: add your transforms below following the same pattern.
#    Mahmoud's train_master.py will pick them up automatically.
# ══════════════════════════════════════════════════════════════════════════

def _base_crop():
    """All new augs build on top of the base SimCLR crop."""
    return T.RandomResizedCrop(32, scale=(0.2, 1.0))

# ── Placeholder entries — Natalie fills in the transform pipelines ─────────
# For each entry, replace T.Compose([_base_crop(), T.ToTensor(), ...])
# with your actual pipeline.

EXTENDED_REGISTRY = {

    # ── Natalie: fill these in ────────────────────────────────────────

    "randaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.RandAugment(num_ops=2, magnitude=9),   # NATALIE: tune params
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "RandAugment",
        "description": "AutoML-style random augmentation policy search.",
    },

    "autoaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.AutoAugment(policy=T.AutoAugmentPolicy.CIFAR10),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "AutoAugment (CIFAR-10 policy)",
        "description": "Learned CIFAR-10 augmentation policy (Cubuk et al.).",
    },

    "trivialaugment": {
        "transform": T.Compose([
            _base_crop(),
            T.TrivialAugmentWide(),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "TrivialAugmentWide",
        "description": "Parameter-free augmentation (Müller & Hutter, 2021).",
    },

    "solarize": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomSolarize(threshold=128, p=0.2),   # NATALIE: tune threshold
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Solarize",
        "description": "Exp8 + RandomSolarize (inspired by SimCLRv2).",
    },

    "equalize": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomEqualize(p=0.2),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Equalize",
        "description": "Exp8 + RandomEqualize histogram normalization.",
    },

    "sharpness": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomAdjustSharpness(sharpness_factor=2, p=0.3),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Sharpness",
        "description": "Exp8 + RandomAdjustSharpness.",
    },

    "rotation": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomRotation(degrees=15),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Rotation",
        "description": "Exp8 + RandomRotation ±15°.",
    },

    "perspective": {
        "transform": T.Compose([
            _base_crop(),
            T.RandomHorizontalFlip(),
            T.RandomPerspective(distortion_scale=0.2, p=0.3),
            T.RandomApply([T.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
            T.RandomGrayscale(p=0.2),
            T.ToTensor(),
            T.Normalize(CIFAR_MEAN, CIFAR_STD),
        ]),
        "name":        "Full SimCLR + Perspective",
        "description": "Exp8 + RandomPerspective distortion.",
    },

    # NOTE: Mixup and CutMix require batch-level operations — they are
    # implemented in the training loop via torchvision.transforms.v2.
    # Natalie: implement these as collate_fn wrappers in augmentations.py
    # and register them here by referencing your collate_fn.
    # Mahmoud's DataLoader accepts an optional collate_fn parameter.
    "mixup": {
        "transform": transform_exp8,   # NATALIE: replace with Mixup-aware transform
        "name":        "Mixup (placeholder)",
        "description": "Batch-level Mixup — requires custom collate_fn.",
        "collate_fn":  None,           # NATALIE: set to your mixup_collate_fn
    },

    "cutmix": {
        "transform": transform_exp8,   # NATALIE: replace with CutMix-aware transform
        "name":        "CutMix (placeholder)",
        "description": "Batch-level CutMix — requires custom collate_fn.",
        "collate_fn":  None,           # NATALIE: set to your cutmix_collate_fn
    },
}


# ══════════════════════════════════════════════════════════════════════════
# 3. AugmentedDataset — returns (view1, view2), label
# ══════════════════════════════════════════════════════════════════════════

class AugmentedDataset(Dataset):
    """
    Wraps any torchvision dataset to return two augmented views.
    Compatible with both NT-Xent (labels ignored) and SupCon (labels used).
    """
    def __init__(self, dataset, transform):
        self.dataset   = dataset
        self.transform = SimCLRViewGenerator(transform, n_views=2)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, label = self.dataset[idx]
        views = self.transform(img)      # [view_1, view_2]
        return (views[0], views[1]), label


# ══════════════════════════════════════════════════════════════════════════
# 4. DATALOADER FACTORIES
# ══════════════════════════════════════════════════════════════════════════

def _resolve_transform(exp_id: int, aug_name: str):
    """
    Resolves which transform to use.
    Priority: aug_name (extended registry) > exp_id (midterm registry)
    """
    if aug_name is not None:
        aug_name = aug_name.lower().strip()
        if aug_name not in EXTENDED_REGISTRY:
            available = list(EXTENDED_REGISTRY.keys())
            raise ValueError(
                f"Unknown aug_name='{aug_name}'. "
                f"Available: {available}"
            )
        info = EXTENDED_REGISTRY[aug_name]
        print(f"  [Dataset] Augmentation (extended): {info['name']}")
        print(f"            {info['description']}")
        return info["transform"], info.get("collate_fn", None)

    # Fallback to midterm exp_id
    if exp_id not in EXP_REGISTRY:
        raise ValueError(
            f"Invalid exp_id={exp_id}. Choose from {list(EXP_REGISTRY.keys())}."
        )
    info = EXP_REGISTRY[exp_id]
    print(f"  [Dataset] Augmentation (midterm): {info['name']}")
    print(f"            {info['description']}")
    return info["transform"], None


def get_train_dataloader(
    data_dir: str,
    batch_size: int   = 128,
    num_workers: int  = 4,
    shuffle: bool     = True,
    exp_id: int       = 8,
    aug_name: str     = None,
) -> DataLoader:
    """
    Returns a training DataLoader yielding ((view1, view2), label).

    Args:
        data_dir    : root directory for CIFAR-10 download
        batch_size  : training batch size
        num_workers : DataLoader worker processes
        shuffle     : shuffle training data
        exp_id      : midterm ablation config 1–8 (overridden by aug_name)
        aug_name    : named augmentation from EXTENDED_REGISTRY
    """
    transform, collate_fn = _resolve_transform(exp_id, aug_name)

    os.makedirs(data_dir, exist_ok=True)
    base_dataset = CIFAR10(root=data_dir, train=True, download=True)
    augmented    = AugmentedDataset(base_dataset, transform)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,      # ensures consistent batch size for NT-Xent
        persistent_workers=(num_workers > 0),
    )
    if collate_fn is not None:
        loader_kwargs["collate_fn"] = collate_fn

    return DataLoader(augmented, **loader_kwargs)


def get_supcon_dataloader(
    data_dir: str,
    batch_size: int     = 256,
    num_workers: int    = 4,
    label_fraction: float = 0.1,
    exp_id: int         = 8,
    aug_name: str       = None,
    seed: int           = 42,
) -> DataLoader:
    """
    Returns a labeled DataLoader for Bonus #4 (SupCon training).
    Subsamples `label_fraction` of the CIFAR-10 training set, stratified
    per class so class balance is maintained.

    Args:
        label_fraction: 0.1 = 10% = 5,000 images (500 per class)
    """
    transform, collate_fn = _resolve_transform(exp_id, aug_name)
    rng = np.random.RandomState(seed)

    os.makedirs(data_dir, exist_ok=True)
    base_dataset  = CIFAR10(root=data_dir, train=True, download=True)
    labels_array  = np.array(base_dataset.targets)

    # Stratified subsampling
    selected = []
    for cls in range(10):
        cls_idx = np.where(labels_array == cls)[0].tolist()
        n_keep  = max(1, int(len(cls_idx) * label_fraction))
        chosen  = rng.choice(cls_idx, size=n_keep, replace=False).tolist()
        selected.extend(chosen)

    print(f"  [SupCon] {len(selected):,} labeled images "
          f"({label_fraction*100:.0f}% of 50k, stratified per class)")

    subset    = Subset(base_dataset, selected)
    augmented = AugmentedDataset(subset, transform)

    loader_kwargs = dict(
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
        persistent_workers=(num_workers > 0),
    )
    if collate_fn is not None:
        loader_kwargs["collate_fn"] = collate_fn

    return DataLoader(augmented, **loader_kwargs)


def get_eval_dataloader(
    data_dir: str,
    train: bool      = False,
    batch_size: int  = 256,
    num_workers: int = 4,
) -> DataLoader:
    """
    Returns an evaluation DataLoader (no augmentation, no shuffling).
    Used for: t-SNE generation, linear probe caching, FAISS index building.
    """
    os.makedirs(data_dir, exist_ok=True)

    eval_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    dataset = CIFAR10(
        root=data_dir, train=train, download=True, transform=eval_transform
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=(num_workers > 0),
    )


**`src/Final Work/loss.py`** (314 lines) — NT-Xent and SupCon losses.

In [ ]:
%%writefile "src/Final Work/loss.py"
"""
loss.py — NT-Xent + SupConLoss
Group 20, CISC 867, Queen's University, Spring 2026
Author (Student B): Mahmoud Alyosify

Two loss functions in one file:
  • NTXentLoss   — used for ALL self-supervised (unlabeled) pretraining
  • SupConLoss   — Bonus #4, Khosla et al. (NeurIPS 2020)
                   used for the hybrid 10%-label supervised-contrastive run

SupCon paper: https://arxiv.org/abs/2004.11362

Key design difference vs NT-Xent
────────────────────────────────
NT-Xent defines ONE positive per anchor (its augmented twin).
SupCon defines ALL same-class embeddings in the batch as positives.
When labels are present, the model can exploit the richer supervision
signal while still using unlabeled negatives for discrimination.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


# ══════════════════════════════════════════════════════════════════════
# 1. NT-Xent Loss  (unchanged from midterm, kept here for single import)
# ══════════════════════════════════════════════════════════════════════

class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy loss.
    For a batch of N images → 2N views (two augmented copies each).
    Each view's positive is its twin; all 2(N-1) other views are negatives.

    Args:
        temperature (float): τ scaling factor. Default 0.5 (paper default).
    """

    def __init__(self, temperature: float = 0.5):
        super().__init__()
        self.temperature = temperature

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> torch.Tensor:
        """
        Args:
            z1, z2: projection embeddings, shape (N, D). NOT yet normalized.
        Returns:
            Scalar NT-Xent loss averaged over both views.
        """
        N = z1.shape[0]
        device = z1.device

        # ℓ₂-normalize so cosine sim = dot product
        z1 = F.normalize(z1, dim=1)
        z2 = F.normalize(z2, dim=1)

        # Stack into (2N, D) — [z1_0, z1_1, …, z2_0, z2_1, …]
        z = torch.cat([z1, z2], dim=0)                         # (2N, D)

        # Full similarity matrix scaled by temperature
        sim = torch.mm(z, z.T) / self.temperature             # (2N, 2N)

        # Mask out self-similarity (diagonal)
        diag_mask = torch.eye(2 * N, dtype=torch.bool, device=device)
        sim = sim.masked_fill(diag_mask, float("-inf"))

        # Positive targets: view i pairs with view i+N (and vice versa)
        targets = torch.cat([
            torch.arange(N, 2 * N, device=device),
            torch.arange(0, N, device=device),
        ])

        return F.cross_entropy(sim, targets)


# ══════════════════════════════════════════════════════════════════════
# 2. SupCon Loss  (Bonus #4)  — Khosla et al., NeurIPS 2020
# ══════════════════════════════════════════════════════════════════════

class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., NeurIPS 2020).

    Supports two modes controlled by `labels`:
      • labels=None  → self-supervised mode (same as NT-Xent with two views)
      • labels given → supervised mode: all same-class embeddings are positives

    For Bonus #4 we use SUPERVISED mode with 10% of CIFAR-10 labels.

    Mathematical formulation (paper Eq. 2):
    ┌──────────────────────────────────────────────────────────────────┐
    │  L = Σᵢ  -1/|P(i)| · Σₚ∈P(i)  log  exp(zᵢ·zₚ/τ)             │
    │                                      ──────────────────────     │
    │                                      Σₐ∈A(i) exp(zᵢ·zₐ/τ)     │
    │                                                                  │
    │  P(i) = set of positives for anchor i (same class, not self)    │
    │  A(i) = all indices except i (the denominator negatives)        │
    └──────────────────────────────────────────────────────────────────┘

    Args:
        temperature (float): τ. Recommended 0.07–0.5 for SupCon.
        contrast_mode (str): 'all'  — use all views as anchors (default)
                             'one'  — use only first view as anchor
        base_temperature (float): τ_base for loss re-scaling. Default 0.07
                                   as in the original paper.
    """

    def __init__(
        self,
        temperature: float = 0.07,
        contrast_mode: str = "all",
        base_temperature: float = 0.07,
    ):
        super().__init__()
        assert temperature > 0, "Temperature must be positive."
        assert contrast_mode in ("all", "one"), \
            "contrast_mode must be 'all' or 'one'."

        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(
        self,
        features: torch.Tensor,
        labels: torch.Tensor = None,
        mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Args:
            features: shape (N, n_views, D) — ℓ₂-normalized embeddings.
                      For two augmented views: n_views = 2.
                      Each slice features[:, v, :] is one view of the batch.
            labels:   shape (N,) — class labels for the N images.
                      Pass None for self-supervised mode (treats each sample
                      as its own class, recovers standard contrastive loss).
            mask:     shape (N, N) — binary mask where mask[i,j]=1 means i
                      and j form a positive pair. Overrides labels if given.
        Returns:
            Scalar supervised contrastive loss.
        """
        device = features.device

        # ── Input validation ─────────────────────────────────────────
        if features.ndim == 2:
            # Caller passed (N, D) — assume single view, unsqueeze
            features = features.unsqueeze(1)

        batch_size, n_views, feat_dim = features.shape

        # ── Build positive-pair mask ──────────────────────────────────
        if mask is not None:
            # Explicit mask provided — trust the caller
            mask = mask.float().to(device)

        elif labels is not None:
            # Supervised: mask[i,j] = 1 iff label[i] == label[j]
            labels = labels.contiguous().view(-1, 1)          # (N, 1)
            mask = torch.eq(labels, labels.T).float().to(device)  # (N, N)

        else:
            # Self-supervised fallback: each sample is its own class
            mask = torch.eye(batch_size, dtype=torch.float32, device=device)

        # ── Flatten views into one big (N*n_views, D) tensor ─────────
        # Anchor features — determined by contrast_mode
        if self.contrast_mode == "one":
            anchor_feat = features[:, 0, :]                   # (N, D)
            anchor_count = 1
        else:  # "all"
            # Stack all views: each becomes an anchor
            anchor_feat = features.view(
                batch_size * n_views, feat_dim
            )                                                  # (N*v, D)
            anchor_count = n_views

        # Contrast features = ALL views of ALL samples
        contrast_feat = features.view(
            batch_size * n_views, feat_dim
        )                                                      # (N*v, D)
        contrast_count = n_views

        # ── Similarity matrix ─────────────────────────────────────────
        # anchor_dot_contrast: (N*anchor_count, N*contrast_count)
        anchor_dot_contrast = (
            torch.mm(anchor_feat, contrast_feat.T) / self.temperature
        )

        # For numerical stability: subtract row-wise max
        logits_max, _ = anchor_dot_contrast.max(dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # ── Expand mask to cover all view combinations ─────────────────
        # Original mask is (N, N). We tile it to (N*anchor_count, N*contrast_count)
        # so that every view of sample i treats every view of a same-class
        # sample j as a positive.
        mask_tiled = mask.repeat(anchor_count, contrast_count)   # (N*a, N*c)

        # Remove self-contrast (an anchor must not compare with itself)
        self_contrast_mask = torch.scatter(
            torch.ones_like(mask_tiled),
            1,
            torch.arange(batch_size * anchor_count, device=device).view(-1, 1),
            0,
        )
        mask_tiled = mask_tiled * self_contrast_mask

        # ── Log-softmax over negatives ────────────────────────────────
        exp_logits = torch.exp(logits) * self_contrast_mask    # zero out self
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-9)

        # ── Mean over positives ───────────────────────────────────────
        # For each anchor, average log-prob over its positives
        mean_log_prob_pos = (mask_tiled * log_prob).sum(1) / (
            mask_tiled.sum(1) + 1e-9
        )

        # ── Final loss (temperature re-scaling as in paper Eq. 2) ─────
        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss


# ══════════════════════════════════════════════════════════════════════
# 3. Loss Factory  — used by train_master.py and train_supcon.py
# ══════════════════════════════════════════════════════════════════════

def get_loss(loss_type: str, temperature: float = 0.5) -> nn.Module:
    """
    Factory function so all training scripts import losses by name.

    Args:
        loss_type   : 'ntxent' | 'supcon'
        temperature : τ value (same for both losses)
    Returns:
        Instantiated loss module.
    """
    if loss_type == "ntxent":
        return NTXentLoss(temperature=temperature)

    if loss_type == "supcon":
        # SupCon paper uses τ=0.07 by default; we allow override
        return SupConLoss(
            temperature=temperature,
            contrast_mode="all",
            base_temperature=0.07,
        )

    raise ValueError(
        f"Unknown loss '{loss_type}'. Choose 'ntxent' or 'supcon'."
    )


# ══════════════════════════════════════════════════════════════════════
# 4. Smoke Tests
# ══════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    import math

    print("=" * 60)
    print("Smoke Test — NTXentLoss")
    print("=" * 60)

    crit = NTXentLoss(temperature=0.5)
    N, D = 32, 128

    z1 = torch.randn(N, D)
    z2 = torch.randn(N, D)
    loss_rand = crit(z1, z2)
    loss_expected = math.log(2 * N - 1)

    print(f"Random embeddings loss : {loss_rand.item():.4f}")
    print(f"Expected (uniform)     : {loss_expected:.4f}")

    loss_perfect = crit(z1, z1.clone())
    print(f"Identical embeddings   : {loss_perfect.item():.6f}  (should be ~0)")
    assert loss_rand.shape == torch.Size([]), "Not a scalar!"
    print("PASSED ✓\n")

    # ──────────────────────────────────────────────────────────────────
    print("=" * 60)
    print("Smoke Test — SupConLoss (supervised mode, Bonus #4)")
    print("=" * 60)

    supcon = SupConLoss(temperature=0.07)
    N_sup = 16
    n_views = 2
    D_sup = 128
    n_classes = 10

    # Simulate two augmented views: features shape (N, n_views, D)
    feats = F.normalize(
        torch.randn(N_sup, n_views, D_sup), dim=2
    )
    labels = torch.randint(0, n_classes, (N_sup,))

    loss_sup = supcon(feats, labels=labels)
    print(f"SupCon loss (supervised, 2 views) : {loss_sup.item():.4f}")
    assert loss_sup.shape == torch.Size([]), "Not a scalar!"
    assert not torch.isnan(loss_sup), "NaN detected!"
    print("PASSED ✓\n")

    # Self-supervised fallback (no labels)
    loss_self = supcon(feats, labels=None)
    print(f"SupCon loss (self-supervised mode) : {loss_self.item():.4f}")
    assert not torch.isnan(loss_self), "NaN detected!"
    print("PASSED ✓\n")

    print("=" * 60)
    print("All loss tests passed.")
    print("=" * 60)


**`src/Final Work/model.py`** (182 lines) — ResNet encoder with projection head.

In [ ]:
%%writefile "src/Final Work/model.py"
import torch
import torch.nn as nn
import torchvision.models as models


class ProjectionHead(nn.Module):

    def __init__(
        self,
        input_dim=512,
        hidden_dim=512,
        output_dim=128
    ):

        super().__init__()

        self.layers = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
                bias=False
            ),

            nn.BatchNorm1d(hidden_dim),

            nn.ReLU(inplace=True),

            nn.Linear(
                hidden_dim,
                output_dim
            )
        )

    def forward(self, x):

        return self.layers(x)


class SimCLRResNet18(nn.Module):

    def __init__(self, projection_dim=128):

        super().__init__()

        backbone = models.resnet18(weights=None)

        # adjust first layer for CIFAR-10 images
        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove original maxpool
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,

            backbone.avgpool
        )

        self.feature_dim = 512

        self.projector = ProjectionHead(
            input_dim=self.feature_dim,
            hidden_dim=self.feature_dim,
            output_dim=projection_dim
        )

    def forward(self, x):

        features = self.encoder(x)

        features = torch.flatten(
            features,
            start_dim=1
        )

        projections = self.projector(features)

        return features, projections


class SimCLRResNet50(nn.Module):
    """
    ResNet-50 backbone for SimCLR with CIFAR-10 stem adjustment.
    Output: (h, z) where h is the 2048-dim hidden representation
    and z is the 128-dim projection (or custom projection_dim).
    """

    def __init__(self, projection_dim=128):

        super().__init__()

        backbone = models.resnet50(weights=None)

        # adjust first layer for CIFAR-10 images (32x32)
        backbone.conv1 = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        # remove original maxpool
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(

            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,

            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        # global average pooling
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

        # projection head: ResNet50 outputs 2048-dim features
        self.projection = ProjectionHead(
            input_dim=2048,
            hidden_dim=2048,
            output_dim=projection_dim
        )

    def forward(self, x):
        """
        Args:
            x: (N, 3, 32, 32) CIFAR-10 batch
        Returns:
            h: (N, 2048) encoder output
            z: (N, projection_dim) projection output
        """
        x = self.encoder(x)            # (N, 2048, 1, 1)
        x = self.pool(x)               # (N, 2048, 1, 1)
        h = x.flatten(start_dim=1)     # (N, 2048)
        z = self.projection(h)         # (N, projection_dim)
        return h, z


def get_simclr_model(
    backbone="resnet18",
    projection_dim=128
):

    if backbone == "resnet18":

        return SimCLRResNet18(
            projection_dim=projection_dim
        )

    elif backbone == "resnet50":

        return SimCLRResNet50(
            projection_dim=projection_dim
        )

    raise ValueError(
        f"Backbone '{backbone}' is not supported. "
        f"Supported: resnet18, resnet50"
    )

**`src/Final Work/train_master.py`** (709 lines) — SimCLR training script (final phase).

In [ ]:
%%writefile "src/Final Work/train_master.py"
"""
train_master.py — SimCLR Final-Phase Master Training Script
Group 20, CISC 867, Queen's University, Spring 2026
Author (Student B): Mahmoud Alyosify

PURPOSE
───────
This is THE production training script for the final 200-epoch ResNet-50
SimCLR run on the RTX 5000 Ada (32 GB VRAM). It also supports:
  • Quick hyperparameter sweeps (τ, batch size, LR)
  • The 8 midterm ablation experiments (--exp_id 1-8)
  • SupCon Bonus #4 (--loss supcon --label_fraction 0.1)

PERFORMANCE FEATURES ENABLED
─────────────────────────────
1. torch.amp.autocast    — FP16 activations, halves memory bandwidth
2. torch.compile()       — TorchInductor/Triton fusion, ~25-35% speedup
3. cudnn.benchmark=True  — fastest conv algorithm auto-select
4. pin_memory + non_blocking — overlap CPU→GPU transfers with compute
5. drop_last=True        — ensures every batch has exactly N samples
                           (critical for NT-Xent similarity matrix)

USAGE EXAMPLES
──────────────
# The 200-epoch master run (wait for Natalie's best aug combo):
python train_master.py --epochs 200 --batch_size 1024 --backbone resnet50

# Quick 8-experiment ablation (all 8 midterm configs in sequence):
python train_master.py --run_all_ablations --epochs 20 --batch_size 128

# Temperature sweep:
python train_master.py --temperature 0.1 --exp_id 8 --epochs 50
python train_master.py --temperature 0.5 --exp_id 8 --epochs 50
python train_master.py --temperature 1.0 --exp_id 8 --epochs 50

# Batch size sweep:
python train_master.py --batch_size 256 --exp_id 8 --epochs 50
python train_master.py --batch_size 512 --exp_id 8 --epochs 50
python train_master.py --batch_size 1024 --exp_id 8 --epochs 50

# Bonus #4 — SupCon with 10% labels:
python train_master.py --loss supcon --label_fraction 0.1 --epochs 200

HAND-OFFS
─────────
→ Mirna : checkpoints at outputs/<run_id>/checkpoints/  (.pth files)
           training CSV log at outputs/<run_id>/logs/training_log.csv
→ Natalie: training CSV log to verify convergence with her augmentations
"""

import os
import sys
import time
import math
import json
import argparse
import subprocess
from datetime import datetime
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.cuda.amp import GradScaler
from sklearn.manifold import TSNE

# ── Local imports ──────────────────────────────────────────────────────────
sys.path.insert(0, str(Path(__file__).parent))
from model import get_simclr_model
from loss import get_loss
from dataset import get_train_dataloader, get_eval_dataloader


# ══════════════════════════════════════════════════════════════════════════
# 1. ARGUMENT PARSER
# ══════════════════════════════════════════════════════════════════════════

def parse_args():
    p = argparse.ArgumentParser(
        description="SimCLR Final-Phase Training — Group 20"
    )

    # ── Data ──────────────────────────────────────────────────────────
    p.add_argument("--data_dir",      type=str,   default="./data")
    p.add_argument("--output_dir",    type=str,   default="./outputs")
    p.add_argument("--exp_id",        type=int,   default=8,
        help="Augmentation config 1–8 (midterm ablations). "
             "Ignored if --aug_name is set.")
    p.add_argument("--aug_name",      type=str,   default=None,
        help="Named augmentation from Natalie's extended registry "
             "(e.g. 'randaugment', 'cutmix'). Overrides --exp_id.")

    # ── Model ─────────────────────────────────────────────────────────
    p.add_argument("--backbone",      type=str,   default="resnet50",
        choices=["resnet18", "resnet50"])
    p.add_argument("--projection_dim", type=int,  default=128)

    # ── Loss ──────────────────────────────────────────────────────────
    p.add_argument("--loss",          type=str,   default="ntxent",
        choices=["ntxent", "supcon"],
        help="'ntxent' for standard SimCLR. "
             "'supcon' for Bonus #4 hybrid training.")
    p.add_argument("--temperature",   type=float, default=0.5,
        help="NT-Xent τ. Sweep {0.1, 0.5, 1.0}.")
    p.add_argument("--label_fraction", type=float, default=0.1,
        help="Fraction of labels used for SupCon. Only active when "
             "--loss supcon. Default 0.10 = 10%%.")

    # ── Training ──────────────────────────────────────────────────────
    p.add_argument("--epochs",        type=int,   default=200)
    p.add_argument("--batch_size",    type=int,   default=1024)
    p.add_argument("--lr",            type=float, default=0.03,
        help="Peak LR for cosine schedule. "
             "Linear scaling rule: lr = 0.03 * batch_size / 256")
    p.add_argument("--auto_lr",       action="store_true", default=True,
        help="Apply linear LR scaling rule automatically.")
    p.add_argument("--weight_decay",  type=float, default=1e-4)
    p.add_argument("--warmup_epochs", type=int,   default=10,
        help="Linear warmup. Paper uses 10 epochs for 200-epoch runs.")
    p.add_argument("--optimizer",     type=str,   default="adamw",
        choices=["adamw", "sgd", "lars"],
        help="'lars' recommended for very large batches (BS >= 4096). "
             "'adamw' works well for BS 1024 on CIFAR-10.")

    # ── Checkpointing ─────────────────────────────────────────────────
    p.add_argument("--save_every",    type=int,   default=50,
        help="Save checkpoint every N epochs. "
             "Always saves at 50/100/150/200 regardless.")
    p.add_argument("--resume",        type=str,   default=None,
        help="Path to checkpoint .pth to resume from.")

    # ── Compute ───────────────────────────────────────────────────────
    p.add_argument("--num_workers",   type=int,   default=4)
    p.add_argument("--seed",          type=int,   default=42)
    p.add_argument("--no_compile",    action="store_true", default=False,
        help="Disable torch.compile (useful for debugging).")

    # ── Convenience ───────────────────────────────────────────────────
    p.add_argument("--run_all_ablations", action="store_true", default=False,
        help="Run all 8 midterm experiments sequentially in subprocesses.")
    p.add_argument("--tsne_final_only",   action="store_true", default=True)

    return p.parse_args()


# ══════════════════════════════════════════════════════════════════════════
# 2. REPRODUCIBILITY
# ══════════════════════════════════════════════════════════════════════════

def set_seed(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        cudnn.deterministic = True


# ══════════════════════════════════════════════════════════════════════════
# 3. LEARNING RATE SCHEDULE
# ══════════════════════════════════════════════════════════════════════════

def build_scheduler(optimizer, warmup_epochs, total_epochs, steps_per_epoch):
    """
    Linear warmup followed by cosine annealing (step-level granularity).
    Using step-level (not epoch-level) ensures smooth LR even for large batches
    where each epoch has very few steps.
    """
    warmup_steps = warmup_epochs * steps_per_epoch
    total_steps  = total_epochs  * steps_per_epoch

    def lr_lambda(step):
        if step < warmup_steps:
            # Linear ramp from 0 → 1
            return float(step) / max(warmup_steps, 1)
        # Cosine annealing from 1 → 0
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1.0 + math.cos(math.pi * progress))

    return optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ══════════════════════════════════════════════════════════════════════════
# 4. OPTIMIZER FACTORY
# ══════════════════════════════════════════════════════════════════════════

def build_optimizer(model, args):
    """
    Three optimizer options:
      adamw — best for BS 512–1024, stable, our primary choice
      sgd   — used in the original SimCLR paper with LARS for large batches
      lars  — wraps SGD with layer-adaptive rate scaling; needed for BS >=4096
    """
    params = model.parameters()

    if args.optimizer == "adamw":
        return optim.AdamW(params, lr=args.lr, weight_decay=args.weight_decay)

    if args.optimizer == "sgd":
        return optim.SGD(
            params, lr=args.lr,
            momentum=0.9, weight_decay=args.weight_decay, nesterov=True
        )

    if args.optimizer == "lars":
        # LARS is not in core PyTorch — use a pure-Python wrapper
        # that applies layer-wise adaptive rate scaling on top of SGD.
        try:
            from torch.optim import SGD

            class LARS(optim.Optimizer):
                """Simplified LARS (Layerwise Adaptive Rate Scaling)."""
                def __init__(self, params, lr, momentum=0.9,
                             weight_decay=1e-6, eta=1e-3):
                    defaults = dict(lr=lr, momentum=momentum,
                                    weight_decay=weight_decay, eta=eta)
                    super().__init__(params, defaults)

                @torch.no_grad()
                def step(self, closure=None):
                    loss = None
                    if closure is not None:
                        with torch.enable_grad():
                            loss = closure()
                    for group in self.param_groups:
                        for p in group["params"]:
                            if p.grad is None:
                                continue
                            param_norm = p.data.norm(2)
                            grad_norm  = p.grad.norm(2)
                            if param_norm > 0 and grad_norm > 0:
                                adaptive_lr = (
                                    group["eta"] * param_norm / (
                                        grad_norm
                                        + group["weight_decay"] * param_norm
                                        + 1e-9
                                    )
                                )
                                adaptive_lr = min(adaptive_lr, group["lr"])
                            else:
                                adaptive_lr = group["lr"]
                            d_p = p.grad + group["weight_decay"] * p.data
                            if "momentum_buffer" not in self.state[p]:
                                self.state[p]["momentum_buffer"] = torch.zeros_like(p.data)
                            buf = self.state[p]["momentum_buffer"]
                            buf.mul_(group["momentum"]).add_(d_p)
                            p.data.add_(buf, alpha=-adaptive_lr)
                    return loss

            return LARS(params, lr=args.lr, weight_decay=args.weight_decay)

        except Exception as e:
            print(f"[WARN] LARS build failed ({e}), falling back to AdamW.")
            return optim.AdamW(params, lr=args.lr,
                               weight_decay=args.weight_decay)

    raise ValueError(f"Unknown optimizer '{args.optimizer}'.")


# ══════════════════════════════════════════════════════════════════════════
# 5. CHECKPOINT UTILITIES
# ══════════════════════════════════════════════════════════════════════════

def save_checkpoint(state: dict, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(state, path)
    print(f"  [Checkpoint] Saved → {path}")


def load_checkpoint(path: str, model, optimizer, scheduler, scaler, device):
    print(f"  [Resume] Loading checkpoint from {path}")
    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_loss   = ckpt.get("best_loss", float("inf"))
    loss_history = ckpt.get("loss_history", [])
    print(f"  [Resume] Resuming from epoch {start_epoch}, best loss {best_loss:.4f}")
    return start_epoch, best_loss, loss_history


# ══════════════════════════════════════════════════════════════════════════
# 6. t-SNE VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def generate_tsne(model, device, data_dir, epoch, output_dir,
                  num_workers=4, run_id="run"):
    print(f"\n  [t-SNE] Generating for epoch {epoch}...")
    model.eval()

    loader = get_eval_dataloader(
        data_dir, train=False, batch_size=512, num_workers=num_workers
    )
    all_feats, all_labels = [], []
    for imgs, lbls in loader:
        feats, _ = model(imgs.to(device))
        all_feats.append(feats.cpu().numpy())
        all_labels.append(lbls.numpy())
        if sum(x.shape[0] for x in all_feats) >= 2000:
            break

    feats  = np.concatenate(all_feats)[:2000]
    labels = np.concatenate(all_labels)[:2000]
    
    if np.isnan(feats).any():
        print("[t-SNE] Skipped because of NaN features")
        return

    emb = TSNE(
        n_components=2,
        random_state=42,
        perplexity=30
    ).fit_transform(feats)

    class_names = [
        "airplane", "automobile", "bird", "cat", "deer",
        "dog", "frog", "horse", "ship", "truck"
    ]
    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(emb[:, 0], emb[:, 1], c=labels, cmap="tab10",
                    alpha=0.6, s=8)
    cb = plt.colorbar(sc, ax=ax, ticks=range(10))
    cb.ax.set_yticklabels(class_names)
    ax.set_title(f"t-SNE — Epoch {epoch}  ({run_id})", fontsize=13,
                 fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])

    save_path = os.path.join(output_dir, "plots", f"tsne_epoch_{epoch:03d}.png")
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [t-SNE] Saved → {save_path}")
    model.train()


# ══════════════════════════════════════════════════════════════════════════
# 7. LOSS CURVE PLOT
# ══════════════════════════════════════════════════════════════════════════

def plot_loss_curve(loss_history, best_loss, output_dir, run_id):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(range(1, len(loss_history) + 1), loss_history,
            lw=2, marker="o", markersize=4)
    ax.axhline(y=best_loss, ls="--", alpha=0.6,
               label=f"Best loss = {best_loss:.4f}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("NT-Xent / SupCon Loss")
    ax.set_title(f"Training Loss — {run_id}", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(True, alpha=0.3)
    path = os.path.join(output_dir, "plots", "training_loss_curve.png")
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  [Plot] Loss curve saved → {path}")


# ══════════════════════════════════════════════════════════════════════════
# 8. DATASET HELPERS FOR SUPCON (10% LABEL FILTERING)
# ══════════════════════════════════════════════════════════════════════════

def get_supcon_dataloader(data_dir, batch_size, num_workers,
                          label_fraction=0.1, exp_id=8, seed=42):
    """
    For Bonus #4: returns a DataLoader that yields
        (view1, view2), label
    but only for `label_fraction` of CIFAR-10 training images
    (stratified per class so class balance is maintained).
    """
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader, Subset
    from dataset import AugmentedDataset, EXP_REGISTRY
    import random as _random

    rng = np.random.RandomState(seed)

    base = CIFAR10(root=data_dir, train=True, download=True)
    labels_array = np.array(base.targets)

    # Stratified sampling — keep `label_fraction` per class
    selected_indices = []
    for cls in range(10):
        cls_idx = np.where(labels_array == cls)[0].tolist()
        n_keep  = max(1, int(len(cls_idx) * label_fraction))
        chosen  = rng.choice(cls_idx, size=n_keep, replace=False).tolist()
        selected_indices.extend(chosen)

    subset    = Subset(base, selected_indices)
    transform = EXP_REGISTRY[exp_id]["transform"]
    augmented = AugmentedDataset(subset, transform)

    print(f"  [SupCon] Using {len(selected_indices):,} images "
          f"({label_fraction*100:.0f}% of CIFAR-10)")

    return DataLoader(
        augmented,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True,
    )


# ══════════════════════════════════════════════════════════════════════════
# 9. RUN ALL ABLATIONS (convenience wrapper)
# ══════════════════════════════════════════════════════════════════════════

def run_all_ablations(args):
    """
    Spawns 8 subprocesses — one per midterm augmentation config.
    Each uses --exp_id 1..8 and the epoch/batch settings you passed.
    Useful for regenerating all 8 t-SNE plots and loss curves at once.
    """
    print("\n" + "=" * 60)
    print("Running all 8 ablation experiments sequentially")
    print("=" * 60)

    for exp in range(1, 9):
        print(f"\n{'─'*50}")
        print(f"  Ablation Exp {exp}/8")
        print(f"{'─'*50}")

        cmd = [
            sys.executable, __file__,
            "--exp_id",       str(exp),
            "--epochs",       str(args.epochs),
            "--batch_size",   str(args.batch_size),
            "--backbone",     args.backbone,
            "--loss",         "ntxent",
            "--temperature",  str(args.temperature),
            "--output_dir",   os.path.join(args.output_dir, f"exp_{exp}"),
            "--data_dir",     args.data_dir,
            "--num_workers",  str(args.num_workers),
            "--seed",         str(args.seed),
        ]
        if args.no_compile:
            cmd.append("--no_compile")

        try:
            subprocess.run(cmd, check=True)
            print(f"  [OK] Experiment {exp} finished.")
        except subprocess.CalledProcessError as e:
            print(f"  [ERROR] Experiment {exp} crashed — code {e.returncode}")

    print("\n" + "=" * 60)
    print("All 8 ablations done.")
    print("=" * 60)


# ══════════════════════════════════════════════════════════════════════════
# 10. MAIN TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════

def train(args):

    # ── Seed & cuDNN ────────────────────────────────────────────────
    set_seed(args.seed)
    cudnn.benchmark = True    # fastest conv algo selection

    # ── Run ID for organized output dirs ────────────────────────────
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    aug_tag = args.aug_name if args.aug_name else f"exp{args.exp_id}"
    run_id  = (
        f"{args.backbone}_{aug_tag}_"
        f"{args.loss}_τ{args.temperature}_"
        f"bs{args.batch_size}_ep{args.epochs}_{ts}"
    )
    output_dir = os.path.join(args.output_dir, run_id)

    for sub in ("checkpoints", "plots", "logs"):
        os.makedirs(os.path.join(output_dir, sub), exist_ok=True)

    # ── Save run config to JSON (for reproducibility) ───────────────
    config_path = os.path.join(output_dir, "run_config.json")
    with open(config_path, "w") as f:
        json.dump(vars(args), f, indent=2)
    print(f"\n  [Config] Saved run config → {config_path}")

    # ── Device ──────────────────────────────────────────────────────
    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = device.type == "cuda"
    print(f"  [Device] {device}  |  AMP={use_amp}")
    if device.type == "cuda":
        print(f"  [GPU]    {torch.cuda.get_device_name(0)}")
        print(f"  [VRAM]   {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    # ── Auto LR scaling ─────────────────────────────────────────────
    # Linear scaling rule: lr = base_lr × batch_size / 256
    # Reference: Goyal et al. (2017) "Accurate, Large Minibatch SGD"
    if args.auto_lr:
        args.lr = 0.03 * args.batch_size / 256
        print(f"  [LR]     Auto-scaled to {args.lr:.4f} "
              f"(0.03 × {args.batch_size} / 256)")

    # ── Data ────────────────────────────────────────────────────────
    if args.loss == "supcon":
        train_loader = get_supcon_dataloader(
            args.data_dir,
            args.batch_size,
            args.num_workers,
            label_fraction=args.label_fraction,
            exp_id=args.exp_id,
            seed=args.seed,
        )
    else:
        train_loader = get_train_dataloader(
            args.data_dir,
            batch_size=args.batch_size,
            num_workers=args.num_workers,
            exp_id=args.exp_id,
            aug_name=args.aug_name,   # None → falls back to exp_id
        )

    steps_per_epoch = len(train_loader)
    total_samples   = len(train_loader.dataset)

    print(f"\n  [Data]   {total_samples:,} training samples")
    print(f"           {steps_per_epoch} steps/epoch  |  BS={args.batch_size}")

    # ── Model ───────────────────────────────────────────────────────
    model = get_simclr_model(
        backbone=args.backbone,
        projection_dim=args.projection_dim,
    ).to(device)

    # torch.compile: TorchInductor fuses kernels → ~25-35% speedup on Ada
    # Disabled in first-run debugging with --no_compile
    if not args.no_compile and hasattr(torch, "compile"):
        try:
            model = torch.compile(model)
            print("  [Compile] torch.compile() enabled (TorchInductor backend)")
        except Exception as e:
            print(f"  [Compile] torch.compile skipped: {e}")
    else:
        print("  [Compile] torch.compile disabled")

    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  [Model]  {args.backbone} — {total_params:.2f}M parameters")

    # ── Loss ────────────────────────────────────────────────────────
    criterion = get_loss(args.loss, args.temperature).to(device)
    print(f"  [Loss]   {args.loss.upper()}  τ={args.temperature}")

    # ── Optimizer + Scheduler + Scaler ──────────────────────────────
    optimizer = build_optimizer(model, args)
    scheduler = build_scheduler(
        optimizer, args.warmup_epochs, args.epochs, steps_per_epoch
    )
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    # ── Resume from checkpoint ──────────────────────────────────────
    start_epoch  = 1
    best_loss    = float("inf")
    loss_history = []

    if args.resume:
        start_epoch, best_loss, loss_history = load_checkpoint(
            args.resume, model, optimizer, scheduler, scaler, device
        )

    # ── Training loop ───────────────────────────────────────────────
    log_path = os.path.join(output_dir, "logs", "training_log.csv")
    with open(log_path, "w") as f:
        f.write("epoch,loss,lr,time_s\n")

    print(f"\n{'─'*55}")
    print(f"  {'Epoch':>6}  {'Loss':>10}  {'LR':>12}  {'Time':>8}")
    print(f"{'─'*55}")

    # Fixed checkpoint milestone epochs for Mirna's linear probe
    MILESTONE_EPOCHS = {50, 100, 150, 200}

    for epoch in range(start_epoch, args.epochs + 1):
        model.train()
        running_loss = 0.0
        t0 = time.time()

        for batch in train_loader:
            (x1, x2), labels_batch = batch

            x1 = x1.to(device, non_blocking=True)
            x2 = x2.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                _, z1 = model(x1)
                _, z2 = model(x2)

                if args.loss == "ntxent":
                    loss = criterion(z1, z2)
                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"[WARN] NaN loss at epoch {epoch}")
                    continue
                elif args.loss == "supcon":
                    # SupCon expects (N, n_views, D)
                    import torch.nn.functional as F
                    z1n = F.normalize(z1, dim=1)
                    z2n = F.normalize(z2, dim=1)
                    features = torch.stack([z1n, z2n], dim=1)  # (N, 2, D)
                    lbl = labels_batch.to(device, non_blocking=True)
                    loss = criterion(features, labels=lbl)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)    
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            if scaler.get_scale() == scale_before:
                scheduler.step()

            running_loss += loss.item()

        avg_loss   = running_loss / steps_per_epoch
        current_lr = optimizer.param_groups[0]["lr"]
        elapsed    = time.time() - t0

        loss_history.append(avg_loss)

        is_best = avg_loss < best_loss
        if is_best:
            best_loss = avg_loss

        print(f"  {epoch:>6}  {avg_loss:>10.4f}  {current_lr:>12.6f}  {elapsed:>7.1f}s"
              f"{'  ★' if is_best else ''}")

        with open(log_path, "a") as f:
            f.write(f"{epoch},{avg_loss:.6f},{current_lr:.8f},{elapsed:.2f}\n")

        # ── Save checkpoints ──────────────────────────────────────────
        # Save at: milestone epochs, best loss, user-defined interval
        should_save = (
            epoch in MILESTONE_EPOCHS
            or epoch % args.save_every == 0
            or epoch == args.epochs
            or is_best
        )

        if should_save:
            tag    = "BEST" if is_best else f"ep{epoch:03d}"
            ckpt_path = os.path.join(
                output_dir, "checkpoints",
                f"simclr_{args.backbone}_{aug_tag}_{tag}.pth"
            )
            save_checkpoint(
                {
                    "epoch": epoch,
                    "model_state_dict":     model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scheduler_state_dict": scheduler.state_dict(),
                    "scaler_state_dict":    scaler.state_dict(),
                    "loss": avg_loss,
                    "best_loss": best_loss,
                    "loss_history": loss_history,
                    "args": vars(args),
                    "run_id": run_id,
                },
                ckpt_path,
            )

        # ── t-SNE (final epoch only by default) ──────────────────────
        if not args.tsne_final_only or epoch == args.epochs:
            generate_tsne(
                model, device, args.data_dir, epoch,
                output_dir, args.num_workers, run_id
            )

    # ── Final loss curve ────────────────────────────────────────────
    plot_loss_curve(loss_history, best_loss, output_dir, run_id)

    # ── Save encoder-only weights (for Mirna + ONNX export) ─────────
    encoder_path = os.path.join(output_dir, "simclr_encoder_final.pth")
    torch.save(model.state_dict(), encoder_path)
    print(f"\n  [Export] Encoder weights → {encoder_path}")

    # ── Summary ─────────────────────────────────────────────────────
    print("\n" + "=" * 55)
    print(f"  Run complete: {run_id}")
    print(f"  Best loss  : {best_loss:.4f}")
    print(f"  Epochs     : {args.epochs}")
    print(f"  Log CSV    : {log_path}")
    print(f"  Checkpoints: {output_dir}/checkpoints/")
    print("=" * 55 + "\n")


# ══════════════════════════════════════════════════════════════════════════
# 11. ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    args = parse_args()

    if args.run_all_ablations:
        run_all_ablations(args)
    else:
        train(args)


**`src/Final Work/supcon/dataset_subset.py`** (232 lines) — stratified 10% CIFAR-10 subset for SupCon.

In [ ]:
%%writefile "src/Final Work/supcon/dataset_subset.py"
"""
dataset_subset.py — Stratified 10% Subset Sampler for SupCon Training
──────────────────────────────────────────────────────────────────────
Creates a perfectly balanced subset of CIFAR-10 using stratified sampling.

Why 10%? (Semi-supervised angle)
  In semi-supervised learning, we assume labels are expensive. By training on 
  just 10% of labels using SupCon, we show that contrastive learning can build
  strong representations from limited supervision.
  
  Expected improvement: 60-75% linear eval accuracy
  vs 40-50% with supervised CE loss on 10% labels
  vs 70-80% with 100% labels + CE loss

Stratified Sampling:
  Instead of random sampling (which might get 600 cats, 400 dogs):
  - Take exactly 500 images from each of 10 CIFAR-10 classes
  - Total: 500 * 10 = 5,000 images (exactly 10% of 50,000 training set)
  - This ensures balanced class distribution during training
"""

import numpy as np
import torch
from torch.utils.data import Subset
from torchvision.datasets import CIFAR10


CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

NUM_CLASSES = 10
SAMPLES_PER_CLASS = 500  # 10% of 5000 per class


def get_stratified_subset(
    dataset,
    fraction=0.1,
    num_classes=10,
    random_seed=42,
    verbose=True
):
    """
    Create a stratified subset of a dataset (e.g., CIFAR-10).
    
    Args:
        dataset (torch.utils.data.Dataset): Full dataset (e.g., CIFAR10)
            Must have `targets` attribute (list of labels)
        
        fraction (float): Fraction of data to sample (default 0.1 = 10%)
        
        num_classes (int): Number of classes in dataset (default 10 for CIFAR-10)
        
        random_seed (int): RNG seed for reproducibility
        
        verbose (bool): Print sampling statistics
    
    Returns:
        subset (torch.utils.data.Subset): Subset containing exactly
            `fraction * len(dataset)` samples, balanced across classes
    
    Example:
        >>> train_set = CIFAR10('./data', train=True, download=True)
        >>> subset = get_stratified_subset(train_set, fraction=0.1)
        >>> print(f"Subset size: {len(subset)}")  # Should be ~5000
        >>> dataloader = DataLoader(subset, batch_size=512, shuffle=True)
    """
    
    np.random.seed(random_seed)
    torch.manual_seed(random_seed)
    
    # ── Extract targets from dataset ─────────────────────────────────
    if hasattr(dataset, 'targets'):
        targets = np.array(dataset.targets)
    elif hasattr(dataset, 'labels'):
        targets = np.array(dataset.labels)
    else:
        raise ValueError(
            f"Dataset {type(dataset)} has no 'targets' or 'labels' attribute. "
            "Cannot extract labels for stratification."
        )
    
    total_samples = len(dataset)
    samples_per_class = int(total_samples * fraction / num_classes)
    
    indices = []
    
    # ── Stratify: sample exactly `samples_per_class` from each class ──
    for class_idx in range(num_classes):
        # Find all indices with this class label
        class_indices = np.where(targets == class_idx)[0]
        
        # Randomly sample `samples_per_class` from this class
        sampled_indices = np.random.choice(
            class_indices,
            size=min(samples_per_class, len(class_indices)),
            replace=False
        )
        
        indices.extend(sampled_indices)
    
    indices = np.array(indices)
    
    # ── Shuffle the combined indices ─────────────────────────────────
    np.random.shuffle(indices)
    
    # ── Create Subset ────────────────────────────────────────────────
    subset = Subset(dataset, indices.tolist())
    
    # ── Print statistics ─────────────────────────────────────────────
    if verbose:
        print("\n" + "="*70)
        print("  STRATIFIED SUBSET SAMPLING")
        print("="*70)
        print(f"  Original dataset size:  {total_samples:,} samples")
        print(f"  Subset fraction:        {fraction:.1%}")
        print(f"  Subset size:            {len(subset):,} samples")
        print(f"  Samples per class:      {samples_per_class:,}")
        print(f"  Number of classes:      {num_classes}")
        print()
        
        # Verify balance
        subset_targets = targets[indices]
        for class_idx in range(num_classes):
            count = np.sum(subset_targets == class_idx)
            class_name = CIFAR10_CLASSES[class_idx] if num_classes == 10 else f"class_{class_idx}"
            print(f"    {class_name:15s}: {count:4d} samples")
        
        print("="*70 + "\n")
    
    return subset


def get_balanced_dataloader(
    dataset,
    fraction=0.1,
    batch_size=512,
    num_workers=4,
    shuffle=True,
    random_seed=42
):
    """
    Convenience function: create stratified subset + DataLoader in one call.
    
    Args:
        dataset: Full dataset (CIFAR-10)
        fraction: Fraction to subset (0.1 = 10%)
        batch_size: Batch size for DataLoader
        num_workers: Number of workers for DataLoader
        shuffle: Whether to shuffle the DataLoader
        random_seed: RNG seed
    
    Returns:
        dataloader (torch.utils.data.DataLoader)
        subset (torch.utils.data.Subset)
    
    Example:
        >>> from torchvision.datasets import CIFAR10
        >>> train_set = CIFAR10('./data', train=True, download=True)
        >>> loader, subset = get_balanced_dataloader(train_set, batch_size=512)
        >>> for batch_idx, (images, labels) in enumerate(loader):
        ...     print(f"Batch {batch_idx}: {images.shape}, {labels.shape}")
    """
    
    from torch.utils.data import DataLoader
    
    subset = get_stratified_subset(
        dataset,
        fraction=fraction,
        random_seed=random_seed
    )
    
    dataloader = DataLoader(
        subset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True  # Ensures all batches have exactly batch_size samples
    )
    
    return dataloader, subset


# ─────────────────────────────────────────────────────────────────
# TEST & VERIFICATION
# ─────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    """
    Quick test to verify stratified sampling works correctly.
    
    Run with: python dataset_subset.py
    """
    
    from torchvision.datasets import CIFAR10
    from torch.utils.data import DataLoader
    
    # Download CIFAR-10 (if not already)
    print("Loading CIFAR-10...")
    train_set = CIFAR10('./data', train=True, download=True)
    
    # Create 10% stratified subset
    print("\nCreating 10% stratified subset...")
    subset = get_stratified_subset(train_set, fraction=0.1, verbose=True)
    
    # Verify with DataLoader
    print("Creating DataLoader from subset...")
    dataloader = DataLoader(
        subset,
        batch_size=512,
        shuffle=True,
        num_workers=0,  # Use 0 for testing
        drop_last=True
    )
    
    print(f"Number of batches: {len(dataloader)}")
    
    # Check first batch
    images, labels = next(iter(dataloader))
    print(f"\nFirst batch:")
    print(f"  Images shape: {images.shape}")
    print(f"  Labels shape: {labels.shape}")
    print(f"  Unique labels in batch: {torch.unique(labels).tolist()}")
    print(f"  Label counts in batch:")
    for class_idx in range(10):
        count = (labels == class_idx).sum().item()
        if count > 0:
            print(f"    Class {class_idx}: {count}")
    
    print("\n✅ Stratified sampling test passed!")


**`src/Final Work/supcon/loss_supcon.py`** (116 lines) — SupCon loss (Khosla et al., 2020).

In [ ]:
%%writefile "src/Final Work/supcon/loss_supcon.py"
"""
loss_supcon.py — Supervised Contrastive Loss (Khosla et al., 2020)
───────────────────────────────────────────────────────────────────
Official implementation from: https://github.com/HobbitLong/SupContrast

Mathematical formulation (Khosla et al., NeurIPS 2020, Eq. 2):

    L_sup = (1/N) Σᵢ  (-1/|P(i)|) Σₚ∈P(i)  log [
        exp(zᵢ·zₚ/τ) / Σₐ∈A(i) exp(zᵢ·zₐ/τ)
    ]

Where:
  • zᵢ, zₚ: normalized embeddings
  • P(i): positive set (same class as i, excluding i)
  • A(i): all indices except i (in-batch negatives)
  • τ: temperature scaling (default 0.1, per paper Section 4.5)
"""

import torch
import torch.nn as nn


class SupConLoss(nn.Module):
    """
    Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    
    Adapted from: https://github.com/HobbitLong/SupContrast
    
    It also supports the unsupervised contrastive loss in SimCLR.
    """

    def __init__(self, temperature=0.1, contrast_mode='all',
                 base_temperature=0.1):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """
        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = features.device

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             ' at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # Modified to handle edge case where mask.sum(1) could be 0
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(
            mask_pos_pairs < 1e-6, torch.ones_like(mask_pos_pairs), mask_pos_pairs
        )
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = -(self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss


**`src/Final Work/supcon/train_supcon.py`** (480 lines) — SupCon training on the 10% subset.

In [ ]:
%%writefile "src/Final Work/supcon/train_supcon.py"

import os
import sys
import time
import json
import argparse
from pathlib import Path
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from torchvision import transforms

# Ensure we can import from src/
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
sys.path.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))

# -- Local imports ----------------------------------------------------
try:
    from loss_supcon import SupConLoss
    from dataset_subset import get_stratified_subset
    from model import SimCLRResNet50, ProjectionHead
    from augmentations import SimCLRViewGenerator, CIFAR_MEAN, CIFAR_STD
except ImportError as e:
    print(f"Error importing local modules: {e}")
    print("Make sure you're running from the project folder")
    sys.exit(1)


# -- CIFAR-10 configuration -------------------------------------------
NUM_CLASSES = 10
CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]


class AugmentedDataset(torch.utils.data.Dataset):
    """
    Wrapper to return two augmented views + labels for SupCon training.
    
    Each __getitem__ returns:
        (view_1, view_2, label)
    
    where view_1 and view_2 are independent augmentations of the same image.
    """
    
    def __init__(self, cifar_dataset, augmentation_transform):
        self.dataset = cifar_dataset
        self.augmentation = augmentation_transform
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        image, label = self.dataset[idx]
        
        # Two independent augmentations
        view_1 = self.augmentation(image)
        view_2 = self.augmentation(image)
        
        return view_1, view_2, label


def build_augmentation():
    """
    Build the augmentation pipeline for SupCon training.
    
    Uses SimCLR's augmentation strategy:
      • RandomResizedCrop (0.2-1.0 scale)
      • RandomHorizontalFlip
      • RandomApply ColorJitter
      • RandomApply GaussianBlur (if available)
      • RandomGrayscale
      • Normalize with CIFAR-10 statistics
    """
    
    s = 0.5  # Color distortion strength (from SimCLR paper)
    
    augmentation = transforms.Compose([
        transforms.RandomResizedCrop(32, scale=(0.2, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomApply([
            transforms.ColorJitter(
                brightness=0.8*s,
                contrast=0.8*s,
                saturation=0.8*s,
                hue=0.2*s
            )
        ], p=0.8),
        transforms.RandomApply([
            transforms.GaussianBlur(kernel_size=3)
        ], p=0.1),
        transforms.RandomGrayscale(p=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
    ])
    
    return augmentation


def create_datasets(data_dir, fraction=0.1, batch_size=512, num_workers=4):
    """
    Create stratified training subset + DataLoader for SupCon.
    
    Args:
        data_dir: Where to download/cache CIFAR-10
        fraction: Fraction of data (0.1 = 10%)
        batch_size: Batch size
        num_workers: Number of workers
    
    Returns:
        train_loader: DataLoader
        subset: The Subset object (for statistics)
    """
    
    # Download CIFAR-10 (no transforms here, augmentation happens in AugmentedDataset)
    train_set = CIFAR10(
        data_dir,
        train=True,
        download=True,
        transform=None
    )
    
    # Create stratified subset (5000 samples for 10%)
    subset = get_stratified_subset(
        train_set,
        fraction=fraction,
        verbose=True
    )
    
    # Wrap with augmentation
    augmentation = build_augmentation()
    augmented_subset = AugmentedDataset(subset, augmentation)
    
    # Create DataLoader
    train_loader = DataLoader(
        augmented_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )
    
    return train_loader, subset


def build_model_and_optimizer(learning_rate, use_lars=True):
    """
    Build ResNet-50 encoder + projection head + optimizer.
    
    Uses SGD + momentum 0.9 (Khosla et al., 2020, Section 4.1):
      • LR = 0.05 for batch_size=512
      • Weight decay = 1e-4
      • Momentum = 0.9
    
    Args:
        learning_rate: Base learning rate (0.05 for batch 512)
        use_lars: Use LARS optimizer (recommended for batch >= 4096)
    
    Returns:
        encoder: ResNet-50 backbone
        optimizer: SGD or LARS
    """
    
    # ResNet-50 encoder
    encoder = SimCLRResNet50(projection_dim=128)
    encoder.eval()  # Will be set to train() in training loop
    
    # Collect parameters
    params = list(encoder.parameters())
    
    if use_lars:
        try:
            from torchlars import LARS
            base_optimizer = optim.SGD(
                params,
                lr=learning_rate,
                momentum=0.9,
                weight_decay=1e-4
            )
            optimizer = LARS(base_optimizer)
        except ImportError:
            print("⚠️ torchlars not found. Using SGD + momentum (SupCon default).")
            optimizer = optim.SGD(
                params,
                lr=learning_rate,
                momentum=0.9,
                weight_decay=1e-4
            )
    else:
        # SGD + momentum 0.9 (matches SupCon paper exactly)
        optimizer = optim.SGD(
            params,
            lr=learning_rate,
            momentum=0.9,
            weight_decay=1e-4
        )
    
    return encoder, optimizer


def build_scheduler(optimizer, total_epochs, warmup_epochs=10):
    """Build cosine annealing scheduler (warmup handled per-step in train_epoch)."""
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_epochs, eta_min=0
    )
    return scheduler


def warmup_learning_rate(warmup_epochs, warmup_from, warmup_to,
                         epoch, batch_id, total_batches, optimizer):
    """Per-step linear warmup (matching official SupContrast repo)."""
    if epoch <= warmup_epochs:
        p = (batch_id + (epoch - 1) * total_batches) / \
            (warmup_epochs * total_batches)
        lr = warmup_from + p * (warmup_to - warmup_from)
        for param_group in optimizer.param_groups:
            param_group['lr'] = lr


def train_epoch(
    epoch,
    train_loader,
    encoder,
    criterion,
    optimizer,
    warmup_epochs,
    warmup_from,
    warmup_to,
    device
):
    """
    Single epoch of SupCon training (FP32).
    Per-step warmup during first warmup_epochs (matching official repo).
    """
    
    encoder.train()
    loss_meter = 0.0
    time_start = time.time()
    total_batches = len(train_loader)
    
    for batch_idx, (view1, view2, labels) in enumerate(train_loader):
        view1 = view1.to(device)
        view2 = view2.to(device)
        labels = labels.to(device)
        
        # Per-step warmup (official SupContrast approach)
        warmup_learning_rate(warmup_epochs, warmup_from, warmup_to,
                             epoch, batch_idx, total_batches, optimizer)
        
        optimizer.zero_grad()
        
        # Forward pass (FP32)
        _, z1 = encoder(view1)
        _, z2 = encoder(view2)
        
        # L2-normalize projections
        z1 = torch.nn.functional.normalize(z1, dim=1)
        z2 = torch.nn.functional.normalize(z2, dim=1)
        
        # Stack for SupConLoss: [N, 2, 128]
        features = torch.stack([z1, z2], dim=1)
        
        # Compute loss
        loss = criterion(features, labels)
        
        # Backward
        loss.backward()
        optimizer.step()
        
        loss_meter += loss.item()
    
    time_elapsed = time.time() - time_start
    loss_avg = loss_meter / len(train_loader)
    
    return loss_avg, time_elapsed


def train(args):
    """
    Main training loop for SupCon Stage 1.
    """
    
    # -- Setup --------------------------------------------------------
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    cudnn.benchmark = True
    
    # Create output directory
    run_id = f"supcon_resnet50_frac{int(args.fraction*100)}_bs{args.batch_size}_ep{args.epochs}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    output_dir = Path(f"./outputs/{run_id}")
    output_dir.mkdir(parents=True, exist_ok=True)
    
    (output_dir / "checkpoints").mkdir(exist_ok=True)
    (output_dir / "logs").mkdir(exist_ok=True)
    (output_dir / "plots").mkdir(exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"  SUPERVISED CONTRASTIVE LEARNING (SupCon) -- STAGE 1")
    print(f"  Khosla et al. (2020)")
    print(f"{'='*70}")
    print(f"\n  [Config] Run ID: {run_id}")
    print(f"  [Device] {device}")
    if device.type == 'cuda':
        print(f"  [GPU]    {torch.cuda.get_device_name(0)}")
        vram = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"  [VRAM]   {vram:.1f} GB")
    
    # -- Create datasets and loaders ----------------------------------
    print(f"\n  [Data] Loading CIFAR-10 ({args.fraction:.1%} subset)...")
    train_loader, subset = create_datasets(
        args.data_dir,
        fraction=args.fraction,
        batch_size=args.batch_size,
        num_workers=args.num_workers
    )
    print(f"         Subset size: {len(subset):,} samples")
    print(f"         Batch size: {args.batch_size}")
    print(f"         Steps/epoch: {len(train_loader)}")
    
    # -- Build model --------------------------------------------------
    print(f"\n  [Model] ResNet-50 + 128-dim projection")
    encoder, optimizer = build_model_and_optimizer(
        args.learning_rate,
        use_lars=args.use_lars
    )
    encoder = encoder.to(device)
    
    print(f"         Parameters: {sum(p.numel() for p in encoder.parameters()) / 1e6:.2f}M")
    
    # -- Loss and scheduler -------------------------------------------
    criterion = SupConLoss(temperature=0.1)  # Paper Section 4.5: "All our results used τ=0.1"
    
    warmup_epochs = 10
    warmup_from = 0.01
    scheduler = build_scheduler(optimizer, args.epochs, warmup_epochs=warmup_epochs)
    
    # Compute warmup target LR (from official SupContrast repo)
    import math
    warmup_to = args.learning_rate * (
        1 + math.cos(math.pi * warmup_epochs / args.epochs)) / 2
    
    print(f"  [Loss]  SupConLoss (tau={criterion.temperature})")
    print(f"  [Optim] {'LARS' if args.use_lars else 'SGD'} (momentum=0.9, wd=1e-4)")
    print(f"  [Sched] Cosine annealing + warmup ({warmup_epochs} epochs, from {warmup_from})")
    
    # -- Training loop ------------------------------------------------
    print(f"\n{'-'*70}")
    print(f"   Epoch      Loss          LR        Time")
    print(f"{'-'*70}")
    
    log_data = []
    best_loss = float('inf')
    best_epoch = 0
    
    for epoch in range(1, args.epochs + 1):
        loss, time_taken = train_epoch(
            epoch,
            train_loader,
            encoder,
            criterion,
            optimizer,
            warmup_epochs,
            warmup_from,
            warmup_to,
            device
        )
        
        # Step cosine scheduler after warmup ends
        if epoch > warmup_epochs:
            scheduler.step()
        
        current_lr = optimizer.param_groups[0]['lr']
        
        # Logging
        log_data.append({
            'epoch': epoch,
            'loss': loss,
            'learning_rate': current_lr,
            'time': time_taken
        })
        
        # Print
        marker = "*" if loss < best_loss else " "
        print(f"   {epoch:3d}    {loss:.4f}     {current_lr:.6f}   {time_taken:6.1f}s  {marker}")
        
        # Save best checkpoint
        if loss < best_loss:
            best_loss = loss
            best_epoch = epoch
            torch.save(
                encoder.state_dict(),
                output_dir / "checkpoints" / "supcon_best.pth"
            )
        
        # Save periodic checkpoints
        if epoch % 50 == 0:
            torch.save(
                encoder.state_dict(),
                output_dir / "checkpoints" / f"supcon_epoch_{epoch:03d}.pth"
            )
    
    print(f"{'-'*70}\n")
    
    # -- Save results -------------------------------------------------
    print(f"  [Checkpoint] Best model > {output_dir}/checkpoints/supcon_best.pth")
    
    # Save encoder for linear evaluation
    torch.save(
        encoder.state_dict(),
        output_dir / "supcon_encoder_final.pth"
    )
    print(f"  [Encoder] Saved for Phase 2 > {output_dir}/supcon_encoder_final.pth")
    
    # Save CSV log
    import csv
    with open(output_dir / "logs" / "training_log.csv", 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'loss', 'learning_rate', 'time'])
        writer.writeheader()
        writer.writerows(log_data)
    print(f"  [Log] CSV saved > {output_dir}/logs/training_log.csv")
    
    # Save config
    config = vars(args)
    config['run_id'] = run_id
    config['best_loss'] = best_loss
    config['best_epoch'] = best_epoch
    with open(output_dir / "run_config.json", 'w') as f:
        json.dump(config, f, indent=2)
    
    print(f"\n{'='*70}")
    print(f"  ✅ SupCon training complete!")
    print(f"  Best loss: {best_loss:.4f} (epoch {best_epoch})")
    print(f"  Total epochs: {args.epochs}")
    print(f"  Output directory: {output_dir}")
    print(f"{'='*70}\n")


def parse_args():
    """Parse command-line arguments."""
    
    parser = argparse.ArgumentParser(
        description="SupCon Stage 1 Training -- Khosla et al. (2020)"
    )
    
    # Data
    parser.add_argument('--data_dir', type=str, default='./data',
                        help='Path to CIFAR-10 data')
    parser.add_argument('--fraction', type=float, default=0.1,
                        help='Fraction of data to use (0.1 = 10%%)')
    parser.add_argument('--num_workers', type=int, default=0,
                        help='DataLoader workers (0 for Windows compatibility)')
    
    # Model
    parser.add_argument('--backbone', type=str, default='resnet50',
                        help='Backbone architecture')
    
    # Training
    parser.add_argument('--epochs', type=int, default=200,
                        help='Number of epochs')
    parser.add_argument('--batch_size', type=int, default=512,
                        help='Batch size')
    parser.add_argument('--learning_rate', type=float, default=0.05,
                        help='Base learning rate (SupCon paper: 0.05 for batch 512)')
    parser.add_argument('--use_lars', action='store_true', default=False,
                        help='Use LARS optimizer (if available)')
    
    return parser.parse_args()


if __name__ == '__main__':
    args = parse_args()
    train(args)


**`src/Final Work/run_final_ablations_18_may.py`** (91 lines) — launches the final augmentation ablations, one folder per experiment.

In [ ]:
%%writefile "src/Final Work/run_final_ablations_18_may.py"


#import's part
import os
import subprocess
import shutil
import sys
import time


# Isolates the environment and launches an individual SimCLR experiment
def setup_and_run(exp_id, folder_name, base_output_dir):
    # Set up source tracking and target workspace paths
    src_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "src")
    target_dir = os.path.join(base_output_dir, folder_name)

    print("\n" + "="*80)
    print(f" Preparing and Running Jittered Experiment {exp_id} in {folder_name} (200 Epochs)")
    print("="*80)

    # Initialize isolated environment directory
    os.makedirs(target_dir, exist_ok=True)

    # Clone source repository code to the execution target workspace
    for file_name in os.listdir(src_dir):
        src_file = os.path.join(src_dir, file_name)
        dst_file = os.path.join(target_dir, file_name)
        if os.path.isdir(src_file):
            if os.path.exists(dst_file):
                # Wipe stale directories to isolate data
                shutil.rmtree(dst_file)
            shutil.copytree(src_file, dst_file)
            print(f"Copied directory {file_name} to {folder_name}")
        elif file_name.endswith(".py"):
            # Duplicate scripts with metadata preserved
            shutil.copy2(src_file, dst_file)
            print(f"Copied {file_name} to {folder_name}")

    # Define training CLI execution flags and hyperparameters
    cmd = [
        sys.executable, "train_master.py",
        "--epochs", "200",
        "--batch_size", "512",
        "--backbone", "resnet50",
        # Bypass torch.compile overhead across dynamic steps
        "--no_compile",
        "--exp_id", str(exp_id),
        "--output_dir", "./outputs"
    ]

    print(f"\nExecuting: {' '.join(cmd)}")

    try:
        # Launch training process inside isolated directory context
        result = subprocess.run(cmd, cwd=target_dir)
        if result.returncode != 0:
            print(f"\nExperiment {exp_id} failed with error code {result.returncode}")
        else:
            print(f"\nExperiment {exp_id} completed successfully!")
    except Exception as e:
        print(f"\nFailed to run Experiment {exp_id}: {e}")

if __name__ == "__main__":
    # Color jitter ablation execution matrix
    experiments = [
        (38, "v9_rotation_jitter_exp38"),
        (39, "v10_weakbaseline_jitter_exp39"),
        (40, "v11_blur_jitter_exp40"),
        (41, "v12_flipblur_jitter_exp41"),
        (42, "v13_cutout_jitter_exp42")
    ]

    # Define core output directories
    base_dir = os.path.dirname(os.path.abspath(__file__))
    pipeline_dir = os.path.join(base_dir, "All Experiment SimCLR 18 May 2026")
    os.makedirs(pipeline_dir, exist_ok=True)

    start_time = time.time()

    # Run the ablation sequence sequentially
    for exp_id, folder in experiments:
        setup_and_run(exp_id, folder, pipeline_dir)

    total_time = (time.time() - start_time) / 3600
    print("\n" + "="*80)
    print(f" ALL 5 JITTERED ABLATIONS COMPLETED! Total Time: {total_time:.2f} hours")
    print("="*80)

    # Trigger automated linear probe validation pipeline
    eval_script = os.path.join(base_dir, "evaluate_and_plot_beast.py")
    subprocess.run([sys.executable, eval_script, pipeline_dir], cwd=base_dir)

**`src/Final Work/run_beast_experiments.py`** (76 lines) — launches the 200-epoch experiments.

In [ ]:
%%writefile "src/Final Work/run_beast_experiments.py"
import os
import subprocess
import shutil
import sys
import time

def setup_and_run(exp_id, folder_name, base_output_dir):
    src_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), "src")
    target_dir = os.path.join(base_output_dir, folder_name)
    
    print("\n" + "="*80)
    print(f" Preparing and Running BEAST Experiment {exp_id} in {folder_name} (200 Epochs)")
    print("="*80)
    
    os.makedirs(target_dir, exist_ok=True)
    
    # 1. Copy src files and folders to the target folder
    for file_name in os.listdir(src_dir):
        src_file = os.path.join(src_dir, file_name)
        dst_file = os.path.join(target_dir, file_name)
        if os.path.isdir(src_file):
            if os.path.exists(dst_file):
                shutil.rmtree(dst_file)
            shutil.copytree(src_file, dst_file)
            print(f"Copied directory {file_name} to {folder_name}")
        elif file_name.endswith(".py"):
            shutil.copy2(src_file, dst_file)
            print(f"Copied {file_name} to {folder_name}")
            
    # 2. Run the training script from within the target folder
    cmd = [
        sys.executable, "train_master.py",
        "--epochs", "200",
        "--batch_size", "512",
        "--backbone", "resnet50",
        "--no_compile",
        "--exp_id", str(exp_id),
        "--output_dir", "./outputs"
    ]
    
    print(f"\nExecuting: {' '.join(cmd)}")
    
    try:
        result = subprocess.run(cmd, cwd=target_dir)
        if result.returncode != 0:
            print(f"\nExperiment {exp_id} failed with error code {result.returncode}")
        else:
            print(f"\nExperiment {exp_id} completed successfully!")
    except Exception as e:
        print(f"\nFailed to run Experiment {exp_id}: {e}")

if __name__ == "__main__":
    # We run the Baseline with Jitter (Exp 8) and The Ultimate Beast (Exp 37 in the registry, which maps to N29)
    experiments = [
        (8, "v7_baseline_with_jitter"),
        (37, "v8_ultimate_beast")
    ]
    
    base_dir = os.path.dirname(os.path.abspath(__file__))
    pipeline_dir = os.path.join(base_dir, "All Experiment SimCLR 18 May 2026")
    os.makedirs(pipeline_dir, exist_ok=True)
    
    start_time = time.time()
    
    for exp_id, folder in experiments:
        setup_and_run(exp_id, folder, pipeline_dir)
        
    total_time = (time.time() - start_time) / 3600
    print("\n" + "="*80)
    print(f" BEAST EXPERIMENTS COMPLETED! Total Time: {total_time:.2f} hours")
    print("="*80)
    
    print("\n[+] Starting Automated Linear Probe Evaluation and Plotting...")
    eval_script = os.path.join(base_dir, "evaluate_and_plot_beast.py")
    subprocess.run([sys.executable, eval_script, pipeline_dir], cwd=base_dir)
    print("\n[+] Pipeline fully completed. Please check the 'All Experiment SimCLR 18 May 2026/Results' folder for the final outputs!")


**`src/Final Work/run_top_5_production.py`** (70 lines) — runs five selected augmentation configurations for 200 epochs.

In [ ]:
%%writefile "src/Final Work/run_top_5_production.py"
import os
import subprocess
import shutil
import sys
import time

def setup_and_run(exp_id, folder_name):
    base_dir = os.path.dirname(os.path.abspath(__file__))
    src_dir = os.path.join(base_dir, "src")
    target_dir = os.path.join(base_dir, folder_name)
    
    print("\n" + "="*80)
    print(f" Preparing and Running Experiment {exp_id} in {folder_name} (200 Epochs)")
    print("="*80)
    
    # 1. Copy src files and folders to the target folder
    for file_name in os.listdir(src_dir):
        src_file = os.path.join(src_dir, file_name)
        dst_file = os.path.join(target_dir, file_name)
        if os.path.isdir(src_file):
            if os.path.exists(dst_file):
                shutil.rmtree(dst_file)
            shutil.copytree(src_file, dst_file)
            print(f"Copied directory {file_name} to {folder_name}")
        elif file_name.endswith(".py"):
            shutil.copy2(src_file, dst_file)
            print(f"Copied {file_name} to {folder_name}")
            
    # 2. Run the training script from within the target folder
    # We use the same python executable that is running this script
    cmd = [
        sys.executable, "train_master.py",
        "--epochs", "200",
        "--batch_size", "512",
        "--backbone", "resnet50",
        "--no_compile",
        "--exp_id", str(exp_id),
        "--output_dir", "./outputs"
    ]
    
    print(f"\nExecuting: {' '.join(cmd)}")
    
    try:
        # Run subprocess and stream output
        result = subprocess.run(cmd, cwd=target_dir)
        if result.returncode != 0:
            print(f"\nExperiment {exp_id} failed with error code {result.returncode}")
        else:
            print(f"\nExperiment {exp_id} completed successfully!")
    except Exception as e:
        print(f"\nFailed to run Experiment {exp_id}: {e}")

if __name__ == "__main__":
    experiments = [
        (36, "v2_pure_rotation"),
        (35, "v3_weak_baseline"),
        (9,  "v4_crop_blur"),
        (13, "v5_crop_flip_blur"),
        (10, "v6_crop_cutout")
    ]
    
    start_time = time.time()
    
    for exp_id, folder in experiments:
        setup_and_run(exp_id, folder)
        
    total_time = (time.time() - start_time) / 3600
    print("\n" + "="*80)
    print(f" ALL 5 EXPERIMENTS COMPLETED! Total Time: {total_time:.2f} hours")
    print("="*80)


**`src/Final Work/evaluate_all_probes.py`** (150 lines) — linear-probe evaluation of every trained encoder.

In [ ]:
%%writefile "src/Final Work/evaluate_all_probes.py"
import os
import sys
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))
from model import get_simclr_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimCLREncoderOnly(nn.Module):
    def __init__(self, simclr_model, repr_dim=2048):
        super().__init__()
        self.simclr_model = simclr_model
        self.repr_dim = getattr(simclr_model, 'repr_dim', repr_dim)

    def forward(self, x):
        h, z = self.simclr_model(x)
        return h

def extract_features(encoder, loader, device):
    encoder.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            out = encoder(imgs.to(device))
            feats = out[0] if isinstance(out, tuple) else out
            all_feats.append(feats.cpu())
            all_labels.append(lbls)
    return torch.cat(all_feats), torch.cat(all_labels)

def main():
    base_dir = os.path.dirname(os.path.abspath(__file__))
    
    checkpoints = [
        (36, "v2_pure_rotation", r"v2_pure_rotation\outputs\resnet50_exp36_ntxent_τ0.5_bs512_ep200_20260517_153808\simclr_encoder_final.pth"),
        (35, "v3_weak_baseline", r"v3_weak_baseline\outputs\resnet50_exp35_ntxent_τ0.5_bs512_ep200_20260517_174004\simclr_encoder_final.pth"),
        (9, "v4_crop_blur", r"v4_crop_blur\outputs\resnet50_exp9_ntxent_τ0.5_bs512_ep200_20260517_194204\simclr_encoder_final.pth"),
        (13, "v5_crop_flip_blur", r"v5_crop_flip_blur\outputs\resnet50_exp13_ntxent_τ0.5_bs512_ep200_20260517_214535\simclr_encoder_final.pth"),
        (10, "v6_crop_cutout", r"v6_crop_cutout\outputs\resnet50_exp10_ntxent_τ0.5_bs512_ep200_20260517_234848\simclr_encoder_final.pth")
    ]

    CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
    CIFAR_STD  = [0.2023, 0.1994, 0.2010]
    test_transform = T.Compose([
        T.ToTensor(),
        T.Normalize(CIFAR_MEAN, CIFAR_STD)
    ])

    data_dir = os.path.join(base_dir, 'data')
    os.makedirs(data_dir, exist_ok=True)
    
    plain_train = DataLoader(
        CIFAR10(data_dir, train=True, download=True, transform=test_transform),
        batch_size=512, shuffle=False, num_workers=0, pin_memory=False
    )
    plain_test = DataLoader(
        CIFAR10(data_dir, train=False, download=True, transform=test_transform),
        batch_size=512, shuffle=False, num_workers=0, pin_memory=False
    )

    results = []

    for exp_id, folder, ckpt_rel in checkpoints:
        ckpt_path = os.path.join(base_dir, ckpt_rel)
        if not os.path.exists(ckpt_path):
            print(f"File not found: {ckpt_path}")
            continue
            
        print(f"\n_______________________________________________________")
        print(f"Evaluating Experiment {exp_id} ({folder})")
        print(f"_______________________________________________________")
        
        # Loading our model
        model = get_simclr_model(backbone='resnet50', projection_dim=128)
        ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
        
        if 'model_state_dict' in ckpt:
            model.load_state_dict(ckpt['model_state_dict'])
        else:
            model.load_state_dict(ckpt)
        
        model.eval()
        FEATURE_DIM = getattr(model, 'repr_dim', 2048)
        encoder = SimCLREncoderOnly(model, repr_dim=FEATURE_DIM).to(device)
        
        print('Extracting train features...')
        train_feats, train_labels = extract_features(encoder, plain_train, device)
        print('Extracting test features...')
        test_feats, test_labels = extract_features(encoder, plain_test, device)
        
        probe_train_ds = TensorDataset(train_feats, train_labels)
        probe_test_ds  = TensorDataset(test_feats,  test_labels)

        probe_train_loader = DataLoader(probe_train_ds, batch_size=512, shuffle=True)
        probe_test_loader  = DataLoader(probe_test_ds,  batch_size=512, shuffle=False)

        probe = nn.Linear(FEATURE_DIM, 10).to(device)
        probe_criterion = nn.CrossEntropyLoss()
        probe_optimizer = optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-4)

        PROBE_EPOCHS = 50
        print(f"{'Epoch':>6}  {'Train Acc':>9}  {'Test Acc':>8}")
        print('-' * 30)

        for ep in range(1, PROBE_EPOCHS + 1):
            probe.train()
            correct, total = 0, 0
            for feats, lbls in probe_train_loader:
                feats, lbls = feats.to(device), lbls.to(device)
                probe_optimizer.zero_grad()
                out  = probe(feats)
                loss = probe_criterion(out, lbls)
                loss.backward()
                probe_optimizer.step()
                correct += out.argmax(1).eq(lbls).sum().item()
                total   += lbls.size(0)
            tr_acc = 100.0 * correct / total
            
            probe.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for feats, lbls in probe_test_loader:
                    feats, lbls = feats.to(device), lbls.to(device)
                    out = probe(feats)
                    correct += out.argmax(1).eq(lbls).sum().item()
                    total   += lbls.size(0)
            te_acc = 100.0 * correct / total
            
            if ep % 10 == 0 or ep == 1:
                print(f"{ep:>6}  {tr_acc:>8.2f}%  {te_acc:>7.2f}%")

        probe_final_acc = te_acc
        print('_' * 30)
        print(f'Linear Probe Final Test Acc: {probe_final_acc:.2f}%')
        results.append((exp_id, folder, probe_final_acc))

    print("\n" + "_"*60)
    print(" FINAL RESULTS SUMMARY (LINEAR PROBE - RESNET50) ")
    print("_"*60)
    for r in results:
        print(f"Exp {r[0]:<2} | {r[1]:<20} | Top-1 Test Acc: {r[2]:.2f}%")
        
if __name__ == '__main__':
    main()


**`src/Final Work/evaluate_and_plot_beast.py`** (217 lines) — evaluation and plots of the 200-epoch experiments.

In [ ]:
%%writefile "src/Final Work/evaluate_and_plot_beast.py"
import os
import sys
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as T
from torchvision.datasets import CIFAR10
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))
from model import get_simclr_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class SimCLREncoderOnly(nn.Module):
    def __init__(self, simclr_model, repr_dim=2048):
        super().__init__()
        self.simclr_model = simclr_model
        self.repr_dim = getattr(simclr_model, 'repr_dim', repr_dim)

    def forward(self, x):
        h, z = self.simclr_model(x)
        return h

def extract_features(encoder, loader, device):
    encoder.eval()
    all_feats, all_labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            out = encoder(imgs.to(device))
            feats = out[0] if isinstance(out, tuple) else out
            all_feats.append(feats.cpu())
            all_labels.append(lbls)
    return torch.cat(all_feats), torch.cat(all_labels)

def evaluate_model(ckpt_path, plain_train, plain_test):
    safe_path = str(ckpt_path).encode('ascii', 'replace').decode('ascii')
    print(f"\n___ Evaluating: {safe_path} ___")
    model = get_simclr_model(backbone='resnet50', projection_dim=128)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    
    if 'model_state_dict' in ckpt:
        model.load_state_dict(ckpt['model_state_dict'])
    else:
        model.load_state_dict(ckpt)
    
    model.eval()
    FEATURE_DIM = getattr(model, 'repr_dim', 2048)
    encoder = SimCLREncoderOnly(model, repr_dim=FEATURE_DIM).to(device)
    
    train_feats, train_labels = extract_features(encoder, plain_train, device)
    test_feats, test_labels = extract_features(encoder, plain_test, device)
    
    probe_train_ds = TensorDataset(train_feats, train_labels)
    probe_test_ds  = TensorDataset(test_feats,  test_labels)

    probe_train_loader = DataLoader(probe_train_ds, batch_size=512, shuffle=True)
    probe_test_loader  = DataLoader(probe_test_ds,  batch_size=512, shuffle=False)

    probe = nn.Linear(FEATURE_DIM, 10).to(device)
    probe_criterion = nn.CrossEntropyLoss()
    probe_optimizer = optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-4)

    for ep in range(1, 51):
        probe.train()
        for feats, lbls in probe_train_loader:
            feats, lbls = feats.to(device), lbls.to(device)
            probe_optimizer.zero_grad()
            out  = probe(feats)
            loss = probe_criterion(out, lbls)
            loss.backward()
            probe_optimizer.step()
            
    probe.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for feats, lbls in probe_test_loader:
            feats, lbls = feats.to(device), lbls.to(device)
            out = probe(feats)
            correct += out.argmax(1).eq(lbls).sum().item()
            total   += lbls.size(0)
    te_acc = 100.0 * correct / total
    print(f"Final Test Acc: {te_acc:.2f}%")
    return te_acc

def generate_plot(results_dict, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    
    sns.set_theme(style="whitegrid")
    sns.set_context("paper", font_scale=1.5)
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'axes.titlesize': 16,
        'figure.dpi': 300
    })

    experiments = [
        "Exp 36\n(Pure Rotation)",
        "Exp 35\n(Weak Baseline)",
        "Exp 9\n(Crop + Blur)",
        "Exp 13\n(Crop+Flip+Blur)",
        "Exp 10\n(Crop + Cutout)"
    ]

    acc_without_jitter = [34.40, 59.22, 63.01, 64.49, 66.27]
    
    acc_with_jitter = [0.0, 0.0, 0.0, 0.0, 0.0]
    
    #Mapping the folder names to the correct index in the acc_with_jitter array
    folder_to_index = {
        "v9_rotation_jitter_exp38": 0,
        "v10_weakbaseline_jitter_exp39": 1,
        "v11_blur_jitter_exp40": 2,
        "v12_flipblur_jitter_exp41": 3,
        "v13_cutout_jitter_exp42": 4,

        # Keeping the old ones as they are for backwards compatibility
        "v7_baseline_with_jitter": 1,
        "v8_ultimate_beast": 4
    }
    
    for folder_name, idx in folder_to_index.items():
        if folder_name in results_dict:
            acc_with_jitter[idx] = results_dict[folder_name]

    df = pd.DataFrame({
        'Experiment': experiments * 2,
        'Top-1 Accuracy (%)': acc_without_jitter + acc_with_jitter,
        'Condition': ['Base Augmentation\n(Without Color Jitter)'] * 5 + ['Base + Color Jitter\n(With Jitter)'] * 5
    })

    fig, ax = plt.subplots(figsize=(11, 6.5))
    palette = {"Base Augmentation\n(Without Color Jitter)": "#1f77b4", "Base + Color Jitter\n(With Jitter)": "#2ca02c"}
    
    barplot = sns.barplot(
        data=df, 
        x='Experiment', 
        y='Top-1 Accuracy (%)', 
        hue='Condition',
        palette=palette,
        edgecolor='black',
        linewidth=1.2,
        ax=ax
    )

    for p in barplot.patches:
        height = p.get_height()
        if height > 0:
            ax.annotate(f'{height:.2f}%', 
                        (p.get_x() + p.get_width() / 2., height), 
                        ha='center', va='bottom', 
                        xytext=(0, 5), 
                        textcoords='offset points',
                        fontsize=11, fontweight='bold')

    ax.set_title('Impact of Color Jitter on Shortcut Learning in SimCLR\n(Linear Probe Top-1 Test Accuracy)', pad=20, fontweight='bold')
    ax.set_ylabel('Top-1 Test Accuracy (%)', fontweight='bold', labelpad=15)
    ax.set_xlabel('Structural Augmentation Strategy', fontweight='bold', labelpad=15)
    ax.set_ylim(0, 100)
    ax.legend(title='', loc='upper left', bbox_to_anchor=(0.02, 0.98), framealpha=0.9, edgecolor='black', fancybox=True)
    
    plt.tight_layout()
    out_png = os.path.join(output_dir, 'ablation_shortcut_learning_final.png')
    out_pdf = os.path.join(output_dir, 'ablation_shortcut_learning_final.pdf')
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')

def main():
    if len(sys.argv) < 2:
        print("Usage: python evaluate_and_plot_beast.py <pipeline_dir>")
        sys.exit(1)
        
    pipeline_dir = sys.argv[1]
    print(f"Scanning for models in {pipeline_dir}...")
    
    #Setting up the data loaders
    CIFAR_MEAN = [0.4914, 0.4822, 0.4465]
    CIFAR_STD  = [0.2023, 0.1994, 0.2010]
    test_transform = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])

    data_dir = os.path.join(os.path.dirname(os.path.abspath(__file__)), 'data')
    os.makedirs(data_dir, exist_ok=True)
    
    plain_train = DataLoader(CIFAR10(data_dir, train=True, download=True, transform=test_transform), batch_size=512, shuffle=False, num_workers=0)
    plain_test = DataLoader(CIFAR10(data_dir, train=False, download=True, transform=test_transform), batch_size=512, shuffle=False, num_workers=0)

    results_dict = {}
    
    #Find all simclr_encoder_final.pth
    for root, dirs, files in os.walk(pipeline_dir):
        if "simclr_encoder_final.pth" in files:
            ckpt_path = os.path.join(root, "simclr_encoder_final.pth")
            #Extracting folder name from path
            parts = Path(ckpt_path).parts
            try:
                exp_folder = parts[-4]
                acc = evaluate_model(ckpt_path, plain_train, plain_test)
                results_dict[exp_folder] = acc
            except Exception as e:
                safe_err_path = str(ckpt_path).encode('ascii', 'replace').decode('ascii')
                print(f"Error evaluating {safe_err_path}: {e}")

    results_dir = os.path.join(pipeline_dir, "Results")
    generate_plot(results_dict, results_dir)

if __name__ == '__main__':
    from pathlib import Path
    main()


**`src/Final Work/plot_ablation.py`** (96 lines) — figures of the ablation study.

In [ ]:
%%writefile "src/Final Work/plot_ablation.py"
#import's part
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

def main():
    # Configure plot aesthetics to meet IEEE academic publication standards
    # This ensures high readability, serif fonts, and appropriate scaling for papers
    sns.set_theme(style="whitegrid")
    sns.set_context("paper", font_scale=1.5)
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 14,
        'xtick.labelsize': 12,
        'ytick.labelsize': 12,
        'legend.fontsize': 12,
        'axes.titlesize': 16,
        'figure.dpi': 300
    })
    # Define the structural augmentation configurations tested in the pipeline
    experiments = [
        "Exp 36\n(Pure Rotation)",
        "Exp 35\n(Weak Baseline)",
        "Exp 9\n(Crop + Blur)",
        "Exp 13\n(Crop+Flip+Blur)",
        "Exp 10\n(Crop + Cutout)"
    ]

    # Baseline Top-1 test accuracy metrics (models trained without color jitter)
    acc_without_jitter = [34.40, 59.22, 63.01, 64.49, 66.27]

    # Placeholders for upcoming experiments where color jitter is introduced
    # to prevent the model from relying on color histograms as a shortcut
    acc_with_jitter = [0.0, 0.0, 0.0, 0.0, 0.0]

    # Construct the dataset for seaborn, doubling the experiments list to match both conditions
    df = pd.DataFrame({
        'Experiment': experiments * 2,
        'Top-1 Accuracy (%)': acc_without_jitter + acc_with_jitter,
        'Condition': ['Base Augmentation\n(Without Color Jitter)'] * 5 + ['Base + Color Jitter\n(With Jitter)'] * 5
    })

    # Initialize the figure canvas with dimensions optimized for double-column papers
    fig, ax = plt.subplots(figsize=(11, 6.5))

    # Define a custom, colorblind-friendly palette to clearly contrast the two conditions
    palette = {"Base Augmentation\n(Without Color Jitter)": "#1f77b4", "Base + Color Jitter\n(With Jitter)": "#2ca02c"}

    # Render the grouped bar chart
    barplot = sns.barplot(
        data=df,
        x='Experiment',
        y='Top-1 Accuracy (%)',
        hue='Condition',
        palette=palette,
        edgecolor='black',
        linewidth=1.2,
        ax=ax
    )

    # Iterate through the generated bars to overlay exact accuracy percentages
    # This prevents the reader from having to guess values from the y-axis
    for p in barplot.patches:
        height = p.get_height()
        # Only label completed experiments (non-zero bars)
        if height > 0:
            ax.annotate(f'{height:.2f}%',
                        (p.get_x() + p.get_width() / 2., height),
                        ha='center', va='bottom',
                        xytext=(0, 5),
                        textcoords='offset points',
                        fontsize=11, fontweight='bold')

    # Apply descriptive labels, title, and strictly bound the y-axis to 100%
    ax.set_title('Impact of Color Jitter on Shortcut Learning in SimCLR\n(Linear Probe Top-1 Test Accuracy)', pad=20, fontweight='bold')
    ax.set_ylabel('Top-1 Test Accuracy (%)', fontweight='bold', labelpad=15)
    ax.set_xlabel('Structural Augmentation Strategy', fontweight='bold', labelpad=15)
    ax.set_ylim(0, 100)

    # Customize the legend to integrate seamlessly into the upper left without obscuring data
    ax.legend(title='', loc='upper left', bbox_to_anchor=(0.02, 0.98), framealpha=0.9, edgecolor='black', fancybox=True)

    # Adjust layout to prevent label clipping
    plt.tight_layout()

    # Export the finalized charts in both raster (PNG) and vector (PDF) formats
    out_png = 'ablation_shortcut_learning.png'
    out_pdf = 'ablation_shortcut_learning.pdf'
    plt.savefig(out_png, dpi=300, bbox_inches='tight')
    plt.savefig(out_pdf, bbox_inches='tight')
    print(f"Academic charts successfully saved to {out_png} and {out_pdf}")

if __name__ == "__main__":
    main()


**`run_ablations.py`** (177 lines) — first ablation launcher (repository root).

In [ ]:
%%writefile run_ablations.py
import os
import sys
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def run_all_experiments():

    total_epochs = 20
    batch_size = 128
    model_backbone = "resnet18"

    print(f"\nStarting all SimCLR experiments using {model_backbone}\n")

    for exp_num in range(1, 9):

        print("-" * 60)
        print(f"Running Experiment #{exp_num}")
        print("-" * 60)

        save_dir = f"./outputs/exp_{exp_num}"

        cmd = [
            sys.executable,
            "src/train.py",
            "--exp_id", str(exp_num),
            "--epochs", str(total_epochs),
            "--batch_size", str(batch_size),
            "--backbone", model_backbone,
            "--output_dir", save_dir
        ]

        try:
            subprocess.run(cmd, check=True)
            print(f"Experiment {exp_num} finished successfully.\n")

        except subprocess.CalledProcessError as err:
            print(f"Experiment {exp_num} crashed! Error code: {err.returncode}\n")

    print("=" * 60)
    print("All experiments are done.")
    print("=" * 60)

    create_comparison_plots()


def create_comparison_plots():

    experiments = {
        1: "Crop Only",
        2: "Crop + Flip",
        3: "Crop + Color",
        4: "Crop + Grayscale",
        5: "Crop + Flip + Color",
        6: "Crop + Flip + Grayscale",
        7: "Crop + Color + Grayscale",
        8: "Full SimCLR"
    }

    # t-SNE Results

    print("\nCreating t-SNE comparison figure...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    fig.suptitle(
        "t-SNE Visualization Comparison (Epoch 20)",
        fontsize=16,
        fontweight="bold"
    )

    for index, (exp_num, exp_name) in enumerate(experiments.items()):

        row = index // 4
        col = index % 4

        ax = axes[row][col]

        image_path = f"./outputs/exp_{exp_num}/plots/tsne_epoch_020.png"

        if os.path.exists(image_path):

            image = mpimg.imread(image_path)

            ax.imshow(image)

            ax.set_title(
                f"Exp {exp_num} - {exp_name}",
                fontsize=10,
                fontweight="bold"
            )

        else:
            ax.text(
                0.5,
                0.5,
                f"Experiment {exp_num}\nImage Missing",
                ha="center",
                va="center",
                fontsize=11
            )

        ax.axis("off")

    plt.tight_layout()

    tsne_output = "./outputs/tsne_comparison_all_experiments.png"

    fig.savefig(tsne_output, dpi=150, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved t-SNE comparison to:\n{tsne_output}")

    # Loss Curves

    print("\nCreating loss curve comparison figure...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))

    fig.suptitle(
        "Training Loss Comparison",
        fontsize=16,
        fontweight="bold"
    )

    for index, (exp_num, exp_name) in enumerate(experiments.items()):

        row = index // 4
        col = index % 4

        ax = axes[row][col]

        image_path = f"./outputs/exp_{exp_num}/plots/training_loss_curve.png"

        if os.path.exists(image_path):

            image = mpimg.imread(image_path)

            ax.imshow(image)

            ax.set_title(
                f"Exp {exp_num} - {exp_name}",
                fontsize=10,
                fontweight="bold"
            )

        else:
            ax.text(
                0.5,
                0.5,
                f"Experiment {exp_num}\nImage Missing",
                ha="center",
                va="center",
                fontsize=11
            )

        ax.axis("off")

    plt.tight_layout()

    loss_output = "./outputs/loss_comparison_all_experiments.png"

    fig.savefig(loss_output, dpi=150, bbox_inches="tight")

    plt.close(fig)

    print(f"Saved loss comparison to:\n{loss_output}")

    print("\nFinished generating all comparison plots.")


if __name__ == "__main__":
    run_all_experiments()

## Appendix B: the visual-search demo

Streamlit app: ONNX-exported SimCLR encoder and a FAISS index over the CIFAR-10 test set (the model files and
the index are binary and are not part of this notebook).

**`deployment/app.py`** (436 lines) — Streamlit visual-search app: upload or pick an image, nearest CIFAR-10 test images by cosine similarity.

In [ ]:
%%writefile deployment/app.py
"""
================================================================================
  SimCLR ResNet-50 Visual Search Engine GUI
  -----------------------------------------
  A premium, interactive web-based graphical user interface (GUI) built with
  Streamlit, powered by ONNX Runtime and FAISS for real-time visual similarity
  retrieval on CIFAR-10.

  Key Features:
    1. Premium Dark Mode UI with tailored CSS glassmorphism styling
    2. Upload any local image or select a random test image from CIFAR-10
    3. Lightning-fast feature extraction (2048-d) via optimized ONNX Runtime
    4. Sub-millisecond exact Cosine Similarity search using FAISS vector database
    5. Clean visual results grid with similarity scores and class matches
    6. Dedicated interactive "Ablation Study Dashboard" showing the Exp 38-42 findings

  Usage:
    streamlit run app.py
================================================================================
"""

import os
import sys
import time
import json
import random
import numpy as np
from PIL import Image

import streamlit as st

# Windows console encoding fix
sys.stdout.reconfigure(encoding='utf-8')
sys.stderr.reconfigure(encoding='utf-8')

# Ensure we can import from src/
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
sys.path.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))

# ==========================================================================
# Caching Assets for Instant Performance
# ==========================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DEPLOY_DIR = os.path.join(BASE_DIR, "deployment")
ONNX_PATH = os.path.join(DEPLOY_DIR, "simclr_encoder_exp41.onnx")
INDEX_PATH = os.path.join(DEPLOY_DIR, "cifar10_index.faiss")
META_PATH = os.path.join(DEPLOY_DIR, "metadata.json")

CIFAR_MEAN = np.array([0.4914, 0.4822, 0.4465], dtype=np.float32)
CIFAR_STD = np.array([0.2023, 0.1994, 0.2010], dtype=np.float32)


@st.cache_resource
def load_onnx_model():
    """Load ONNX inference session and cache it."""
    import onnxruntime as ort
    # CPU is fast enough for single image inference, ensures compatibility
    providers = ['CPUExecutionProvider']
    return ort.InferenceSession(ONNX_PATH, providers=providers)


@st.cache_resource
def load_faiss_index():
    """Load FAISS index and cache it."""
    import faiss
    return faiss.read_index(INDEX_PATH)


@st.cache_data
def load_metadata():
    """Load metadata dictionary and cache it."""
    with open(META_PATH, "r", encoding="utf-8") as f:
        return json.load(f)


# -- Check that deployment files exist ---------------------------------------
missing_files = []
for p, name in [(ONNX_PATH, "ONNX Model"), (INDEX_PATH, "FAISS Index"), (META_PATH, "Metadata JSON")]:
    if not os.path.exists(p):
        missing_files.append(name)

if missing_files:
    st.error(f"❌ Missing deployment assets: {', '.join(missing_files)}")
    st.warning("👉 Please run `export_onnx.py` then `build_faiss.py` to generate these files before running the GUI!")
    st.stop()

# Load cached assets
ort_sess = load_onnx_model()
faiss_index = load_faiss_index()
metadata = load_metadata()

# ==========================================================================
# Preprocessing Logic
# ==========================================================================
def preprocess_image(pil_img):
    """
    Resizes image to 32x32 and applies CIFAR-10 mean/std normalization.
    Returns: numpy array of shape (1, 3, 32, 32)
    """
    # 1. Resize to 32x32
    img_resized = pil_img.resize((32, 32), Image.Resampling.BILINEAR)
    
    # 2. Convert to numpy array and scale to [0, 1]
    img_np = np.array(img_resized, dtype=np.float32) / 255.0
    
    # Handle grayscale images if uploaded
    if len(img_np.shape) == 2:
        img_np = np.stack([img_np, img_np, img_np], axis=-1)
    elif img_np.shape[2] == 4:
        img_np = img_np[:, :, :3] # Drop alpha channel
        
    # 3. Normalize: (x - mean) / std
    img_normalized = (img_np - CIFAR_MEAN) / CIFAR_STD
    
    # 4. Transpose to channel-first (HWC -> CHW) and add batch dimension (1, C, H, W)
    img_transposed = np.transpose(img_normalized, (2, 0, 1))
    img_batch = np.expand_dims(img_transposed, axis=0)
    
    return img_batch


def perform_search(features_batch, top_k=5):
    """
    Normalizes features, queries FAISS, and returns matched metadata.
    """
    # 1. L2-Normalize vector
    norm = np.linalg.norm(features_batch, axis=1, keepdims=True)
    features_normalized = features_batch / np.maximum(norm, 1e-12)
    
    # 2. Search FAISS Index
    similarities, indices = faiss_index.search(features_normalized, top_k)
    
    results = []
    for rank in range(top_k):
        vector_id = int(indices[0][rank])
        score = float(similarities[0][rank])
        
        # Load from metadata catalog
        match_info = metadata["images"][vector_id]
        
        results.append({
            "rank": rank + 1,
            "id": vector_id,
            "similarity": score,
            "image_path": os.path.join(DEPLOY_DIR, match_info["image_path"]),
            "class_name": match_info["class_name"],
            "label_id": match_info["label_id"]
        })
        
    return results


# ==========================================================================
# Streamlit UI Configuration
# ==========================================================================
st.set_page_config(
    page_title="SimCLR ResNet-50 Visual Search",
    page_icon="🔍",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for Premium Styling
st.markdown("""
<style>
    /* Dark theme customizations */
    .stApp {
        background-color: #0F172A;
        color: #E2E8F0;
    }
    
    /* Title and Header designs */
    h1 {
        background: linear-gradient(90deg, #38BDF8, #818CF8);
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        font-family: 'Outfit', sans-serif;
        font-weight: 800;
    }
    
    /* Custom Card/Glassmorphism block */
    .glass-card {
        background: rgba(30, 41, 59, 0.7);
        border: 1px solid rgba(255, 255, 255, 0.05);
        border-radius: 12px;
        padding: 20px;
        margin-bottom: 20px;
        box-shadow: 0 4px 30px rgba(0, 0, 0, 0.2);
        backdrop-filter: blur(5px);
    }
    
    /* Metric styling */
    .metric-value {
        font-size: 2.2rem;
        font-weight: 700;
        color: #38BDF8;
    }
    .metric-label {
        font-size: 0.9rem;
        color: #94A3B8;
        text-transform: uppercase;
        letter-spacing: 0.05em;
    }
</style>
""", unsafe_allow_html=True)

# ==========================================================================
# Sidebar Configuration
# ==========================================================================
st.sidebar.markdown("### ⚙️ Engine Settings")
top_k_slider = st.sidebar.slider("Number of results (K)", min_value=3, max_value=20, value=6, step=1)

st.sidebar.markdown("---")
st.sidebar.markdown("### 🏆 Trained Backbone Model")
st.sidebar.markdown("""
* **Architecture**: ResNet-50 (CIFAR STEM adjusted)
* **Training Type**: Self-Supervised (SimCLR)
* **Best Experiment**: **Exp 41**
* **Augmentations**: Crop + Flip + Blur + Color Jitter
* **Midterm Baseline**: 64.49%
* **Final Linear Probe Accuracy**: **84.30%** (+19.81% Gain!)
* **Embedding Dimension**: 2048
""")

st.sidebar.markdown("---")
st.sidebar.markdown("### ⚡ Backend Pipeline")
st.sidebar.markdown("""
* **Inference Engine**: ONNX Runtime CPU
* **Vector Database**: FAISS (IndexFlatIP)
* **Similarity Metric**: Exact Cosine Similarity
* **Database Size**: 10,000 Images
""")

st.sidebar.caption("Group 20, CISC 867, Queen's University, Spring 2026")

# ==========================================================================
# Main Title
# ==========================================================================
st.title("🔍 Real-time SimCLR Image Retrieval Engine")
st.markdown("##### Self-Supervised Representation Learning with ResNet-50 & FAISS Indexing")

# Define Tabs
tab1, tab2 = st.tabs(["🚀 Real-Time Search", "📊 Ablation Study Dashboard"])

# ==========================================================================
# TAB 1: Visual Search Engine
# ==========================================================================
with tab1:
    col_left, col_right = st.columns([1, 2], gap="large")
    
    # Initialize session state for random query image selection
    if "random_img_id" not in st.session_state:
        st.session_state.random_img_id = None
        
    query_image = None
    query_info = None

    with col_left:
        st.markdown("### 📥 Select Query Image")
        
        # Method 1: Upload a file
        uploaded_file = st.file_uploader("Upload an image...", type=["jpg", "png", "jpeg"])
        
        # Method 2: Pick a random image from the database
        st.markdown("<p style='text-align: center; color: #94A3B8; margin: 10px 0;'>— OR —</p>", unsafe_allow_html=True)
        if st.button("🎲 Pick a Random Test Image from Database", use_container_width=True):
            st.session_state.random_img_id = random.randint(0, len(metadata["images"]) - 1)
            
        # Display selected/uploaded image
        if uploaded_file is not None:
            st.session_state.random_img_id = None # Clear random image
            query_image = Image.open(uploaded_file).convert("RGB")
            st.image(query_image, caption="Uploaded Query Image", use_container_width=True)
            
        elif st.session_state.random_img_id is not None:
            idx = st.session_state.random_img_id
            img_info = metadata["images"][idx]
            local_path = os.path.join(DEPLOY_DIR, img_info["image_path"])
            
            if os.path.exists(local_path):
                query_image = Image.open(local_path).convert("RGB")
                query_info = img_info
                st.image(
                    query_image, 
                    caption=f"Selected Reference Image #{idx} (Class: {img_info['class_name']})", 
                    use_container_width=True
                )
            else:
                st.error("Reference image file not found.")

    with col_right:
        st.markdown("### 🎯 Visual Similarity Search Results")
        
        if query_image is not None:
            t_start = time.time()
            
            # 1. Preprocess
            batch_img = preprocess_image(query_image)
            
            # 2. ONNX Inference
            onnx_outputs = ort_sess.run(None, {"images": batch_img})
            feature_vector = onnx_outputs[0] # (1, 2048)
            
            # 3. FAISS Query
            t_search_start = time.time()
            results = perform_search(feature_vector, top_k=top_k_slider)
            
            t_total_ms = (time.time() - t_start) * 1000
            t_search_ms = (time.time() - t_search_start) * 1000
            
            # Show Metrics
            m_col1, m_col2, m_col3 = st.columns(3)
            with m_col1:
                st.markdown(f'<div class="glass-card"><div class="metric-value">{t_search_ms:.2f} ms</div><div class="metric-label">FAISS Query Time</div></div>', unsafe_allow_html=True)
            with m_col2:
                st.markdown(f'<div class="glass-card"><div class="metric-value">{t_total_ms:.1f} ms</div><div class="metric-label">Total Pipeline Latency</div></div>', unsafe_allow_html=True)
            with m_col3:
                # Calculate class precision
                if query_info is not None:
                    matches = sum(1 for r in results if r["class_name"] == query_info["class_name"])
                    precision = (matches / len(results)) * 100
                    st.markdown(f'<div class="glass-card"><div class="metric-value">{precision:.1f}%</div><div class="metric-label">Top-{len(results)} Class Precision</div></div>', unsafe_allow_html=True)
                else:
                    st.markdown(f'<div class="glass-card"><div class="metric-value">N/A</div><div class="metric-label">Upload class unknown</div></div>', unsafe_allow_html=True)
            
            # Display Results Grid
            st.markdown("#### Retreived Nearest Neighbors")
            
            # Create dynamic grid columns
            grid_cols = st.columns(3)
            
            for idx, res in enumerate(results):
                col = grid_cols[idx % 3]
                
                with col:
                    # Determine styling color based on class match if query info exists
                    is_match = False
                    border_color = "rgba(255, 255, 255, 0.05)"
                    bg_color = "rgba(30, 41, 59, 0.7)"
                    badge_html = f'<span style="background-color: #475569; padding: 2px 8px; border-radius: 4px; font-size: 0.8rem; font-weight: 600; color: white;">{res["class_name"]}</span>'
                    
                    if query_info is not None:
                        is_match = (res["class_name"] == query_info["class_name"])
                        if is_match:
                            border_color = "rgba(16, 185, 129, 0.3)"
                            bg_color = "rgba(6, 78, 59, 0.3)"
                            badge_html = f'<span style="background-color: #10B981; padding: 2px 8px; border-radius: 4px; font-size: 0.8rem; font-weight: 600; color: white;">{res["class_name"]} (MATCH)</span>'
                        else:
                            border_color = "rgba(239, 68, 68, 0.2)"
                            bg_color = "rgba(127, 29, 29, 0.1)"
                            badge_html = f'<span style="background-color: #EF4444; padding: 2px 8px; border-radius: 4px; font-size: 0.8rem; font-weight: 600; color: white;">{res["class_name"]}</span>'
                    
                    # Custom card container
                    card_content = f"""
                    <div style="background: {bg_color}; border: 1px solid {border_color}; border-radius: 8px; padding: 12px; margin-bottom: 12px; text-align: center;">
                        <div style="color: #38BDF8; font-size: 1.1rem; font-weight: 700; margin-bottom: 8px;">Rank #{res["rank"]}</div>
                        <div style="margin-bottom: 10px;">{badge_html}</div>
                        <div style="color: #94A3B8; font-size: 0.9rem; margin-bottom: 4px;">Cosine Similarity:</div>
                        <div style="color: #E2E8F0; font-size: 1.4rem; font-weight: 700; margin-bottom: 10px;">{res["similarity"] * 100:.2f}%</div>
                        <div style="color: #64748B; font-size: 0.8rem;">Vector ID: {res["id"]}</div>
                    </div>
                    """
                    
                    # Render matched image and custom card
                    if os.path.exists(res["image_path"]):
                        matched_img = Image.open(res["image_path"])
                        col.image(matched_img, use_container_width=True)
                    else:
                        col.error("Image file missing.")
                        
                    col.markdown(card_content, unsafe_allow_html=True)
                    
        else:
            st.info("💡 Please upload a custom image or click the button to select a random test image to query the visual search engine!")

# ==========================================================================
# TAB 2: Ablation Study Dashboard
# ==========================================================================
with tab2:
    st.markdown("### 📊 Experiment Ablation Study & Color Jitter Findings")
    st.markdown("##### Evaluating Natalie's Midterm Pipelines against Mahmoud's Final Jittered Re-runs")
    
    st.markdown("""
    This section presents the results of the complete **Color Jitter Shortcut-Learning Ablation Study** (Experiments 38-42).
    By comparing the self-supervised representations learned with and without photometric distortion (Color Jitter), we demonstrate how SimCLR ResNet-50 encoders learn rich semantic representations instead of exploiting simple pixel-level color shortcuts.
    """)
    
    # 3x2 Grid for Ablation Statistics
    ablation_data = [
        {"exp": "Exp 38 vs 36", "desc": "Pure Discrete Rotation", "base": "34.40%", "jitter": "51.21%", "gain": "+16.81 pp"},
        {"exp": "Exp 39 vs 35", "desc": "Weak Spatial Baseline", "base": "59.22%", "jitter": "80.53%", "gain": "+21.31 pp"},
        {"exp": "Exp 40 vs 9",  "desc": "Crop + Gaussian Blur", "base": "63.01%", "jitter": "80.65%", "gain": "+17.64 pp"},
        {"exp": "Exp 41 vs 13", "desc": "Crop + Flip + Blur", "base": "64.49%", "jitter": "84.30%", "gain": "+19.81 pp"},
        {"exp": "Exp 42 vs 10", "desc": "Crop + Random Cutout", "base": "66.27%", "jitter": "81.21%", "gain": "+14.94 pp"}
    ]
    
    cols_ablation = st.columns(5)
    for i, data in enumerate(ablation_data):
        with cols_ablation[i]:
            card_html = f"""
            <div style="background: rgba(30, 41, 59, 0.5); border: 1px solid rgba(56, 189, 248, 0.2); border-radius: 8px; padding: 15px; text-align: center;">
                <div style="color: #38BDF8; font-size: 1.1rem; font-weight: 700;">{data["exp"]}</div>
                <div style="color: #94A3B8; font-size: 0.85rem; height: 35px; overflow: hidden; margin-top: 4px;">{data["desc"]}</div>
                <hr style="border: 0; border-top: 1px solid rgba(255,255,255,0.05); margin: 8px 0;"/>
                <div style="color: #94A3B8; font-size: 0.8rem;">Without Jitter: <b>{data["base"]}</b></div>
                <div style="color: #10B981; font-size: 0.85rem; font-weight: 600; margin-top: 2px;">With Jitter: {data["jitter"]}</div>
                <div style="background-color: rgba(16, 185, 129, 0.15); color: #10B981; border-radius: 4px; padding: 2px 6px; font-size: 0.9rem; font-weight: 700; margin-top: 8px; display: inline-block;">
                    {data["gain"]} Gain
                </div>
            </div>
            """
            st.markdown(card_html, unsafe_allow_html=True)
            
    st.markdown("---")
    st.markdown("#### 🔑 Key Project Takeaways")
    col1, col2 = st.columns(2)
    
    with col1:
        st.markdown("""
        ##### 1. The Shortcut-Learning Problem 🧩
        Without Color Jitter, contrastive self-supervised encoders exploit low-level **color histograms** as a shortcut to maximize mutual information, rather than learning general shapes and semantic features. This results in weaker downstream representations, showing a severe performance limit (e.g. baseline Crop+Flip+Blur achieves only **64.49%**).
        
        ##### 2. The Color Jitter Shield 🛡️
        Adding photometric distortion (color jittering) forces the model to ignore color profiles and focus on invariant structures, spatial boundaries, and contours. This single ablation yields a massive average boost of **+18.1 pp** across all settings, pushing our best encoder (Exp 41) to a stellar **84.30% Top-1 accuracy**!
        """)
        
    with col2:
        st.markdown("""
        ##### 3. Model Architecture & Stem Tuning ⚙️
        Modifying the standard ResNet-50 conv1 stem from $7\\times7$ (stride 2) to a custom $3\\times3$ (stride 1) and removing the initial MaxPool was crucial to preserve the resolution of $32\\times32$ CIFAR-10 images. 
        
        ##### 4. Near Foundation-Model Upper Bound 🏆
        Our zero-shot CLIP ViT-B/32 foundation model evaluation sets the academic upper bound at **88.80%**. Our custom-trained SimCLR ResNet-50 achieves **95% of this performance** (**84.30%**) while using **8,000x less training data**!
        """)

    st.success("🎉 Weights & Biases Dashboard has archived all 5 experiments complete with training curves, checkpoints, and t-SNE files.")


**`deployment/build_faiss.py`** (251 lines) — builds the FAISS index and metadata for the CIFAR-10 test set.

In [ ]:
%%writefile deployment/build_faiss.py
"""
================================================================================
  Build FAISS Index for CIFAR-10 Test Set
  ---------------------------------------
  Generates the FAISS vector database and metadata mapping for the real-time
  visual search engine GUI.

  What this script does:
    1. Loads the exported ONNX ResNet-50 encoder (simclr_encoder_exp41.onnx)
    2. Loads the CIFAR-10 test set (10,000 images) using the evaluation pipeline
    3. Runs batched ONNX inference on the CPU/GPU to extract 2048-d features
    4. L2-normalizes the features to enable exact Cosine Similarity search
    5. Saves the raw test images to disk as PNGs for easy loading in the GUI
    6. Constructs a FAISS IndexFlatIP (Inner Product) search database
    7. Saves a JSON metadata mapping linking vector IDs to image paths/classes
    8. Performs a validation query to verify search accuracy

  Outputs:
    - ./deployment/cifar10_index.faiss     (FAISS vector database)
    - ./deployment/metadata.json           (Vector ID to Image/Label mapping)
    - ./deployment/test_images/*.png       (10,000 reference images)

  Usage:
    python build_faiss.py
================================================================================
"""

import os
import sys
import time
import json
import numpy as np
from tqdm import tqdm
from PIL import Image

# Windows console encoding fix
sys.stdout.reconfigure(encoding='utf-8')
sys.stderr.reconfigure(encoding='utf-8')

# Ensure we can import from src/
sys.path.append(os.path.dirname(os.path.abspath(__file__)))
sys.path.append(os.path.join(os.path.dirname(os.path.abspath(__file__)), "src"))

try:
    import torch
    from torchvision.datasets import CIFAR10
except ImportError:
    print("[ERROR] PyTorch / TorchVision not found.")
    sys.exit(1)

try:
    import onnxruntime as ort
except ImportError:
    print("[ERROR] onnxruntime not found. Install with: pip install onnxruntime")
    sys.exit(1)

try:
    import faiss
except ImportError:
    print("[ERROR] faiss not found. Install with: pip install faiss-cpu")
    sys.exit(1)

from src.dataset import get_eval_dataloader, CIFAR_MEAN, CIFAR_STD

# ==========================================================================
# Configuration
# ==========================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
ONNX_PATH = os.path.join(BASE_DIR, "deployment", "simclr_encoder_exp41.onnx")

DEPLOY_DIR = os.path.join(BASE_DIR, "deployment")
IMAGES_DIR = os.path.join(DEPLOY_DIR, "test_images")
INDEX_PATH = os.path.join(DEPLOY_DIR, "cifar10_index.faiss")
META_PATH = os.path.join(DEPLOY_DIR, "metadata.json")

BATCH_SIZE = 256
DATA_DIR = "./data"

CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]


def main():
    print("=" * 72)
    print("  FAISS Vector Index Builder")
    print("  Backbone: ResNet-50 (Exp 41 SimCLR Encoder)")
    print("=" * 72)

    # -- 1. Load ONNX model --------------------------------------------------
    if not os.path.exists(ONNX_PATH):
        print(f"[ERROR] ONNX model not found at: {ONNX_PATH}")
        print("        Please run export_onnx.py first!")
        sys.exit(1)

    print(f"\n[1/6] Loading ONNX model...")
    t0 = time.time()
    
    # Auto-select best execution provider (CUDA if available, else CPU)
    providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
    try:
        ort_sess = ort.InferenceSession(ONNX_PATH, providers=providers)
        active_provider = ort_sess.get_providers()[0]
        print(f"      ONNX session loaded successfully!")
        print(f"      Execution Provider: {active_provider}")
    except Exception as e:
        print(f"[ERROR] Failed to load ONNX model: {e}")
        sys.exit(1)
        
    print(f"      Load time: {time.time() - t0:.1f}s")

    # -- 2. Load Dataset -----------------------------------------------------
    print(f"\n[2/6] Loading CIFAR-10 test set...")
    # Load normalized images for inference
    eval_loader = get_eval_dataloader(
        data_dir=DATA_DIR,
        train=False,
        batch_size=BATCH_SIZE,
        num_workers=4
    )
    
    # Load raw dataset to extract original unnormalized PIL images to save to disk
    raw_dataset = CIFAR10(root=DATA_DIR, train=False, download=True)
    n_images = len(raw_dataset)
    print(f"      Loaded {n_images:,} test images.")

    # -- 3. Extract Features -------------------------------------------------
    print(f"\n[3/6] Extracting 2048-d SimCLR features...")
    features_list = []
    
    t_feat_start = time.time()
    for batch_images, _ in tqdm(eval_loader, desc="      Inference", unit="batch"):
        # Convert PyTorch tensor to numpy array for ONNX
        batch_numpy = batch_images.numpy().astype(np.float32)
        # Run inference
        outputs = ort_sess.run(None, {"images": batch_numpy})
        features_list.append(outputs[0])
        
    features = np.concatenate(features_list, axis=0)
    print(f"      Extracted feature matrix shape: {features.shape}")
    print(f"      Inference speed: {n_images / (time.time() - t_feat_start):.0f} img/s")

    # -- 4. L2-Normalize for Cosine Similarity -------------------------------
    print(f"\n[4/6] Normalizing features for exact Cosine Similarity...")
    norms = np.linalg.norm(features, axis=1, keepdims=True)
    # Avoid division by zero
    features_normalized = features / np.maximum(norms, 1e-12)
    print(f"      L2 normalization complete.")

    # -- 5. Save Raw PNG Images and Create Metadata --------------------------
    print(f"\n[5/6] Exporting original images to disk & creating metadata...")
    os.makedirs(IMAGES_DIR, exist_ok=True)
    
    metadata = {
        "classes": CIFAR10_CLASSES,
        "images": []
    }
    
    # Save unnormalized PIL images and generate mapping dictionary
    for i in tqdm(range(n_images), desc="      Exporting PNGs", unit="img"):
        img, label = raw_dataset[i]
        
        # Save image file
        img_filename = f"{i:05d}.png"
        img_path = os.path.join(IMAGES_DIR, img_filename)
        img.save(img_path, format="PNG")
        
        # Relative path for portability (allows moving the deployment folder)
        rel_img_path = f"test_images/{img_filename}"
        
        metadata["images"].append({
            "id": i,
            "image_path": rel_img_path,
            "label_id": int(label),
            "class_name": CIFAR10_CLASSES[label]
        })
        
    # Write metadata.json
    with open(META_PATH, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=4, ensure_ascii=False)
        
    print(f"      Exported {n_images:,} images to {IMAGES_DIR}")
    print(f"      Saved metadata catalog to {META_PATH}")

    # -- 6. Build and Save FAISS Index ---------------------------------------
    print(f"\n[6/6] Building FAISS Inner Product (Cosine) index...")
    t_faiss = time.time()
    
    dimension = features_normalized.shape[1] # 2048
    
    # We use IndexFlatIP (Inner Product) on L2 normalized features
    # to yield exact Cosine Similarity search results.
    index = faiss.IndexFlatIP(dimension)
    
    # Add vectors to index
    index.add(features_normalized)
    
    # Write index to disk
    faiss.write_index(index, INDEX_PATH)
    print(f"      Constructed FAISS IndexFlatIP.")
    print(f"      Saved FAISS index to {INDEX_PATH}")
    print(f"      FAISS build time: {time.time() - t_faiss:.3f}s")

    # ==========================================================================
    # Validation Query
    # ==========================================================================
    print("\n" + "=" * 72)
    print("  Validation: Performing Sample Visual Search")
    print("=" * 72)
    
    # Query with the first image in the test set
    query_id = 0
    query_class = metadata["images"][query_id]["class_name"]
    query_vec = features_normalized[query_id:query_id+1]
    
    print(f"  Query Vector ID: {query_id} (Class: {query_class})")
    
    # Search for top 5 nearest neighbors
    top_k = 5
    t_search = time.time()
    similarities, indices = index.search(query_vec, top_k)
    search_time = (time.time() - t_search) * 1000
    
    print(f"  Search completed in {search_time:.3f} ms")
    print(f"  Top {top_k} Nearest Neighbors:")
    print(f"  ┌──────┬──────────┬─────────────┬────────────┐")
    print(f"  │ Rank │ VectorID │ Class Name  │ Similarity │")
    print(f"  ├──────┼──────────┼─────────────┼────────────┤")
    for rank in range(top_k):
        neighbor_id = int(indices[0][rank])
        neighbor_class = metadata["images"][neighbor_id]["class_name"]
        score = float(similarities[0][rank])
        print(f"  │ {rank+1:4d} │ {neighbor_id:8d} │ {neighbor_class:<11s} │ {score:10.4f} │")
    print(f"  └──────┴──────────┴─────────────┴────────────┘")
    
    # Check if the top match is the image itself (score ~ 1.0)
    success = (indices[0][0] == query_id) and (similarities[0][0] > 0.99)
    # Check top-5 precision (how many have the same class)
    same_class_count = sum(1 for idx in indices[0] if metadata["images"][int(idx)]["class_name"] == query_class)
    precision = (same_class_count / top_k) * 100
    
    print(f"\n  Index Self-Consistency : {'[PASS]' if success else '[FAIL]'}")
    print(f"  Top-5 Search Precision : {precision:.1f}% ({same_class_count}/{top_k} same class)")
    print("=" * 72)
    print("  SUCCESS: FAISS deployment artifacts generated and verified!")
    print("=" * 72)


if __name__ == "__main__":
    main()


**`deployment/export_onnx.py`** (205 lines) — exports the SimCLR encoder to ONNX.

In [ ]:
%%writefile deployment/export_onnx.py

  #Export SimCLR Encoder to ONNX


import os
import sys
import time
import numpy as np

# Windows console encoding fix (tau char in folder names)
sys.stdout.reconfigure(encoding='utf-8')
sys.stderr.reconfigure(encoding='utf-8')

import torch
import torch.nn as nn
import torchvision.models as models


# ==========================================================================
# 1. Rebuild the encoder-only architecture (no projection head)
# ==========================================================================
class SimCLREncoder(nn.Module):
    """
    ResNet-50 encoder with CIFAR-10 stem modification.
    Outputs 2048-d feature vectors (no projection head).

    Architecture matches SimCLRResNet50.encoder + pool + flatten
    from model.py exactly, so we can load the same state_dict keys.
    """

    def __init__(self):
        super().__init__()

        backbone = models.resnet50(weights=None)

        # CIFAR-10 stem: 3x3 conv, stride 1, no maxpool
        backbone.conv1 = nn.Conv2d(
            3, 64, kernel_size=3, stride=1, padding=1, bias=False
        )
        backbone.maxpool = nn.Identity()

        self.encoder = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
            backbone.layer1,
            backbone.layer2,
            backbone.layer3,
            backbone.layer4,
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x):
        """
        Args:
            x: (N, 3, 32, 32) CIFAR-10 images, normalized
        Returns:
            h: (N, 2048) feature embeddings
        """
        x = self.encoder(x)
        x = self.pool(x)
        h = x.flatten(start_dim=1)
        return h


# ==========================================================================
# 2. Configuration
# ==========================================================================
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
EXP_ROOT = os.path.join(BASE_DIR, "All Experiment SimCLR 18 May 2026")
EXP41_FOLDER = "v12_flipblur_jitter_exp41"

OUTPUT_DIR = os.path.join(BASE_DIR, "deployment")
ONNX_FILENAME = "simclr_encoder_exp41.onnx"

# CIFAR-10 input shape
INPUT_SHAPE = (1, 3, 32, 32)


def find_checkpoint():
    """Auto-discover the .pth checkpoint inside Exp 41's output folder."""
    outputs_dir = os.path.join(EXP_ROOT, EXP41_FOLDER, "outputs")
    if not os.path.isdir(outputs_dir):
        print(f"[ERROR] Outputs directory not found: {outputs_dir}")
        sys.exit(1)

    for run_dir in os.listdir(outputs_dir):
        pth = os.path.join(outputs_dir, run_dir, "simclr_encoder_final.pth")
        if os.path.exists(pth):
            return pth

    print(f"[ERROR] No simclr_encoder_final.pth found in {outputs_dir}")
    sys.exit(1)


def main():
    print("=" * 72)
    print("  SimCLR Encoder -> ONNX Export")
    print("  Experiment 41: Crop+Flip+Blur+Jitter (84.30% Top-1)")
    print("=" * 72)

    # -- Find checkpoint -------------------------------------------------
    ckpt_path = find_checkpoint()
    ckpt_size = os.path.getsize(ckpt_path) / (1024 * 1024)
    print(f"\n[1/4] Checkpoint found:")
    print(f"      {ckpt_path}")
    print(f"      Size: {ckpt_size:.1f} MB")

    # -- Build encoder and load weights ----------------------------------
    print("\n[2/4] Building encoder and loading weights...")
    t0 = time.time()

    model = SimCLREncoder()
    full_state = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    # Filter: keep only encoder.* and pool.* keys (drop projection.*)
    encoder_keys = {
        k: v for k, v in full_state.items()
        if k.startswith("encoder.") or k.startswith("pool.")
    }
    projection_keys = [k for k in full_state if k.startswith("projection.")]

    model.load_state_dict(encoder_keys, strict=True)
    model.eval()

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"      Loaded {len(encoder_keys)} encoder keys ({n_params:.1f}M params)")
    print(f"      Dropped {len(projection_keys)} projection head keys")
    print(f"      Load time: {time.time() - t0:.1f}s")

    # -- Export to ONNX --------------------------------------------------
    print("\n[3/4] Exporting to ONNX...")
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    onnx_path = os.path.join(OUTPUT_DIR, ONNX_FILENAME)

    dummy_input = torch.randn(*INPUT_SHAPE)

    t1 = time.time()
    torch.onnx.export(
        model,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=17,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["features"],
        dynamic_axes={
            "images":   {0: "batch_size"},
            "features": {0: "batch_size"},
        },
        dynamo=False,
    )

    onnx_size = os.path.getsize(onnx_path) / (1024 * 1024)
    print(f"      Exported: {onnx_path}")
    print(f"      ONNX size: {onnx_size:.1f} MB")
    print(f"      Export time: {time.time() - t1:.1f}s")

    # -- Validate ONNX vs PyTorch ----------------------------------------
    print("\n[4/4] Validating ONNX against PyTorch...")
    try:
        import onnxruntime as ort

        session = ort.InferenceSession(onnx_path)
        test_input = np.random.randn(*INPUT_SHAPE).astype(np.float32)

        # PyTorch output
        with torch.no_grad():
            pt_out = model(torch.from_numpy(test_input)).numpy()

        # ONNX output
        onnx_out = session.run(None, {"images": test_input})[0]

        max_err = np.max(np.abs(pt_out - onnx_out))
        mean_err = np.mean(np.abs(pt_out - onnx_out))

        status = "PASS" if max_err < 1e-4 else "FAIL"
        print(f"      Max  error: {max_err:.2e}")
        print(f"      Mean error: {mean_err:.2e}")
        print(f"      Validation: [{status}]")

        if status == "FAIL":
            print("      [WARNING] Error exceeds threshold. Check opset compatibility.")

    except ImportError:
        print("      [SKIP] onnxruntime not installed. Install with:")
        print("             pip install onnxruntime")
        print("      (ONNX file was exported successfully regardless.)")

    # -- Summary ---------------------------------------------------------
    print("\n" + "=" * 72)
    print(f"  ONNX EXPORT COMPLETE")
    print(f"  File   : {onnx_path}")
    print(f"  Size   : {onnx_size:.1f} MB")
    print(f"  Input  : images (batch, 3, 32, 32) float32")
    print(f"  Output : features (batch, 2048) float32")
    print(f"  Ready for FAISS indexing and GUI deployment!")
    print("=" * 72)


if __name__ == "__main__":
    main()


## Appendix C: W&B sync and the first asset generator

**`wandb/sync_to_wandb.py`** (246 lines) — uploads the logs of the course-project ablations (experiments 38–42) to Weights & Biases.

In [ ]:
%%writefile wandb/sync_to_wandb.py
"""
================================================================================
  Sync SimCLR Ablation Experiments (38-42) to Weights & Biases
  ------------------------------------------------------------
  Reads training_log.csv and run_config.json from each experiment's
  output folder and logs them as individual W&B runs under one project.
  Also uploads loss curves, t-SNE plots, and the final encoder checkpoint
  as artifacts.

  Prerequisites:
      pip install wandb
      wandb login          <-- must be done first!

  Usage:
      python sync_to_wandb.py
================================================================================
"""

import os
import sys
import csv
import json
import glob
import time

# -- Dependency check --------------------------------------------------------
try:
    import wandb
except ImportError:
    print("[ERROR] wandb not found. Install with:  pip install wandb")
    sys.exit(1)


# ==========================================================================
# Configuration
# ==========================================================================
WANDB_PROJECT = "SimCLR-Augmentation-Ablation"
WANDB_ENTITY  = None   # Uses your default entity (mahmoudalyosify)

BASE_DIR = os.path.dirname(os.path.abspath(__file__))
EXPERIMENT_ROOT = os.path.join(BASE_DIR, "All Experiment SimCLR 18 May 2026")

# -- Experiment registry: (folder_name, exp_id, human-readable name) -----
EXPERIMENTS = [
    ("v9_rotation_jitter_exp38",       38, "Exp38: Pure Rotation + Jitter"),
    ("v10_weakbaseline_jitter_exp39",  39, "Exp39: Weak Baseline + Jitter"),
    ("v11_blur_jitter_exp40",          40, "Exp40: Crop+Blur + Jitter"),
    ("v12_flipblur_jitter_exp41",      41, "Exp41: Crop+Flip+Blur + Jitter"),
    ("v13_cutout_jitter_exp42",        42, "Exp42: Crop+Cutout + Jitter"),
]

# -- Linear probe results (from evaluate_and_plot_beast.py) --------------
PROBE_ACCURACY = {
    38: 51.21,
    39: 80.53,
    40: 80.65,
    41: 84.30,
    42: 81.21,
}

# -- Corresponding base experiment accuracy (without jitter) -------------
BASE_ACCURACY = {
    38: 34.40,   # Exp 36: Pure Rotation
    39: 59.22,   # Exp 35: Weak Baseline
    40: 63.01,   # Exp 9:  Crop + Blur
    41: 64.49,   # Exp 13: Crop + Flip + Blur
    42: 66.27,   # Exp 10: Crop + Cutout
}


def find_run_dir(exp_folder):
    """Find the single run output directory inside an experiment folder."""
    outputs_dir = os.path.join(exp_folder, "outputs")
    if not os.path.isdir(outputs_dir):
        return None
    subdirs = [d for d in os.listdir(outputs_dir) 
               if os.path.isdir(os.path.join(outputs_dir, d))]
    if len(subdirs) == 1:
        return os.path.join(outputs_dir, subdirs[0])
    # If multiple, pick the one matching the pattern
    for d in subdirs:
        if d.startswith("resnet50_exp"):
            return os.path.join(outputs_dir, d)
    return None


def load_config(run_dir):
    """Load run_config.json."""
    config_path = os.path.join(run_dir, "run_config.json")
    if os.path.exists(config_path):
        with open(config_path, 'r') as f:
            return json.load(f)
    return {}


def load_training_log(run_dir):
    """Load training_log.csv and return list of dicts."""
    log_path = os.path.join(run_dir, "logs", "training_log.csv")
    if not os.path.exists(log_path):
        return []
    rows = []
    with open(log_path, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            rows.append({
                "epoch": int(row["epoch"]),
                "loss":  float(row["loss"]),
                "lr":    float(row["lr"]),
                "time_s": float(row["time_s"]),
            })
    return rows


def sync_experiment(folder_name, exp_id, display_name):
    """Sync a single experiment to W&B."""
    exp_folder = os.path.join(EXPERIMENT_ROOT, folder_name)
    if not os.path.isdir(exp_folder):
        print(f"  [SKIP] Folder not found: {folder_name}")
        return False

    run_dir = find_run_dir(exp_folder)
    if run_dir is None:
        print(f"  [SKIP] No run output found in: {folder_name}")
        return False

    # -- Load data -------------------------------------------------------
    config = load_config(run_dir)
    training_log = load_training_log(run_dir)

    if not training_log:
        print(f"  [SKIP] No training log found in: {run_dir}")
        return False

    # -- Compute summary metrics -----------------------------------------
    final_loss = training_log[-1]["loss"]
    best_loss = min(row["loss"] for row in training_log)
    init_loss = training_log[0]["loss"]
    total_time_h = sum(row["time_s"] for row in training_log) / 3600
    probe_acc = PROBE_ACCURACY.get(exp_id, None)
    base_acc  = BASE_ACCURACY.get(exp_id, None)
    jitter_gain = (probe_acc - base_acc) if (probe_acc and base_acc) else None

    # -- Initialize W&B run ----------------------------------------------
    run = wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=display_name,
        config={
            **config,
            "experiment_id": exp_id,
            "pipeline": folder_name,
            "has_color_jitter": True,
            "probe_accuracy": probe_acc,
            "base_accuracy_no_jitter": base_acc,
            "jitter_gain_pp": jitter_gain,
        },
        tags=["ablation", "color-jitter", f"exp{exp_id}", "resnet50", "200ep"],
        notes=f"SimCLR ablation experiment {exp_id}: {display_name}. "
              f"200 epochs, BS=512, ResNet-50 on CIFAR-10.",
        reinit=True,
    )

    # -- Log epoch-level metrics -----------------------------------------
    print(f"  Logging {len(training_log)} epochs...")
    for row in training_log:
        wandb.log({
            "epoch":         row["epoch"],
            "train/loss":    row["loss"],
            "train/lr":      row["lr"],
            "train/time_s":  row["time_s"],
        }, step=row["epoch"])

    # -- Log summary metrics ---------------------------------------------
    run.summary["final_loss"]     = final_loss
    run.summary["best_loss"]      = best_loss
    run.summary["initial_loss"]   = init_loss
    run.summary["loss_reduction"] = init_loss - best_loss
    run.summary["total_time_h"]   = round(total_time_h, 2)
    if probe_acc is not None:
        run.summary["probe_accuracy"] = probe_acc
    if base_acc is not None:
        run.summary["base_accuracy_no_jitter"] = base_acc
    if jitter_gain is not None:
        run.summary["jitter_gain_pp"] = round(jitter_gain, 2)

    # -- Upload plots as images ------------------------------------------
    plots_dir = os.path.join(run_dir, "plots")
    if os.path.isdir(plots_dir):
        for img_file in glob.glob(os.path.join(plots_dir, "*.png")):
            img_name = os.path.splitext(os.path.basename(img_file))[0]
            wandb.log({f"plots/{img_name}": wandb.Image(img_file)})
            print(f"    Uploaded: {img_name}.png")

    # -- Upload final encoder as artifact --------------------------------
    encoder_path = os.path.join(run_dir, "simclr_encoder_final.pth")
    if os.path.exists(encoder_path):
        artifact = wandb.Artifact(
            name=f"simclr-encoder-exp{exp_id}",
            type="model",
            description=f"SimCLR ResNet-50 encoder from {display_name}",
            metadata={
                "exp_id": exp_id,
                "best_loss": best_loss,
                "probe_accuracy": probe_acc,
            }
        )
        artifact.add_file(encoder_path)
        run.log_artifact(artifact)
        print(f"    Uploaded artifact: simclr-encoder-exp{exp_id}")

    run.finish()
    return True


def main():
    print("=" * 72)
    print("  SimCLR Ablation -> Weights & Biases Sync")
    print("  Project: " + WANDB_PROJECT)
    print("=" * 72)
    print(f"  Experiment root: {EXPERIMENT_ROOT}")
    print(f"  Experiments to sync: {len(EXPERIMENTS)}")
    print("-" * 72)

    success_count = 0
    t_start = time.time()

    for folder_name, exp_id, display_name in EXPERIMENTS:
        print(f"\n[Exp {exp_id}] {display_name}")
        print(f"  Folder: {folder_name}")
        ok = sync_experiment(folder_name, exp_id, display_name)
        if ok:
            success_count += 1
            print(f"  [OK] Synced successfully!")
        else:
            print(f"  [FAIL] Could not sync.")

    elapsed = time.time() - t_start
    print("\n" + "=" * 72)
    print(f"  SYNC COMPLETE: {success_count}/{len(EXPERIMENTS)} experiments")
    print(f"  Total time: {elapsed:.1f}s")
    print(f"  View at: https://wandb.ai/mahmoudalyosify/{WANDB_PROJECT}")
    print("=" * 72)


if __name__ == "__main__":
    main()


**`scripts/make_assets_v1.py`** (727 lines) — asset generator of the first version of the paper (TMLR template, `paper/old_tmlr/`); replaced by `scripts/make_assets.py`.

In [ ]:
%%writefile scripts/make_assets_v1.py
"""Build every number, table and figure of the paper from results/.

    python scripts/make_assets.py

Writes
    paper/generated/numbers.tex   \\newcommand macros for every number quoted in the text
    paper/generated/*.tex         tables
    paper/figures/*.pdf           figures
    results/summary.csv           one row per run
Missing runs are skipped (their macros print "--"), so the script can be run at any time.
"""
import glob
import json
import os
import re
import sys

import matplotlib
import numpy as np
import pandas as pd

matplotlib.use("Agg")
import matplotlib.pyplot as plt   # noqa: E402

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "scripts"))
from stats import compare, test_labels   # noqa: E402

RUNS = os.path.join(ROOT, "results", "runs")
GEN = os.path.join(ROOT, "paper", "generated")
FIG = os.path.join(ROOT, "paper", "figures")
os.makedirs(GEN, exist_ok=True)
os.makedirs(FIG, exist_ok=True)

# entity colours (fixed per method, never by rank)
COL = {"simclr": "#2a78d6", "supcon": "#eb6834", "union": "#1baf7a", "sub": "#7f7f7f",
       "ce": "#6250d6", "fixmatch": "#e87ba4", "mixmatch": "#008300", "byol": "#eda100", "simsiam": "#e34948"}
INK, MUTED, GRID = "#1f1f1f", "#5f5f5f", "#e6e6e6"
plt.rcParams.update({
    "font.family": "serif", "font.size": 8.5, "axes.titlesize": 9, "axes.labelsize": 8.5,
    "legend.fontsize": 7.5, "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID,
    "grid.linewidth": 0.6, "lines.linewidth": 1.6, "lines.markersize": 5, "legend.frameon": False,
    "pdf.fonttype": 42, "savefig.bbox": "tight", "savefig.pad_inches": 0.02,
})


# ----------------------------------------------------------------------------
# loading
# ----------------------------------------------------------------------------

def load():
    rows = []
    for path in sorted(glob.glob(os.path.join(RUNS, "*", "result.json"))):
        d = os.path.dirname(path)
        name = os.path.basename(d)
        if name.startswith("pilot") or name.startswith("smoke"):
            continue
        r = json.load(open(path))
        row = {"name": name, "config": re.sub(r"_s\d+$", "", name), "seed": r["args"].get("seed"),
               "train_min": r.get("train_time_s", 0) / 60, "steps": r.get("steps")}
        row.update(r["metrics"])
        pj = os.path.join(d, "probes.json")
        if os.path.exists(pj):
            row.update({"p_" + k: v for k, v in json.load(open(pj)).items()})
        rows.append(row)
    return pd.DataFrame(rows)


DF = load()
# every macro used in the paper exists even before its run has finished ("--")
DEFAULT_MACROS = """radiusTen spreadTen radiusOne spreadOne radiusFull spreadFull radiusKc spreadKc neffKc neffClFifty
oodSimclrCifar oodSimclrOod oodSimclrPCifar oodSimclrPOod oodSimclrTopWrong oodUnionCifar oodUnionOod oodUnionPCifar
oodUnionPOod oodUnionTopWrong oodSupconCifar oodSupconOod oodSupconPCifar oodSupconPOod oodSupconTopWrong oodCeCifar
oodCeOod oodCePCifar oodCePOod oodCeTopWrong histProbe clipCifar clipResSixteen clipResThirtyTwo clipResSixtyFour
clipResOneTwoEight clipResTwoTwoFour clipTest nOod""".split()
DF.to_csv(os.path.join(ROOT, "results", "summary.csv"), index=False)
MACROS = {}


def vals(config, key):
    if DF.empty or key not in DF.columns:
        return np.array([])
    return DF.loc[DF["config"] == config, key].dropna().to_numpy(dtype=float)


def ms(config, key, digits=1, pm=True):
    v = vals(config, key)
    if len(v) == 0:
        return "--"
    if len(v) == 1 or not pm:
        return f"{v.mean():.{digits}f}"
    return f"{v.mean():.{digits}f}{{\\scriptsize$\\pm${v.std(ddof=1):.{digits}f}}}"


def mean(config, key):
    v = vals(config, key)
    return float(v.mean()) if len(v) else float("nan")


def macro(name, value):
    MACROS[name] = value


def fmt(x, d=1, sign=False):
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return "--"
    return f"{x:+.{d}f}" if sign else f"{x:.{d}f}"


# ----------------------------------------------------------------------------
# headline numbers + significance tests
# ----------------------------------------------------------------------------
CORE = {"simclr": "simclr_all_e100", "supcon": "supcon_lab_f0.1_e100",
        "sub": "simclr_sub_f0.1_e100", "union": "union_f0.1_e100"}
Y = test_labels()


def stat_macros(tag, a, b, key):
    try:
        r = compare(a, b, Y, key, RUNS, n_boot=10000)
    except (FileNotFoundError, ValueError):
        return None
    macro(f"diff{tag}", fmt(r["diff"], 1, True))
    macro(f"ci{tag}", f"[{r['ci_lo']:+.1f}, {r['ci_hi']:+.1f}]")
    p = r["p_boot"]
    macro(f"p{tag}", "<10^{-4}" if p <= 1e-4 else f"={p:.3f}" if p >= 0.001 else f"={p:.1e}")
    return r


STATS = []
for key, label in (("linear", "Lin"), ("knn", "Knn")):
    for tag, a, b in (("UnionSimclr", "union", "simclr"), ("UnionSupcon", "union", "supcon"),
                      ("SupconSimclr", "supcon", "simclr"), ("SimclrSub", "simclr", "sub"),
                      ("SupconSub", "supcon", "sub")):
        r = stat_macros(tag + label, CORE[a], CORE[b], key)
        if r:
            STATS.append((label, a, b, r))
for tag, a, b in (("UnionSimclr", "union", "simclr"), ("UnionSupcon", "union", "supcon"), ("SupconSimclr", "supcon", "simclr")):
    for key, label in (("finetune", "Ft"),):
        r = stat_macros(tag + label, "ft_" + CORE[a], "ft_" + CORE[b], key)
        if r:
            STATS.append((label, a, b, r))
r = stat_macros("LongLin", "simclr_all_e200", "supcon_lab_f0.1_e200", "linear")
if r:
    STATS.append(("Lin", "simclr@e200", "supcon@e200", r))
r = stat_macros("LongUnionLin", "union_f0.1_e200", "supcon_lab_f0.1_e200", "linear")
if r:
    STATS.append(("Lin", "union@e200", "supcon@e200", r))
r = stat_macros("LongSimclrUnionLin", "simclr_all_e200", "union_f0.1_e200", "linear")
if r:
    STATS.append(("Lin", "simclr@e200", "union@e200", r))
for tag, a_, b_ in (("FtLongSupcon", "ft_simclr_all_e200", "ft_supcon_lab_f0.1_e200"),
                    ("FtLongUnion", "ft_simclr_all_e200", "ft_union_f0.1_e200")):
    r = stat_macros(tag, a_, b_, "finetune")
    if r:
        STATS.append(("Ft", a_.replace("ft_", "") + "@ft", b_.replace("ft_", "") + "@ft", r))
r = stat_macros("ShortLin", "supcon_lab_f0.1_e10", "simclr_all_e10", "linear")
if r:
    STATS.append(("Lin", "supcon@e10", "simclr@e10", r))
# the original report's budget: SupCon trained 100 epochs over 5,000 images (= 10 full-data epochs)
r = stat_macros("BudgetLin", "supcon_lab_f0.1_e10", "simclr_all_e100", "linear")
if r:
    STATS.append(("Lin", "supcon@e10", "simclr@e100", r))

for m, c in CORE.items():
    for key, short in (("linear_acc", "Lin"), ("knn_acc", "Knn"), ("p_linear_0.1", "LinTen"),
                       ("seen_cdnv", "CdnvSeen"), ("heldout_cdnv", "CdnvHeld"), ("test_cdnv", "CdnvTest"),
                       ("test_ncc_acc", "Ncc"), ("test_etf_dev", "Etf")):
        macro(f"{m}{short}", ms(c, key, 2 if "cdnv" in key or "etf" in key else 1, pm=False))
    macro(f"{m}LinPm", ms(c, "linear_acc"))
    macro(f"{m}FtPm", ms("ft_" + c, "finetune_acc"))
    macro(f"{m}Ft", ms("ft_" + c, "finetune_acc", pm=False))
macro("supconTenLin", ms("supcon_lab_f0.1_e10", "linear_acc", pm=False))
macro("supconTenLinPm", ms("supcon_lab_f0.1_e10", "linear_acc"))
macro("simclrTenLin", ms("simclr_all_e10", "linear_acc", pm=False))
macro("gapBudget", fmt(mean("simclr_all_e100", "linear_acc") - mean("supcon_lab_f0.1_e10", "linear_acc")))
macro("gapMatched", fmt(mean("simclr_all_e100", "linear_acc") - mean("supcon_lab_f0.1_e100", "linear_acc"), 1, True))
macro("rFiftySupconTen", ms("r50_supcon_lab_f0.1_e10", "linear_acc", pm=False))
macro("rFiftySupconTenLow", ms("r50_supcon_lab_f0.1_e10_lr0.05", "linear_acc", pm=False))
macro("rFiftySimclrTen", ms("r50_simclr_all_e10", "linear_acc", pm=False))
macro("gapBudgetSimclr", fmt(mean("supcon_lab_f0.1_e10", "linear_acc") - mean("simclr_all_e10", "linear_acc"), 1, True))
for e, w in ((25, "TwentyFive"), (50, "Fifty"), (200, "TwoHundred")):
    macro(f"supcon{w}Lin", ms(f"supcon_lab_f0.1_e{e}", "linear_acc", pm=False))
    macro(f"simclr{w}Lin", ms(f"simclr_all_e{e}", "linear_acc", pm=False))
    macro(f"union{w}Lin", ms(f"union_f0.1_e{e}", "linear_acc", pm=False))
    macro(f"supcon{w}LinPm", ms(f"supcon_lab_f0.1_e{e}", "linear_acc"))
    macro(f"simclr{w}LinPm", ms(f"simclr_all_e{e}", "linear_acc"))
    macro(f"union{w}LinPm", ms(f"union_f0.1_e{e}", "linear_acc"))
    macro(f"supcon{w}Knn", ms(f"supcon_lab_f0.1_e{e}", "knn_acc", pm=False))
    macro(f"simclr{w}Knn", ms(f"simclr_all_e{e}", "knn_acc", pm=False))
    macro(f"union{w}Knn", ms(f"union_f0.1_e{e}", "knn_acc", pm=False))
macro("unionUnifTwoHundredLin", ms("unionunif_f0.1_e200", "linear_acc", pm=False))
macro("unionUnifLin", ms("unionunif_f0.1_e100", "linear_acc", pm=False))
macro("gapLong", fmt(mean("simclr_all_e200", "linear_acc") - mean("supcon_lab_f0.1_e200", "linear_acc"), 1))
macro("gapShort", fmt(mean("supcon_lab_f0.1_e10", "linear_acc") - mean("simclr_all_e10", "linear_acc"), 1))
macro("ceTen", ms("ce_f0.1_e100", "classifier_acc", pm=False))
macro("ceFull", ms("ce_f1.0_e100", "classifier_acc", pm=False))
for k in ("fixmatch", "mixmatch"):
    macro(f"{k}Acc", ms(f"{k}_f0.1", "classifier_acc", pm=False))
    macro(f"{k}Lin", ms(f"{k}_f0.1", "linear_acc", pm=False))
for k in ("byol", "simsiam"):
    macro(f"{k}Lin", ms(f"{k}_all_e100", "linear_acc", pm=False))
    macro(f"{k}Knn", ms(f"{k}_all_e100", "knn_acc", pm=False))
    macro(f"{k}Ft", ms(f"ft_{k}_all_e100", "finetune_acc", pm=False))
macro("nSeeds", str(len(vals(CORE["simclr"], "linear_acc"))))

# ----------------------------------------------------------------------------
# Table: main results at two budgets and three protocols
# ----------------------------------------------------------------------------
def write(name, text):
    with open(os.path.join(GEN, name), "w") as f:
        f.write(text)


MAIN_ROWS = [
    ("SimCLR", "all 50k", "--", "simclr_all_e{}"),
    ("SimCLR (subset)", "5k", "--", "simclr_sub_f0.1_e{}"),
    ("SupCon (label-only)", "5k", "5k", "supcon_lab_f0.1_e{}"),
    ("Union", "all 50k", "5k", "union_f0.1_e{}"),
    ("Union, uniform batches", "all 50k", "5k", "unionunif_f0.1_e{}"),
]
PROTS = (("linear_acc", None), ("p_linear_0.1", None), ("finetune_acc", "ft_"))


def cell(cfg, key, prefix, best):
    c = (prefix or "") + cfg
    out = ms(c, key)
    m = mean(c, key)
    if np.isfinite(m) and best is not None and abs(m - best) < 0.05:
        out = r"\textbf{" + out + "}"
    return out


lines = [r"\begin{tabular}{l cc ccc ccc}", r"\toprule",
         r" & \multicolumn{2}{c}{pretraining uses} & \multicolumn{3}{c}{9,700 steps (100 epochs)} & \multicolumn{3}{c}{19,400 steps (200 epochs)} \\",
         r"\cmidrule(lr){2-3}\cmidrule(lr){4-6}\cmidrule(lr){7-9}",
         r"method & images & labels & probe, 50k & probe, 5k & fine-tune, 5k & probe, 50k & probe, 5k & fine-tune, 5k \\", r"\midrule"]
best = {}
for e in (100, 200):
    for key, prefix in PROTS:
        ms_ = [mean((prefix or "") + r[3].format(e), key) for r in MAIN_ROWS]
        ms_ = [m for m in ms_ if np.isfinite(m)]
        best[(e, key)] = max(ms_) if ms_ else None
for label, imgs, labs, pat in MAIN_ROWS:
    cells = [cell(pat.format(e), key, prefix, best[(e, key)]) for e in (100, 200) for key, prefix in PROTS]
    lines.append(f"{label} & {imgs} & {labs} & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_main.tex", "\n".join(lines) + "\n")

# kNN and short-budget table (appendix)
lines = [r"\begin{tabular}{l ccccc}", r"\toprule", r"method & 970 & 2,425 & 4,850 & 9,700 & 19,400 steps \\", r"\midrule"]
for label, pat in (("SimCLR", "simclr_all_e{}"), ("SupCon (label-only)", "supcon_lab_f0.1_e{}"), ("Union", "union_f0.1_e{}")):
    for key, nm in (("linear_acc", "linear"), ("knn_acc", "$k$NN")):
        lines.append(f"{label}, {nm} & " + " & ".join(ms(pat.format(e), key) for e in (10, 25, 50, 100, 200)) + r" \\")
lines += [r"\midrule",
          r"ResNet-50 SupCon (label-only), lr 0.25 & " + ms("r50_supcon_lab_f0.1_e10", "linear_acc") + r" & & & " + ms("r50_supcon_lab_f0.1_e100", "linear_acc") + r" & \\",
          r"ResNet-50 SupCon (label-only), lr 0.05 & " + ms("r50_supcon_lab_f0.1_e10_lr0.05", "linear_acc") + r" & & & & \\",
          r"ResNet-50 SimCLR & " + ms("r50_simclr_all_e10", "linear_acc") + r" & & & " + ms("r50_simclr_all_e100", "linear_acc") + r" & \\",
          r"\bottomrule", r"\end{tabular}"]
write("table_budget.tex", "\n".join(lines) + "\n")

for e, w in ((100, ""), (200, "TwoHundred")):
    for k, pat in (("simclr", "simclr_all_e{}"), ("supcon", "supcon_lab_f0.1_e{}"), ("union", "union_f0.1_e{}"),
                   ("unionUnif", "unionunif_f0.1_e{}"), ("sub", "simclr_sub_f0.1_e{}")):
        macro(f"{k}{w}Ft", ms("ft_" + pat.format(e), "finetune_acc", pm=False))
        macro(f"{k}{w}FtPm", ms("ft_" + pat.format(e), "finetune_acc"))
        macro(f"{k}{w}ProbeFive", ms(pat.format(e), "p_linear_0.1", pm=False))
        macro(f"{k}{w}ProbeTrain", ms(pat.format(e), "p_linear_0.1_train", pm=False))
        macro(f"{k}{w}CdnvLab", ms(pat.format(e), "p_cdnv_labeled_0.1", 2, pm=False))
        macro(f"{k}{w}CdnvTestLab", ms(pat.format(e), "p_cdnv_test_0.1", 2, pm=False))

# significance table
lines = [r"\begin{tabular}{llcc}", r"\toprule", r"comparison & protocol & difference [95\% CI] & $p$ \\", r"\midrule"]
NICE = {"simclr": "SimCLR", "supcon": "SupCon (label-only)", "union": "Union", "sub": "SimCLR (subset)",
        "supcon@e10": "SupCon, 970 steps", "simclr@e100": "SimCLR, 9,700 steps", "simclr@e10": "SimCLR, 970 steps",
        "simclr@e200": "SimCLR, 19,400 steps", "supcon@e200": "SupCon, 19,400 steps", "union@e200": "Union, 19,400 steps",
        "simclr_all_e200@ft": "SimCLR, 19,400 steps", "supcon_lab_f0.1_e200@ft": "SupCon, 19,400 steps",
        "union_f0.1_e200@ft": "Union, 19,400 steps"}
PROT = {"Lin": "linear (all labels)", "Knn": "$k$NN (all labels)", "Ft": "fine-tune (5k labels)"}
for label, a, b, r in STATS:
    p = r["p_boot"]
    ps = r"$<10^{-4}$" if p <= 1e-4 else f"{p:.3f}" if p >= 0.001 else f"{p:.1e}"
    lines.append(f"{NICE[a]} $-$ {NICE[b]} & {PROT[label]} & {r['diff']:+.2f} [{r['ci_lo']:+.2f}, {r['ci_hi']:+.2f}] & {ps} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_stats.tex", "\n".join(lines) + "\n")

# ----------------------------------------------------------------------------
# Figure 1: the epoch trap (accuracy vs. gradient steps)
# ----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(6.9, 2.5), gridspec_kw={"width_ratios": [1.25, 1]})
ax = axes[0]
series = [("SimCLR, all images", "simclr", "simclr_all_e{}"), ("SupCon, 10% labeled only", "supcon", "supcon_lab_f0.1_e{}"),
          ("Union", "union", "union_f0.1_e{}")]
for label, k, pat in series:
    xs, ys, es = [], [], []
    for e in (10, 25, 50, 100, 200):
        v = vals(pat.format(e), "linear_acc")
        if len(v):
            steps = DF.loc[DF["config"] == pat.format(e), "steps"].iloc[0]
            xs.append(steps)
            ys.append(v.mean())
            es.append(v.std(ddof=1) if len(v) > 1 else 0)
    if xs:
        ax.errorbar(xs, ys, yerr=es, color=COL[k], marker="o", label=label, capsize=2, ms=4)
# the original report's comparison, in step units of this study
r_sup, r_sim = vals("supcon_lab_f0.1_e10", "linear_acc"), vals("simclr_all_e100", "linear_acc")
if len(r_sup) and len(r_sim):
    x0 = DF.loc[DF["config"] == "supcon_lab_f0.1_e10", "steps"].iloc[0]
    x1 = DF.loc[DF["config"] == "simclr_all_e100", "steps"].iloc[0]
    ax.annotate("", xy=(x0, r_sup.mean()), xytext=(x1, r_sim.mean()),
                arrowprops=dict(arrowstyle="<->", color=MUTED, lw=0.8, ls="--", shrinkA=4, shrinkB=4))
    ax.text(np.sqrt(x0 * x1) * 0.95, (r_sup.mean() + r_sim.mean()) / 2 - 1.3,
            "\"100 epochs\" each:\nthe comparison\nin our report", fontsize=6.5, color=MUTED, ha="center", va="top")
ax.set_xscale("log")
ax.set_xlabel("gradient steps (batch 512)")
ax.set_ylabel("linear-probe accuracy (%)")
ax.set_title("(a) Labels buy speed, images buy accuracy", loc="left")
ax.legend(loc="lower right")

ax = axes[1]
groups = [("SimCLR", "simclr", "simclr_all_e{}"), ("SupCon\n(label-only)", "supcon", "supcon_lab_f0.1_e{}"),
          ("Union", "union", "union_f0.1_e{}")]
w = 0.36
for i, (label, k, pat) in enumerate(groups):
    for j, (key, prefix, hatch, lab) in enumerate((("linear_acc", "", None, "probe trained on all 50k labels"),
                                                     ("finetune_acc", "ft_", "////", "fine-tuned on the 5k labels"))):
        v = vals(prefix + pat.format(200), key)
        if not len(v):
            continue
        x = i + (j - 0.5) * (w + 0.03)
        ax.bar(x, v.mean(), w, color=COL[k] if hatch is None else "white", edgecolor=COL[k], hatch=hatch, linewidth=1.0,
               label=lab if i == 0 else None)
        if len(v) > 1:
            ax.errorbar(x, v.mean(), yerr=v.std(ddof=1), color=INK, capsize=2, lw=0.8)
        ax.text(x, v.mean() + 0.35, f"{v.mean():.1f}", ha="center", va="bottom", fontsize=6.5, color=INK)
ax.set_xticks(range(len(groups)))
ax.set_xticklabels([g[0] for g in groups], fontsize=7.5)
ax.set_ylim(78, 93.5)
ax.set_ylabel("test accuracy (%)")
ax.set_title("(b) Labels spent in pretraining are gone", loc="left")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor=MUTED, label="probe trained on all 50k labels"),
                   Patch(facecolor="white", edgecolor=MUTED, hatch="////", label="fine-tuned on the 5k labels only")],
          loc="upper right", fontsize=6.5, bbox_to_anchor=(1.0, 1.02))
ax.grid(axis="x", visible=False)
fig.savefig(os.path.join(FIG, "fig_main.pdf"))
plt.close(fig)

# fraction figure: (a) all-label probe, (b) fine-tuning with the labeled subset only
fig, axes = plt.subplots(1, 2, figsize=(6.9, 2.3))
fr = [0.01, 0.05, 0.1, 0.2]
FTF = {"simclr": lambda f: "ft_simclr_all_e100" if f == 0.1 else f"ft_simclr_all_e100_f{f}",
       "supcon": lambda f: f"ft_supcon_lab_f{f}_e100", "union": lambda f: f"ft_union_f{f}_e100"}
for ax, mode in ((axes[0], "probe"), (axes[1], "ft")):
    for label, k, pat in (("SimCLR, all images", "simclr", None), ("SimCLR, labeled images only", "sub", "simclr_sub_f{}_e100"),
                          ("SupCon, labeled images only", "supcon", "supcon_lab_f{}_e100"), ("Union", "union", "union_f{}_e100")):
        if mode == "ft" and k == "sub":
            continue
        xs, ys = [], []
        for f in fr:
            if mode == "probe":
                v = vals(CORE["simclr"] if pat is None else pat.format(f), "linear_acc")
            else:
                v = vals(FTF[k](f), "finetune_acc")
            if len(v):
                xs.append(100 * f)
                ys.append(v.mean())
        if xs:
            ax.plot(xs, ys, color=COL[k], marker="o", ls="--" if k == "sub" else "-", label=label, ms=4)
    ax.set_xscale("log")
    ax.set_xticks([1, 5, 10, 20])
    ax.set_xticklabels(["1%", "5%", "10%", "20%"])
    ax.set_xlabel("labeled fraction")
axes[0].set_ylabel("test accuracy (%)")
axes[0].set_title("(a) Probe trained on all labels", loc="left")
axes[1].set_title("(b) Fine-tuned on the labeled images only", loc="left")
axes[0].legend(loc="lower right", fontsize=6.3)
fig.savefig(os.path.join(FIG, "fig_fraction.pdf"))
plt.close(fig)

# fraction-sweep table (both protocols)
fr = [0.01, 0.05, 0.1, 0.2]
lines = [r"\begin{tabular}{l cccc cccc}", r"\toprule",
         r" & \multicolumn{4}{c}{linear probe, all labels} & \multicolumn{4}{c}{fine-tuned, labeled subset only} \\",
         r"\cmidrule(lr){2-5}\cmidrule(lr){6-9}",
         r"labeled fraction & 1\% & 5\% & 10\% & 20\% & 1\% & 5\% & 10\% & 20\% \\", r"\midrule"]
FT = {"simclr": lambda f: "ft_simclr_all_e100" if f == 0.1 else f"ft_simclr_all_e100_f{f}",
      "sub": lambda f: None,
      "supcon": lambda f: f"ft_supcon_lab_f{f}_e100",
      "union": lambda f: f"ft_union_f{f}_e100"}
for label, k, pat in (("SimCLR (all images)", "simclr", None), ("SimCLR (labeled images only)", "sub", "simclr_sub_f{}_e100"),
                      ("SupCon (label-only)", "supcon", "supcon_lab_f{}_e100"), ("Union", "union", "union_f{}_e100")):
    cells = [ms(CORE["simclr"] if pat is None else pat.format(f), "linear_acc", pm=False) for f in fr]
    cells += [ms(FT[k](f), "finetune_acc", pm=False) if FT[k](f) else "--" for f in fr]
    lines.append(label + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_fraction.tex", "\n".join(lines) + "\n")
for f, w in ((0.01, "One"), (0.05, "Five"), (0.2, "Twenty")):
    macro(f"fracSimclrFt{w}", ms(FT["simclr"](f), "finetune_acc", pm=False))
    macro(f"fracSupconFt{w}", ms(FT["supcon"](f), "finetune_acc", pm=False))
    macro(f"fracUnionFt{w}", ms(FT["union"](f), "finetune_acc", pm=False))
    macro(f"fracSupconLin{w}", ms(f"supcon_lab_f{f}_e100", "linear_acc", pm=False))
    macro(f"fracUnionLin{w}", ms(f"union_f{f}_e100", "linear_acc", pm=False))
    macro(f"fracSubLin{w}", ms(f"simclr_sub_f{f}_e100", "linear_acc", pm=False))

# ----------------------------------------------------------------------------
# Figure: coverage vs spread (diversity of the labeled pool)
# ----------------------------------------------------------------------------
div_path = os.path.join(ROOT, "results", "diversity.json")
if os.path.exists(div_path):
    div = json.load(open(div_path))["simclr"]["pools"]
    pts = []
    for name, v in div.items():
        f, kind, s = v["fraction"], v["kind"], v["seed"]
        if s != 0 or f >= 1:
            continue
        c = f"supcon_lab_f{f}_e100" if kind == "random" else f"supcon_lab_{kind}_f{f}_e100"
        acc = vals(c, "linear_acc")
        acc_sub = vals(c.replace("supcon_lab", "simclr_sub"), "linear_acc")
        if len(acc):
            pts.append((kind, f, v, acc.mean(), acc_sub.mean() if len(acc_sub) else None))
    if pts:
        fig, axes = plt.subplots(1, 2, figsize=(6.9, 2.3), sharey=True)
        mk = {"random": "o", "cluster10": "s", "cluster50": "D", "kcenter": "^"}
        nm = {"random": "random pool", "cluster10": "redundant pool (k=10)", "cluster50": "redundant pool (k=50)",
              "kcenter": "k-center pool"}
        for ax, key, xl in ((axes[0], "radius", "coverage radius (lower = better covered)"),
                            (axes[1], "spread_aug", "within-class spread of augmented views")):
            seen = set()
            for kind, f, v, acc, acc_sub in pts:
                ax.scatter(v[key], acc, marker=mk[kind], s=30, color=COL["supcon"],
                           edgecolor="white", linewidth=0.6, zorder=3, label=None if kind in seen else nm[kind])
                if acc_sub is not None:
                    ax.scatter(v[key], acc_sub, marker=mk[kind], s=30, facecolor="white", edgecolor=COL["simclr"],
                               linewidth=1.0, zorder=3)
                seen.add(kind)
                if kind == "random":
                    ax.annotate(f"{100 * f:g}%", (v[key], acc), textcoords="offset points", xytext=(4, 2), fontsize=6.5, color=MUTED)
            ax.set_xlabel(xl)
        axes[0].invert_xaxis()
        axes[0].set_ylabel("linear-probe acc. (%)")
        axes[0].set_title("(a) Coverage of unseen images", loc="left")
        axes[1].set_title("(b) Spread within the pool", loc="left")
        from matplotlib.lines import Line2D
        handles = [Line2D([], [], marker=mk[k], ls="", color=MUTED, label=nm[k]) for k in mk if k in {p_[0] for p_ in pts}]
        handles += [Line2D([], [], marker="o", ls="", color=COL["supcon"], label="SupCon (filled)"),
                    Line2D([], [], marker="o", ls="", markerfacecolor="white", markeredgecolor=COL["simclr"], label="SimCLR (hollow)")]
        axes[1].legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=6.3)
        fig.savefig(os.path.join(FIG, "fig_coverage.pdf"))
        plt.close(fig)
    # diversity table
    rows = [r"\begin{tabular}{l r cccc cc c}", r"\toprule",
            r" & & \multicolumn{4}{c}{diversity of the pool} & \multicolumn{2}{c}{SupCon on the pool} & SimCLR on the pool \\",
            r"\cmidrule(lr){3-6}\cmidrule(lr){7-8}\cmidrule(lr){9-9}",
            r"pool & images & spread (pairs) & spread (aug.\ views) & Vendi & coverage radius & probe, 50k & probe, own labels & probe, 50k \\", r"\midrule"]
    order = [f"random_f{f}_s0" for f in (0.01, 0.05, 0.1, 0.2, 1.0)] + ["cluster10_f0.1_s0", "cluster50_f0.1_s0", "kcenter_f0.1_s0"]
    nice = {"random": "random", "cluster10": "redundant ($k{=}10$)", "cluster50": "redundant ($k{=}50$)", "kcenter": "$k$-center"}
    for name in order:
        if name not in div:
            continue
        v = div[name]
        if v["fraction"] >= 1:
            c = CORE["supcon"].replace("f0.1", "f1.0")
        elif v["kind"] == "random":
            c = f"supcon_lab_f{v['fraction']}_e100"
        else:
            c = f"supcon_lab_{v['kind']}_f{v['fraction']}_e100"
        lab = nice[v["kind"]] + (f" {100 * v['fraction']:g}\\%" if v["kind"] == "random" else "")
        rows.append(f"{lab} & {v['n']:,} & {v['spread']:.3f} & {v['spread_aug']:.3f} & {v['vendi']:.1f} & "
                    f"{v['radius']:.4f} & {ms(c, 'linear_acc', pm=False)} & {ms(c, 'linear_lab_acc', pm=False)} & "
                    f"{ms(c.replace('supcon_lab', 'simclr_sub'), 'linear_acc', pm=False) if v['fraction'] < 1 else ms(CORE['simclr'], 'linear_acc', pm=False)} \\\\")
        if v["kind"] == "random" and v["fraction"] == 0.1:
            macro("radiusTen", f"{v['radius']:.3f}")
            macro("spreadTen", f"{v['spread_aug']:.3f}")
        if v["kind"] == "random" and v["fraction"] == 0.01:
            macro("radiusOne", f"{v['radius']:.3f}")
            macro("spreadOne", f"{v['spread_aug']:.3f}")
        if v["kind"] == "random" and v["fraction"] == 1.0:
            macro("radiusFull", f"{v['radius']:.3f}")
            macro("spreadFull", f"{v['spread_aug']:.3f}")
        if v["kind"] == "kcenter":
            macro("radiusKc", f"{v['radius']:.3f}")
            macro("spreadKc", f"{v['spread_aug']:.3f}")
            macro("neffKc", f"{v['n_eff']:,.0f}")
        if v["kind"] == "cluster50":
            macro("neffClFifty", f"{v['n_eff']:,.0f}")
    rows += [r"\bottomrule", r"\end{tabular}"]
    write("table_diversity.tex", "\n".join(rows) + "\n")
for k in ("cluster10", "cluster50", "kcenter"):
    nm_ = {"cluster10": "clTen", "cluster50": "clFifty", "kcenter": "kc"}[k]
    macro(nm_ + "Acc", ms(f"supcon_lab_{k}_f0.1_e100", "linear_acc", pm=False))
    macro(nm_ + "LabAcc", ms(f"supcon_lab_{k}_f0.1_e100", "linear_lab_acc", pm=False))
macro("supconLabAcc", ms("supcon_lab_f0.1_e100", "linear_lab_acc", pm=False))
macro("subKcAcc", ms("simclr_sub_kcenter_f0.1_e100", "linear_acc", pm=False))
macro("subClFiftyAcc", ms("simclr_sub_cluster50_f0.1_e100", "linear_acc", pm=False))

# ----------------------------------------------------------------------------
# Figure: geometry -- collapse on seen vs unseen images
# ----------------------------------------------------------------------------
geo = [("SimCLR", "simclr"), ("SimCLR (subset)", "sub"), ("SupCon (label-only)", "supcon"), ("Union", "union")]
have = [g for g in geo if len(vals(CORE[g[1]], "seen_cdnv"))]
if have:
    fig, ax = plt.subplots(figsize=(3.3, 2.2))
    w = 0.38
    for i, (label, k) in enumerate(have):
        c = CORE[k]
        seen = vals(c, "seen_cdnv").mean()
        # images that were not in the pretraining pool: held-out training images for label-only pools,
        # the test set otherwise (identical numbers within 0.02 for full-pool methods)
        held = vals(c, "heldout_cdnv")
        held = held.mean() if len(held) else vals(c, "test_cdnv").mean()
        ax.bar(i - w / 2 - 0.01, seen, w, color=COL[k], edgecolor="white", linewidth=0.8)
        ax.bar(i + w / 2 + 0.01, held, w, color="white", edgecolor=COL[k], hatch="////", linewidth=1.0)
        ax.text(i - w / 2, seen, f"{seen:.2f}", ha="center", va="bottom", fontsize=6.5, color=INK)
        ax.text(i + w / 2, held, f"{held:.2f}", ha="center", va="bottom", fontsize=6.5, color=INK)
    ax.set_xticks(range(len(have)))
    ax.set_xticklabels([h[0].replace(" (", "\n(") for h in have], fontsize=6.8)
    ax.set_ylabel("class-distance normalised\nvariance (CDNV, lower = tighter)")
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=MUTED, label="images seen in pretraining"),
                       Patch(facecolor="white", edgecolor=MUTED, hatch="////", label="unseen images")], loc="upper right", fontsize=6.5)
    ax.grid(axis="x", visible=False)
    fig.savefig(os.path.join(FIG, "fig_geometry.pdf"))
    plt.close(fig)

# geometry table
lines = [r"\begin{tabular}{l ccc ccc}", r"\toprule",
         r" & \multicolumn{3}{c}{CDNV $\downarrow$} & \multicolumn{3}{c}{test set} \\",
         r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}",
         r"method & seen & unseen & ratio & NCC acc. & ETF dev.\ $\downarrow$ & equinorm CV $\downarrow$ \\", r"\midrule"]
for label, k in geo:
    c = CORE[k]
    s, h = mean(c, "seen_cdnv"), mean(c, "heldout_cdnv")
    if not np.isfinite(h):
        h = mean(c, "test_cdnv")
    lines.append(f"{label} & {fmt(s, 2)} & {fmt(h, 2)} & {fmt(h / s if np.isfinite(s) else float('nan'), 1)}$\\times$ & "
                 f"{fmt(mean(c, 'test_ncc_acc'))} & {fmt(mean(c, 'test_etf_dev'), 3)} & {fmt(mean(c, 'test_etf_norm_cv'), 3)} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_geometry.tex", "\n".join(lines) + "\n")
for k in ("simclr", "supcon", "union", "sub"):
    c = CORE[k]
    s, h = mean(c, "seen_cdnv"), mean(c, "heldout_cdnv")
    if not np.isfinite(h):
        h = mean(c, "test_cdnv")
    macro(f"{k}CdnvRatio", fmt(h / s, 1) if np.isfinite(s) and s > 0 else "--")

# ----------------------------------------------------------------------------
# Table: baselines
# ----------------------------------------------------------------------------
lines = [r"\begin{tabular}{l l c ccc}", r"\toprule",
         r"method & family & labels used & linear (all labels) & $k$NN & accuracy with 5k labels$^\ddagger$ \\", r"\midrule"]
BASE = [("Supervised CE", "supervised", "5k", "ce_f0.1_e100", "classifier_acc"),
        ("SimCLR", "contrastive SSL", "--", CORE["simclr"], "ft"),
        ("BYOL", "non-contrastive SSL", "--", "byol_all_e100", "ft"),
        ("SimSiam", "non-contrastive SSL", "--", "simsiam_all_e100", "ft"),
        ("SupCon (label-only)", "supervised contrastive", "5k", CORE["supcon"], "ft"),
        ("MixMatch$^\\S$", "semi-supervised", "5k", "mixmatch_f0.1", "classifier_acc"),
        ("FixMatch$^\\S$", "semi-supervised", "5k", "fixmatch_f0.1", "classifier_acc"),
        ("Union", "semi-supervised contrastive", "5k", CORE["union"], "ft"),
        ("Supervised CE, all labels (reference)", "supervised", "50k", "ce_f1.0_e100", "classifier_acc")]
for label, fam, labs, c, key in BASE:
    last = ms("ft_" + c, "finetune_acc") if key == "ft" else ms(c, key)
    lines.append(f"{label} & {fam} & {labs} & {ms(c, 'linear_acc')} & {ms(c, 'knn_acc')} & {last} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_baselines.tex", "\n".join(lines) + "\n")

# ----------------------------------------------------------------------------
# CIFAR-100 and ResNet-50
# ----------------------------------------------------------------------------
lines = [r"\begin{tabular}{l ccc ccc}", r"\toprule",
         r" & \multicolumn{3}{c}{CIFAR-100, ResNet-18} & \multicolumn{3}{c}{CIFAR-10, ResNet-50} \\",
         r"\cmidrule(lr){2-4}\cmidrule(lr){5-7}", r"method & probe, 50k & $k$NN & fine-tune, 5k & probe, 50k & $k$NN & fine-tune, 5k \\", r"\midrule"]
for label, k in (("SimCLR", "simclr_all_e100"), ("SimCLR (subset)", "simclr_sub_f0.1_e100"),
                 ("SupCon (label-only)", "supcon_lab_f0.1_e100"), ("Union", "union_f0.1_e100")):
    lines.append(f"{label} & {ms('c100_' + k, 'linear_acc')} & {ms('c100_' + k, 'knn_acc')} & {ms('ft_c100_' + k, 'finetune_acc')} & "
                 f"{ms('r50_' + k, 'linear_acc')} & {ms('r50_' + k, 'knn_acc')} & {ms('ft_r50_' + k, 'finetune_acc')} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_generalization.tex", "\n".join(lines) + "\n")
for k, short in (("simclr_all_e100", "Simclr"), ("supcon_lab_f0.1_e100", "Supcon"), ("union_f0.1_e100", "Union"),
                 ("simclr_sub_f0.1_e100", "Sub")):
    macro(f"cHundred{short}", ms("c100_" + k, "linear_acc", pm=False))
    macro(f"rFifty{short}", ms("r50_" + k, "linear_acc", pm=False))
    macro(f"cHundred{short}Ft", ms("ft_c100_" + k, "finetune_acc", pm=False))
    macro(f"rFifty{short}Ft", ms("ft_r50_" + k, "finetune_acc", pm=False))

# ----------------------------------------------------------------------------
# colour shortcut + augmentation tiers
# ----------------------------------------------------------------------------
AUG = [("crop + flip", "Tier 1", "simclr_exp2_e100"), ("crop + flip + jitter", "Tier 2", "simclr_exp5_e100"),
       ("crop + flip + grayscale", "Tier 2", "simclr_exp6_e100"), ("crop + flip + jitter + grayscale", "Tier 3", CORE["simclr"])]
lines = [r"\begin{tabular}{l l ccc ccc}", r"\toprule",
         r" & & \multicolumn{3}{c}{accuracy (\%)} & \multicolumn{3}{c}{colour reliance} \\",
         r"\cmidrule(lr){3-5}\cmidrule(lr){6-8}",
         r"augmentation & tier & linear & gray test & $\Delta$ & CKA(hist.) & CKA(label) & hist.\ $R^2$ \\", r"\midrule"]
for label, tier, c in AUG:
    lin, gray = mean(c, "linear_acc"), mean(c, "linear_acc_gray")
    lines.append(f"{label} & {tier} & {fmt(lin)} & {fmt(gray)} & {fmt(gray - lin, 1, True)} & "
                 f"{fmt(mean(c, 'cka_hist'), 2)} & {fmt(mean(c, 'cka_label'), 2)} & {fmt(mean(c, 'hist_r2'), 2)} \\\\")
hp = os.path.join(ROOT, "results", "hist_probe.json")
if os.path.exists(hp):
    h = json.load(open(hp))["cifar10_bins8"]["acc"]
    lines.append(r"\midrule")
    lines.append(f"colour histogram only ($8^3$ bins, no encoder) & -- & {h:.1f} & -- & -- & 1.00 & -- & 1.00 \\\\")
    macro("histProbe", f"{h:.1f}")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_color.tex", "\n".join(lines) + "\n")
lines = [r"\begin{tabular}{l l ccccc}", r"\toprule",
         r"augmentation & tier & linear & CDNV$\downarrow$ & ETF dev.$\downarrow$ & $\cos$ mean (target $-0.111$) & equinorm CV$\downarrow$ \\", r"\midrule"]
for label, tier, c in AUG:
    lines.append(f"{label} & {tier} & {fmt(mean(c, 'linear_acc'))} & {fmt(mean(c, 'test_cdnv'), 2)} & {fmt(mean(c, 'test_etf_dev'), 3)} & "
                 f"{fmt(mean(c, 'test_etf_cos_mean'), 3)} & {fmt(mean(c, 'test_etf_norm_cv'), 3)} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_etf.tex", "\n".join(lines) + "\n")
for short, c in (("Two", "simclr_exp2_e100"), ("Five", "simclr_exp5_e100"), ("Six", "simclr_exp6_e100"), ("Eight", CORE["simclr"])):
    macro(f"aug{short}Lin", fmt(mean(c, "linear_acc")))
    macro(f"aug{short}Gray", fmt(mean(c, "linear_acc_gray") - mean(c, "linear_acc"), 1, True))
    macro(f"aug{short}Cka", fmt(mean(c, "cka_hist"), 2))
    macro(f"aug{short}CkaLabel", fmt(mean(c, "cka_label"), 2))
    macro(f"aug{short}GrayDrop", fmt(mean(c, "linear_acc") - mean(c, "linear_acc_gray"), 1))
    macro(f"aug{short}GrayAcc", fmt(mean(c, "linear_acc_gray"), 1))
    macro(f"aug{short}Etf", fmt(mean(c, "test_etf_dev"), 3))
    macro(f"aug{short}Cdnv", fmt(mean(c, "test_cdnv"), 2))
macro("supconNoJitterLin", ms("supcon_lab_f0.1_exp6_e100", "linear_acc", pm=False))
macro("augJitterCost", fmt(mean(CORE["simclr"], "linear_acc") - mean("simclr_exp6_e100", "linear_acc"), 1))
macro("supconJitterCost", fmt(mean(CORE["supcon"], "linear_acc") - mean("supcon_lab_f0.1_exp6_e100", "linear_acc"), 1))

# ----------------------------------------------------------------------------
# OOD resolution study
# ----------------------------------------------------------------------------
ood_path = os.path.join(ROOT, "results", "ood_resolution.json")
if os.path.exists(ood_path):
    ood = json.load(open(ood_path))
    enc = ood["encoders"]
    lines = [r"\begin{tabular}{l cc cc}", r"\toprule",
             r" & \multicolumn{2}{c}{dog recall (\%)} & \multicolumn{2}{c}{retrieval P@5 (\%)} \\",
             r"\cmidrule(lr){2-3}\cmidrule(lr){4-5}",
             r"encoder & CIFAR-10 dogs & ImageNet dogs$\to$32px & CIFAR-10 dogs & ImageNet dogs$\to$32px \\", r"\midrule"]
    names = {"simclr_all_e100_s0": "SimCLR", "simclr_all_e200_s0": "SimCLR, 19,400 steps",
             "supcon_lab_f0.1_e100_s0": "SupCon (label-only)",
             "union_f0.1_e100_s0": "Union", "ce_f1.0_e100_s0": "Supervised CE (all labels)",
             "simclr_sub_f0.1_e100_s0": "SimCLR (subset)"}
    for run, r in enc.items():
        lo, hi = r["dog_recall_ood_ci"]
        lines.append(f"{names.get(run, run)} & {r['dog_recall_cifar']:.1f} & {r['dog_recall_ood']:.1f} {{\\scriptsize[{lo:.1f}, {hi:.1f}]}} & "
                     f"{r['p_at_5_cifar']:.1f} & {r['p_at_5_ood']:.1f} \\\\")
        key = {"simclr_all_e100_s0": "Simclr", "union_f0.1_e100_s0": "Union", "supcon_lab_f0.1_e100_s0": "Supcon",
               "ce_f1.0_e100_s0": "Ce"}.get(run)
        if key:
            macro(f"ood{key}Cifar", f"{r['dog_recall_cifar']:.1f}")
            macro(f"ood{key}Ood", f"{r['dog_recall_ood']:.1f}")
            macro(f"ood{key}PCifar", f"{r['p_at_5_cifar']:.1f}")
            macro(f"ood{key}POod", f"{r['p_at_5_ood']:.1f}")
            top = max(r["ood_prediction_hist"].items(), key=lambda kv: kv[1] if kv[0] != "dog" else -1)
            macro(f"ood{key}TopWrong", f"{top[0]} ({top[1]:.0f}\\%)")
    if "clip" in ood:
        c = ood["clip"]
        res = c["dog_recall_ood_by_resolution"]
        lines.append(r"\midrule")
        lines.append(f"CLIP ViT-B/32 zero-shot (reference) & {c['dog_recall_cifar']:.1f} & {res['32']:.1f} & -- & -- \\\\")
        macro("clipCifar", f"{c['dog_recall_cifar']:.1f}")
        for rr, v in res.items():
            macro("clipRes" + {"16": "Sixteen", "32": "ThirtyTwo", "64": "SixtyFour", "128": "OneTwoEight", "224": "TwoTwoFour"}[rr], f"{v:.1f}")
        macro("clipTest", f"{c['test_acc']:.1f}")
    lines += [r"\bottomrule", r"\end{tabular}"]
    write("table_ood.tex", "\n".join(lines) + "\n")
    macro("nOod", f"{ood['n_ood']:,}")

# ----------------------------------------------------------------------------
# validation-split design ablation for the union objective (pilot runs)
# ----------------------------------------------------------------------------
PIL = [("SimCLR, $\\tau{=}0.5$", "pilot_simclr_t0.5"), ("SimCLR, $\\tau{=}0.2$", "pilot_simclr_t0.2"),
       ("SimCLR, $\\tau{=}0.1$", "pilot_simclr_t0.1"),
       ("SupCon label-only, $\\tau{=}0.1$", "pilot_supconL_t0.1"), ("SupCon label-only, $\\tau{=}0.5$", "pilot_supconL_t0.5"),
       ("Union, $\\tau{=}0.1$ for all anchors", "pilot_union_t0.1"), ("Union, $\\tau{=}0.5$ for all anchors", "pilot_union_t0.5"),
       ("Union, $\\tau_\\ell{=}0.1$, $\\tau_u{=}0.5$, uniform batches", "pilot_union_mixed"),
       ("Union, $\\tau_\\ell{=}0.1$, $\\tau_u{=}0.5$, 50\\% labeled per batch", "pilot_union_mixed_sh0.5"),
       ("Union, $\\tau_\\ell{=}0.1$, $\\tau_u{=}0.5$, 25\\% labeled per batch (chosen)", "pilot_union_mixed_sh0.25")]
lines = [r"\begin{tabular}{l ccc}", r"\toprule", r"configuration (validation split, 50 epochs) & linear & $k$NN & linear, 10\% labels \\", r"\midrule"]
for label, n in PIL:
    p = os.path.join(RUNS, n, "result.json")
    if os.path.exists(p):
        m = json.load(open(p))["metrics"]
        lines.append(f"{label} & {m['linear_acc']:.2f} & {m['knn_acc']:.2f} & {m.get('linear_lab_acc', float('nan')):.2f} \\\\".replace("nan", "--"))
lines += [r"\bottomrule", r"\end{tabular}"]
write("table_pilot.tex", "\n".join(lines) + "\n")

# ----------------------------------------------------------------------------
# Appendix figure: colour-jitter pairs of the original report (ResNet-50, 200 epochs, single runs,
# NVIDIA RTX 5000 Ada). Numbers are copied from the report (Table V); no run of this study.
# ----------------------------------------------------------------------------
REPORT = [("pure rotation\n(36 / 38)", 34.40, 51.21), ("weak crop\n(35 / 39)", 59.22, 80.53),
          ("crop + blur\n(9 / 40)", 63.01, 80.65), ("crop + flip + blur\n(13 / 41)", 64.49, 84.30),
          ("crop + cutout\n(10 / 42)", 66.27, 81.21)]
fig, ax = plt.subplots(figsize=(6.0, 2.2))
x = np.arange(len(REPORT))
w = 0.38
ax.bar(x - w / 2 - 0.01, [r[1] for r in REPORT], w, color="#86b6ef", edgecolor="white", label="without colour jitter")
ax.bar(x + w / 2 + 0.01, [r[2] for r in REPORT], w, color=COL["simclr"], edgecolor="white", label="with colour jitter")
for i, (_, a, b) in enumerate(REPORT):
    ax.text(i - w / 2, a + 1, f"{a:.1f}", ha="center", va="bottom", fontsize=6.5, color=INK)
    ax.text(i + w / 2, b + 1, f"{b:.1f}", ha="center", va="bottom", fontsize=6.5, color=INK)
ax.set_xticks(x)
ax.set_xticklabels([r[0] for r in REPORT], fontsize=6.8)
ax.set_ylabel("linear-probe acc. (%)")
ax.set_ylim(0, 95)
ax.grid(axis="x", visible=False)
ax.legend(loc="upper left", ncol=2)
fig.savefig(os.path.join(FIG, "fig_report_jitter.pdf"))
plt.close(fig)

# ----------------------------------------------------------------------------
# write macros
# ----------------------------------------------------------------------------
for k in DEFAULT_MACROS:
    MACROS.setdefault(k, "--")
for name in ("table_ood.tex", "table_diversity.tex"):
    if not os.path.exists(os.path.join(GEN, name)):
        write(name, "\\begin{tabular}{l}\\toprule (not yet computed) \\\\ \\bottomrule\\end{tabular}\n")
with open(os.path.join(GEN, "numbers.tex"), "w") as f:
    f.write("% generated by scripts/make_assets.py -- do not edit\n")
    for k, v in sorted(MACROS.items()):
        assert re.fullmatch(r"[A-Za-z]+", k), k
        f.write(f"\\newcommand{{\\{k}}}{{{v}\\xspace}}\n")
print(f"{len(DF)} runs, {len(MACROS)} macros, tables in {GEN}, figures in {FIG}")


## Appendix D: code cells of the earlier notebooks

The two exploratory notebooks of the course project, code cells only (Markdown cells and outputs omitted). They
are shown as text so that running this notebook does not run them; open the original notebooks to execute them.

### `notebooks/visualize-pairs.ipynb` (3 code cells)

*Cell 1*

```python
import matplotlib.pyplot as plt
import random
import torch
import sys
sys.path.append('../src')
from dataset import train_raw
from augmentations import CIFAR_MEAN, CIFAR_STD
from augmentations import transform_exp1, transform_exp2, transform_exp3, transform_exp4, transform_exp5, transform_exp6, transform_exp7, transform_exp8
```

*Cell 2*

```python
def show_positive_pairs(transform, title, n=2):
    """
    Visualizes the stochastic nature of the augmentation pipeline by comparing 
    the original image with its corresponding generated positive pairs.
    """
    # Initialize a grid to display original images alongside two augmented views.
    fig, axes = plt.subplots(n, 3, figsize=(9, 3 * n))
    
    # Broadcastable tensors for de-normalization to revert the data back to its 
    # original pixel intensity range for accurate visualization.
    mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
    std = torch.tensor(CIFAR_STD).view(3, 1, 1)

    for i in range(n):
        # Randomly sample an image from the raw dataset to ensure unbiased visualization.
        img, _ = train_raw[random.randint(0, len(train_raw)-1)]

        # Apply the stochastic transformation twice to produce distinct views 
        # (simulating the contrastive learning input).
        view1 = transform(img)
        view2 = transform(img)

        # Plotting the raw input image as a baseline reference.
        axes[i, 0].imshow(img)
        axes[i, 0].set_title("Original Image")
        axes[i, 0].axis("off")
  
        # De-normalization process: Reverses the standardization applied by T.Normalize.
        # Required for correct RGB color rendering in Matplotlib.  
        view1_denorm = view1 * std + mean
        view2_denorm = view2 * std + mean

        # Visualize 'View A' after re-arranging dimensions to (H, W, C) and clipping.
        axes[i, 1].imshow(view1_denorm.permute(1, 2, 0).clip(0, 1))
        axes[i, 1].set_title("Positive View A")
        axes[i, 1].axis("off")
        
        # Visualize 'View B' to demonstrate the variance within the same positive pair.
        axes[i, 2].imshow(view2_denorm.permute(1, 2, 0).clip(0, 1))
        axes[i, 2].set_title("Positive View B")
        axes[i, 2].axis("off")

    # Final layout adjustments for high-quality figure reporting.
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()
```

*Cell 3*

```python
# EXP 1: Baseline spatial variance testing.
show_positive_pairs(
    transform_exp1,
    "Experiment 1 — Crop Only Positive Pairs"
)

# EXP 2: Assessing the influence of horizontal symmetry on spatial features.
show_positive_pairs(
    transform_exp2,
    "Experiment 2 — Crop + Flip Positive Pairs"
)

# EXP 3: Evaluating photometric distortion through aggressive color jittering.
show_positive_pairs(
    transform_exp3,
    "Experiment 3 — Crop + ColorJitter Positive Pairs"
)


# EXP 4: Investigating the removal of chromatic cues for shape-based learning.
show_positive_pairs(
    transform_exp4,
    "Experiment 4 — Crop + Grayscale Positive Pairs"
)


# EXP 5: Testing the interaction between geometric flipping and color noise.
show_positive_pairs(
    transform_exp5,
    "Experiment 5 — Crop + Flip + ColorJitter Positive Pairs"
)


# EXP 6: Combining geometric variance with complete chromatic information loss.
show_positive_pairs(
    transform_exp6,
    "Experiment 6 — Crop + Flip + Grayscale Positive Pairs"
)


# EXP 7: Evaluating heavy photometric suite without geometric symmetry.
show_positive_pairs(
    transform_exp7,
    "Experiment 7 — Crop + ColorJitter + Grayscale Positive Pairs"
)


# EXP 8: Final validation of the holistic SimCLR augmentation policy.
show_positive_pairs(
    transform_exp8,
    "Experiment 8 — Full Combination Positive Pairs"
)
```

### `notebooks/Linear evaluation notebook/supervised-baseline-and-linear-probe-evaluation.ipynb` (17 code cells)

*Cell 1*

```python
uv install torch torchvision torchaudio
```

*Cell 2*

```python
import os, time, random, csv
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Force GPU usage if available
print(f'CUDA Available: {torch.cuda.is_available()}')
print(f'CUDA Version: {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU Count: {torch.cuda.device_count()}')
    torch.cuda.set_device(0)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
if device.type == 'cuda':
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Path('logs').mkdir(exist_ok=True)
Path('checkpoints').mkdir(exist_ok=True)
print('Folders : logs/ and checkpoints/ ready')
```

*Cell 3*

```python
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_set = datasets.CIFAR10('./data', train=True,  download=True, transform=train_transform)
test_set  = datasets.CIFAR10('./data', train=False, download=True, transform=test_transform)

BATCH = 256

train_loader = DataLoader(train_set, batch_size=BATCH, shuffle=True,
                          num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_set,  batch_size=BATCH, shuffle=False,
                          num_workers=0, pin_memory=False)

CLASSES = train_set.classes
print(f'Train   : {len(train_set):,} images → {len(train_loader)} batches of {BATCH}')
print(f'Test    : {len(test_set):,} images → {len(test_loader)} batches')
print(f'Classes : {CLASSES}')

# Quick sanity check
imgs, lbls = next(iter(train_loader))
print(f'Batch shape: {imgs.shape}  ✓  (this confirms DataLoader works)')
```

*Cell 4*

```python
# Unnormalize helper
def unnorm(t, mean=CIFAR_MEAN, std=CIFAR_STD):
    m = torch.tensor(mean).view(3,1,1)
    s = torch.tensor(std).view(3,1,1)
    return (t * s + m).clamp(0, 1)

imgs, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
fig.suptitle('Sample CIFAR-10 training images (after normalization reversed)',
             fontsize=11, y=1.01)
for i, ax in enumerate(axes.flat):
    ax.imshow(unnorm(imgs[i]).permute(1, 2, 0).numpy())
    ax.set_title(CLASSES[labels[i]], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('logs/sample_images.png', bbox_inches='tight', dpi=120)
plt.show()
print('Saved → logs/sample_images.png')
```

*Cell 5*

```python
def build_resnet50_cifar(num_classes=10):
    """ResNet50 with CIFAR-10 stem modifications (per SimCLR Appendix B.9)."""
    model = models.resnet50(weights=None)
    
    # Swap 7×7/stride-2 → 3×3/stride-1  (no resolution loss on small images)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()          # remove the early MaxPool
    model.fc      = nn.Linear(2048, num_classes)  # 10-class head
    
    return model

model = build_resnet50_cifar().to(device)

n_total     = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_trainable/1e6:.2f}M trainable / {n_total/1e6:.2f}M total')

# Quick sanity check
with torch.no_grad():
    dummy = torch.randn(4, 3, 32, 32).to(device)
    out   = model(dummy)
print(f'Output shape: {out.shape}  ✓  (batch=4, classes=10)')
```

*Cell 6*

```python
EPOCHS       = 90
LR           = 0.1
WEIGHT_DECAY = 1e-4
WARMUP       = 5

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR,
                      momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / WARMUP
    t = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * (1.0 + np.cos(np.pi * t))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

use_amp = device.type == 'cuda'
scaler  = torch.amp.GradScaler('cuda') if use_amp else None

print(f'Optimizer : SGD  lr={LR}, momentum=0.9, wd={WEIGHT_DECAY}')
print(f'Schedule  : {WARMUP}-ep warmup + cosine annealing')
print(f'AMP/FP16  : {use_amp}')
print()

# Quick smoke test: 2 mini-batches to confirm nothing hangs
print('Running 2-batch smoke test...')
model.train()
for i, (imgs, lbls) in enumerate(train_loader):
    if i == 2: break
    imgs, lbls = imgs.to(device), lbls.to(device)
    optimizer.zero_grad()
    if use_amp:
        with torch.amp.autocast('cuda'):
            out  = model(imgs)
            loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        out  = model(imgs)
        loss = criterion(out, lbls)
        loss.backward()
        optimizer.step()
    print(f'  batch {i+1}: loss={loss.item():.4f}')

print()
print('Smoke test passed! Safe to run full training.')

# Reseting the optimizer after smoke test
optimizer = optim.SGD(model.parameters(), lr=LR,
                      momentum=0.9, weight_decay=WEIGHT_DECAY, nesterov=True)
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
```

*Cell 7*

```python
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    """Single pass. optimizer=None → eval mode (no gradients)."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, correct, total = 0.0, 0, 0

    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=False)
        lbls = lbls.to(device, non_blocking=False)

        if training:
            optimizer.zero_grad()
            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    out  = model(imgs)
                    loss = criterion(out, lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                out  = model(imgs)
                loss = criterion(out, lbls)
                loss.backward()
                optimizer.step()
        else:
            with torch.no_grad():
                out  = model(imgs)
                loss = criterion(out, lbls)

        total_loss += loss.item() * imgs.size(0)
        correct    += out.argmax(1).eq(lbls).sum().item()
        total      += imgs.size(0)

    return total_loss / total, 100.0 * correct / total

print('run_epoch defined.')
```

*Cell 8*

```python
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': [], 'lr': []}

best_acc, best_epoch = 0.0, 0
t0 = time.time()

if device.type != 'cuda':
    print("Running on CPU. This will be VERY slow (~4-8 hours for 90 epochs).")
    print("    Each epoch takes 3-5 minutes. Progress will print after each one.")
    print()

print(f"{'Ep':>4}  {'TrLoss':>7}  {'TrAcc':>7}  {'TeLoss':>7}  {'TeAcc':>7}  {'LR':>8}")
print('─' * 50)

for ep in range(1, EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, scaler)
    te_loss, te_acc = run_epoch(model, test_loader,  criterion)
    scheduler.step()

    cur_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['test_loss'].append(te_loss)
    history['test_acc'].append(te_acc)
    history['lr'].append(cur_lr)

    if te_acc > best_acc:
        best_acc, best_epoch = te_acc, ep
        torch.save({'epoch': ep, 'state_dict': model.state_dict(),
                    'test_acc': te_acc}, 'checkpoints/baseline_best.pth')

    if ep % 25 == 0:
        torch.save({'epoch': ep, 'state_dict': model.state_dict()},
                   f'checkpoints/baseline_ep{ep:03d}.pth')

    print(f"{ep:>4}  {tr_loss:>7.4f}  {tr_acc:>6.2f}%  {te_loss:>7.4f}  {te_acc:>6.2f}%  {cur_lr:>8.5f}", flush=True)

elapsed = time.time() - t0
print('─' * 50)
print(f'Done in {elapsed/60:.1f} min  |  Best: {best_acc:.2f}% @ epoch {best_epoch}')
```

*Cell 9*

```python
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_range = range(1, EPOCHS + 1)

# Loss
ax = axes[0]
ax.plot(epochs_range, history['train_loss'], label='Train', color='#2563eb', lw=1.8)
ax.plot(epochs_range, history['test_loss'],  label='Test',  color='#dc2626', lw=1.8)
ax.set_title('Cross-Entropy Loss', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(); ax.grid(alpha=0.3)

# Accuracy
ax = axes[1]
ax.plot(epochs_range, history['train_acc'], label='Train', color='#2563eb', lw=1.8)
ax.plot(epochs_range, history['test_acc'],  label='Test',  color='#dc2626', lw=1.8)
ax.axhline(best_acc, color='#16a34a', ls='--', lw=1.2, label=f'Best Test = {best_acc:.2f}%')
ax.set_title('Top-1 Accuracy', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
ax.legend(); ax.grid(alpha=0.3)

# LR schedule
ax = axes[2]
ax.plot(epochs_range, history['lr'], color='#7c3aed', lw=1.8)
ax.set_title('Learning Rate Schedule', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('LR')
ax.grid(alpha=0.3)

plt.suptitle('Supervised ResNet50 Baseline — CIFAR-10 Training', 
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
```

*Cell 10*

```python
print('=' * 50)
print('  SUPERVISED BASELINE — FINAL RESULTS')
print('=' * 50)
print(f'  Architecture : ResNet50 (CIFAR-10 stem)')
print(f'  Dataset      : CIFAR-10 (50k train / 10k test)')
print(f'  Epochs       : {EPOCHS}')
print(f'  Batch size   : {BATCH}')
print(f'  Optimizer    : SGD + cosine annealing')
print(f'  Seed         : {SEED}')
print('─' * 50)
print(f'  Best Test Acc: {best_acc:.2f}%  (epoch {best_epoch})')
print(f'  Final Test   : {history["test_acc"][-1]:.2f}%')
print(f'  Training time: {elapsed/60:.1f} min')
```

*Cell 11*

```python
class LinearProbe(nn.Module):
    """
    Frozen encoder + single trainable linear layer.
    """
    def __init__(self, encoder, feature_dim=2048, num_classes=10):
        super().__init__()
        self.encoder = encoder
        for param in self.encoder.parameters():
            param.requires_grad = False
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            out = self.encoder(x)
            # SimCLR_ResNet50 returns (h, z) — take h only
            features = out[0] if isinstance(out, tuple) else out
        return self.classifier(features)


def extract_features(encoder, loader, device):
    """
    One-time forward pass to cache all features.
    """
    encoder.eval()
    all_feats, all_labels = [], []

    with torch.no_grad():
        for imgs, lbls in loader:
            out = encoder(imgs.to(device))
            # SimCLR_ResNet50 returns (h, z) — we want h
            feats = out[0] if isinstance(out, tuple) else out
            all_feats.append(feats.cpu())
            all_labels.append(lbls)

    return torch.cat(all_feats), torch.cat(all_labels)
print('LinearProbe and extract_features defined.')
```

*Cell 12*

```python
import sys, os

def _find_repo_root():
    candidate = os.path.abspath('.')
    for _ in range(6):
        if os.path.isfile(os.path.join(candidate, 'src', 'model.py')):
            return candidate
        candidate = os.path.dirname(candidate)
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None:
    REPO_ROOT = r'D:\Deep Learning Project\Midterm'   # fallback
    print(f'Auto-detect failed — using: {REPO_ROOT}')
else:
    print(f'Repo root: {REPO_ROOT}')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

assert os.path.isfile(os.path.join(REPO_ROOT, 'src', 'model.py')), \
    f"src/model.py not found in {REPO_ROOT}"
print(f'src/model.py reachable')



def simclr_ckpt_path(exp_id: int, epoch: int = 20) -> str:
    """
    Build the checkpoint path for a given experiment and epoch.

    Args:
        exp_id : 1–8  (matches Natalie's augmentation configs)
        epoch  : which epoch checkpoint to load (default: 20)

    Returns:
        Full absolute path to the .pth file.

    Example:
        simclr_ckpt_path(8, 20)
        → 'D:\\Deep Learning Project\\Midterm\\outputs\\exp_8\\checkpoints\\simclr_exp8_resnet18_epoch_020.pth'
    """
    rel = os.path.join(
        'outputs', f'exp_{exp_id}', 'checkpoints',
        f'simclr_exp{exp_id}_resnet18_epoch_{epoch:03d}.pth'
    )
    return os.path.join(REPO_ROOT, rel)


#  ENCODER WRAPPER

class SimCLREncoderOnly(nn.Module):
    """Wraps a SimCLR model and returns only the 512-dim encoder output h."""
    def __init__(self, simclr_model, repr_dim=512):
        super().__init__()
        self.encoder  = simclr_model.encoder
        # Don't read repr_dim from model — __pycache__ mismatch can break it
        self.repr_dim = getattr(simclr_model, 'repr_dim', repr_dim)

    def forward(self, x):
        h = self.encoder(x)
        return torch.flatten(h, start_dim=1)   # (N, repr_dim)



#  LOADER

def get_simclr_encoder(exp_id: int, epoch: int = 20):
    """
    Load a SimCLR checkpoint for experiment exp_id at a given epoch.

    Returns: (encoder, FEATURE_DIM)
      encoder     : SimCLREncoderOnly — frozen, outputs (N, 512)
      FEATURE_DIM : 512  (ResNet-18)
    """
    # Force reimport — avoids loading a stale __pycache__ version
    import importlib
    if 'src.model' in sys.modules:
        importlib.reload(sys.modules['src.model'])
    from src.model import get_simclr_model

    ckpt_path = simclr_ckpt_path(exp_id, epoch)
    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            f'Checkpoint not found:\n  {ckpt_path}\n'
            f'Run run_ablations.py first, or check exp_id/epoch.'
        )

    ckpt  = torch.load(ckpt_path, map_location=device, weights_only=False)
    model = get_simclr_model(backbone='resnet18', projection_dim=128)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    ep   = ckpt.get('epoch', epoch)
    loss = ckpt.get('loss', float('nan'))
    print(f'Loaded  exp_{exp_id}  epoch={ep}  NT-Xent={loss:.4f}')
    print(f'Path: {ckpt_path}')

    # repr_dim = 512 for ResNet-18 (hardcoded as fallback — safe regardless of cache)
    feat_dim = getattr(model, 'repr_dim', 512)
    wrapper  = SimCLREncoderOnly(model, repr_dim=feat_dim).to(device)
    return wrapper, feat_dim

EXP_ID  = 8  
EPOCH   = 20   

encoder, FEATURE_DIM = get_simclr_encoder(EXP_ID, EPOCH)

# ── Verification ──────────────────────────────────────────────────────────────
total     = sum(p.numel() for p in encoder.parameters())
trainable = sum(p.numel() for p in encoder.parameters() if p.requires_grad)
print()
print(f'Experiment     : Exp {EXP_ID}  (epoch {EPOCH})')
print(f'Feature dim    : {FEATURE_DIM}   (ResNet-18 = 512)')
print(f'Encoder params : {total/1e6:.2f}M  |  Trainable: {trainable/1e6:.2f}M (must be 0.00)')

with torch.no_grad():
    _out   = encoder(torch.randn(4, 3, 32, 32).to(device))
    _shape = _out.shape
print(f'Output shape   : {list(_shape)}   expected [4, {FEATURE_DIM}]')
assert _shape == (4, FEATURE_DIM), f'Shape mismatch! got {_shape}'
print()
print(f' Ready — FEATURE_DIM={FEATURE_DIM} will be used in all probe cells below')
```

*Cell 13*

```python
plain_train = DataLoader(
    datasets.CIFAR10('./data', train=True, download=False, transform=test_transform),
    batch_size=512, shuffle=False, num_workers=0, pin_memory=False
)
plain_test = DataLoader(
    datasets.CIFAR10('./data', train=False, download=False, transform=test_transform),
    batch_size=512, shuffle=False, num_workers=0, pin_memory=False
)

print('Extracting train features...', end=' ', flush=True)
t0 = time.time()
train_feats, train_labels = extract_features(encoder, plain_train, device)
print(f'done in {time.time()-t0:.1f}s , shape: {train_feats.shape}')

print('Extracting test features... ', end=' ', flush=True)
t0 = time.time()
test_feats, test_labels = extract_features(encoder, plain_test, device)
print(f'done in {time.time()-t0:.1f}s , shape: {test_feats.shape}')
```

*Cell 14*

```python
from torch.utils.data import TensorDataset

probe_train_ds = TensorDataset(train_feats, train_labels)
probe_test_ds  = TensorDataset(test_feats,  test_labels)

probe_train_loader = DataLoader(probe_train_ds, batch_size=512, shuffle=True)
probe_test_loader  = DataLoader(probe_test_ds,  batch_size=512, shuffle=False)

#Probe model: single linear layer
probe = nn.Linear(FEATURE_DIM, 10).to(device)
probe_criterion = nn.CrossEntropyLoss()
probe_optimizer = optim.Adam(probe.parameters(), lr=1e-3, weight_decay=1e-4)

PROBE_EPOCHS = 50
probe_history = {'train_acc': [], 'test_acc': []}

print(f"{'Epoch':>6}  {'Train Acc':>9}  {'Test Acc':>8}")
print('─' * 30)

for ep in range(1, PROBE_EPOCHS + 1):
    # Train
    probe.train()
    correct, total = 0, 0
    for feats, lbls in probe_train_loader:
        feats, lbls = feats.to(device), lbls.to(device)
        probe_optimizer.zero_grad()
        out  = probe(feats)
        loss = probe_criterion(out, lbls)
        loss.backward()
        probe_optimizer.step()
        correct += out.argmax(1).eq(lbls).sum().item()
        total   += lbls.size(0)
    tr_acc = 100.0 * correct / total
    
    # Eval
    probe.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for feats, lbls in probe_test_loader:
            feats, lbls = feats.to(device), lbls.to(device)
            out = probe(feats)
            correct += out.argmax(1).eq(lbls).sum().item()
            total   += lbls.size(0)
    te_acc = 100.0 * correct / total
    
    probe_history['train_acc'].append(tr_acc)
    probe_history['test_acc'].append(te_acc)
    
    if ep % 10 == 0 or ep == 1:
        print(f"{ep:>6}  {tr_acc:>8.2f}%  {te_acc:>7.2f}%")

probe_final_acc = probe_history['test_acc'][-1]
print('─' * 30)
print(f'Linear Probe Final Test Acc: {probe_final_acc:.2f}%')
```

*Cell 15*

```python
from sklearn.metrics import confusion_matrix, classification_report
import itertools

probe.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for feats, lbls in probe_test_loader:
        out = probe(feats.to(device))
        all_preds.extend(out.argmax(1).cpu().tolist())
        all_true.extend(lbls.tolist())

# Classification report
print(classification_report(all_true, all_preds, target_names=CLASSES))

# Per-class accuracy bar chart
cm = confusion_matrix(all_true, all_preds)
per_class_acc = cm.diagonal() / cm.sum(axis=1) * 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = ['#16a34a' if a >= 90 else '#2563eb' if a >= 80 else '#dc2626' for a in per_class_acc]
bars = ax1.barh(CLASSES, per_class_acc, color=colors, height=0.6)
ax1.axvline(probe_final_acc, color='#7c3aed', ls='--', lw=1.5, label=f'Overall: {probe_final_acc:.1f}%')
for bar, acc in zip(bars, per_class_acc):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{acc:.1f}%', va='center', fontsize=8)
ax1.set_xlim(0, 105)
ax1.set_xlabel('Accuracy (%)')
ax1.set_title('Per-Class Accuracy (Linear Probe)', fontweight='bold')
ax1.legend()
ax1.grid(axis='x', alpha=0.3)

# Confusion matrix
im = ax2.imshow(cm, interpolation='nearest', cmap='Blues')
ax2.set_xticks(range(10)); ax2.set_yticks(range(10))
ax2.set_xticklabels(CLASSES, rotation=45, ha='right', fontsize=8)
ax2.set_yticklabels(CLASSES, fontsize=8)
ax2.set_xlabel('Predicted'); ax2.set_ylabel('True')
ax2.set_title('Confusion Matrix', fontweight='bold')
for i, j in itertools.product(range(10), range(10)):
    ax2.text(j, i, cm[i, j], ha='center', va='center',
             color='white' if cm[i, j] > cm.max()*0.5 else 'black', fontsize=7)
plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()
```

*Cell 16*

```python
# ══════════════════════════════════════════════════════════════════
#  Results Summary Table — Experiment 8 Only (Midterm)
# ══════════════════════════════════════════════════════════════════

W = 72

print('╔' + '═'*W + '╗')
print('║' + ' MIDTERM RESULTS SUMMARY '.center(W) + '║')
print('╠' + '═'*W + '╣')
print(f'║  {"Method":<35}  {"Backbone":<10}  {"Top-1 Acc":>10}  ║')
print('╠' + '─'*W + '╣')

# ── Supervised Baseline ───────────────────────────────────────────
print(f'║  {"Supervised Baseline (end-to-end)":<35}  {"ResNet-50":<10}  {"93.77%":>10}  ║')

# ── Linear Probe on Supervised Encoder (sanity check) ─────────────
print(f'║  {"Linear Probe on Supervised Encoder":<35}  {"ResNet-50":<10}  {"93.89%":>10}  ║')

print('╠' + '─'*W + '╣')

# ── SimCLR Experiment 8 ───────────────────────────────────────────
print(f'║  {"SimCLR Probe (Exp 8, 20 ep.) *":<35}  {"ResNet-18":<10}  {"72.14%":>10}  ║')

print('╠' + '─'*W + '╣')

# ── Paper Reference ───────────────────────────────────────────────
print(f'║  {"SimCLR Paper (Chen et al., 2020)":<35}  {"ResNet-50":<10}  {"94.00%":>10}  ║')
print(f'║  {"Supervised Ceiling (Chen et al.)":<35}  {"ResNet-50":<10}  {"95.10%":>10}  ║')

print('╚' + '═'*W + '╝')

print()
print('* Exp 8 = Full SimCLR Pipeline: Crop + Flip + ColorJitter + Grayscale')
print('  NT-Xent Loss @ epoch 20 : 3.9676')
print('  Note: SimCLR uses ResNet-18 (midterm) vs ResNet-50 (baseline).')
print('  Full 200-epoch ResNet-50 training scheduled for final report.')
print()

# ── SimCLR Probe Training History ─────────────────────────────────
print('SimCLR Linear Probe Training (Exp 8, Frozen ResNet-18 Encoder):')
print(f'  {"Epoch":>6}  {"Train Acc":>10}  {"Test Acc":>10}')
print('  ' + '─'*30)

# Probe history data from training
probe_log = [
    ( 1, 59.65, 64.47),
    (10, 70.35, 69.41),
    (20, 71.71, 70.90),
    (30, 72.63, 71.68),
    (40, 73.18, 71.99),
    (50, 73.60, 72.14),
]
for ep, tr, te in probe_log:
    print(f'  {ep:>6}  {tr:>9.2f}%  {te:>9.2f}%')

print('  ' + '─'*30)
print(f'  Final Test Accuracy: 72.14%')
```

*Cell 17*

```python
import csv

# Baseline training history
with open('logs/baseline_history.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['epoch', 'train_loss', 'train_acc', 'test_loss', 'test_acc', 'lr'])
    for i in range(len(history['train_loss'])):
        w.writerow([i+1,
                    f"{history['train_loss'][i]:.4f}",
                    f"{history['train_acc'][i]:.2f}",
                    f"{history['test_loss'][i]:.4f}",
                    f"{history['test_acc'][i]:.2f}",
                    f"{history['lr'][i]:.6f}"])

# Linear probe history
with open('logs/probe_history.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['epoch', 'train_acc', 'test_acc'])
    for i in range(len(probe_history['train_acc'])):
        w.writerow([i+1,
                    f"{probe_history['train_acc'][i]:.2f}",
                    f"{probe_history['test_acc'][i]:.2f}"])
print('Training histories saved to logs/baseline_history.csv and logs/probe_history.csv')
```